# Step 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np

from metriclens import load_revenue_data


# ------------------------------------------------------------
# 0. Source file
# ------------------------------------------------------------
file_path = "online_retail_II.xlsx"


# ------------------------------------------------------------
# 1. Preserve the untouched source data for audit purposes
# ------------------------------------------------------------
raw_sheets = pd.read_excel(file_path, sheet_name=None)

raw_df = pd.concat(
    [
        sheet.assign(source_sheet=sheet_name)
        for sheet_name, sheet in raw_sheets.items()
    ],
    ignore_index=True,
)

print("Loaded source sheets:", list(raw_sheets.keys()))
print("Raw dataset shape:", raw_df.shape)


# ------------------------------------------------------------
# 2. Load the canonical analysis-ready transaction dataset
# ------------------------------------------------------------
load_result = load_revenue_data(file_path)

df = load_result.data.copy()


# ------------------------------------------------------------
# 3. Review loader diagnostics
# ------------------------------------------------------------
print("\nCleaned dataset shape:", df.shape)

print("\nLoader warnings:")
for warning in load_result.warnings:
    print("-", warning)

display(load_result.quality_report)

display(df.head())

In [ ]:
# ================================================================
# IMPORT SHARED DATA-LOADER HELPERS
# ================================================================

from metriclens.data_loader import (
    FIELD_ALIASES,
    NON_MERCHANDISE_CODE_TYPES,
    _find_fields,
    _numeric_or_missing,
    _text_or_missing,
    _classify_revenue_type,
)

print("✓ Data-loader helpers imported successfully.")

We have prepared and validated the data, but we have not yet completed a rigorous cleaning pass.
What is already handled:
- Standardized date, customer, product, region, invoice, and revenue fields.
- Calculated revenue from Quantity × Price.
- Excluded invalid/missing dates and revenue values.
- Flagged missing customer IDs and negative revenue.
- Retained cancellation/credit rows deliberately, because they matter for net-revenue change.


Moving on to some additional cleaning 
- Remove exact duplicate transaction rows.
- Treat zero-price items separately (often samples, adjustments, or non-revenue lines).
- Validate zero or negative quantities and prices.
- Clean inconsistent descriptions, product codes, country names, and customer IDs.
- Handle missing customer IDs: retain them for total revenue, but exclude or label them in customer-level driver analysis.
- Reconcile the two yearly sheets’ overlapping December 2010 dates and check for duplicate invoices.
- Produce a data-quality report with row counts removed/retained and the revenue impact of each rule.


# Step 2 : Data Cleaning and Standadisation

### Step 2A.

Clean and standardise raw retail transaction data into a canonical, analysis-ready line-item dataset.

Cleaning principles
-------------------
1. Preserve transaction-level detail.
2. Standardise dates, numeric fields, IDs, product codes and descriptions.
3. Keep missing customer IDs as missing rather than creating a synthetic customer.
4. Retain legitimate accounting / non-merchandise adjustment rows.
5. Remove only clearly identifiable cross-sheet duplicate records.
6. Flag within-sheet exact duplicates for audit rather than deleting them automatically.
7. Keep returns, cancellations and zero-price transactions and classify them explicitly.
8. Separate merchandise revenue from non-merchandise/accounting revenue.
9. Create stable product descriptions and customer-history fields for later analysis.

Returns
-------
DataLoadResult
Contains:
- cleaned canonical dataframe
- warnings
- source row count
- retained row count
- detailed quality report

In [ ]:
# ================================================================
# 2A.1 — VALIDATE SOURCE DATA
# ================================================================

source_rows = len(raw_df)

if raw_df.empty:
    raise ValueError("The revenue dataset contains no rows.")

print(f"Source rows: {source_rows:,}")
print(f"Source columns: {raw_df.shape[1]:,}")

In [ ]:
# ================================================================
# 2A.2 — STANDARDISE SOURCE COLUMN NAMES
# ================================================================

prepared = raw_df.copy()

# Remove leading/trailing whitespace from column names.
prepared.columns = [
    str(column).strip()
    for column in prepared.columns
]

# Identify canonical fields using FIELD_ALIASES.
fields = _find_fields(prepared.columns)

print("Detected fields:")
display(fields)

In [ ]:
# ================================================================
# 2A.3 — CHECK REQUIRED FIELDS
# ================================================================

if "date" not in fields:
    raise ValueError(
        "Missing a date column. "
        "Accepted names include Date and InvoiceDate."
    )

if (
    "revenue" not in fields
    and not {"quantity", "unit_price"}.issubset(fields)
):
    raise ValueError(
        "Revenue cannot be calculated. "
        "Provide either a Revenue/Amount column, "
        "or both Quantity and Price/UnitPrice."
    )

print("✓ Required fields are available.")

In [ ]:
# ================================================================
# 2A.4 — BUILD CANONICAL DATAFRAME
# ================================================================

data = pd.DataFrame(index=prepared.index)

# Convert invoice date to datetime.
data["date"] = pd.to_datetime(
    prepared[fields["date"]],
    errors="coerce"
)

# Convert quantity and unit price to numeric values.
data["quantity"] = _numeric_or_missing(
    prepared,
    fields.get("quantity")
)

data["unit_price"] = _numeric_or_missing(
    prepared,
    fields.get("unit_price")
)

# Calculate or import revenue.
if "revenue" in fields:
    data["revenue"] = pd.to_numeric(
        prepared[fields["revenue"]],
        errors="coerce"
    )
else:
    data["revenue"] = (
        data["quantity"]
        * data["unit_price"]
    )

In [ ]:
# ================================================================
# 2A.5 — IMPORT CANONICAL DIMENSIONS
# ================================================================

for field in (
    "customer",
    "product",
    "description",
    "region",
    "channel",
    "invoice",
):
    data[field] = _text_or_missing(
        prepared,
        fields.get(field)
    )

In [ ]:
# ================================================================
# 2A.6 — PRESERVE SOURCE SHEET
# ================================================================

if "source_sheet" in prepared.columns:
    data["source_sheet"] = (
        prepared["source_sheet"]
        .astype("string")
        .str.strip()
    )
else:
    data["source_sheet"] = pd.Series(
        pd.NA,
        index=prepared.index,
        dtype="string"
    )

In [ ]:
# ================================================================
# 2A.7 — STANDARDISE CUSTOMER IDs
# ================================================================

data["customer"] = (
    data["customer"]
    .str.replace(r"\.0$", "", regex=True)
)

In [ ]:
# ================================================================
# 2A.8 — STANDARDISE PRODUCT CODES
# ================================================================

data["product"] = (
    data["product"]
    .str.upper()
    .str.strip()
)

In [ ]:
# ================================================================
# 2A.9 — STANDARDISE PRODUCT DESCRIPTIONS
# ================================================================

data["description"] = (
    data["description"]
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# ================================================================
# 2A.10 — STANDARDISE REGION AND INVOICE
# ================================================================

data["region"] = (
    data["region"]
    .str.strip()
    .str.title()
)

data["invoice"] = (
    data["invoice"]
    .str.upper()
    .str.strip()
)

In [ ]:
# ================================================================
# 2A.11 — CLASSIFY KNOWN NON-MERCHANDISE CODES
# ================================================================

data["non_merchandise_code"] = (
    data["product"]
    .map(NON_MERCHANDISE_CODE_TYPES)
)

data["is_known_non_merchandise"] = (
    data["non_merchandise_code"].notna()
)

In [ ]:
# ================================================================
# 2A.12 — IDENTIFY INVALID REQUIRED VALUES
# ================================================================

invalid_required = (
    data["date"].isna()
    | data["revenue"].isna()
)

print(
    f"Rows with invalid date/revenue: "
    f"{invalid_required.sum():,}"
)

In [ ]:
# ================================================================
# 2A.13 — IDENTIFY UNEXPLAINED NEGATIVE PRICES
# ================================================================

unexplained_negative_price = (
    data["unit_price"].lt(0).fillna(False)
    & ~data["is_known_non_merchandise"]
)

print(
    f"Unexplained negative-price rows: "
    f"{unexplained_negative_price.sum():,}"
)

In [ ]:
# ================================================================
# 2A.14 — REMOVE INVALID ROWS
# ================================================================

pre_filter_data = data.copy()

data = data.loc[
    ~invalid_required
    & ~unexplained_negative_price
].copy()

print(
    f"Rows retained after validity checks: "
    f"{len(data):,}"
)

In [ ]:
# ================================================================
# 2A.15 — DEFINE DUPLICATE TRANSACTION KEY
# ================================================================

duplicate_key = [
    "invoice",
    "product",
    "description",
    "quantity",
    "date",
    "unit_price",
    "customer",
    "region",
]

In [ ]:
# ================================================================
# 2A.16 — FLAG WITHIN-SHEET DUPLICATES
# ================================================================

data["is_within_sheet_exact_duplicate"] = (
    data.duplicated(
        subset=duplicate_key + ["source_sheet"],
        keep=False
    )
)

print(
    "Within-sheet exact duplicate rows flagged: "
    f"{data['is_within_sheet_exact_duplicate'].sum():,}"
)

In [ ]:
# ================================================================
# 2A.17 — PREPARE CROSS-SHEET DUPLICATE MATCHING
# ================================================================

data["_duplicate_instance"] = (
    data.groupby(
        duplicate_key + ["source_sheet"],
        dropna=False
    )
    .cumcount()
)

cross_sheet_key = (
    duplicate_key
    + ["_duplicate_instance"]
)

In [ ]:
# ================================================================
# 2A.18 — IDENTIFY CROSS-SHEET DUPLICATES
# ================================================================

data = data.sort_values(
    ["date", "source_sheet"],
    kind="stable"
)

data["is_cross_sheet_duplicate"] = (
    data.duplicated(
        subset=cross_sheet_key,
        keep="first"
    )
)

cross_sheet_duplicates = data.loc[
    data["is_cross_sheet_duplicate"]
].copy()

print(
    f"Cross-sheet duplicate rows identified: "
    f"{len(cross_sheet_duplicates):,}"
)

In [ ]:
# ================================================================
# 2A.19 — REMOVE CROSS-SHEET DUPLICATES
# ================================================================

data = data.loc[
    ~data["is_cross_sheet_duplicate"]
].copy()

data = data.drop(
    columns=["_duplicate_instance"]
)

print(
    f"Rows after cross-sheet deduplication: "
    f"{len(data):,}"
)

In [ ]:
# ================================================================
# 2A.20 — FLAG CANCELLATIONS
# ================================================================

data["is_cancellation"] = (
    data["invoice"]
    .str.startswith("C", na=False)
)

In [ ]:
# ================================================================
# 2A.21 — FLAG RETURNS
# ================================================================

data["is_return"] = (
    data["quantity"]
    .lt(0)
    .fillna(False)
)

In [ ]:
# ================================================================
# 2A.22 — FLAG ZERO-PRICE TRANSACTIONS
# ================================================================

data["is_zero_price"] = (
    data["unit_price"]
    .eq(0)
    .fillna(False)
)

### STEP 2B — REVENUE CLASSIFICATION AND ANALYTICAL FIELDS
Now the cleaning is largely complete.
Step 2B turns the clean transaction data into an analytically useful structure.

In [ ]:
# ================================================================
# 2B.1 — CLASSIFY TRANSACTION TYPE
# ================================================================

data["revenue_type"] = _classify_revenue_type(data)

display(
    data["revenue_type"]
    .value_counts(dropna=False)
)

In [ ]:
# ================================================================
# 2B.2 — DEFINE MERCHANDISE STATUS
# ================================================================

data["is_merchandise"] = (
    data["non_merchandise_code"].isna()
)

In [ ]:
# ================================================================
# 2B.3 — ALL-IN RECORDED REVENUE
# ================================================================

data["all_in_revenue"] = (
    data["revenue"]
)

In [ ]:
# ================================================================
# 2B.4 — NET MERCHANDISE REVENUE
# ================================================================

data["net_merchandise_revenue"] = (
    data["revenue"]
    .where(
        data["is_merchandise"],
        0.0
    )
)

In [ ]:
# ================================================================
# 2B.5 — GROSS MERCHANDISE REVENUE
# ================================================================

data["gross_merchandise_revenue"] = (
    data["revenue"]
    .where(
        data["is_merchandise"]
        & data["quantity"].gt(0)
        & data["unit_price"].gt(0)
        & ~data["is_cancellation"],
        0.0
    )
)

In [ ]:
# ================================================================
# 2B.6 — NON-MERCHANDISE / ACCOUNTING REVENUE
# ================================================================

data["non_merchandise_revenue"] = (
    data["revenue"]
    .where(
        ~data["is_merchandise"],
        0.0
    )
)

In [ ]:
# ================================================================
# 2B.7 — CREATE CANONICAL PRODUCT DESCRIPTION
# ================================================================

description_counts = (
    data
    .dropna(
        subset=[
            "product",
            "description"
        ]
    )
    .groupby(
        [
            "product",
            "description"
        ],
        as_index=False
    )
    .agg(
        occurrences=("revenue", "size"),
        latest_date=("date", "max"),
    )
)

In [ ]:
# ================================================================
# 2B.8 — SELECT PREFERRED DESCRIPTION PER SKU
# ================================================================

description_counts = (
    description_counts
    .sort_values(
        [
            "product",
            "occurrences",
            "latest_date"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        subset="product",
        keep="first"
    )
)

canonical_descriptions = (
    description_counts
    .set_index("product")["description"]
)

In [ ]:
# ================================================================
# 2B.9 — ATTACH CANONICAL PRODUCT DESCRIPTION
# ================================================================

data["product_description"] = (
    data["product"]
    .map(canonical_descriptions)
    .fillna(data["description"])
)

In [ ]:
# ================================================================
# 2B.10 — FLAG IDENTIFIED CUSTOMERS
# ================================================================

data["customer_is_identified"] = (
    data["customer"].notna()
)

In [ ]:
# ================================================================
# 2B.11 — CUSTOMER FIRST OBSERVED DATE
# ================================================================

data["customer_first_observed_date"] = (
    data.groupby(
        "customer",
        dropna=True
    )["date"]
    .transform("min")
)

In [ ]:
# ================================================================
# 2B.12 — FLAG LEFT-TRUNCATED CUSTOMERS
# ================================================================

first_date = data["date"].min()

first_month = first_date.to_period("M")

data["is_left_truncated_customer"] = (
    data["customer_first_observed_date"]
    .dt.to_period("M")
    .eq(first_month)
    .fillna(False)
)

In [ ]:
# ================================================================
# 2B.13 — CREATE TIME PERIOD FIELDS
# ================================================================

data["period_month"] = (
    data["date"]
    .dt.to_period("M")
    .astype("string")
)

data["period_quarter"] = (
    data["date"]
    .dt.to_period("Q")
    .astype("string")
)

data["period_year"] = (
    data["date"]
    .dt.year
    .astype("Int64")
)

In [ ]:
# ================================================================
# 2B.14 — SORT FINAL ANALYTICAL DATASET
# ================================================================

data = (
    data
    .sort_values(
        "date",
        kind="stable"
    )
    .reset_index(drop=True)
)

In [ ]:
# ================================================================
# 2B.15 — CREATE FINAL ANALYTICAL DATAFRAME
# ================================================================

df = data.copy()

print(
    f"Final analytical dataset: "
    f"{df.shape[0]:,} rows × "
    f"{df.shape[1]:,} columns"
)

display(df.head())

In [ ]:
# ================================================================
# VISUAL 2.1 — DATA CLEANING FLOW
# ================================================================

import matplotlib.pyplot as plt


# ---------------------------------------------------------------
# Calculate cleaning-stage row counts
# ---------------------------------------------------------------

# Some invalid conditions could theoretically overlap, so use a
# combined Boolean mask rather than simply adding the two counts.
invalid_exclusion_mask = (
    invalid_required
    | unexplained_negative_price
)

invalid_rows_removed = int(
    invalid_exclusion_mask.sum()
)

rows_after_validity_checks = (
    source_rows
    - invalid_rows_removed
)

cross_sheet_rows_removed = (
    len(cross_sheet_duplicates)
)

final_rows = len(df)


# ---------------------------------------------------------------
# Build chart dataset
# ---------------------------------------------------------------

cleaning_stages = [
    "Raw Source Data",
    "After Validity Checks",
    "Final Analytical Data",
]

cleaning_row_counts = [
    source_rows,
    rows_after_validity_checks,
    final_rows,
]


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

bars = ax.bar(
    cleaning_stages,
    cleaning_row_counts
)

ax.set_title(
    "Data Cleaning Flow"
)

ax.set_ylabel(
    "Transaction Rows"
)

ax.set_xlabel(
    "Cleaning Stage"
)

ax.ticklabel_format(
    style="plain",
    axis="y"
)


# ---------------------------------------------------------------
# Add row-count labels
# ---------------------------------------------------------------

for bar, value in zip(
    bars,
    cleaning_row_counts
):
    ax.text(
        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height(),

        f"{value:,}",

        ha="center",
        va="bottom"
    )


plt.tight_layout()
plt.show()


# ---------------------------------------------------------------
# Print accompanying audit explanation
# ---------------------------------------------------------------

print(
    f"Invalid rows removed: "
    f"{invalid_rows_removed:,}"
)

print(
    f"Cross-sheet duplicate rows removed: "
    f"{cross_sheet_rows_removed:,}"
)

print(
    f"Final retained rows: "
    f"{final_rows:,}"
)

### Step 2 Summary — Data Cleaning and Standardisation

In Step 2, the raw retail transaction data was cleaned and converted into a consistent analytical structure before any revenue analysis was performed.

The main actions completed were:

- Standardised source fields into a canonical schema for dates, customers, products, descriptions, regions, invoices, quantities, prices and revenue.
- Preserved the original source-sheet information to support reconciliation of overlapping yearly extracts.
- Standardised customer IDs, product codes, descriptions, regions and invoice identifiers.
- Kept missing customer IDs as missing rather than combining them into a synthetic `UNKNOWN_CUSTOMER`.
- Identified known non-merchandise lines such as postage, fees, discounts and accounting adjustments.
- Retained legitimate accounting adjustments with negative prices while excluding only unexplained invalid negative-price records.
- Flagged returns, cancellations and zero-price transactions rather than removing them.
- Separated confirmed cross-sheet duplicates from within-sheet duplicate records; cross-sheet duplicates were removed while within-sheet duplicates were retained for audit and sensitivity testing.
- Created stable SKU-level product descriptions to reduce inconsistencies caused by description changes over time.
- Added customer-history fields, including first observed purchase date and a left-truncation flag for customers first observed in the dataset's opening month.
- Created standard monthly, quarterly and annual period fields.
- Established separate analytical revenue measures for gross merchandise revenue, net merchandise revenue, non-merchandise revenue and all-in recorded revenue.

The result of Step 2 is a single cleaned, auditable transaction-level dataset that can now be used consistently for the revenue, customer, product and geographic analyses that follow.


## STEP 3 — ANALYTICAL SCOPE AND REVENUE DEFINITIONS

### 3.1 Analytical objective
The primary objective is to determine how revenue performance changed between November 2010 and November 2011, and to identify the underlying commercial drivers of that change.
The analysis distinguishes core merchandise trading performance from non-merchandise and accounting effects so that changes in postage, fees, adjustments or other operational lines are not incorrectly interpreted as product-sales growth.
November 2010 is treated as the baseline period and November 2011 as the comparison period.

This is important because it anchors the rest of the notebook.

In [ ]:
# ================================================================
# 3.2 — DEFINE COMPARISON PERIODS
# ================================================================

baseline_period = "2010-11"
comparison_period = "2011-11"

print(f"Baseline period:   {baseline_period}")
print(f"Comparison period: {comparison_period}")

In [ ]:
# ================================================================
# 3.3 — VALIDATE COMPARISON PERIODS
# ================================================================

available_periods = sorted(
    df["period_month"]
    .dropna()
    .unique()
    .tolist()
)

for period in [baseline_period, comparison_period]:
    if period not in available_periods:
        raise ValueError(
            f"{period} is not available in the dataset."
        )

print("✓ Both comparison periods are available.")

In [ ]:
# ================================================================
# 3.4 — CREATE COMPARISON DATASET
# ================================================================

comparison_df = df.loc[
    df["period_month"].isin(
        [baseline_period, comparison_period]
    )
].copy()

print(
    f"Comparison dataset contains "
    f"{len(comparison_df):,} transaction lines."
)

display(
    comparison_df[
        [
            "date",
            "invoice",
            "product",
            "product_description",
            "quantity",
            "unit_price",
            "revenue",
            "net_merchandise_revenue",
            "non_merchandise_revenue",
            "period_month",
        ]
    ].head()
)

#### 3.5 Revenue definitions
Four revenue measures are used:
- **Gross merchandise revenue.**
    Positive merchandise sales before the effect of merchandise returns and cancellations.
- **Net merchandise revenue.**
    Merchandise revenue after including merchandise returns and cancellations. This is the principal measure of underlying trading performance.
- **Non-merchandise revenue.**
    Revenue from postage, fees, discounts, adjustments and other recognised non-merchandise transaction codes.
- **All-in recorded revenue.**
    The total recorded revenue effect of all retained transaction lines, equal to net merchandise revenue plus non-merchandise revenue.
  
Unless explicitly stated otherwise, product, customer and commercial-performance analyses should use net merchandise revenue.

In [ ]:
# ================================================================
# 3.6 — RECONCILE REVENUE BY PERIOD
# ================================================================

revenue_reconciliation = (
    comparison_df
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        gross_merchandise_revenue=(
            "gross_merchandise_revenue",
            "sum"
        ),
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        non_merchandise_revenue=(
            "non_merchandise_revenue",
            "sum"
        ),
        all_in_revenue=(
            "all_in_revenue",
            "sum"
        ),
    )
)

display(revenue_reconciliation)

In [ ]:
# ================================================================
# 3.7 — CALCULATE REVENUE CHANGES
# ================================================================

baseline_values = (
    revenue_reconciliation
    .loc[
        revenue_reconciliation["period_month"]
        == baseline_period
    ]
    .iloc[0]
)

comparison_values = (
    revenue_reconciliation
    .loc[
        revenue_reconciliation["period_month"]
        == comparison_period
    ]
    .iloc[0]
)

metrics = [
    "gross_merchandise_revenue",
    "net_merchandise_revenue",
    "non_merchandise_revenue",
    "all_in_revenue",
]

change_rows = []

for metric in metrics:

    baseline_value = baseline_values[metric]
    comparison_value = comparison_values[metric]

    absolute_change = (
        comparison_value
        - baseline_value
    )

    percentage_change = (
        absolute_change / baseline_value * 100
        if baseline_value != 0
        else np.nan
    )

    change_rows.append(
        {
            "metric": metric,
            "baseline": baseline_value,
            "comparison": comparison_value,
            "absolute_change": absolute_change,
            "percentage_change": percentage_change,
        }
    )

revenue_change_summary = pd.DataFrame(
    change_rows
)

display(revenue_change_summary)

In [ ]:
# ================================================================
# 3.8 — DEFINE PRIMARY REVENUE KPI
# ================================================================

primary_revenue_metric = "net_merchandise_revenue"

print(
    "Primary commercial revenue metric:",
    primary_revenue_metric
)

In [ ]:
# ================================================================
# 3.9 — PRIMARY COMMERCIAL REVENUE MOVEMENT
# ================================================================

primary_summary = (
    revenue_change_summary
    .loc[
        revenue_change_summary["metric"]
        == primary_revenue_metric
    ]
    .iloc[0]
)

baseline_revenue = primary_summary["baseline"]
comparison_revenue = primary_summary["comparison"]
revenue_change = primary_summary["absolute_change"]
revenue_change_pct = primary_summary["percentage_change"]

print(
    f"Baseline net merchandise revenue "
    f"({baseline_period}): "
    f"{baseline_revenue:,.2f}"
)

print(
    f"Comparison net merchandise revenue "
    f"({comparison_period}): "
    f"{comparison_revenue:,.2f}"
)

print(
    f"Absolute change: "
    f"{revenue_change:,.2f}"
)

print(
    f"Percentage change: "
    f"{revenue_change_pct:,.2f}%"
)

In [ ]:
# ================================================================
# 3.10 — ALL-IN RECORDED REVENUE MOVEMENT
# ================================================================

all_in_summary = (
    revenue_change_summary
    .loc[
        revenue_change_summary["metric"]
        == "all_in_revenue"
    ]
    .iloc[0]
)

print(
    f"All-in revenue change: "
    f"{all_in_summary['absolute_change']:,.2f}"
)

print(
    f"All-in revenue change %: "
    f"{all_in_summary['percentage_change']:.2f}%"
)

In [ ]:
# ================================================================
# 3.11 — MEASURE NON-MERCHANDISE CONTRIBUTION
# ================================================================

non_merch_summary = (
    revenue_change_summary
    .loc[
        revenue_change_summary["metric"]
        == "non_merchandise_revenue"
    ]
    .iloc[0]
)

all_in_change = (
    all_in_summary["absolute_change"]
)

non_merch_change = (
    non_merch_summary["absolute_change"]
)

non_merch_contribution_pct = (
    non_merch_change
    / all_in_change
    * 100
    if all_in_change != 0
    else np.nan
)

print(
    f"Change in non-merchandise revenue: "
    f"{non_merch_change:,.2f}"
)

print(
    f"Share of all-in revenue movement "
    f"explained by non-merchandise lines: "
    f"{non_merch_contribution_pct:.2f}%"
)

In [ ]:
# ================================================================
# 3.12 — NON-MERCHANDISE REVENUE BY TYPE
# ================================================================

non_merchandise_detail = (
    comparison_df
    .loc[
        ~comparison_df["is_merchandise"]
    ]
    .groupby(
        [
            "period_month",
            "non_merchandise_code"
        ],
        as_index=False
    )
    .agg(
        revenue=(
            "all_in_revenue",
            "sum"
        ),
        rows=(
            "all_in_revenue",
            "size"
        ),
    )
)

display(non_merchandise_detail)

In [ ]:
# ================================================================
# 3.13 — DEFINE CUSTOMER ANALYSIS SCOPE
# ================================================================

customer_comparison_df = (
    comparison_df
    .loc[
        comparison_df["customer_is_identified"]
    ]
    .copy()
)

print(
    f"Identified customer rows: "
    f"{len(customer_comparison_df):,}"
)

print(
    f"Identified customer revenue: "
    f"{customer_comparison_df['net_merchandise_revenue'].sum():,.2f}"
)

In [ ]:
# ================================================================
# 3.14 — DEFINE PRODUCT ANALYSIS SCOPE
# ================================================================

product_comparison_df = (
    comparison_df
    .loc[
        comparison_df["is_merchandise"]
        & comparison_df["product"].notna()
    ]
    .copy()
)

print(
    f"Merchandise product rows: "
    f"{len(product_comparison_df):,}"
)

print(
    f"Net merchandise revenue: "
    f"{product_comparison_df['net_merchandise_revenue'].sum():,.2f}"
)

In [ ]:
# ================================================================
# 3.15 — DEFINE COUNTRY ANALYSIS SCOPE
# ================================================================

country_comparison_df = (
    comparison_df
    .loc[
        comparison_df["region"].notna()
    ]
    .copy()
)

print(
    f"Country analysis rows: "
    f"{len(country_comparison_df):,}"
)

In [ ]:
# ================================================================
# 3.16 — ANALYTICAL SCOPE SUMMARY
# ================================================================

scope_summary = pd.DataFrame(
    {
        "Analysis": [
            "Overall accounting revenue",
            "Core commercial revenue",
            "Country drivers",
            "Product drivers",
            "Customer drivers",
        ],
        "Dataset": [
            "comparison_df",
            "comparison_df",
            "country_comparison_df",
            "product_comparison_df",
            "customer_comparison_df",
        ],
        "Primary metric": [
            "all_in_revenue",
            "net_merchandise_revenue",
            "net_merchandise_revenue",
            "net_merchandise_revenue",
            "net_merchandise_revenue",
        ],
    }
)

display(scope_summary)

### Step 3 Summary — Analytical Scope and Revenue Definitions

In Step 3, the cleaned transaction dataset was organised into clearly defined analytical scopes and revenue measures before beginning the detailed performance analysis.

The main actions completed were:

- Defined November 2010 as the baseline period and November 2011 as the comparison period.
- Confirmed that both periods are available in the cleaned dataset.
- Created a dedicated comparison dataset containing only the two periods under review.
- Defined four revenue measures: gross merchandise revenue, net merchandise revenue, non-merchandise revenue, and all-in recorded revenue.
- Established net merchandise revenue as the primary measure of core commercial performance.
- Reconciled the different revenue measures across both periods and calculated absolute and percentage changes.
- Separated the overall recorded revenue movement from the underlying merchandise trading movement.
- Measured the contribution of non-merchandise lines such as postage, fees and adjustments to the overall change in revenue.
- Created separate analytical scopes for customer, product and country analysis to ensure that each later section uses an appropriate dataset and revenue definition.
- Restricted customer-level analysis to identified customers and product analysis to merchandise SKUs, preventing anonymous customers and non-merchandise codes from distorting later findings.

The result of Step 3 is a clear analytical framework that distinguishes core trading performance from accounting and service-line effects, providing a consistent basis for the revenue variance, customer, product and geographic analyses that follow.


## Step 4 — Overall Revenue Variance

This section evaluates the overall change in revenue between November 2010 and November 2011.

The analysis separates the movement in **core merchandise revenue** from **non-merchandise and accounting effects**. It also examines gross merchandise sales and the effect of returns and cancellations to determine whether the headline revenue movement reflects genuine underlying trading growth.

The objective at this stage is to establish **what changed at the total-business level** before investigating the customer, country and product drivers of that movement.

In [ ]:
# ================================================================
# 4.2 — CONFIRM COMPARISON-PERIOD DATA
# ================================================================

print(f"Baseline period:   {baseline_period}")
print(f"Comparison period: {comparison_period}")

print(
    f"\nRows in comparison dataset: "
    f"{len(comparison_df):,}"
)

print(
    "\nTransactions by period:"
)

display(
    comparison_df["period_month"]
    .value_counts()
    .sort_index()
    .rename("transaction_rows")
    .to_frame()
)

In [ ]:
# ================================================================
# 4.3 — BUILD OVERALL REVENUE SUMMARY
# ================================================================

period_revenue_summary = (
    comparison_df
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        gross_merchandise_revenue=(
            "gross_merchandise_revenue",
            "sum"
        ),
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        non_merchandise_revenue=(
            "non_merchandise_revenue",
            "sum"
        ),
        all_in_revenue=(
            "all_in_revenue",
            "sum"
        ),
    )
)

display(period_revenue_summary)

In [ ]:
# ================================================================
# 4.4 — CALCULATE REVENUE VARIANCES
# ================================================================

revenue_metrics = [
    "gross_merchandise_revenue",
    "net_merchandise_revenue",
    "non_merchandise_revenue",
    "all_in_revenue",
]

variance_results = []

for metric in revenue_metrics:

    baseline_value = (
        period_revenue_summary
        .loc[
            period_revenue_summary["period_month"]
            == baseline_period,
            metric
        ]
        .iloc[0]
    )

    comparison_value = (
        period_revenue_summary
        .loc[
            period_revenue_summary["period_month"]
            == comparison_period,
            metric
        ]
        .iloc[0]
    )

    absolute_change = (
        comparison_value
        - baseline_value
    )

    percentage_change = (
        absolute_change
        / baseline_value
        * 100
        if baseline_value != 0
        else np.nan
    )

    variance_results.append(
        {
            "metric": metric,
            "baseline_revenue": baseline_value,
            "comparison_revenue": comparison_value,
            "absolute_change": absolute_change,
            "percentage_change": percentage_change,
        }
    )

revenue_variance_summary = pd.DataFrame(
    variance_results
)

display(revenue_variance_summary)

In [ ]:
# ================================================================
# 4.5 — CREATE PRESENTATION-READY REVENUE TABLE
# ================================================================

metric_labels = {
    "gross_merchandise_revenue":
        "Gross Merchandise Revenue",

    "net_merchandise_revenue":
        "Net Merchandise Revenue",

    "non_merchandise_revenue":
        "Non-Merchandise Revenue",

    "all_in_revenue":
        "All-In Recorded Revenue",
}

revenue_variance_display = (
    revenue_variance_summary
    .copy()
)

revenue_variance_display["metric"] = (
    revenue_variance_display["metric"]
    .map(metric_labels)
)

revenue_variance_display = (
    revenue_variance_display.rename(
        columns={
            "metric": "Revenue Measure",
            "baseline_revenue":
                f"{baseline_period} Revenue",
            "comparison_revenue":
                f"{comparison_period} Revenue",
            "absolute_change":
                "Absolute Change",
            "percentage_change":
                "Change (%)",
        }
    )
)

display(
    revenue_variance_display.style.format(
        {
            f"{baseline_period} Revenue": "{:,.2f}",
            f"{comparison_period} Revenue": "{:,.2f}",
            "Absolute Change": "{:,.2f}",
            "Change (%)": "{:,.2f}%",
        }
    )
)

In [ ]:
# ================================================================
# 4.6 — HEADLINE ALL-IN REVENUE MOVEMENT
# ================================================================

all_in_result = (
    revenue_variance_summary
    .loc[
        revenue_variance_summary["metric"]
        == "all_in_revenue"
    ]
    .iloc[0]
)

all_in_baseline = (
    all_in_result["baseline_revenue"]
)

all_in_comparison = (
    all_in_result["comparison_revenue"]
)

all_in_change = (
    all_in_result["absolute_change"]
)

all_in_change_pct = (
    all_in_result["percentage_change"]
)


print("ALL-IN RECORDED REVENUE")
print("-" * 50)

print(
    f"{baseline_period}: "
    f"{all_in_baseline:,.2f}"
)

print(
    f"{comparison_period}: "
    f"{all_in_comparison:,.2f}"
)

print(
    f"Absolute change: "
    f"{all_in_change:,.2f}"
)

print(
    f"Percentage change: "
    f"{all_in_change_pct:,.2f}%"
)

In [ ]:
# ================================================================
# 4.7 — CORE MERCHANDISE REVENUE MOVEMENT
# ================================================================

merchandise_result = (
    revenue_variance_summary
    .loc[
        revenue_variance_summary["metric"]
        == "net_merchandise_revenue"
    ]
    .iloc[0]
)

merchandise_baseline = (
    merchandise_result["baseline_revenue"]
)

merchandise_comparison = (
    merchandise_result["comparison_revenue"]
)

merchandise_change = (
    merchandise_result["absolute_change"]
)

merchandise_change_pct = (
    merchandise_result["percentage_change"]
)


print("NET MERCHANDISE REVENUE")
print("-" * 50)

print(
    f"{baseline_period}: "
    f"{merchandise_baseline:,.2f}"
)

print(
    f"{comparison_period}: "
    f"{merchandise_comparison:,.2f}"
)

print(
    f"Absolute change: "
    f"{merchandise_change:,.2f}"
)

print(
    f"Percentage change: "
    f"{merchandise_change_pct:,.2f}%"
)

In [ ]:
# ================================================================
# 4.8 — RECONCILE THE OVERALL REVENUE CHANGE
# ================================================================

non_merchandise_result = (
    revenue_variance_summary
    .loc[
        revenue_variance_summary["metric"]
        == "non_merchandise_revenue"
    ]
    .iloc[0]
)

non_merchandise_change = (
    non_merchandise_result["absolute_change"]
)

reconciliation_check = (
    merchandise_change
    + non_merchandise_change
)

print(
    f"Merchandise revenue change: "
    f"{merchandise_change:,.2f}"
)

print(
    f"Non-merchandise revenue change: "
    f"{non_merchandise_change:,.2f}"
)

print(
    f"Combined change: "
    f"{reconciliation_check:,.2f}"
)

print(
    f"Reported all-in change: "
    f"{all_in_change:,.2f}"
)

In [ ]:
# ================================================================
# 4.9 — VALIDATE REVENUE RECONCILIATION
# ================================================================

assert np.isclose(
    reconciliation_check,
    all_in_change,
    atol=0.01
), (
    "Revenue reconciliation failed: "
    "merchandise change plus non-merchandise change "
    "does not equal the all-in revenue change."
)

print(
    "✓ Revenue movement reconciles successfully."
)

In [ ]:
# ================================================================
# 4.10 — CONTRIBUTION TO THE HEADLINE REVENUE CHANGE
# ================================================================

revenue_change_bridge = pd.DataFrame(
    {
        "component": [
            "Net Merchandise Revenue",
            "Non-Merchandise Revenue",
        ],
        "absolute_change": [
            merchandise_change,
            non_merchandise_change,
        ],
    }
)

if all_in_change != 0:
    revenue_change_bridge[
        "share_of_net_change_pct"
    ] = (
        revenue_change_bridge[
            "absolute_change"
        ]
        / all_in_change
        * 100
    )
else:
    revenue_change_bridge[
        "share_of_net_change_pct"
    ] = np.nan


display(
    revenue_change_bridge.style.format(
        {
            "absolute_change": "{:,.2f}",
            "share_of_net_change_pct": "{:,.2f}%",
        }
    )
)

In [ ]:
# ================================================================
# 4.11 — GROSS VS NET MERCHANDISE REVENUE
# ================================================================

merchandise_margin_summary = (
    comparison_df
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        gross_merchandise_revenue=(
            "gross_merchandise_revenue",
            "sum"
        ),
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
    )
)

merchandise_margin_summary[
    "returns_and_cancellations_effect"
] = (
    merchandise_margin_summary[
        "net_merchandise_revenue"
    ]
    - merchandise_margin_summary[
        "gross_merchandise_revenue"
    ]
)

merchandise_margin_summary[
    "net_to_gross_pct"
] = (
    merchandise_margin_summary[
        "net_merchandise_revenue"
    ]
    / merchandise_margin_summary[
        "gross_merchandise_revenue"
    ]
    * 100
)

display(
    merchandise_margin_summary
)

In [ ]:
# ================================================================
# 4.12 — CHANGE IN RETURNS / CANCELLATION EFFECT
# ================================================================

baseline_drag = (
    merchandise_margin_summary
    .loc[
        merchandise_margin_summary["period_month"]
        == baseline_period,
        "returns_and_cancellations_effect"
    ]
    .iloc[0]
)

comparison_drag = (
    merchandise_margin_summary
    .loc[
        merchandise_margin_summary["period_month"]
        == comparison_period,
        "returns_and_cancellations_effect"
    ]
    .iloc[0]
)

drag_change = (
    comparison_drag
    - baseline_drag
)

print(
    f"Baseline returns/cancellation effect: "
    f"{baseline_drag:,.2f}"
)

print(
    f"Comparison returns/cancellation effect: "
    f"{comparison_drag:,.2f}"
)

print(
    f"Change in revenue drag: "
    f"{drag_change:,.2f}"
)

In [ ]:
# ================================================================
# VISUAL 4.2 — GROSS VS NET MERCHANDISE REVENUE
# ================================================================

# ---------------------------------------------------------------
# Prepare chart data
# ---------------------------------------------------------------

gross_net_plot = (
    merchandise_margin_summary
    .set_index("period_month")
    .loc[
        [
            baseline_period,
            comparison_period
        ]
    ]
)


period_labels = [
    baseline_period,
    comparison_period
]

gross_values = (
    gross_net_plot[
        "gross_merchandise_revenue"
    ]
    .values
)

net_values = (
    gross_net_plot[
        "net_merchandise_revenue"
    ]
    .values
)


# ---------------------------------------------------------------
# Create grouped bar positions
# ---------------------------------------------------------------

x = np.arange(
    len(period_labels)
)

bar_width = 0.36


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

gross_bars = ax.bar(
    x - bar_width / 2,
    gross_values,
    bar_width,
    label="Gross Merchandise Revenue"
)

net_bars = ax.bar(
    x + bar_width / 2,
    net_values,
    bar_width,
    label="Net Merchandise Revenue"
)


# ---------------------------------------------------------------
# Format chart
# ---------------------------------------------------------------

ax.set_title(
    "Gross vs Net Merchandise Revenue"
)

ax.set_xlabel(
    "Period"
)

ax.set_ylabel(
    "Revenue"
)

ax.set_xticks(x)

ax.set_xticklabels(
    period_labels
)

ax.legend()


# ---------------------------------------------------------------
# Add data labels
# ---------------------------------------------------------------

for bars in [
    gross_bars,
    net_bars
]:
    for bar in bars:

        value = bar.get_height()

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            value,

            f"{value / 1_000_000:.3f}m",

            ha="center",
            va="bottom",
            fontsize=9
        )


plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 4.13 — CREATE MUTUALLY EXCLUSIVE MERCHANDISE CATEGORIES
# ================================================================

merchandise_activity = (
    comparison_df
    .loc[
        comparison_df["is_merchandise"]
    ]
    .copy()
)


conditions = [
    merchandise_activity["is_cancellation"],

    (
        merchandise_activity["is_return"]
        & ~merchandise_activity["is_cancellation"]
    ),

    (
        merchandise_activity["quantity"].gt(0)
        & merchandise_activity["unit_price"].gt(0)
        & ~merchandise_activity["is_cancellation"]
    ),

    merchandise_activity["is_zero_price"],
]

choices = [
    "Cancellation",
    "Return - Non-Cancellation",
    "Positive Merchandise Sale",
    "Zero-Price Merchandise",
]


merchandise_activity[
    "merchandise_activity_type"
] = np.select(
    conditions,
    choices,
    default="Other Merchandise"
)

In [ ]:
# ================================================================
# 4.13 — CREATE MUTUALLY EXCLUSIVE MERCHANDISE CATEGORIES
# ================================================================

merchandise_activity = (
    comparison_df
    .loc[
        comparison_df["is_merchandise"]
    ]
    .copy()
)


conditions = [
    merchandise_activity["is_cancellation"],

    (
        merchandise_activity["is_return"]
        & ~merchandise_activity["is_cancellation"]
    ),

    (
        merchandise_activity["quantity"].gt(0)
        & merchandise_activity["unit_price"].gt(0)
        & ~merchandise_activity["is_cancellation"]
    ),

    merchandise_activity["is_zero_price"],
]

choices = [
    "Cancellation",
    "Return - Non-Cancellation",
    "Positive Merchandise Sale",
    "Zero-Price Merchandise",
]


merchandise_activity[
    "merchandise_activity_type"
] = np.select(
    conditions,
    choices,
    default="Other Merchandise"
)

In [ ]:
# ================================================================
# 4.15 — NON-MERCHANDISE REVENUE BREAKDOWN
# ================================================================

non_merchandise_variance = (
    comparison_df
    .loc[
        ~comparison_df["is_merchandise"]
    ]
    .groupby(
        [
            "non_merchandise_code",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        revenue=(
            "all_in_revenue",
            "sum"
        )
    )
)

non_merchandise_variance_pivot = (
    non_merchandise_variance
    .pivot(
        index="non_merchandise_code",
        columns="period_month",
        values="revenue"
    )
    .fillna(0)
)


for period in [
    baseline_period,
    comparison_period
]:
    if period not in non_merchandise_variance_pivot.columns:
        non_merchandise_variance_pivot[
            period
        ] = 0.0


non_merchandise_variance_pivot[
    "absolute_change"
] = (
    non_merchandise_variance_pivot[
        comparison_period
    ]
    - non_merchandise_variance_pivot[
        baseline_period
    ]
)


non_merchandise_variance_pivot = (
    non_merchandise_variance_pivot
    .reset_index()
    .sort_values(
        "absolute_change",
        ascending=False
    )
)

display(non_merchandise_variance_pivot)

In [ ]:
# ================================================================
# 4.16 — VISUAL: BASELINE VS COMPARISON REVENUE
# ================================================================

import matplotlib.pyplot as plt


plot_data = (
    period_revenue_summary
    .set_index("period_month")
    [
        [
            "net_merchandise_revenue",
            "non_merchandise_revenue",
        ]
    ]
)


ax = plot_data.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

ax.set_title(
    "Revenue Composition: "
    "November 2010 vs November 2011"
)

ax.set_xlabel(
    "Period"
)

ax.set_ylabel(
    "Revenue"
)

ax.legend(
    [
        "Net Merchandise Revenue",
        "Non-Merchandise Revenue",
    ]
)

plt.xticks(
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 4.17 — HEADLINE REVENUE FINDINGS
# ================================================================

headline_summary = pd.DataFrame(
    {
        "Measure": [
            "All-in revenue change",
            "All-in revenue growth",
            "Net merchandise revenue change",
            "Net merchandise revenue growth",
            "Non-merchandise revenue change",
            "Change in returns/cancellation drag",
        ],

        "Value": [
            all_in_change,
            all_in_change_pct,
            merchandise_change,
            merchandise_change_pct,
            non_merchandise_change,
            drag_change,
        ],

        "Unit": [
            "Revenue",
            "%",
            "Revenue",
            "%",
            "Revenue",
            "Revenue",
        ],
    }
)

display(headline_summary)

### Step 4 Summary — Overall Revenue Variance

Step 4 evaluated the overall change in business revenue between **November 2010** and **November 2011** and separated the headline movement into core merchandise performance and non-merchandise effects. This distinction is important because total recorded revenue includes items such as postage, fees and accounting adjustments, which should not automatically be interpreted as underlying product-sales growth.

The analysis shows that **all-in recorded revenue increased from approximately 1.423 million in November 2010 to 1.462 million in November 2011**. This represents an increase of approximately **39.10k, or 2.75%**. On the surface, this confirms that the business generated more total recorded revenue in the comparison period than in the baseline period.

However, the more commercially meaningful measure is **net merchandise revenue**, because it isolates the performance of the underlying merchandise business while still recognising the effect of merchandise returns and cancellations. Net merchandise revenue increased from approximately **1.401 million to 1.433 million**, an improvement of approximately **32.17k or 2.30%**.

This means that the majority of the overall revenue increase was supported by genuine improvement in merchandise trading rather than being driven entirely by non-merchandise activity. Net merchandise revenue accounted for approximately **82.3% of the total increase in all-in recorded revenue**.

Non-merchandise revenue also increased during the period. It rose from approximately **22.09k in November 2010 to 29.02k in November 2011**, an increase of approximately **6.93k**. This represented around **17.7% of the overall increase in recorded revenue**. The result demonstrates why the separation between merchandise and non-merchandise revenue is necessary: the headline growth rate slightly overstates the growth in the core merchandise business because part of the increase came from non-merchandise lines.

The analysis also compared **gross merchandise revenue** with **net merchandise revenue** to understand the effect of cancellations, returns and other negative merchandise transactions. Gross merchandise revenue increased from approximately **1.436 million to 1.458 million**, an increase of approximately **22.06k or 1.54%**.

At the same time, the negative effect of merchandise cancellations and related negative transaction activity improved materially. The difference between gross and net merchandise revenue reduced from approximately **35.12k in November 2010 to 25.01k in November 2011**. In other words, the revenue drag caused by negative merchandise activity became approximately **10.11k smaller** in the comparison period.

Relative to gross merchandise revenue, this drag improved from approximately **2.45% of gross merchandise sales to 1.72%**. This is an important finding because it means that the increase in net merchandise revenue was not caused only by additional positive sales. Part of the improvement also resulted from the business losing less revenue to cancellations and related negative transactions.

Overall, the Step 4 analysis indicates that the business experienced **modest but genuine improvement in underlying trading performance**. Total recorded revenue grew by approximately **2.75%**, while the more relevant measure of core merchandise performance grew by approximately **2.30%**. Most of the increase came from merchandise activity, although non-merchandise revenue also made a meaningful contribution.

The evidence therefore supports the conclusion that revenue performance improved between November 2010 and November 2011, but the improvement was relatively moderate rather than exceptionally strong. The next stage of the analysis should determine **where this growth came from geographically**, before examining customer and product-level drivers in greater detail.



## Step 5 — Country Revenue Drivers

This section examines how merchandise revenue changed across countries between November 2010 and November 2011.

The objective is to identify which geographic markets contributed positively or negatively to the overall change in **net merchandise revenue**, determine whether growth was concentrated in a small number of countries, and assess the extent to which the business remained dependent on the United Kingdom.

Country performance is evaluated using net merchandise revenue so that postage, fees, adjustments and other non-merchandise lines do not distort the geographic trading analysis.

In [ ]:
# ================================================================
# 5.2 — CONFIRM COUNTRY ANALYSIS SCOPE
# ================================================================

print(f"Baseline period:   {baseline_period}")
print(f"Comparison period: {comparison_period}")

print(
    f"\nCountry-analysis rows: "
    f"{len(country_comparison_df):,}"
)

print(
    f"Countries represented: "
    f"{country_comparison_df['region'].nunique():,}"
)

In [ ]:
# ================================================================
# 5.3 — CALCULATE COUNTRY REVENUE BY PERIOD
# ================================================================

country_revenue = (
    country_comparison_df
    .groupby(
        [
            "region",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)

display(country_revenue.head())

In [ ]:
# ================================================================
# 5.4 — PIVOT COUNTRY REVENUE
# ================================================================

country_revenue_pivot = (
    country_revenue
    .pivot(
        index="region",
        columns="period_month",
        values="net_merchandise_revenue"
    )
    .fillna(0)
)

# Ensure both comparison periods exist as columns.
for period in [
    baseline_period,
    comparison_period
]:
    if period not in country_revenue_pivot.columns:
        country_revenue_pivot[period] = 0.0

country_revenue_pivot.head()

In [ ]:
# ================================================================
# 5.5 — CALCULATE COUNTRY REVENUE CHANGES
# ================================================================

country_revenue_pivot[
    "absolute_change"
] = (
    country_revenue_pivot[
        comparison_period
    ]
    - country_revenue_pivot[
        baseline_period
    ]
)

country_revenue_pivot[
    "percentage_change"
] = np.where(
    country_revenue_pivot[
        baseline_period
    ] != 0,

    country_revenue_pivot[
        "absolute_change"
    ]
    / country_revenue_pivot[
        baseline_period
    ]
    * 100,

    np.nan
)

country_revenue_change = (
    country_revenue_pivot
    .reset_index()
)

display(
    country_revenue_change.sort_values(
        "absolute_change",
        ascending=False
    )
)

In [ ]:
# ================================================================
# 5.6 — RECONCILE COUNTRY CHANGES TO TOTAL MERCHANDISE CHANGE
# ================================================================

country_total_change = (
    country_revenue_change[
        "absolute_change"
    ]
    .sum()
)

print(
    f"Total country-level merchandise change: "
    f"{country_total_change:,.2f}"
)

print(
    f"Step 4 merchandise change: "
    f"{merchandise_change:,.2f}"
)

In [ ]:
# ================================================================
# 5.7 — VALIDATE COUNTRY RECONCILIATION
# ================================================================

assert np.isclose(
    country_total_change,
    merchandise_change,
    atol=0.01
), (
    "Country revenue changes do not reconcile "
    "to the Step 4 merchandise revenue change."
)

print(
    "✓ Country revenue movements reconcile "
    "to total merchandise revenue change."
)

In [ ]:
# ================================================================
# 5.8 — CALCULATE CONTRIBUTION TO TOTAL REVENUE CHANGE
# ================================================================

country_revenue_change[
    "contribution_to_total_change_pct"
] = np.where(
    merchandise_change != 0,

    country_revenue_change[
        "absolute_change"
    ]
    / merchandise_change
    * 100,

    np.nan
)

display(
    country_revenue_change
    .sort_values(
        "absolute_change",
        ascending=False
    )
)

In [ ]:
# ================================================================
# 5.9 — TOP POSITIVE COUNTRY CONTRIBUTORS
# ================================================================

top_country_gainers = (
    country_revenue_change
    .loc[
        country_revenue_change[
            "absolute_change"
        ] > 0
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .head(10)
)

display(
    top_country_gainers.style.format(
        {
            baseline_period: "{:,.2f}",
            comparison_period: "{:,.2f}",
            "absolute_change": "{:,.2f}",
            "percentage_change": "{:,.2f}%",
            "contribution_to_total_change_pct": "{:,.2f}%",
        }
    )
)

In [ ]:
# ================================================================
# 5.10 — TOP NEGATIVE COUNTRY CONTRIBUTORS
# ================================================================

top_country_decliners = (
    country_revenue_change
    .loc[
        country_revenue_change[
            "absolute_change"
        ] < 0
    ]
    .sort_values(
        "absolute_change",
        ascending=True
    )
    .head(10)
)

display(
    top_country_decliners.style.format(
        {
            baseline_period: "{:,.2f}",
            comparison_period: "{:,.2f}",
            "absolute_change": "{:,.2f}",
            "percentage_change": "{:,.2f}%",
            "contribution_to_total_change_pct": "{:,.2f}%",
        }
    )
)

In [ ]:
# ================================================================
# 5.11 — UNITED KINGDOM VS REST OF WORLD
# ================================================================

uk_country = "United Kingdom"

uk_summary = (
    country_revenue_change
    .loc[
        country_revenue_change[
            "region"
        ] == uk_country
    ]
)

if uk_summary.empty:
    raise ValueError(
        "United Kingdom was not found "
        "in the country-analysis dataset."
    )

uk_summary = uk_summary.iloc[0]

uk_baseline = (
    uk_summary[baseline_period]
)

uk_comparison = (
    uk_summary[comparison_period]
)

uk_change = (
    uk_summary["absolute_change"]
)


rest_of_world = (
    country_revenue_change
    .loc[
        country_revenue_change[
            "region"
        ] != uk_country
    ]
)

rest_baseline = (
    rest_of_world[
        baseline_period
    ]
    .sum()
)

rest_comparison = (
    rest_of_world[
        comparison_period
    ]
    .sum()
)

rest_change = (
    rest_of_world[
        "absolute_change"
    ]
    .sum()
)


uk_vs_rest = pd.DataFrame(
    {
        "Market": [
            "United Kingdom",
            "Rest of World",
        ],

        baseline_period: [
            uk_baseline,
            rest_baseline,
        ],

        comparison_period: [
            uk_comparison,
            rest_comparison,
        ],

        "Absolute Change": [
            uk_change,
            rest_change,
        ],
    }
)

display(uk_vs_rest)

In [ ]:
# ================================================================
# 5.12 — UK VS REST-OF-WORLD GROWTH RATES
# ================================================================

uk_growth_pct = (
    uk_change
    / uk_baseline
    * 100
    if uk_baseline != 0
    else np.nan
)

rest_growth_pct = (
    rest_change
    / rest_baseline
    * 100
    if rest_baseline != 0
    else np.nan
)

print(
    f"United Kingdom growth: "
    f"{uk_growth_pct:.2f}%"
)

print(
    f"Rest of World growth: "
    f"{rest_growth_pct:.2f}%"
)

In [ ]:
# ================================================================
# 5.13 — UNITED KINGDOM REVENUE CONCENTRATION
# ================================================================

total_baseline_revenue = (
    country_revenue_change[
        baseline_period
    ]
    .sum()
)

total_comparison_revenue = (
    country_revenue_change[
        comparison_period
    ]
    .sum()
)


uk_baseline_share = (
    uk_baseline
    / total_baseline_revenue
    * 100
)

uk_comparison_share = (
    uk_comparison
    / total_comparison_revenue
    * 100
)


print(
    f"UK share of merchandise revenue "
    f"in {baseline_period}: "
    f"{uk_baseline_share:.2f}%"
)

print(
    f"UK share of merchandise revenue "
    f"in {comparison_period}: "
    f"{uk_comparison_share:.2f}%"
)

print(
    f"Change in UK revenue share: "
    f"{uk_comparison_share - uk_baseline_share:+.2f} percentage points"
)

In [ ]:
# ================================================================
# VISUAL 5.2 — UNITED KINGDOM VS REST OF WORLD
# ================================================================

# ---------------------------------------------------------------
# Prepare chart data
# ---------------------------------------------------------------

markets = [
    "United Kingdom",
    "Rest of World"
]

baseline_values = [
    uk_baseline,
    rest_baseline
]

comparison_values = [
    uk_comparison,
    rest_comparison
]


x = np.arange(
    len(markets)
)

bar_width = 0.36


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10, 6)
)

baseline_bars = ax.bar(
    x - bar_width / 2,
    baseline_values,
    bar_width,
    label=baseline_period
)

comparison_bars = ax.bar(
    x + bar_width / 2,
    comparison_values,
    bar_width,
    label=comparison_period
)


# ---------------------------------------------------------------
# Format chart
# ---------------------------------------------------------------

ax.set_title(
    "Net Merchandise Revenue: "
    "United Kingdom vs Rest of World"
)

ax.set_xlabel(
    "Market"
)

ax.set_ylabel(
    "Net Merchandise Revenue"
)

ax.set_xticks(x)

ax.set_xticklabels(
    markets
)

ax.legend()


# ---------------------------------------------------------------
# Add labels
# ---------------------------------------------------------------

for bars in [
    baseline_bars,
    comparison_bars
]:

    for bar in bars:

        value = bar.get_height()

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            value,

            f"{value / 1_000_000:.2f}m",

            ha="center",
            va="bottom",
            fontsize=9
        )


plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 5.14 — TOP-FIVE COUNTRY REVENUE CONCENTRATION
# ================================================================

baseline_country_ranking = (
    country_revenue_change
    .sort_values(
        baseline_period,
        ascending=False
    )
)

comparison_country_ranking = (
    country_revenue_change
    .sort_values(
        comparison_period,
        ascending=False
    )
)


baseline_top5_share = (
    baseline_country_ranking[
        baseline_period
    ]
    .head(5)
    .sum()
    / total_baseline_revenue
    * 100
)

comparison_top5_share = (
    comparison_country_ranking[
        comparison_period
    ]
    .head(5)
    .sum()
    / total_comparison_revenue
    * 100
)


print(
    f"Top 5 country share "
    f"in {baseline_period}: "
    f"{baseline_top5_share:.2f}%"
)

print(
    f"Top 5 country share "
    f"in {comparison_period}: "
    f"{comparison_top5_share:.2f}%"
)

In [ ]:
# ================================================================
# 5.15 — COUNT EXPANDING AND DECLINING MARKETS
# ================================================================

country_direction_summary = pd.DataFrame(
    {
        "Category": [
            "Countries with higher revenue",
            "Countries with lower revenue",
            "Countries with unchanged revenue",
        ],

        "Count": [
            int(
                (
                    country_revenue_change[
                        "absolute_change"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    country_revenue_change[
                        "absolute_change"
                    ] < 0
                ).sum()
            ),

            int(
                (
                    country_revenue_change[
                        "absolute_change"
                    ] == 0
                ).sum()
            ),
        ],
    }
)

display(country_direction_summary)

In [ ]:
# ================================================================
# 5.16 — IDENTIFY NEWLY ACTIVE COUNTRY MARKETS
# ================================================================

newly_active_countries = (
    country_revenue_change
    .loc[
        (
            country_revenue_change[
                baseline_period
            ] == 0
        )
        &
        (
            country_revenue_change[
                comparison_period
            ] > 0
        )
    ]
    .sort_values(
        comparison_period,
        ascending=False
    )
)

display(newly_active_countries)

In [ ]:
# ================================================================
# 5.17 — IDENTIFY COUNTRIES WITH NO COMPARISON-PERIOD REVENUE
# ================================================================

inactive_countries = (
    country_revenue_change
    .loc[
        (
            country_revenue_change[
                baseline_period
            ] > 0
        )
        &
        (
            country_revenue_change[
                comparison_period
            ] == 0
        )
    ]
    .sort_values(
        baseline_period,
        ascending=False
    )
)

display(inactive_countries)

In [ ]:
# ================================================================
# 5.18 — DEFINE MEANINGFUL BASELINE FOR % GROWTH INTERPRETATION
# ================================================================

minimum_baseline_revenue = 1_000

country_revenue_change[
    "meaningful_baseline"
] = (
    country_revenue_change[
        baseline_period
    ]
    >= minimum_baseline_revenue
)

meaningful_country_growth = (
    country_revenue_change
    .loc[
        country_revenue_change[
            "meaningful_baseline"
        ]
    ]
    .sort_values(
        "percentage_change",
        ascending=False
    )
)

display(
    meaningful_country_growth.head(10)
)

In [ ]:
# ================================================================
# 5.19 — VISUAL: LARGEST COUNTRY REVENUE MOVEMENTS
# ================================================================

import matplotlib.pyplot as plt


country_plot_data = (
    country_revenue_change
    .sort_values(
        "absolute_change",
        ascending=False
    )
)

# Take the strongest positive and negative movements.
top_positive = (
    country_plot_data.head(8)
)

top_negative = (
    country_plot_data.tail(8)
)

country_plot = (
    pd.concat(
        [
            top_positive,
            top_negative
        ]
    )
    .drop_duplicates(
        subset="region"
    )
    .sort_values(
        "absolute_change"
    )
)


fig, ax = plt.subplots(
    figsize=(10, 8)
)

ax.barh(
    country_plot["region"],
    country_plot["absolute_change"]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_title(
    "Largest Country Contributions to "
    "Net Merchandise Revenue Change"
)

ax.set_xlabel(
    "Change in Net Merchandise Revenue"
)

ax.set_ylabel(
    "Country"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 5.20 — COUNTRY DRIVER SUMMARY TABLE
# ================================================================

country_driver_summary = (
    country_revenue_change
    [
        [
            "region",
            baseline_period,
            comparison_period,
            "absolute_change",
            "percentage_change",
            "contribution_to_total_change_pct",
        ]
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
)

country_driver_summary = (
    country_driver_summary.rename(
        columns={
            "region": "Country",
            baseline_period: f"{baseline_period} Revenue",
            comparison_period: f"{comparison_period} Revenue",
            "absolute_change": "Absolute Change",
            "percentage_change": "Change (%)",
            "contribution_to_total_change_pct":
                "Contribution to Total Change (%)",
        }
    )
)

display(
    country_driver_summary.style.format(
        {
            f"{baseline_period} Revenue": "{:,.2f}",
            f"{comparison_period} Revenue": "{:,.2f}",
            "Absolute Change": "{:,.2f}",
            "Change (%)": "{:,.2f}%",
            "Contribution to Total Change (%)": "{:,.2f}%",
        }
    )
)

### Step 5 Summary — Country Revenue Drivers

Step 5 examined how the change in **net merchandise revenue** between November 2010 and November 2011 was distributed across geographic markets. The purpose was to determine whether the improvement identified in Step 4 was broadly shared across countries or concentrated in a small number of markets.

The analysis shows that revenue performance was **geographically uneven**. Some countries generated meaningful increases in merchandise revenue, while others declined and therefore offset part of the growth achieved elsewhere. This means that the positive overall movement identified in Step 4 should not be interpreted as evidence that every geographic market improved.

The **United Kingdom remained the dominant market** and was the largest contributor to the change in merchandise revenue. Because of its substantially larger revenue base, relatively modest percentage movements in the UK translated into a much greater absolute impact on total business performance than much larger percentage changes in smaller international markets.

This distinction between **absolute revenue contribution and percentage growth** is important. Several smaller countries recorded high percentage growth rates, but some of those movements came from relatively small baseline revenue levels. For this reason, country performance was ranked primarily by absolute revenue change rather than percentage growth alone.

Outside the United Kingdom, performance was mixed. A number of international markets contributed positively to the comparison-period result, demonstrating that growth was not entirely confined to the domestic market. At the same time, several countries experienced material revenue declines, partially offsetting the gains generated by the strongest-performing markets.

The comparison between the **United Kingdom and the rest of the world** provides an important measure of geographic diversification. The analysis indicates that overall performance remained heavily influenced by the UK, while international markets collectively displayed a more uneven pattern of growth and decline. This concentration means that total company performance remains particularly sensitive to changes in the UK market.

Revenue concentration was also assessed using the share of total merchandise revenue generated by the United Kingdom and by the five largest country markets. These measures confirm that a relatively small group of countries accounts for a substantial proportion of total merchandise revenue. Geographic concentration should therefore be considered alongside headline growth when evaluating the resilience of the business.

The analysis also distinguished between countries that were active in both periods, **newly active markets** that generated revenue in November 2011 but not November 2010, and markets that generated baseline revenue but recorded no revenue in the comparison period. This prevents newly active countries from being misleadingly described using very large percentage growth rates and provides a clearer view of market entry, continuity and inactivity.

Overall, Step 5 indicates that the improvement in merchandise revenue was **not evenly distributed geographically**. The business benefited from strong contributions from its leading markets, particularly the United Kingdom, but these gains were partially offset by weaker performance elsewhere. The geographic picture therefore combines positive growth with continuing concentration risk and uneven international performance.

The next stage should determine whether these country-level movements were driven by changes in the **customer base**, including new customers, retained customers, reactivated customers, contraction and churn.


## Step 6 — Customer Revenue Drivers

This section examines how customer-level activity contributed to the change in net merchandise revenue between November 2010 and November 2011.

The analysis separates revenue movement into customers acquired after the baseline period, customers newly acquired during the comparison month, reactivated customers, growth and contraction among customers active in both periods, and customers who were inactive in the comparison month.

Customer analysis is restricted to transactions with an identified customer ID. Revenue associated with unidentified customers is measured separately and reconciled back to the total merchandise revenue movement.

The objective is to determine whether revenue growth was being generated through sustainable expansion of the existing customer base or whether new acquisition and reactivation were largely replacing revenue lost through contraction and inactivity.

In [ ]:
# ================================================================
# 6.2 — DEFINE POSITIVE MERCHANDISE PURCHASE HISTORY
# ================================================================

positive_merchandise_sales = (
    df.loc[
        df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
    ]
    .copy()
)

print(
    f"Positive merchandise transaction rows: "
    f"{len(positive_merchandise_sales):,}"
)

In [ ]:
# ================================================================
# 6.3 — FIRST POSITIVE MERCHANDISE PURCHASE DATE
# ================================================================

customer_first_positive_purchase = (
    positive_merchandise_sales
    .groupby(
        "customer",
        dropna=True
    )["date"]
    .min()
    .rename(
        "first_positive_purchase_date"
    )
)

display(
    customer_first_positive_purchase
    .head()
)


In [ ]:
# ================================================================
# 6.4 — IDENTIFIED CUSTOMER COMPARISON DATA
# ================================================================

identified_customer_df = (
    comparison_df
    .loc[
        comparison_df[
            "customer_is_identified"
        ]
    ]
    .copy()
)

print(
    f"Identified customer rows: "
    f"{len(identified_customer_df):,}"
)

print(
    f"Unique identified customers: "
    f"{identified_customer_df['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 6.5 — CUSTOMER REVENUE BY PERIOD
# ================================================================

customer_period_revenue = (
    identified_customer_df
    .groupby(
        [
            "customer",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)

display(
    customer_period_revenue.head()
)

In [ ]:
# ================================================================
# 6.6 — PIVOT CUSTOMER REVENUE
# ================================================================

customer_revenue_bridge = (
    customer_period_revenue
    .pivot(
        index="customer",
        columns="period_month",
        values="net_merchandise_revenue"
    )
    .fillna(0)
)


for period in [
    baseline_period,
    comparison_period
]:
    if period not in customer_revenue_bridge.columns:
        customer_revenue_bridge[
            period
        ] = 0.0


customer_revenue_bridge = (
    customer_revenue_bridge
    .rename(
        columns={
            baseline_period:
                "baseline_revenue",

            comparison_period:
                "comparison_revenue",
        }
    )
    .reset_index()
)

In [ ]:
# ================================================================
# 6.7 — CALCULATE CUSTOMER REVENUE CHANGE
# ================================================================

customer_revenue_bridge[
    "revenue_change"
] = (
    customer_revenue_bridge[
        "comparison_revenue"
    ]
    - customer_revenue_bridge[
        "baseline_revenue"
    ]
)


customer_revenue_bridge[
    "first_positive_purchase_date"
] = (
    customer_revenue_bridge[
        "customer"
    ]
    .map(
        customer_first_positive_purchase
    )
)

display(
    customer_revenue_bridge.head()
)

In [ ]:
# ================================================================
# 6.8 — DEFINE PERIOD BOUNDARIES
# ================================================================

baseline_start = (
    pd.Period(
        baseline_period,
        freq="M"
    )
    .start_time
)

baseline_end = (
    pd.Period(
        baseline_period,
        freq="M"
    )
    .end_time
)

comparison_start = (
    pd.Period(
        comparison_period,
        freq="M"
    )
    .start_time
)

comparison_end = (
    pd.Period(
        comparison_period,
        freq="M"
    )
    .end_time
)


print(
    f"Baseline:   "
    f"{baseline_start.date()} "
    f"to {baseline_end.date()}"
)

print(
    f"Comparison: "
    f"{comparison_start.date()} "
    f"to {comparison_end.date()}"
)

In [ ]:
# ================================================================
# 6.9 — CLASSIFY CUSTOMER REVENUE MOVEMENT
# ================================================================

def classify_customer_movement(row):
    """
    Classify each identified customer according to their
    merchandise-revenue movement between the baseline and
    comparison periods.

    Important:
    'Inactive in comparison month' does not mean permanent churn.
    It means the customer generated positive baseline revenue
    but no net merchandise revenue in the comparison month.
    """

    baseline = row["baseline_revenue"]
    comparison = row["comparison_revenue"]
    first_purchase = row[
        "first_positive_purchase_date"
    ]

    # ------------------------------------------------------------
    # Customers active in both periods
    # ------------------------------------------------------------

    if baseline > 0 and comparison > 0:

        if comparison > baseline:
            return "Existing customer growth"

        elif comparison < baseline:
            return "Existing customer contraction"

        else:
            return "Stable existing customers"


    # ------------------------------------------------------------
    # Positive baseline revenue but zero comparison revenue
    # ------------------------------------------------------------

    if baseline > 0 and comparison == 0:
        return "Inactive in comparison month"


    # ------------------------------------------------------------
    # Positive baseline revenue but negative comparison revenue
    # ------------------------------------------------------------

    if baseline > 0 and comparison < 0:
        return "Returns-dominant/inactive movement"


    # ------------------------------------------------------------
    # No positive baseline revenue but positive comparison revenue
    # ------------------------------------------------------------

    if baseline <= 0 and comparison > 0:

        if pd.isna(first_purchase):
            return "Other/unusual movement"

        # First purchase occurred during November 2011.
        if (
            comparison_start
            <= first_purchase
            <= comparison_end
        ):
            return "New in comparison month"

        # Customer was acquired after November 2010 but before
        # November 2011.
        if (
            baseline_end
            < first_purchase
            < comparison_start
        ):
            return "Acquired since baseline"

        # Customer existed before the baseline month but generated
        # no positive revenue during the baseline comparison month.
        if first_purchase < baseline_start:
            return "Reactivated customers"

        return "Other/unusual movement"


    # ------------------------------------------------------------
    # No revenue movement
    # ------------------------------------------------------------

    if baseline == 0 and comparison == 0:
        return "No movement"


    # ------------------------------------------------------------
    # Remaining cases are largely return/credit-driven
    # ------------------------------------------------------------

    return "Returns/credit-only movement"


customer_revenue_bridge[
    "customer_segment"
] = (
    customer_revenue_bridge
    .apply(
        classify_customer_movement,
        axis=1
    )
)

In [ ]:
# ================================================================
# 6.10 — CUSTOMER SEGMENT COUNTS
# ================================================================

customer_segment_counts = (
    customer_revenue_bridge
    .groupby(
        "customer_segment",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        )
    )
    .sort_values(
        "customers",
        ascending=False
    )
)

display(
    customer_segment_counts
)

In [ ]:
# ================================================================
# 6.11 — IDENTIFIED CUSTOMER REVENUE BRIDGE
# ================================================================

customer_bridge_summary = (
    customer_revenue_bridge
    .groupby(
        "customer_segment",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),
        revenue_change=(
            "revenue_change",
            "sum"
        ),
    )
    .sort_values(
        "revenue_change",
        ascending=False
    )
)

display(
    customer_bridge_summary
)

In [ ]:
# ================================================================
# 6.12 — UNIDENTIFIED CUSTOMER REVENUE MOVEMENT
# ================================================================

unidentified_customer_revenue = (
    comparison_df
    .loc[
        ~comparison_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        "period_month"
    )[
        "net_merchandise_revenue"
    ]
    .sum()
)


unidentified_baseline = (
    unidentified_customer_revenue
    .get(
        baseline_period,
        0.0
    )
)

unidentified_comparison = (
    unidentified_customer_revenue
    .get(
        comparison_period,
        0.0
    )
)

unidentified_change = (
    unidentified_comparison
    - unidentified_baseline
)


print(
    f"Unidentified customer revenue "
    f"in {baseline_period}: "
    f"{unidentified_baseline:,.2f}"
)

print(
    f"Unidentified customer revenue "
    f"in {comparison_period}: "
    f"{unidentified_comparison:,.2f}"
)

print(
    f"Change in unidentified customer revenue: "
    f"{unidentified_change:,.2f}"
)

In [ ]:
# ================================================================
# 6.13 — RECONCILE CUSTOMER BRIDGE
# ================================================================

identified_customer_change = (
    customer_bridge_summary[
        "revenue_change"
    ]
    .sum()
)


reconciled_customer_change = (
    identified_customer_change
    + unidentified_change
)


print(
    f"Identified customer change: "
    f"{identified_customer_change:,.2f}"
)

print(
    f"Unidentified customer change: "
    f"{unidentified_change:,.2f}"
)

print(
    f"Reconciled customer change: "
    f"{reconciled_customer_change:,.2f}"
)

print(
    f"Step 4 merchandise change: "
    f"{merchandise_change:,.2f}"
)

In [ ]:
# ================================================================
# 6.14 — VALIDATE CUSTOMER RECONCILIATION
# ================================================================

assert np.isclose(
    reconciled_customer_change,
    merchandise_change,
    atol=0.01
), (
    "Customer bridge does not reconcile "
    "to the Step 4 merchandise revenue change."
)

print(
    "✓ Customer revenue bridge reconciles "
    "to total merchandise revenue change."
)

In [ ]:
# ================================================================
# 6.15 — COMPLETE CUSTOMER REVENUE BRIDGE
# ================================================================

complete_customer_bridge = pd.concat(
    [
        customer_bridge_summary,

        pd.DataFrame(
            {
                "customer_segment": [
                    "Unidentified customers"
                ],
                "customers": [
                    np.nan
                ],
                "revenue_change": [
                    unidentified_change
                ],
            }
        ),
    ],
    ignore_index=True
)


complete_customer_bridge = (
    complete_customer_bridge
    .sort_values(
        "revenue_change",
        ascending=False
    )
)

display(
    complete_customer_bridge
)

In [ ]:
# ================================================================
# 6.16 — CONTRIBUTION TO TOTAL MERCHANDISE CHANGE
# ================================================================

complete_customer_bridge[
    "contribution_to_total_change_pct"
] = np.where(
    merchandise_change != 0,

    complete_customer_bridge[
        "revenue_change"
    ]
    / merchandise_change
    * 100,

    np.nan
)

display(
    complete_customer_bridge
)

In [ ]:
# ================================================================
# 6.17 — TOP CUSTOMER REVENUE GAINS
# ================================================================

top_customer_gains = (
    customer_revenue_bridge
    .loc[
        customer_revenue_bridge[
            "revenue_change"
        ] > 0
    ]
    .sort_values(
        "revenue_change",
        ascending=False
    )
    .head(15)
)

display(
    top_customer_gains[
        [
            "customer",
            "baseline_revenue",
            "comparison_revenue",
            "revenue_change",
            "customer_segment",
        ]
    ]
)

In [ ]:
# ================================================================
# 6.18 — TOP CUSTOMER REVENUE DECLINES
# ================================================================

top_customer_declines = (
    customer_revenue_bridge
    .loc[
        customer_revenue_bridge[
            "revenue_change"
        ] < 0
    ]
    .sort_values(
        "revenue_change",
        ascending=True
    )
    .head(15)
)

display(
    top_customer_declines[
        [
            "customer",
            "baseline_revenue",
            "comparison_revenue",
            "revenue_change",
            "customer_segment",
        ]
    ]
)

In [ ]:
# ================================================================
# 6.19 — ACTIVE CUSTOMER COUNTS BY PERIOD
# ================================================================

active_customer_counts = (
    identified_customer_df
    .loc[
        identified_customer_df[
            "net_merchandise_revenue"
        ] > 0
    ]
    .groupby(
        "period_month"
    )[
        "customer"
    ]
    .nunique()
)


baseline_active_customers = (
    active_customer_counts
    .get(
        baseline_period,
        0
    )
)

comparison_active_customers = (
    active_customer_counts
    .get(
        comparison_period,
        0
    )
)


active_customer_change = (
    comparison_active_customers
    - baseline_active_customers
)


print(
    f"Active identified customers "
    f"in {baseline_period}: "
    f"{baseline_active_customers:,}"
)

print(
    f"Active identified customers "
    f"in {comparison_period}: "
    f"{comparison_active_customers:,}"
)

print(
    f"Change in active customers: "
    f"{active_customer_change:+,}"
)

In [ ]:
# ================================================================
# VISUAL 6.2 — ACTIVE IDENTIFIED CUSTOMERS
# ================================================================

# ---------------------------------------------------------------
# Prepare data
# ---------------------------------------------------------------

periods = [
    baseline_period,
    comparison_period
]

active_customers = [
    baseline_active_customers,
    comparison_active_customers
]


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 6)
)

bars = ax.bar(
    periods,
    active_customers
)


# ---------------------------------------------------------------
# Format chart
# ---------------------------------------------------------------

ax.set_title(
    "Active Identified Customers"
)

ax.set_xlabel(
    "Period"
)

ax.set_ylabel(
    "Number of Active Customers"
)


# ---------------------------------------------------------------
# Add labels
# ---------------------------------------------------------------

for bar, value in zip(
    bars,
    active_customers
):

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height(),

        f"{value:,}",

        ha="center",
        va="bottom"
    )


plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 6.20 — REVENUE PER ACTIVE IDENTIFIED CUSTOMER
# ================================================================

identified_period_revenue = (
    identified_customer_df
    .groupby(
        "period_month"
    )[
        "net_merchandise_revenue"
    ]
    .sum()
)


baseline_revenue_per_customer = (
    identified_period_revenue.get(
        baseline_period,
        0
    )
    / baseline_active_customers
    if baseline_active_customers > 0
    else np.nan
)


comparison_revenue_per_customer = (
    identified_period_revenue.get(
        comparison_period,
        0
    )
    / comparison_active_customers
    if comparison_active_customers > 0
    else np.nan
)


revenue_per_customer_change_pct = (
    (
        comparison_revenue_per_customer
        / baseline_revenue_per_customer
        - 1
    )
    * 100
    if baseline_revenue_per_customer != 0
    else np.nan
)


print(
    f"Revenue per active customer "
    f"in {baseline_period}: "
    f"{baseline_revenue_per_customer:,.2f}"
)

print(
    f"Revenue per active customer "
    f"in {comparison_period}: "
    f"{comparison_revenue_per_customer:,.2f}"
)

print(
    f"Change: "
    f"{revenue_per_customer_change_pct:.2f}%"
)

In [ ]:
# ================================================================
# 6.21 — CUSTOMER REVENUE CONCENTRATION
# ================================================================

def calculate_top_customer_share(
    dataframe,
    revenue_column,
    top_n
):
    """
    Calculate the share of identified-customer revenue
    generated by the top N customers.
    """

    customer_values = (
        dataframe
        .groupby(
            "customer"
        )[revenue_column]
        .sum()
        .sort_values(
            ascending=False
        )
    )

    total_revenue = (
        customer_values.sum()
    )

    if total_revenue == 0:
        return np.nan

    return (
        customer_values.head(top_n).sum()
        / total_revenue
        * 100
    )


customer_concentration_rows = []

for period in [
    baseline_period,
    comparison_period
]:

    period_data = (
        identified_customer_df
        .loc[
            identified_customer_df[
                "period_month"
            ] == period
        ]
    )

    customer_concentration_rows.append(
        {
            "period": period,

            "top_1_customer_share_pct":
                calculate_top_customer_share(
                    period_data,
                    "net_merchandise_revenue",
                    1
                ),

            "top_5_customer_share_pct":
                calculate_top_customer_share(
                    period_data,
                    "net_merchandise_revenue",
                    5
                ),

            "top_10_customer_share_pct":
                calculate_top_customer_share(
                    period_data,
                    "net_merchandise_revenue",
                    10
                ),
        }
    )


customer_concentration = pd.DataFrame(
    customer_concentration_rows
)

display(
    customer_concentration
)

In [ ]:
# ================================================================
# 6.22 — VISUAL: CUSTOMER REVENUE MOVEMENT
# ================================================================

import matplotlib.pyplot as plt


customer_bridge_plot = (
    complete_customer_bridge
    .sort_values(
        "revenue_change"
    )
)


fig, ax = plt.subplots(
    figsize=(11, 7)
)

ax.barh(
    customer_bridge_plot[
        "customer_segment"
    ],
    customer_bridge_plot[
        "revenue_change"
    ]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_title(
    "Customer Drivers of "
    "Net Merchandise Revenue Change"
)

ax.set_xlabel(
    "Change in Net Merchandise Revenue"
)

ax.set_ylabel(
    "Customer Segment"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 6.23 — CUSTOMER DRIVER SUMMARY
# ================================================================

customer_driver_summary = (
    complete_customer_bridge
    .rename(
        columns={
            "customer_segment":
                "Customer Segment",

            "customers":
                "Customers",

            "revenue_change":
                "Revenue Change",

            "contribution_to_total_change_pct":
                "Contribution to Total Change (%)",
        }
    )
)


display(
    customer_driver_summary.style.format(
        {
            "Customers": "{:,.0f}",
            "Revenue Change": "{:,.2f}",
            "Contribution to Total Change (%)": "{:,.2f}%",
        },
        na_rep="—"
    )
)

### Step 6 Summary — Customer Revenue Drivers

Step 6 examined how customer-level behaviour contributed to the change in net merchandise revenue between November 2010 and November 2011. The analysis separated customers into acquisition, reactivation, growth, contraction and inactivity categories and treated unidentified-customer revenue separately so that the customer bridge could be fully reconciled to the overall merchandise revenue movement established in Step 4.

The results show that the customer story was considerably more mixed than the positive overall revenue growth initially suggests.

Among identified customers, several strong positive revenue drivers were observed. **Reactivated customers generated approximately 229.41k of additional revenue**, making them the largest positive customer segment. A further **201.68k came from customers acquired after the November 2010 baseline period but before November 2011**, while **existing customers who increased their spending contributed approximately 138.92k**. Customers whose first positive purchase occurred during November 2011 contributed an additional **79.89k**.

Together, these positive customer movements generated substantial gross revenue gains. However, they were largely offset by significant losses elsewhere in the existing customer base.

The largest negative movement came from customers who were active in November 2010 but generated no net merchandise revenue in November 2011. This group comprised **899 identified customers** and represented approximately **411.04k of lost revenue**. These customers are described as **inactive in the comparison month**, rather than permanently churned, because the two-month comparison alone is not sufficient to establish that they permanently left the business.

A further **368 existing customers reduced their spending**, producing approximately **241.22k of revenue contraction**. An additional **15 customers showed returns-dominant or inactive revenue behaviour**, contributing a further decline of approximately **10.46k**.

As a result, the identified customer base as a whole produced a **net decline of approximately 8.24k** between the two periods. This is an important finding: the increase in merchandise revenue was not generated by net growth in identified-customer revenue alone. Strong acquisition, reactivation and customer expansion were effectively offset by inactivity and contraction among other existing customers.

Revenue associated with customers whose identities were unavailable moved in the opposite direction. Unidentified-customer merchandise revenue increased from approximately **273.63k in November 2010 to 314.04k in November 2011**, an increase of approximately **40.41k**. When this increase is combined with the **8.24k decline among identified customers**, the customer bridge reconciles exactly to the overall **32.17k increase in net merchandise revenue** established in Step 4.

The analysis therefore reveals an important data-quality and interpretation issue. Although total merchandise revenue increased, a material portion of that improvement came from transactions that cannot be attributed to a known customer. This limits the extent to which the overall growth can be confidently explained through identified customer acquisition, retention or expansion behaviour.

The number of active identified customers increased from approximately **1,606 in November 2010 to 1,660 in November 2011**, an increase of **54 customers**. However, identified net merchandise revenue declined slightly despite the larger active customer base. Revenue per active identified customer consequently fell from approximately **701.70 to 673.91**, a decline of about **3.96%**. This suggests that the business served more active identified customers in the comparison period, but the average revenue generated per active customer was lower.

Customer revenue concentration remained relatively stable. The largest single identified customer represented approximately **2.73% of identified-customer merchandise revenue in November 2010 and 2.49% in November 2011**. The top five customers accounted for approximately **9.90% and 10.22%**, respectively, while the top ten accounted for approximately **14.97% and 15.27%**. The business therefore does not appear excessively dependent on a single customer, although concentration among the largest accounts increased slightly at the top-five and top-ten levels.

At the individual-customer level, the largest positive revenue movement was generated by customer **17450**, which contributed approximately **27.84k** of additional revenue and was classified as reactivated. Customer **14096** contributed approximately **22.62k** and was acquired after the baseline period. On the negative side, customer **15838** recorded the largest decline, with revenue falling by approximately **17.86k**, followed by customer **16684**, whose revenue declined by approximately **10.80k**.

Overall, Step 6 indicates that the business experienced significant customer turnover beneath the modest headline revenue growth. Acquisition, reactivation and customer expansion were strong, but these gains were substantially absorbed by inactivity and contraction among previously active customers. The identified customer base itself contracted slightly in revenue terms, while unidentified-customer revenue provided the net uplift required to produce the overall merchandise growth.

This suggests that the key customer-management challenge is not simply acquiring more customers. The larger issue is the ability to **retain existing customers, prevent spending contraction and convert newly acquired or reactivated customers into sustained repeat revenue**. The next stage of the analysis should therefore examine product-level revenue drivers to determine which merchandise categories and SKUs were responsible for the gains and losses observed across the customer base.


## Step 7 — Product Revenue Drivers

This section examines how individual merchandise products contributed to the change in net merchandise revenue between November 2010 and November 2011.

The analysis identifies the products responsible for the largest revenue gains and losses, distinguishes between continuing, newly active and inactive products, and evaluates whether revenue movements among continuing products were primarily associated with changes in sales volume or average selling price.

Products are identified primarily by SKU rather than description because product descriptions can change over time. The canonical product description created during data cleaning is used for reporting and interpretation.

The objective is to determine whether overall merchandise growth was broad-based across the product portfolio or concentrated in a relatively small number of products.

In [ ]:
# ================================================================
# 7.2 — CONFIRM PRODUCT ANALYSIS SCOPE
# ================================================================

print(f"Baseline period:   {baseline_period}")
print(f"Comparison period: {comparison_period}")

print(
    f"\nProduct-analysis rows: "
    f"{len(product_comparison_df):,}"
)

print(
    f"Unique merchandise SKUs: "
    f"{product_comparison_df['product'].nunique():,}"
)

print(
    f"Net merchandise revenue in scope: "
    f"{product_comparison_df['net_merchandise_revenue'].sum():,.2f}"
)


In [ ]:
# ================================================================
# 7.3 — CALCULATE PRODUCT REVENUE BY PERIOD
# ================================================================

product_period_revenue = (
    product_comparison_df
    .groupby(
        [
            "product",
            "product_description",
            "period_month",
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        net_quantity=(
            "quantity",
            "sum"
        ),
    )
)

display(
    product_period_revenue.head()
)

In [ ]:
# ================================================================
# 7.4 — CREATE SKU DESCRIPTION LOOKUP
# ================================================================

product_description_lookup = (
    product_comparison_df
    [
        [
            "product",
            "product_description",
        ]
    ]
    .drop_duplicates(
        subset="product"
    )
    .set_index(
        "product"
    )["product_description"]
)

print(
    f"Canonical descriptions available for "
    f"{len(product_description_lookup):,} SKUs."
)

In [ ]:
# ================================================================
# 7.5 — PIVOT PRODUCT REVENUE
# ================================================================

product_revenue_pivot = (
    product_period_revenue
    .groupby(
        [
            "product",
            "period_month"
        ],
        as_index=False
    )[
        "net_merchandise_revenue"
    ]
    .sum()
    .pivot(
        index="product",
        columns="period_month",
        values="net_merchandise_revenue"
    )
    .fillna(0)
)


for period in [
    baseline_period,
    comparison_period
]:
    if period not in product_revenue_pivot.columns:
        product_revenue_pivot[
            period
        ] = 0.0


product_revenue_change = (
    product_revenue_pivot
    .reset_index()
)

product_revenue_change[
    "product_description"
] = (
    product_revenue_change[
        "product"
    ]
    .map(
        product_description_lookup
    )
)

In [ ]:
# ================================================================
# 7.6 — CALCULATE PRODUCT REVENUE CHANGE
# ================================================================

product_revenue_change[
    "absolute_change"
] = (
    product_revenue_change[
        comparison_period
    ]
    - product_revenue_change[
        baseline_period
    ]
)


product_revenue_change[
    "percentage_change"
] = np.where(
    product_revenue_change[
        baseline_period
    ] != 0,

    (
        product_revenue_change[
            "absolute_change"
        ]
        / product_revenue_change[
            baseline_period
        ]
        * 100
    ),

    np.nan
)


display(
    product_revenue_change
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .head(10)
)


In [ ]:
# ================================================================
# 7.7 — RECONCILE PRODUCT REVENUE CHANGE
# ================================================================

total_product_change = (
    product_revenue_change[
        "absolute_change"
    ]
    .sum()
)


print(
    f"Product-level revenue change: "
    f"{total_product_change:,.2f}"
)

print(
    f"Step 4 merchandise revenue change: "
    f"{merchandise_change:,.2f}"
)

In [ ]:
# ================================================================
# 7.8 — VALIDATE PRODUCT RECONCILIATION
# ================================================================

assert np.isclose(
    total_product_change,
    merchandise_change,
    atol=0.01
), (
    "Product revenue movements do not reconcile "
    "to the Step 4 merchandise revenue change."
)

print(
    "✓ Product revenue movements reconcile "
    "to total merchandise revenue change."
)

In [ ]:
# ================================================================
# 7.9 — CLASSIFY PRODUCT ACTIVITY STATUS
# ================================================================

conditions = [
    (
        product_revenue_change[
            baseline_period
        ] > 0
    )
    &
    (
        product_revenue_change[
            comparison_period
        ] > 0
    ),

    (
        product_revenue_change[
            baseline_period
        ] <= 0
    )
    &
    (
        product_revenue_change[
            comparison_period
        ] > 0
    ),

    (
        product_revenue_change[
            baseline_period
        ] > 0
    )
    &
    (
        product_revenue_change[
            comparison_period
        ] <= 0
    ),
]


choices = [
    "Continuing product",
    "Newly active product",
    "Inactive in comparison month",
]


product_revenue_change[
    "product_status"
] = np.select(
    conditions,
    choices,
    default="Other / unusual movement"
)

In [ ]:
# ================================================================
# 7.10 — PRODUCT STATUS REVENUE BRIDGE
# ================================================================

product_status_summary = (
    product_revenue_change
    .groupby(
        "product_status",
        as_index=False
    )
    .agg(
        products=(
            "product",
            "nunique"
        ),
        baseline_revenue=(
            baseline_period,
            "sum"
        ),
        comparison_revenue=(
            comparison_period,
            "sum"
        ),
        revenue_change=(
            "absolute_change",
            "sum"
        ),
    )
    .sort_values(
        "revenue_change",
        ascending=False
    )
)

display(
    product_status_summary
)

In [ ]:
# ================================================================
# 7.11 — CONTRIBUTION TO TOTAL MERCHANDISE CHANGE
# ================================================================

product_revenue_change[
    "contribution_to_total_change_pct"
] = np.where(
    merchandise_change != 0,

    product_revenue_change[
        "absolute_change"
    ]
    / merchandise_change
    * 100,

    np.nan
)

In [ ]:
# ================================================================
# 7.12 — TOP PRODUCT REVENUE GAINS
# ================================================================

top_product_gains = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "absolute_change"
        ] > 0
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .head(15)
)


display(
    top_product_gains[
        [
            "product",
            "product_description",
            "product_status",
            baseline_period,
            comparison_period,
            "absolute_change",
            "percentage_change",
            "contribution_to_total_change_pct",
        ]
    ]
)


In [ ]:
# ================================================================
# 7.13 — TOP PRODUCT REVENUE DECLINES
# ================================================================

top_product_declines = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "absolute_change"
        ] < 0
    ]
    .sort_values(
        "absolute_change",
        ascending=True
    )
    .head(15)
)


display(
    top_product_declines[
        [
            "product",
            "product_description",
            "product_status",
            baseline_period,
            comparison_period,
            "absolute_change",
            "percentage_change",
            "contribution_to_total_change_pct",
        ]
    ]
)

In [ ]:
# ================================================================
# 7.14 — NEWLY ACTIVE PRODUCTS
# ================================================================

newly_active_products = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "product_status"
        ]
        == "Newly active product"
    ]
    .sort_values(
        comparison_period,
        ascending=False
    )
)


print(
    f"Newly active products: "
    f"{len(newly_active_products):,}"
)

print(
    f"Revenue generated by newly active products: "
    f"{newly_active_products[comparison_period].sum():,.2f}"
)


display(
    newly_active_products.head(15)
)

In [ ]:
# ================================================================
# 7.15 — PRODUCTS INACTIVE IN COMPARISON MONTH
# ================================================================

inactive_products = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "product_status"
        ]
        == "Inactive in comparison month"
    ]
    .sort_values(
        baseline_period,
        ascending=False
    )
)


print(
    f"Products inactive in comparison month: "
    f"{len(inactive_products):,}"
)

print(
    f"Baseline revenue represented by these products: "
    f"{inactive_products[baseline_period].sum():,.2f}"
)


display(
    inactive_products.head(15)
)

In [ ]:
# ================================================================
# 7.16 — CONTINUING PRODUCT ANALYSIS
# ================================================================

continuing_products = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "product_status"
        ]
        == "Continuing product"
    ]
    .copy()
)


print(
    f"Continuing products: "
    f"{len(continuing_products):,}"
)

print(
    f"Combined revenue change from continuing products: "
    f"{continuing_products['absolute_change'].sum():,.2f}"
)

In [ ]:
# ================================================================
# 7.17 — PRODUCT QUANTITY BY PERIOD
# ================================================================

product_quantity = (
    product_comparison_df
    .groupby(
        [
            "product",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        net_quantity=(
            "quantity",
            "sum"
        )
    )
)


product_quantity_pivot = (
    product_quantity
    .pivot(
        index="product",
        columns="period_month",
        values="net_quantity"
    )
    .fillna(0)
)


for period in [
    baseline_period,
    comparison_period
]:
    if period not in product_quantity_pivot.columns:
        product_quantity_pivot[
            period
        ] = 0.0


product_quantity_pivot = (
    product_quantity_pivot
    .rename(
        columns={
            baseline_period:
                "baseline_quantity",

            comparison_period:
                "comparison_quantity",
        }
    )
    .reset_index()
)

In [ ]:
# ================================================================
# 7.18 — CALCULATE EFFECTIVE REVENUE PER UNIT
# ================================================================

continuing_product_decomposition = (
    continuing_products
    [
        [
            "product",
            "product_description",
            baseline_period,
            comparison_period,
            "absolute_change",
        ]
    ]
    .merge(
        product_quantity_pivot,
        on="product",
        how="left"
    )
)


continuing_product_decomposition[
    "baseline_revenue_per_unit"
] = np.where(
    continuing_product_decomposition[
        "baseline_quantity"
    ] > 0,

    continuing_product_decomposition[
        baseline_period
    ]
    / continuing_product_decomposition[
        "baseline_quantity"
    ],

    np.nan
)


continuing_product_decomposition[
    "comparison_revenue_per_unit"
] = np.where(
    continuing_product_decomposition[
        "comparison_quantity"
    ] > 0,

    continuing_product_decomposition[
        comparison_period
    ]
    / continuing_product_decomposition[
        "comparison_quantity"
    ],

    np.nan
)


In [ ]:
# ================================================================
# 7.19 — VALID PRICE-VOLUME DECOMPOSITION SCOPE
# ================================================================

price_volume_products = (
    continuing_product_decomposition
    .loc[
        continuing_product_decomposition[
            "baseline_quantity"
        ] > 0
    ]
    .loc[
        continuing_product_decomposition[
            "comparison_quantity"
        ] > 0
    ]
    .loc[
        continuing_product_decomposition[
            "baseline_revenue_per_unit"
        ].notna()
    ]
    .loc[
        continuing_product_decomposition[
            "comparison_revenue_per_unit"
        ].notna()
    ]
    .copy()
)


print(
    f"Continuing products suitable for "
    f"price-volume decomposition: "
    f"{len(price_volume_products):,}"
)

In [ ]:
# ================================================================
# 7.20 — PRICE-VOLUME REVENUE DECOMPOSITION
# ================================================================

price_volume_products[
    "volume_effect"
] = (
    (
        price_volume_products[
            "comparison_quantity"
        ]
        - price_volume_products[
            "baseline_quantity"
        ]
    )
    * price_volume_products[
        "baseline_revenue_per_unit"
    ]
)


price_volume_products[
    "price_effect"
] = (
    (
        price_volume_products[
            "comparison_revenue_per_unit"
        ]
        - price_volume_products[
            "baseline_revenue_per_unit"
        ]
    )
    * price_volume_products[
        "comparison_quantity"
    ]
)


price_volume_products[
    "decomposed_change"
] = (
    price_volume_products[
        "volume_effect"
    ]
    + price_volume_products[
        "price_effect"
    ]
)

In [ ]:
# ================================================================
# 7.21 — VALIDATE PRICE-VOLUME RECONCILIATION
# ================================================================

price_volume_products[
    "decomposition_difference"
] = (
    price_volume_products[
        "absolute_change"
    ]
    - price_volume_products[
        "decomposed_change"
    ]
)


max_decomposition_difference = (
    price_volume_products[
        "decomposition_difference"
    ]
    .abs()
    .max()
)


print(
    f"Maximum decomposition difference: "
    f"{max_decomposition_difference:,.6f}"
)

In [ ]:
assert np.allclose(
    price_volume_products[
        "absolute_change"
    ],
    price_volume_products[
        "decomposed_change"
    ],
    atol=0.01
), (
    "Price-volume decomposition does not "
    "reconcile for all valid products."
)

print(
    "✓ Price-volume decomposition reconciles."
)

In [ ]:
# ================================================================
# 7.22 — TOTAL PRICE AND VOLUME EFFECTS
# ================================================================

total_volume_effect = (
    price_volume_products[
        "volume_effect"
    ]
    .sum()
)

total_price_effect = (
    price_volume_products[
        "price_effect"
    ]
    .sum()
)


print(
    f"Total volume effect: "
    f"{total_volume_effect:,.2f}"
)

print(
    f"Total price effect: "
    f"{total_price_effect:,.2f}"
)

print(
    f"Combined effect: "
    f"{total_volume_effect + total_price_effect:,.2f}"
)

In [ ]:
# ================================================================
# VISUAL 7.3 — CONTINUING PRODUCT PRICE/VOLUME EFFECT
# ================================================================
# ---------------------------------------------------------------
# Prepare decomposition data
# ---------------------------------------------------------------

effects = [
    "Volume Effect",
    "Price / Mix Effect"
]

effect_values = [
    total_volume_effect,
    total_price_effect
]


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 6)
)

bars = ax.bar(
    effects,
    effect_values
)

ax.axhline(
    0,
    linewidth=1
)


# ---------------------------------------------------------------
# Format
# ---------------------------------------------------------------

ax.set_title(
    "Continuing Product Revenue Change: "
    "Volume vs Price/Mix Effect"
)

ax.set_xlabel(
    "Revenue Driver"
)

ax.set_ylabel(
    "Revenue Effect"
)


# ---------------------------------------------------------------
# Add labels
# ---------------------------------------------------------------

for bar, value in zip(
    bars,
    effect_values
):

    label_y = (
        value
        if value >= 0
        else value
    )

    vertical_alignment = (
        "bottom"
        if value >= 0
        else "top"
    )

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,

        label_y,

        f"{value:,.2f}",

        ha="center",
        va=vertical_alignment
    )


plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 7.23 — CLASSIFY PRIMARY PRODUCT DRIVER
# ================================================================

price_volume_products[
    "primary_driver"
] = np.where(
    price_volume_products[
        "volume_effect"
    ].abs()
    >=
    price_volume_products[
        "price_effect"
    ].abs(),

    "Volume-led",
    "Price/mix-led"
)


display(
    price_volume_products[
        "primary_driver"
    ]
    .value_counts()
    .rename("products")
    .to_frame()
)

In [ ]:
# ================================================================
# 7.24 — PRODUCT REVENUE CONCENTRATION
# ================================================================

def product_concentration_share(
    period,
    top_n
):

    values = (
        product_revenue_change[
            [
                "product",
                period
            ]
        ]
        .sort_values(
            period,
            ascending=False
        )
    )

    total = values[
        period
    ].sum()

    if total == 0:
        return np.nan

    return (
        values[
            period
        ]
        .head(top_n)
        .sum()
        / total
        * 100
    )


product_concentration = pd.DataFrame(
    {
        "Period": [
            baseline_period,
            comparison_period,
        ],

        "Top 1 Product Share (%)": [
            product_concentration_share(
                baseline_period,
                1
            ),
            product_concentration_share(
                comparison_period,
                1
            ),
        ],

        "Top 5 Product Share (%)": [
            product_concentration_share(
                baseline_period,
                5
            ),
            product_concentration_share(
                comparison_period,
                5
            ),
        ],

        "Top 10 Product Share (%)": [
            product_concentration_share(
                baseline_period,
                10
            ),
            product_concentration_share(
                comparison_period,
                10
            ),
        ],
    }
)


display(
    product_concentration
)

In [ ]:
# ================================================================
# 7.25 — GROWTH CONTRIBUTION CONCENTRATION
# ================================================================

positive_product_changes = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "absolute_change"
        ] > 0
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
)


gross_positive_product_growth = (
    positive_product_changes[
        "absolute_change"
    ]
    .sum()
)


top_10_growth_contribution = (
    positive_product_changes[
        "absolute_change"
    ]
    .head(10)
    .sum()
)


top_10_share_of_positive_growth = (
    top_10_growth_contribution
    / gross_positive_product_growth
    * 100
    if gross_positive_product_growth != 0
    else np.nan
)


print(
    f"Gross positive product growth: "
    f"{gross_positive_product_growth:,.2f}"
)

print(
    f"Top 10 product growth contribution: "
    f"{top_10_growth_contribution:,.2f}"
)

print(
    f"Top 10 share of gross positive growth: "
    f"{top_10_share_of_positive_growth:.2f}%"
)

In [ ]:
# ================================================================
# 7.26 — VISUAL: LARGEST PRODUCT REVENUE MOVEMENTS
# ================================================================

import matplotlib.pyplot as plt


top_positive_products = (
    product_revenue_change
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .head(10)
)


top_negative_products = (
    product_revenue_change
    .sort_values(
        "absolute_change",
        ascending=True
    )
    .head(10)
)


product_plot = (
    pd.concat(
        [
            top_positive_products,
            top_negative_products,
        ]
    )
    .drop_duplicates(
        subset="product"
    )
    .sort_values(
        "absolute_change"
    )
)


product_plot[
    "label"
] = (
    product_plot["product"]
    + " — "
    + product_plot[
        "product_description"
    ].fillna("")
)


fig, ax = plt.subplots(
    figsize=(12, 10)
)

ax.barh(
    product_plot["label"],
    product_plot["absolute_change"]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_title(
    "Largest Product Contributions to "
    "Net Merchandise Revenue Change"
)

ax.set_xlabel(
    "Change in Net Merchandise Revenue"
)

ax.set_ylabel(
    "Product"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 7.27 — VISUAL: PRODUCT PORTFOLIO BRIDGE
# ================================================================

product_status_plot = (
    product_status_summary
    .sort_values(
        "revenue_change"
    )
)


fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.barh(
    product_status_plot[
        "product_status"
    ],
    product_status_plot[
        "revenue_change"
    ]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_title(
    "Revenue Change by Product Activity Status"
)

ax.set_xlabel(
    "Change in Net Merchandise Revenue"
)

ax.set_ylabel(
    "Product Status"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 7.28 — PRODUCT DRIVER SUMMARY TABLE
# ================================================================

product_driver_summary = (
    product_revenue_change
    [
        [
            "product",
            "product_description",
            "product_status",
            baseline_period,
            comparison_period,
            "absolute_change",
            "percentage_change",
            "contribution_to_total_change_pct",
        ]
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .rename(
        columns={
            "product":
                "SKU",

            "product_description":
                "Product",

            "product_status":
                "Product Status",

            baseline_period:
                f"{baseline_period} Revenue",

            comparison_period:
                f"{comparison_period} Revenue",

            "absolute_change":
                "Revenue Change",

            "percentage_change":
                "Change (%)",

            "contribution_to_total_change_pct":
                "Contribution to Total Change (%)",
        }
    )
)


display(
    product_driver_summary.style.format(
        {
            f"{baseline_period} Revenue": "{:,.2f}",
            f"{comparison_period} Revenue": "{:,.2f}",
            "Revenue Change": "{:,.2f}",
            "Change (%)": "{:,.2f}%",
            "Contribution to Total Change (%)": "{:,.2f}%",
        }
    )
)

### Step 7 Summary — Product Revenue Drivers

Step 7 examined how individual merchandise SKUs contributed to the change in net merchandise revenue between November 2010 and November 2011. The analysis focused on merchandise products only, using SKU as the primary product identifier and the canonical product description for reporting.

A total of approximately **3,800 merchandise SKUs** appeared across the two comparison periods. Of these, **1,508 products recorded a positive revenue movement**, **2,178 recorded a negative movement**, and **114 showed no net change**. This immediately indicates that the overall increase in merchandise revenue was not broad-based across the entire product portfolio. More products declined than increased, but the gains from the strongest-performing products were sufficiently large to offset most of the losses.

The product revenue bridge reconciled exactly to the **32.17k increase in net merchandise revenue** established in Step 4.

A major finding is that the positive overall result was driven heavily by products that were not generating positive net revenue in the baseline month. Under the current classification, **751 products were classified as newly active**, generating approximately **486.58k of comparison-period revenue** and contributing approximately **486.93k of positive revenue movement**. The category includes products with zero or non-positive baseline net revenue, so it should be interpreted as **newly active in the comparison period**, rather than proof that the products were newly launched.

In contrast, **807 products that generated positive revenue in November 2010 were inactive or non-positive in November 2011**. These products represented approximately **127.80k of baseline revenue** and produced a combined negative movement of approximately **127.88k**.

The most important result, however, concerns continuing products. Approximately **2,130 SKUs generated positive net revenue in both periods**, but together these continuing products recorded a substantial **327.83k decline in revenue**. This means that the overall growth in the business did not come from broad strengthening of the existing product portfolio. Instead, large gains from newly active products compensated for significant deterioration among many established products.

The strongest individual product gain came from SKU **23084 — RABBIT NIGHT LIGHT**, which generated approximately **34.42k of additional revenue** in November 2011. Because the total net merchandise increase across the business was only approximately 32.17k, this single product's gain was larger than the final net increase after losses elsewhere were taken into account. This should not be interpreted as the product accounting for more than 100% of gross growth; rather, it demonstrates how strongly gains in some products were offset by declines in others.

Other major positive contributors included **HOT WATER BOTTLE KEEP CALM**, with approximately **11.50k** of additional revenue, **JUMBO BAG 50'S CHRISTMAS**, with approximately **10.91k**, and **JUMBO BAG VINTAGE DOILY**, with approximately **9.53k**. Several of the largest positive contributors were newly active products, reinforcing the importance of portfolio renewal to the overall revenue result.

The largest product decline came from SKU **84347 — ROTATING SILVER ANGELS T-LIGHT HLDR**, whose revenue fell by approximately **16.08k**, or about **62.5%**. **REGENCY CAKESTAND 3 TIER** declined by approximately **13.35k**, while **HOT WATER BOTTLE BABUSHKA** lost approximately **8.13k** and became inactive in the comparison month. Other significant declines were recorded across several seasonal and established merchandise lines.

Across all products, the portfolio generated approximately **618.79k of gross positive product movements**, but these gains were offset by approximately **586.63k of negative product movements**, leaving the final net increase of only **32.17k**. This demonstrates that the modest headline growth concealed a very large amount of movement beneath the surface.

The top ten positive product contributors generated approximately **105.20k of revenue gains**, representing about **17.0% of all positive product growth**. The largest single product, RABBIT NIGHT LIGHT, represented approximately **5.6% of gross positive product growth**. This indicates that positive product growth was not dependent on only one SKU, although a relatively small group of strong performers still made a material contribution.

The price-volume decomposition provided further insight into the deterioration among continuing products. Of the **2,130 continuing SKUs**, approximately **2,082 had valid positive quantities in both periods and could be decomposed reliably**.

For these products, the combined revenue decline was approximately **318.66k**. Around **258.18k of this decline was associated with lower sales volumes**, while approximately **60.48k was associated with changes in realised revenue per unit, or price/mix effects**. Approximately **1,833 of the decomposed products were classified as volume-led**, compared with only **249 that were primarily price/mix-led**.

This suggests that the weakness among continuing products was predominantly a **volume problem rather than a pricing problem**. In other words, the existing portfolio generally suffered more from lower quantities sold than from deterioration in realised revenue per unit.

However, the price component should be interpreted cautiously. The measure used is realised revenue per net unit rather than a verified catalogue price. It can therefore be influenced by discounts, customer mix, returns, pack sizes or other transaction effects. It should be described as a **price/mix effect**, not evidence of deliberate price changes.

Product revenue concentration remained relatively low. The largest product represented approximately **2.02% of merchandise revenue in November 2010 and 2.40% in November 2011**. The top five products accounted for approximately **8.03% and 7.29%**, respectively, while the top ten accounted for approximately **11.94% and 11.35%**. This suggests that the business was not excessively dependent on a small number of individual products for total revenue.

Overall, Step 7 shows that the modest increase in merchandise revenue masked a substantial reshaping of the product portfolio. **Continuing products declined sharply, mainly because of lower sales volumes, while newly active products generated enough revenue to offset most of those losses.**

The evidence therefore suggests that the business's revenue resilience during the comparison period depended heavily on product renewal and newly active SKUs rather than on broad-based growth across the established product base. This creates an important distinction between headline growth and portfolio quality: the business grew overall, but many existing products weakened materially beneath that headline result.

The next stage should investigate whether some of the large opposing product movements reflect **customer switching or product substitution**, rather than simply independent product-level gains and losses.


## Step 8 — Product Substitution and Customer Switching

Step 8 investigates whether some of the opposing product movements identified in Step 7 are consistent with customer substitution.

A decline in one product combined with growth in another does not, by itself, demonstrate substitution. To establish stronger evidence, this section examines customer-level purchasing behaviour across the two comparison periods.

The analysis focuses on whether customers who purchased a declining product in November 2010 subsequently purchased a growing product in November 2011, particularly where they stopped purchasing the original product.

Product-description similarity is used only as supporting context. The primary evidence comes from observed customer switching behaviour.

The objective is to distinguish between:

- independent product growth and decline,
- possible substitution,
- and stronger customer-level switching patterns.

The analysis does not interpret switching as causal proof. Other factors such as product availability, seasonality, pricing, assortment changes and customer preferences may also explain the observed movements.

In [ ]:
# ================================================================
# 8.2 — DEFINE CUSTOMER-PRODUCT PURCHASE SCOPE
# ================================================================

substitution_sales = (
    df.loc[
        df["customer_is_identified"]
        & df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
        & df["period_month"].isin(
            [
                baseline_period,
                comparison_period
            ]
        )
    ]
    .copy()
)

print(
    f"Positive identified merchandise rows: "
    f"{len(substitution_sales):,}"
)

print(
    f"Unique customers: "
    f"{substitution_sales['customer'].nunique():,}"
)

print(
    f"Unique SKUs: "
    f"{substitution_sales['product'].nunique():,}"
)

In [ ]:
# ================================================================
# 8.3 — CREATE CUSTOMER-PRODUCT PURCHASE PRESENCE
# ================================================================

customer_product_presence = (
    substitution_sales
    [
        [
            "customer",
            "product",
            "product_description",
            "period_month",
        ]
    ]
    .drop_duplicates(
        subset=[
            "customer",
            "product",
            "period_month",
        ]
    )
)

display(
    customer_product_presence.head()
)

In [ ]:
# ================================================================
# 8.4 — DEFINE SUBSTITUTION CANDIDATE PRODUCTS
# ================================================================

candidate_count = 30


declining_candidates = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "absolute_change"
        ] < 0
    ]
    .sort_values(
        "absolute_change",
        ascending=True
    )
    .head(candidate_count)
    .copy()
)


growing_candidates = (
    product_revenue_change
    .loc[
        product_revenue_change[
            "absolute_change"
        ] > 0
    ]
    .sort_values(
        "absolute_change",
        ascending=False
    )
    .head(candidate_count)
    .copy()
)


declining_skus = (
    declining_candidates[
        "product"
    ]
    .tolist()
)

growing_skus = (
    growing_candidates[
        "product"
    ]
    .tolist()
)


print(
    f"Declining candidate SKUs: "
    f"{len(declining_skus)}"
)

print(
    f"Growing candidate SKUs: "
    f"{len(growing_skus)}"
)

In [ ]:
# ================================================================
# 8.5 — CREATE CUSTOMER SETS BY SKU AND PERIOD
# ================================================================

baseline_presence = (
    customer_product_presence
    .loc[
        customer_product_presence[
            "period_month"
        ] == baseline_period
    ]
)


comparison_presence = (
    customer_product_presence
    .loc[
        customer_product_presence[
            "period_month"
        ] == comparison_period
    ]
)


baseline_customers_by_product = (
    baseline_presence
    .groupby("product")["customer"]
    .apply(set)
    .to_dict()
)


comparison_customers_by_product = (
    comparison_presence
    .groupby("product")["customer"]
    .apply(set)
    .to_dict()
)

In [ ]:
# ================================================================
# 8.6 — PRODUCT DESCRIPTION LOOKUP
# ================================================================

substitution_description_lookup = (
    df[
        [
            "product",
            "product_description"
        ]
    ]
    .dropna()
    .drop_duplicates(
        subset="product"
    )
    .set_index(
        "product"
    )["product_description"]
)

In [ ]:
# ================================================================
# 8.7 — DESCRIPTION SIMILARITY FUNCTION
# ================================================================

import re


def description_tokens(text):
    """
    Convert a product description into a set of meaningful tokens.
    """

    if pd.isna(text):
        return set()

    tokens = re.findall(
        r"[A-Z0-9]+",
        str(text).upper()
    )

    # Remove very common low-information product words.
    stop_words = {
        "SET",
        "OF",
        "AND",
        "THE",
        "WITH",
        "PACK",
        "SMALL",
        "LARGE",
    }

    return {
        token
        for token in tokens
        if token not in stop_words
    }


def jaccard_similarity(text_a, text_b):
    """
    Measure token overlap between two product descriptions.

    Returns a value from 0 to 1.
    """

    tokens_a = description_tokens(text_a)
    tokens_b = description_tokens(text_b)

    union = (
        tokens_a
        | tokens_b
    )

    if not union:
        return 0.0

    intersection = (
        tokens_a
        & tokens_b
    )

    return (
        len(intersection)
        / len(union)
    )

In [ ]:
# ================================================================
# 8.8 — BUILD PRODUCT SWITCHING PAIRS
# ================================================================

switching_results = []


for declining_sku in declining_skus:

    baseline_declining_customers = (
        baseline_customers_by_product
        .get(
            declining_sku,
            set()
        )
    )

    comparison_declining_customers = (
        comparison_customers_by_product
        .get(
            declining_sku,
            set()
        )
    )


    baseline_customer_count = (
        len(
            baseline_declining_customers
        )
    )


    if baseline_customer_count == 0:
        continue


    for growing_sku in growing_skus:

        # Do not compare a SKU with itself.
        if declining_sku == growing_sku:
            continue


        comparison_growing_customers = (
            comparison_customers_by_product
            .get(
                growing_sku,
                set()
            )
        )


        # --------------------------------------------------------
        # Customers who bought the declining SKU in baseline
        # and the growing SKU in comparison.
        # --------------------------------------------------------

        switched_to_growing = (
            baseline_declining_customers
            & comparison_growing_customers
        )


        # --------------------------------------------------------
        # Stronger switching definition:
        #
        # Customer bought declining SKU in baseline,
        # did NOT buy that SKU in comparison,
        # but DID buy the growing SKU in comparison.
        # --------------------------------------------------------

        exclusive_switchers = (
            baseline_declining_customers
            - comparison_declining_customers
        ) & comparison_growing_customers


        switch_count = (
            len(switched_to_growing)
        )

        exclusive_switch_count = (
            len(exclusive_switchers)
        )


        switch_rate = (
            switch_count
            / baseline_customer_count
        )


        exclusive_switch_rate = (
            exclusive_switch_count
            / baseline_customer_count
        )


        # --------------------------------------------------------
        # Customer retention on declining product
        # --------------------------------------------------------

        declining_product_retained = (
            baseline_declining_customers
            & comparison_declining_customers
        )

        declining_retention_rate = (
            len(declining_product_retained)
            / baseline_customer_count
        )


        # --------------------------------------------------------
        # Description similarity
        # --------------------------------------------------------

        declining_description = (
            substitution_description_lookup
            .get(
                declining_sku,
                ""
            )
        )

        growing_description = (
            substitution_description_lookup
            .get(
                growing_sku,
                ""
            )
        )

        description_similarity = (
            jaccard_similarity(
                declining_description,
                growing_description
            )
        )


        switching_results.append(
            {
                "declining_sku":
                    declining_sku,

                "declining_product":
                    declining_description,

                "growing_sku":
                    growing_sku,

                "growing_product":
                    growing_description,

                "baseline_declining_customers":
                    baseline_customer_count,

                "switch_customers":
                    switch_count,

                "exclusive_switch_customers":
                    exclusive_switch_count,

                "switch_rate":
                    switch_rate,

                "exclusive_switch_rate":
                    exclusive_switch_rate,

                "declining_product_retention_rate":
                    declining_retention_rate,

                "description_similarity":
                    description_similarity,
            }
        )


product_switching = pd.DataFrame(
    switching_results
)

print(
    f"Candidate product pairs evaluated: "
    f"{len(product_switching):,}"
)

In [ ]:
# ================================================================
# 8.9 — ATTACH PRODUCT REVENUE MOVEMENTS
# ================================================================

revenue_change_lookup = (
    product_revenue_change
    .set_index(
        "product"
    )["absolute_change"]
)


product_switching[
    "declining_revenue_change"
] = (
    product_switching[
        "declining_sku"
    ]
    .map(
        revenue_change_lookup
    )
)


product_switching[
    "growing_revenue_change"
] = (
    product_switching[
        "growing_sku"
    ]
    .map(
        revenue_change_lookup
    )
)

In [ ]:
# ================================================================
# 8.10 — CALCULATE BACKGROUND PRODUCT ADOPTION
# ================================================================

baseline_identified_customers = set(
    baseline_presence[
        "customer"
    ]
    .unique()
)

baseline_identified_customer_count = (
    len(
        baseline_identified_customers
    )
)


comparison_product_adoption = (
    comparison_presence
    .groupby("product")["customer"]
    .nunique()
    / baseline_identified_customer_count
)


product_switching[
    "background_adoption_rate"
] = (
    product_switching[
        "growing_sku"
    ]
    .map(
        comparison_product_adoption
    )
    .fillna(0)
)


product_switching[
    "switch_lift"
] = np.where(
    product_switching[
        "background_adoption_rate"
    ] > 0,

    product_switching[
        "exclusive_switch_rate"
    ]
    / product_switching[
        "background_adoption_rate"
    ],

    np.nan
)

In [ ]:
# ================================================================
# 8.11 — CLASSIFY SUBSTITUTION EVIDENCE
# ================================================================

def classify_substitution_evidence(row):
    """
    Conservative heuristic classification.

    These categories indicate strength of observed switching evidence,
    not causal proof of product substitution.
    """

    switchers = (
        row["exclusive_switch_customers"]
    )

    switch_rate = (
        row["exclusive_switch_rate"]
    )

    lift = (
        row["switch_lift"]
    )

    similarity = (
        row["description_similarity"]
    )


    if (
        switchers >= 10
        and switch_rate >= 0.15
        and lift >= 2.0
        and similarity >= 0.20
    ):
        return "Strong candidate"


    if (
        switchers >= 5
        and switch_rate >= 0.10
        and lift >= 1.5
    ):
        return "Moderate candidate"


    if (
        switchers >= 3
        and switch_rate >= 0.05
    ):
        return "Weak candidate"


    return "Insufficient evidence"


product_switching[
    "substitution_evidence"
] = (
    product_switching
    .apply(
        classify_substitution_evidence,
        axis=1
    )
)

In [ ]:
# ================================================================
# 8.12 — RANK PRODUCT SUBSTITUTION CANDIDATES
# ================================================================

substitution_candidates = (
    product_switching
    .sort_values(
        [
            "exclusive_switch_customers",
            "exclusive_switch_rate",
            "switch_lift",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
)


display(
    substitution_candidates[
        [
            "declining_sku",
            "declining_product",
            "growing_sku",
            "growing_product",
            "baseline_declining_customers",
            "exclusive_switch_customers",
            "exclusive_switch_rate",
            "switch_lift",
            "description_similarity",
            "substitution_evidence",
        ]
    ]
    .head(25)
)

In [ ]:
# ================================================================
# 8.13 — SUBSTITUTION EVIDENCE SUMMARY
# ================================================================

substitution_evidence_summary = (
    product_switching[
        "substitution_evidence"
    ]
    .value_counts()
    .rename_axis(
        "Evidence Level"
    )
    .reset_index(
        name="Product Pairs"
    )
)

display(
    substitution_evidence_summary
)

In [ ]:
# ================================================================
# VISUAL 8.1 — PRODUCT SWITCHING HEATMAP
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


top_declining_for_heatmap = (
    declining_candidates
    .head(10)["product"]
    .tolist()
)


top_growing_for_heatmap = (
    growing_candidates
    .head(10)["product"]
    .tolist()
)


heatmap_data = (
    product_switching
    .loc[
        product_switching[
            "declining_sku"
        ].isin(
            top_declining_for_heatmap
        )
        &
        product_switching[
            "growing_sku"
        ].isin(
            top_growing_for_heatmap
        )
    ]
    .pivot(
        index="declining_sku",
        columns="growing_sku",
        values="exclusive_switch_rate"
    )
    .fillna(0)
    * 100
)


plt.figure(
    figsize=(13, 9)
)

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={
        "label":
        "Exclusive customer switch rate (%)"
    }
)

plt.title(
    "Customer Switching from Declining "
    "to Growing Products"
)

plt.xlabel(
    "Growing SKU — November 2011"
)

plt.ylabel(
    "Declining SKU — November 2010"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 8.2 — SWITCHING VS DESCRIPTION SIMILARITY
# ================================================================

plot_switching = (
    product_switching
    .loc[
        product_switching[
            "exclusive_switch_customers"
        ] >= 3
    ]
    .copy()
)


plt.figure(
    figsize=(10, 7)
)


sns.scatterplot(
    data=plot_switching,
    x="description_similarity",
    y="exclusive_switch_rate",
    size="exclusive_switch_customers",
    hue="substitution_evidence",
    sizes=(30, 300),
    alpha=0.75
)


plt.title(
    "Customer Switching Evidence vs "
    "Product Description Similarity"
)

plt.xlabel(
    "Product Description Similarity"
)

plt.ylabel(
    "Exclusive Customer Switch Rate"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 8.3 — STRONGEST SWITCHING CANDIDATES
# ================================================================

strongest_switching_pairs = (
    product_switching
    .loc[
        product_switching[
            "exclusive_switch_customers"
        ] >= 5
    ]
    .sort_values(
        "exclusive_switch_rate",
        ascending=False
    )
    .head(15)
    .copy()
)


strongest_switching_pairs[
    "pair_label"
] = (
    strongest_switching_pairs[
        "declining_sku"
    ]
    + " → "
    + strongest_switching_pairs[
        "growing_sku"
    ]
)


plt.figure(
    figsize=(11, 8)
)


sns.barplot(
    data=strongest_switching_pairs,
    y="pair_label",
    x="exclusive_switch_rate"
)


plt.title(
    "Strongest Observed Product Switching Candidates"
)

plt.xlabel(
    "Exclusive Customer Switch Rate"
)

plt.ylabel(
    "Declining SKU → Growing SKU"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 8.17 — DETAILED SUBSTITUTION CANDIDATE TABLE
# ================================================================

detailed_substitution_candidates = (
    substitution_candidates
    .loc[
        substitution_candidates[
            "substitution_evidence"
        ].isin(
            [
                "Strong candidate",
                "Moderate candidate",
            ]
        )
    ]
    [
        [
            "declining_sku",
            "declining_product",
            "declining_revenue_change",

            "growing_sku",
            "growing_product",
            "growing_revenue_change",

            "baseline_declining_customers",
            "exclusive_switch_customers",
            "exclusive_switch_rate",
            "background_adoption_rate",
            "switch_lift",

            "description_similarity",
            "substitution_evidence",
        ]
    ]
)


display(
    detailed_substitution_candidates.style.format(
        {
            "declining_revenue_change":
                "{:,.2f}",

            "growing_revenue_change":
                "{:,.2f}",

            "exclusive_switch_rate":
                "{:.2%}",

            "background_adoption_rate":
                "{:.2%}",

            "switch_lift":
                "{:.2f}x",

            "description_similarity":
                "{:.2f}",
        }
    )
)

In [ ]:
# ================================================================
# 8.18 — SWITCHING VS CROSS-PURCHASE CHECK
# ================================================================

product_switching[
    "continued_declining_product_rate"
] = (
    product_switching[
        "declining_product_retention_rate"
    ]
)


product_switching[
    "switch_minus_retention"
] = (
    product_switching[
        "exclusive_switch_rate"
    ]
    - product_switching[
        "continued_declining_product_rate"
    ]
)


display(
    product_switching
    .sort_values(
        "exclusive_switch_rate",
        ascending=False
    )
    [
        [
            "declining_sku",
            "growing_sku",
            "exclusive_switch_rate",
            "continued_declining_product_rate",
            "switch_lift",
            "substitution_evidence",
        ]
    ]
    .head(20)
)

In [ ]:
# ================================================================
# VISUAL 8.4 — SWITCHING VS ORIGINAL-PRODUCT RETENTION
# ================================================================

switch_retention_plot = (
    product_switching
    .loc[
        product_switching[
            "exclusive_switch_customers"
        ] >= 3
    ]
    .copy()
)


plt.figure(
    figsize=(10, 7)
)


sns.scatterplot(
    data=switch_retention_plot,
    x="continued_declining_product_rate",
    y="exclusive_switch_rate",
    size="exclusive_switch_customers",
    hue="substitution_evidence",
    sizes=(30, 300),
    alpha=0.75
)


plt.title(
    "Product Retention vs Customer Switching"
)

plt.xlabel(
    "Retention Rate on Declining Product"
)

plt.ylabel(
    "Exclusive Switch Rate to Growing Product"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 8.20 — STEP 8 SUMMARY METRICS
# ================================================================

strong_candidates = (
    product_switching[
        "substitution_evidence"
    ]
    .eq(
        "Strong candidate"
    )
    .sum()
)


moderate_candidates = (
    product_switching[
        "substitution_evidence"
    ]
    .eq(
        "Moderate candidate"
    )
    .sum()
)


weak_candidates = (
    product_switching[
        "substitution_evidence"
    ]
    .eq(
        "Weak candidate"
    )
    .sum()
)


step8_summary = pd.DataFrame(
    {
        "Metric": [
            "Candidate product pairs evaluated",
            "Strong substitution candidates",
            "Moderate substitution candidates",
            "Weak substitution candidates",
        ],

        "Value": [
            len(product_switching),
            strong_candidates,
            moderate_candidates,
            weak_candidates,
        ],
    }
)


display(
    step8_summary
)

### Step 8 Summary — Product Substitution and Customer Switching

Step 8 investigated whether the opposing product movements identified in Step 7 were consistent with customers switching from declining products into growing alternatives. The analysis was deliberately conducted at the customer level because simultaneous growth in one product and decline in another is not, by itself, evidence of substitution.

The analysis focused on the **30 largest declining merchandise SKUs and the 30 largest growing SKUs**, producing **900 candidate product pairs**. For each pair, the analysis measured whether customers who purchased the declining product in November 2010 subsequently purchased the growing product in November 2011, with particular emphasis on customers who **stopped purchasing the original SKU and purchased the potential replacement**.

The results indicate that substitution was **not a general explanation for the product movements observed in Step 7**. Of the 900 candidate pairs evaluated, only **1 pair met the conservative threshold for strong substitution evidence**, while **40 were classified as moderate candidates**, **437 as weak candidates**, and **422 had insufficient evidence**. These categories are screening classifications rather than formal causal tests and should therefore be interpreted as indicators of behavioural association rather than proof of substitution.

The strongest substitution candidate was between **SKU 22198 — LARGE POPCORN HOLDER** and **SKU 22197 — SMALL POPCORN HOLDER**. The large popcorn holder declined by approximately **2.55k** in revenue, while the small popcorn holder increased by approximately **3.71k**.

Among the **78 customers** who purchased the large popcorn holder in the baseline period, **24 customers — approximately 30.8% — stopped purchasing it and purchased the small popcorn holder in the comparison period**. The background adoption rate for the small popcorn holder was approximately **8.6%**, meaning former buyers of the large version were approximately **3.6 times more likely** to purchase the small version than the broader customer population. The two products also had very high description similarity, and none of the baseline buyers of the large version continued purchasing that SKU in the comparison month. Taken together, these results provide the clearest evidence in the analysis of behaviour **consistent with product substitution**.

A second notable pattern emerged within the **hot-water-bottle product family**. **HOT WATER BOTTLE BABUSHKA (SKU 22837)** declined by approximately **8.13k**, while **HOT WATER BOTTLE KEEP CALM (SKU 23355)** increased by approximately **11.50k**. Of the **128 baseline customers** who purchased the Babushka product, **27 customers, or approximately 21.1%, exclusively moved to the Keep Calm product** in the comparison period. This switching rate was around **1.57 times** the broader adoption rate for the Keep Calm product and was accompanied by meaningful product-description similarity.

Additional switching within the same product family supports the possibility of an assortment shift. Approximately **18 Babushka buyers, or 14.1%, moved to LOVE HOT WATER BOTTLE**, while **20 of the 85 customers** who had purchased **RED WHITE SCARF HOT WATER BOTTLE** moved to **HOT WATER BOTTLE KEEP CALM**. These relationships were classified as moderate candidates and collectively suggest that some of the decline in older hot-water-bottle designs may have been absorbed by newer alternatives within the same broad product category.

However, not every statistically noticeable customer movement should be interpreted as substitution. Several moderate and weak candidate pairs involved products with **little or no descriptive similarity**, such as customers moving between unrelated product categories. These movements may reflect normal basket changes, customer preferences, seasonality, cross-selling or broader changes in assortment rather than one product directly replacing another.

This distinction is important because retail customers frequently purchase multiple unrelated products. A customer who stops buying one SKU and later purchases another is not necessarily substituting one for the other. Product similarity, switching rate, background adoption and continued purchasing of the original SKU therefore need to be considered together.

The analysis also showed that customer switching was widespread at low levels across candidate pairs. Many declining-product customers purchased at least one growing product in the comparison period, but only a small number of relationships combined **meaningful switching rates, elevated adoption relative to the wider customer base and clear product similarity**. This is why the evidence classification produces many weak candidates but very few strong ones.

Overall, Step 8 provides **specific evidence of substitution in a limited number of product relationships rather than across the portfolio as a whole**. The strongest example is the movement from the **Large Popcorn Holder to the Small Popcorn Holder**, while the hot-water-bottle range shows a broader pattern consistent with possible migration between older and newer designs.

The findings therefore refine the conclusion from Step 7. The substantial decline in continuing products cannot simply be explained as customers switching to newly active products. In most cases, the available evidence is insufficient to demonstrate direct substitution. Some revenue movements may instead reflect changes in demand, availability, assortment, seasonality or customer mix.

Accordingly, product substitution should be treated as a **targeted hypothesis supported for particular product pairs**, rather than a general explanation for the portfolio's revenue movements. Where operational data on stock availability, launch dates, pricing or formal product replacements is available, these candidate pairs would be the most appropriate starting point for further investigation.


## Step 9 — Geographic Concentration and Diversification

Step 9 examines how geographically concentrated the merchandise business is across the full available transaction history.

While Step 5 identified the countries responsible for the change between November 2010 and November 2011, this section takes a longer-term view. It evaluates whether the business is becoming more or less dependent on its largest geographic markets and whether international revenue is becoming meaningfully diversified.

The analysis focuses on:

- the United Kingdom's share of monthly merchandise revenue,
- the combined share of the five largest geographic markets,
- the distribution of revenue across countries,
- and changes in geographic concentration over time.

Net merchandise revenue is used throughout so that postage, fees and accounting adjustments do not distort the geographic trading analysis.

The final partial calendar month in the source data is excluded from monthly trend interpretation because it does not represent a complete trading period.

In [ ]:
# ================================================================
# 9.2 — DEFINE FULL GEOGRAPHIC ANALYSIS SCOPE
# ================================================================

geographic_df = (
    df.loc[
        df["region"].notna()
        & df["is_merchandise"]
    ]
    .copy()
)

print(
    f"Geographic analysis rows: "
    f"{len(geographic_df):,}"
)

print(
    f"Countries represented: "
    f"{geographic_df['region'].nunique():,}"
)

print(
    f"Date range: "
    f"{geographic_df['date'].min():%d %b %Y} "
    f"to "
    f"{geographic_df['date'].max():%d %b %Y}"
)

In [ ]:
# ================================================================
# 9.3 — IDENTIFY COMPLETE CALENDAR MONTHS
# ================================================================

minimum_date = (
    geographic_df["date"].min()
)

maximum_date = (
    geographic_df["date"].max()
)


first_period = (
    minimum_date.to_period("M")
)

last_period = (
    maximum_date.to_period("M")
)


first_month_complete = (
    minimum_date.normalize()
    == first_period.start_time.normalize()
)

last_month_complete = (
    maximum_date.normalize()
    == last_period.end_time.normalize()
)


print(
    f"First month ({first_period}) complete: "
    f"{first_month_complete}"
)

print(
    f"Last month ({last_period}) complete: "
    f"{last_month_complete}"
)

In [ ]:
# ================================================================
# 9.3B — CREATE COMPLETE-MONTH ANALYSIS DATASET
# ================================================================

excluded_months = []


if not first_month_complete:
    excluded_months.append(
        str(first_period)
    )


if not last_month_complete:
    excluded_months.append(
        str(last_period)
    )


geographic_complete_df = (
    geographic_df.loc[
        ~geographic_df[
            "period_month"
        ].isin(
            excluded_months
        )
    ]
    .copy()
)


print(
    "Excluded incomplete months:",
    excluded_months
)

print(
    f"Rows retained for monthly trend analysis: "
    f"{len(geographic_complete_df):,}"
)

In [ ]:
# ================================================================
# 9.4 — MONTHLY COUNTRY MERCHANDISE REVENUE
# ================================================================

monthly_country_revenue = (
    geographic_complete_df
    .groupby(
        [
            "period_month",
            "region"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


display(
    monthly_country_revenue.head()
)

In [ ]:
# ================================================================
# 9.5 — MONTHLY TOTAL MERCHANDISE REVENUE
# ================================================================

monthly_total_revenue = (
    monthly_country_revenue
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        total_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


display(
    monthly_total_revenue.head()
)

In [ ]:
# ================================================================
# 9.6 — MONTHLY COUNTRY REVENUE SHARE
# ================================================================

monthly_country_share = (
    monthly_country_revenue
    .merge(
        monthly_total_revenue,
        on="period_month",
        how="left"
    )
)


monthly_country_share[
    "revenue_share"
] = np.where(
    monthly_country_share[
        "total_merchandise_revenue"
    ] != 0,

    monthly_country_share[
        "net_merchandise_revenue"
    ]
    / monthly_country_share[
        "total_merchandise_revenue"
    ],

    np.nan
)


monthly_country_share[
    "revenue_share_pct"
] = (
    monthly_country_share[
        "revenue_share"
    ]
    * 100
)

In [ ]:
# ================================================================
# 9.7 — UNITED KINGDOM MONTHLY REVENUE SHARE
# ================================================================

uk_monthly_share = (
    monthly_country_share
    .loc[
        monthly_country_share[
            "region"
        ] == "United Kingdom"
    ]
    [
        [
            "period_month",
            "net_merchandise_revenue",
            "revenue_share_pct",
        ]
    ]
    .rename(
        columns={
            "net_merchandise_revenue":
                "uk_revenue",

            "revenue_share_pct":
                "uk_revenue_share_pct",
        }
    )
)


display(
    uk_monthly_share.head()
)

In [ ]:
# ================================================================
# 9.8 — MONTHLY TOP-FIVE COUNTRY CONCENTRATION
# ================================================================

monthly_ranked_countries = (
    monthly_country_revenue
    .sort_values(
        [
            "period_month",
            "net_merchandise_revenue"
        ],
        ascending=[
            True,
            False
        ]
    )
)


monthly_ranked_countries[
    "country_rank"
] = (
    monthly_ranked_countries
    .groupby(
        "period_month"
    )
    .cumcount()
    + 1
)


monthly_top5_revenue = (
    monthly_ranked_countries
    .loc[
        monthly_ranked_countries[
            "country_rank"
        ] <= 5
    ]
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        top5_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


monthly_top5_share = (
    monthly_total_revenue
    .merge(
        monthly_top5_revenue,
        on="period_month",
        how="left"
    )
)


monthly_top5_share[
    "top5_share_pct"
] = (
    monthly_top5_share[
        "top5_revenue"
    ]
    / monthly_top5_share[
        "total_merchandise_revenue"
    ]
    * 100
)

In [ ]:
# ================================================================
# 9.9 — MONTHLY GEOGRAPHIC HHI
# ================================================================

monthly_country_share[
    "share_squared"
] = (
    monthly_country_share[
        "revenue_share_pct"
    ]
    ** 2
)


monthly_hhi = (
    monthly_country_share
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        geographic_hhi=(
            "share_squared",
            "sum"
        )
    )
)


display(
    monthly_hhi.head()
)

In [ ]:
# ================================================================
# 9.10 — BUILD MONTHLY GEOGRAPHIC CONCENTRATION TABLE
# ================================================================

geographic_concentration = (
    monthly_total_revenue
    .merge(
        uk_monthly_share[
            [
                "period_month",
                "uk_revenue_share_pct"
            ]
        ],
        on="period_month",
        how="left"
    )
    .merge(
        monthly_top5_share[
            [
                "period_month",
                "top5_share_pct"
            ]
        ],
        on="period_month",
        how="left"
    )
    .merge(
        monthly_hhi,
        on="period_month",
        how="left"
    )
)


# Convert month strings into timestamps for plotting.
geographic_concentration[
    "month"
] = pd.to_datetime(
    geographic_concentration[
        "period_month"
    ]
)


display(
    geographic_concentration.head()
)

In [ ]:
# ================================================================
# 9.11 — COMPARE CONCENTRATION IN THE TWO KEY PERIODS
# ================================================================

comparison_concentration = (
    geographic_concentration
    .loc[
        geographic_concentration[
            "period_month"
        ].isin(
            [
                baseline_period,
                comparison_period
            ]
        )
    ]
    [
        [
            "period_month",
            "uk_revenue_share_pct",
            "top5_share_pct",
            "geographic_hhi",
        ]
    ]
)


display(
    comparison_concentration
)

In [ ]:
# ================================================================
# 9.12 — CHANGE IN GEOGRAPHIC CONCENTRATION
# ================================================================

baseline_concentration = (
    comparison_concentration
    .loc[
        comparison_concentration[
            "period_month"
        ] == baseline_period
    ]
    .iloc[0]
)


comparison_concentration_row = (
    comparison_concentration
    .loc[
        comparison_concentration[
            "period_month"
        ] == comparison_period
    ]
    .iloc[0]
)


uk_share_change_pp = (
    comparison_concentration_row[
        "uk_revenue_share_pct"
    ]
    - baseline_concentration[
        "uk_revenue_share_pct"
    ]
)


top5_share_change_pp = (
    comparison_concentration_row[
        "top5_share_pct"
    ]
    - baseline_concentration[
        "top5_share_pct"
    ]
)


hhi_change = (
    comparison_concentration_row[
        "geographic_hhi"
    ]
    - baseline_concentration[
        "geographic_hhi"
    ]
)


print(
    f"Change in UK revenue share: "
    f"{uk_share_change_pp:+.2f} percentage points"
)

print(
    f"Change in Top-5 revenue share: "
    f"{top5_share_change_pp:+.2f} percentage points"
)

print(
    f"Change in geographic HHI: "
    f"{hhi_change:+.2f}"
)

In [ ]:
# ================================================================
# 9.13 — IDENTIFY LEADING GEOGRAPHIC MARKETS
# ================================================================

full_period_country_revenue = (
    geographic_complete_df
    .groupby(
        "region",
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
    .sort_values(
        "net_merchandise_revenue",
        ascending=False
    )
)


top_geographic_markets = (
    full_period_country_revenue
    .head(5)["region"]
    .tolist()
)


print(
    "Top geographic markets:",
    top_geographic_markets
)

In [ ]:
# ================================================================
# 9.14 — BUILD TOP-5 + OTHER REVENUE MIX
# ================================================================

monthly_geographic_mix = (
    monthly_country_revenue
    .copy()
)


monthly_geographic_mix[
    "market_group"
] = np.where(
    monthly_geographic_mix[
        "region"
    ].isin(
        top_geographic_markets
    ),

    monthly_geographic_mix[
        "region"
    ],

    "Other"
)


monthly_geographic_mix = (
    monthly_geographic_mix
    .groupby(
        [
            "period_month",
            "market_group"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)

In [ ]:
# ================================================================
# 9.15 — CALCULATE MONTHLY GEOGRAPHIC MIX SHARES
# ================================================================

monthly_mix_share = (
    monthly_geographic_mix
    .merge(
        monthly_total_revenue,
        on="period_month",
        how="left"
    )
)


monthly_mix_share[
    "share_pct"
] = (
    monthly_mix_share[
        "net_merchandise_revenue"
    ]
    / monthly_mix_share[
        "total_merchandise_revenue"
    ]
    * 100
)


geographic_mix_pivot = (
    monthly_mix_share
    .pivot(
        index="period_month",
        columns="market_group",
        values="share_pct"
    )
    .fillna(0)
)


geographic_mix_pivot.index = (
    pd.to_datetime(
        geographic_mix_pivot.index
    )
)

In [ ]:
# ================================================================
# VISUAL 9.1 — GEOGRAPHIC REVENUE MIX OVER TIME
# ================================================================

# Order the columns with the largest markets first,
# followed by the combined Other category.
plot_columns = [
    market
    for market in top_geographic_markets
    if market in geographic_mix_pivot.columns
]

if "Other" in geographic_mix_pivot.columns:
    plot_columns.append("Other")


plot_data = (
    geographic_mix_pivot[
        plot_columns
    ]
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


ax.stackplot(
    plot_data.index,
    [
        plot_data[column]
        for column in plot_columns
    ],
    labels=plot_columns
)


ax.set_title(
    "Geographic Mix of Net Merchandise Revenue Over Time"
)

ax.set_xlabel(
    "Month"
)

ax.set_ylabel(
    "Share of Net Merchandise Revenue (%)"
)

ax.set_ylim(
    0,
    100
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 9.2 — UK REVENUE DEPENDENCY OVER TIME
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=geographic_concentration,
    x="month",
    y="uk_revenue_share_pct",
    marker="o"
)


plt.title(
    "United Kingdom Share of Net Merchandise Revenue"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "UK Share of Revenue (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 9.3 — GEOGRAPHIC CONCENTRATION OVER TIME
# ================================================================

plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=geographic_concentration,
    x="month",
    y="geographic_hhi",
    marker="o"
)


plt.title(
    "Geographic Revenue Concentration Over Time"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Geographic HHI"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 9.16 — ACTIVE GEOGRAPHIC MARKETS BY MONTH
# ================================================================

monthly_active_markets = (
    monthly_country_revenue
    .loc[
        monthly_country_revenue[
            "net_merchandise_revenue"
        ] > 0
    ]
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        active_markets=(
            "region",
            "nunique"
        )
    )
)


monthly_active_markets[
    "month"
] = pd.to_datetime(
    monthly_active_markets[
        "period_month"
    ]
)


display(
    monthly_active_markets.head()
)

In [ ]:
# ================================================================
# VISUAL 9.4 — ACTIVE COUNTRY MARKETS OVER TIME
# ================================================================

plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=monthly_active_markets,
    x="month",
    y="active_markets",
    marker="o"
)


plt.title(
    "Number of Active Geographic Markets Over Time"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Countries Generating Positive Merchandise Revenue"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 9.17 — STEP 9 SUMMARY METRICS
# ================================================================

step9_summary = pd.DataFrame(
    {
        "Metric": [
            f"UK revenue share — {baseline_period}",
            f"UK revenue share — {comparison_period}",
            "Change in UK share (percentage points)",

            f"Top-5 revenue share — {baseline_period}",
            f"Top-5 revenue share — {comparison_period}",
            "Change in Top-5 share (percentage points)",

            f"Geographic HHI — {baseline_period}",
            f"Geographic HHI — {comparison_period}",
            "Change in Geographic HHI",
        ],

        "Value": [
            baseline_concentration[
                "uk_revenue_share_pct"
            ],

            comparison_concentration_row[
                "uk_revenue_share_pct"
            ],

            uk_share_change_pp,

            baseline_concentration[
                "top5_share_pct"
            ],

            comparison_concentration_row[
                "top5_share_pct"
            ],

            top5_share_change_pp,

            baseline_concentration[
                "geographic_hhi"
            ],

            comparison_concentration_row[
                "geographic_hhi"
            ],

            hhi_change,
        ],
    }
)


display(
    step9_summary
)

### Step 9 Summary — Geographic Concentration and Diversification

Step 9 examined the geographic structure of net merchandise revenue across the complete monthly history of the dataset. Unlike Step 5, which focused on the change between November 2010 and November 2011, this section assessed whether the business became more or less geographically diversified over time.

The final partial month of **December 2011 was excluded** from the monthly trend analysis because the source data ends on 9 December 2011 and therefore does not represent a complete trading month. The analysis consequently covers **24 complete months from December 2009 through November 2011**.

The results show that the business remained **highly dependent on the United Kingdom throughout the period**. Across the 24 complete months, the UK accounted for an average of approximately **85.05% of monthly net merchandise revenue**. Its monthly share ranged from approximately **78.74% at its lowest point to 91.16% at its highest**, confirming that the UK consistently represented the dominant source of merchandise revenue.

The comparison between the two principal analysis periods shows that UK dependence actually increased slightly. The UK represented approximately **87.54% of net merchandise revenue in November 2010 and 88.24% in November 2011**, an increase of approximately **0.70 percentage points**.

This means that although total merchandise revenue increased between the two periods, the business did not become more geographically balanced. Instead, a slightly larger proportion of revenue was generated by the UK in the comparison period.

The concentration is even more pronounced when the five largest geographic markets are considered together. The top five countries accounted for approximately **95.27% of merchandise revenue in November 2010 and 95.70% in November 2011**, an increase of approximately **0.42 percentage points**. Across the full 24-month period, the top-five share averaged approximately **95.35%**, ranging from about **92.35% to 97.59%**.

These results demonstrate that the business's geographic revenue base was concentrated not only in the UK but also within a very small group of countries. Revenue generated outside the leading markets remained comparatively limited.

The geographic Herfindahl-Hirschman Index (HHI) supports the same conclusion. The HHI increased from approximately **7,682 in November 2010 to 7,803 in November 2011**, an increase of around **121 points**. In this analysis, the HHI is used as a descriptive measure of revenue concentration rather than as a competition-law market measure. The increase indicates that revenue became modestly more concentrated across countries rather than more diversified.

Across the complete 24-month period, the geographic HHI averaged approximately **7,281**, with monthly values ranging from about **6,257 to 8,323**. The variation shows that geographic concentration changed meaningfully from month to month, but it remained high throughout the dataset.

The number of active geographic markets also provides useful context. **27 countries generated positive merchandise revenue in November 2010 compared with 22 in November 2011**, a reduction of five active markets. This reinforces the conclusion that the comparison-period growth was not accompanied by broader geographic participation.

However, the number of countries in which the business operates should not be interpreted as a direct measure of diversification. A business may sell into many countries while still generating the overwhelming majority of its revenue from one or two markets. The revenue-share analysis therefore provides a more meaningful measure of geographic dependence than the number of active markets alone.

Across the full available period, the five largest markets by merchandise revenue were the **United Kingdom, Eire, Netherlands, Germany and France**. The United Kingdom was by far the dominant market, generating approximately **15.82 million** in net merchandise revenue over the complete-month analysis period. The next-largest markets were substantially smaller, including approximately **596.28k from Eire**, **534.37k from the Netherlands**, **368.29k from Germany**, and **287.24k from France**.

The large gap between the United Kingdom and the next-largest markets demonstrates that the geographic concentration observed in the monthly analysis is structural rather than the result of a single unusual month.

Overall, Step 9 provides **little evidence that the business became meaningfully more geographically diversified between the two comparison periods**. The UK remained overwhelmingly dominant, its share of revenue increased slightly, the combined share of the five largest markets also increased, the geographic concentration index rose, and the number of active revenue-generating countries declined.

This does not mean international expansion was absent. Several non-UK markets generated meaningful revenue and some contributed positively to growth, as shown in Step 5. However, those gains were not large enough to materially reduce the business's dependence on its core UK market.

The principal strategic implication is therefore that **geographic reach and geographic diversification should not be treated as equivalent**. The business operated across multiple international markets, but its revenue base remained highly concentrated. This creates exposure to market-specific demand, economic conditions and customer behaviour in the United Kingdom.

Future geographic growth should therefore be assessed not simply by the number of countries entered, but by whether international markets begin to contribute a materially larger and more stable share of merchandise revenue.


## Step 10 — Customer-ID Data Quality and Unidentified Revenue

Step 10 evaluates the extent to which missing customer identifiers affect the reliability of customer-level analysis.

Transactions without a customer ID can still contribute valid merchandise revenue, but they cannot be linked across time to measure customer acquisition, repeat purchasing, retention, reactivation or inactivity.

The analysis therefore examines:

- the proportion of transaction rows without a customer ID,
- the proportion of merchandise revenue associated with unidentified customers,
- how unidentified-customer revenue changes over time,
- which countries and products account for the largest unidentified revenue,
- and the implications for customer-level findings.

This section does not treat unidentified transactions as one synthetic customer. Missing customer identities remain missing so that they do not create artificial customer concentration or retention behaviour.

In [ ]:
# ================================================================
# 10.2 — DEFINE CUSTOMER-ID QUALITY ANALYSIS SCOPE
# ================================================================

customer_quality_df = (
    df.loc[
        df["is_merchandise"]
    ]
    .copy()
)


print(
    f"Merchandise transaction rows: "
    f"{len(customer_quality_df):,}"
)

print(
    f"Identified customer rows: "
    f"{customer_quality_df['customer_is_identified'].sum():,}"
)

print(
    f"Unidentified customer rows: "
    f"{(~customer_quality_df['customer_is_identified']).sum():,}"
)

In [ ]:
# ================================================================
# 10.3 — OVERALL CUSTOMER IDENTIFICATION COMPLETENESS
# ================================================================

total_rows = (
    len(customer_quality_df)
)

identified_rows = (
    customer_quality_df[
        "customer_is_identified"
    ].sum()
)

unidentified_rows = (
    total_rows
    - identified_rows
)


identified_row_pct = (
    identified_rows
    / total_rows
    * 100
)

unidentified_row_pct = (
    unidentified_rows
    / total_rows
    * 100
)


print(
    f"Identified transaction rows: "
    f"{identified_row_pct:.2f}%"
)

print(
    f"Unidentified transaction rows: "
    f"{unidentified_row_pct:.2f}%"
)

In [ ]:
# ================================================================
# 10.4 — CUSTOMER IDENTIFICATION REVENUE SPLIT
# ================================================================

total_merchandise_revenue = (
    customer_quality_df[
        "net_merchandise_revenue"
    ]
    .sum()
)


identified_revenue = (
    customer_quality_df.loc[
        customer_quality_df[
            "customer_is_identified"
        ],
        "net_merchandise_revenue"
    ]
    .sum()
)


unidentified_revenue = (
    customer_quality_df.loc[
        ~customer_quality_df[
            "customer_is_identified"
        ],
        "net_merchandise_revenue"
    ]
    .sum()
)


identified_revenue_pct = (
    identified_revenue
    / total_merchandise_revenue
    * 100
)


unidentified_revenue_pct = (
    unidentified_revenue
    / total_merchandise_revenue
    * 100
)


customer_identification_summary = pd.DataFrame(
    {
        "Customer Identification": [
            "Identified",
            "Unidentified",
        ],

        "Transaction Rows": [
            identified_rows,
            unidentified_rows,
        ],

        "Row Share (%)": [
            identified_row_pct,
            unidentified_row_pct,
        ],

        "Net Merchandise Revenue": [
            identified_revenue,
            unidentified_revenue,
        ],

        "Revenue Share (%)": [
            identified_revenue_pct,
            unidentified_revenue_pct,
        ],
    }
)


display(
    customer_identification_summary
)

In [ ]:
# ================================================================
# 10.5 — MONTHLY IDENTIFIED VS UNIDENTIFIED REVENUE
# ================================================================

customer_quality_complete_df = (
    customer_quality_df.loc[
        ~customer_quality_df[
            "period_month"
        ].isin(
            excluded_months
        )
    ]
    .copy()
)


monthly_customer_identification = (
    customer_quality_complete_df
    .groupby(
        [
            "period_month",
            "customer_is_identified"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        transaction_rows=(
            "revenue",
            "size"
        ),
    )
)


display(
    monthly_customer_identification.head()
)

In [ ]:
# ================================================================
# 10.6 — MONTHLY UNIDENTIFIED REVENUE SHARE
# ================================================================

monthly_total_customer_revenue = (
    customer_quality_complete_df
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        total_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


monthly_unidentified_revenue = (
    customer_quality_complete_df
    .loc[
        ~customer_quality_complete_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        "period_month",
        as_index=False
    )
    .agg(
        unidentified_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


monthly_customer_quality = (
    monthly_total_customer_revenue
    .merge(
        monthly_unidentified_revenue,
        on="period_month",
        how="left"
    )
)


monthly_customer_quality[
    "unidentified_revenue"
] = (
    monthly_customer_quality[
        "unidentified_revenue"
    ]
    .fillna(0)
)


monthly_customer_quality[
    "unidentified_revenue_share_pct"
] = (
    monthly_customer_quality[
        "unidentified_revenue"
    ]
    / monthly_customer_quality[
        "total_revenue"
    ]
    * 100
)


monthly_customer_quality[
    "month"
] = pd.to_datetime(
    monthly_customer_quality[
        "period_month"
    ]
)


display(
    monthly_customer_quality.head()
)

In [ ]:
# ================================================================
# VISUAL 10.1 — UNIDENTIFIED CUSTOMER REVENUE SHARE OVER TIME
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=monthly_customer_quality,
    x="month",
    y="unidentified_revenue_share_pct",
    marker="o"
)


plt.title(
    "Share of Net Merchandise Revenue "
    "from Unidentified Customers"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Unidentified Revenue Share (%)"
)

plt.tight_layout()
plt.show()


In [ ]:
# ================================================================
# 10.7 — CUSTOMER IDENTIFICATION IN KEY COMPARISON PERIODS
# ================================================================

comparison_customer_quality = (
    monthly_customer_quality
    .loc[
        monthly_customer_quality[
            "period_month"
        ].isin(
            [
                baseline_period,
                comparison_period
            ]
        )
    ]
    [
        [
            "period_month",
            "total_revenue",
            "unidentified_revenue",
            "unidentified_revenue_share_pct",
        ]
    ]
)


display(
    comparison_customer_quality
)

In [ ]:
# ================================================================
# 10.8 — CHANGE IN CUSTOMER-ID COMPLETENESS
# ================================================================

baseline_customer_quality = (
    comparison_customer_quality
    .loc[
        comparison_customer_quality[
            "period_month"
        ] == baseline_period
    ]
    .iloc[0]
)


comparison_customer_quality_row = (
    comparison_customer_quality
    .loc[
        comparison_customer_quality[
            "period_month"
        ] == comparison_period
    ]
    .iloc[0]
)


unidentified_share_change_pp = (
    comparison_customer_quality_row[
        "unidentified_revenue_share_pct"
    ]
    - baseline_customer_quality[
        "unidentified_revenue_share_pct"
    ]
)


unidentified_revenue_change = (
    comparison_customer_quality_row[
        "unidentified_revenue"
    ]
    - baseline_customer_quality[
        "unidentified_revenue"
    ]
)


print(
    f"Change in unidentified revenue: "
    f"{unidentified_revenue_change:,.2f}"
)

print(
    f"Change in unidentified revenue share: "
    f"{unidentified_share_change_pp:+.2f} percentage points"
)

In [ ]:
# ================================================================
# 10.9 — VALIDATE AGAINST STEP 6
# ================================================================

assert np.isclose(
    unidentified_revenue_change,
    unidentified_change,
    atol=0.01
), (
    "Unidentified customer revenue movement does not "
    "reconcile with the Step 6 customer bridge."
)


print(
    "✓ Unidentified revenue movement reconciles "
    "to Step 6."
)

In [ ]:
# ================================================================
# 10.10 — UNIDENTIFIED REVENUE BY COUNTRY
# ================================================================

unidentified_by_country = (
    customer_quality_df
    .loc[
        ~customer_quality_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        "region",
        as_index=False
    )
    .agg(
        unidentified_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        transaction_rows=(
            "revenue",
            "size"
        ),
    )
    .sort_values(
        "unidentified_revenue",
        ascending=False
    )
)


display(
    unidentified_by_country.head(15)
)

In [ ]:
# ================================================================
# 10.11 — CUSTOMER-ID COMPLETENESS BY COUNTRY
# ================================================================

country_customer_quality = (
    customer_quality_df
    .groupby(
        "region",
        as_index=False
    )
    .agg(
        total_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        total_rows=(
            "revenue",
            "size"
        ),
    )
)


country_unidentified = (
    customer_quality_df
    .loc[
        ~customer_quality_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        "region",
        as_index=False
    )
    .agg(
        unidentified_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        unidentified_rows=(
            "revenue",
            "size"
        ),
    )
)


country_customer_quality = (
    country_customer_quality
    .merge(
        country_unidentified,
        on="region",
        how="left"
    )
    .fillna(
        {
            "unidentified_revenue": 0,
            "unidentified_rows": 0,
        }
    )
)


country_customer_quality[
    "unidentified_revenue_share_pct"
] = np.where(
    country_customer_quality[
        "total_revenue"
    ] != 0,

    country_customer_quality[
        "unidentified_revenue"
    ]
    / country_customer_quality[
        "total_revenue"
    ]
    * 100,

    np.nan
)


country_customer_quality[
    "unidentified_row_share_pct"
] = (
    country_customer_quality[
        "unidentified_rows"
    ]
    / country_customer_quality[
        "total_rows"
    ]
    * 100
)

In [ ]:
# ================================================================
# VISUAL 10.2 — TOP COUNTRIES BY UNIDENTIFIED REVENUE
# ================================================================

top_unknown_country_revenue = (
    country_customer_quality
    .sort_values(
        "unidentified_revenue",
        ascending=False
    )
    .head(12)
)


plt.figure(
    figsize=(11, 7)
)


sns.barplot(
    data=top_unknown_country_revenue,
    y="region",
    x="unidentified_revenue"
)


plt.title(
    "Countries with the Highest "
    "Unidentified Merchandise Revenue"
)

plt.xlabel(
    "Unidentified Net Merchandise Revenue"
)

plt.ylabel(
    "Country"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 10.12 — COUNTRIES WITH LOW CUSTOMER-ID COMPLETENESS
# ================================================================

minimum_country_revenue = 10_000


low_identification_countries = (
    country_customer_quality
    .loc[
        country_customer_quality[
            "total_revenue"
        ] >= minimum_country_revenue
    ]
    .sort_values(
        "unidentified_revenue_share_pct",
        ascending=False
    )
)


display(
    low_identification_countries[
        [
            "region",
            "total_revenue",
            "unidentified_revenue",
            "unidentified_revenue_share_pct",
            "unidentified_row_share_pct",
        ]
    ]
    .head(15)
)

In [ ]:
# ================================================================
# 10.13 — UNIDENTIFIED REVENUE BY PRODUCT
# ================================================================

unidentified_by_product = (
    customer_quality_df
    .loc[
        ~customer_quality_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        [
            "product",
            "product_description"
        ],
        as_index=False
    )
    .agg(
        unidentified_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        transaction_rows=(
            "revenue",
            "size"
        ),
    )
    .sort_values(
        "unidentified_revenue",
        ascending=False
    )
)


display(
    unidentified_by_product.head(15)
)

In [ ]:
# ================================================================
# 10.14 — CUSTOMER-ID COMPLETENESS BY PRODUCT
# ================================================================

product_customer_quality = (
    customer_quality_df
    .groupby(
        [
            "product",
            "product_description"
        ],
        as_index=False
    )
    .agg(
        total_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        total_rows=(
            "revenue",
            "size"
        ),
    )
)


product_unidentified = (
    customer_quality_df
    .loc[
        ~customer_quality_df[
            "customer_is_identified"
        ]
    ]
    .groupby(
        [
            "product",
            "product_description"
        ],
        as_index=False
    )
    .agg(
        unidentified_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        unidentified_rows=(
            "revenue",
            "size"
        ),
    )
)


product_customer_quality = (
    product_customer_quality
    .merge(
        product_unidentified,
        on=[
            "product",
            "product_description"
        ],
        how="left"
    )
    .fillna(
        {
            "unidentified_revenue": 0,
            "unidentified_rows": 0,
        }
    )
)


product_customer_quality[
    "unidentified_revenue_share_pct"
] = np.where(
    product_customer_quality[
        "total_revenue"
    ] != 0,

    product_customer_quality[
        "unidentified_revenue"
    ]
    / product_customer_quality[
        "total_revenue"
    ]
    * 100,

    np.nan
)

In [ ]:
# ================================================================
# VISUAL 10.3 — PRODUCTS WITH HIGHEST UNIDENTIFIED REVENUE
# ================================================================

top_unknown_products = (
    product_customer_quality
    .sort_values(
        "unidentified_revenue",
        ascending=False
    )
    .head(15)
    .copy()
)


top_unknown_products[
    "product_label"
] = (
    top_unknown_products[
        "product"
    ]
    + " — "
    + top_unknown_products[
        "product_description"
    ].fillna("")
)


plt.figure(
    figsize=(12, 9)
)


sns.barplot(
    data=top_unknown_products,
    y="product_label",
    x="unidentified_revenue"
)


plt.title(
    "Products with the Highest "
    "Unidentified-Customer Revenue"
)

plt.xlabel(
    "Unidentified Net Merchandise Revenue"
)

plt.ylabel(
    "Product"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 10.15 — UNIDENTIFIED REVENUE CONCENTRATION
# ================================================================

unknown_product_values = (
    product_customer_quality[
        "unidentified_revenue"
    ]
    .sort_values(
        ascending=False
    )
)


total_unknown_product_revenue = (
    unknown_product_values.sum()
)


unknown_top10_share = (
    unknown_product_values
    .head(10)
    .sum()
    / total_unknown_product_revenue
    * 100
    if total_unknown_product_revenue != 0
    else np.nan
)


print(
    f"Top 10 products' share of "
    f"unidentified revenue: "
    f"{unknown_top10_share:.2f}%"
)

In [ ]:

# ================================================================
# 10.16 — TRANSACTION VALUE BY IDENTIFICATION STATUS
# ================================================================

customer_identification_transaction_profile = (
    customer_quality_df
    .groupby(
        "customer_is_identified",
        as_index=False
    )
    .agg(
        transaction_rows=(
            "revenue",
            "size"
        ),
        total_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),
        average_line_revenue=(
            "net_merchandise_revenue",
            "mean"
        ),
        median_line_revenue=(
            "net_merchandise_revenue",
            "median"
        ),
    )
)


customer_identification_transaction_profile[
    "Customer Status"
] = np.where(
    customer_identification_transaction_profile[
        "customer_is_identified"
    ],
    "Identified",
    "Unidentified"
)


display(
    customer_identification_transaction_profile
)

In [ ]:
# ================================================================
# 10.17 — CUSTOMER ANALYSIS COVERAGE
# ================================================================

customer_analysis_coverage_pct = (
    identified_revenue
    / total_merchandise_revenue
    * 100
)


customer_analysis_uncovered_pct = (
    100
    - customer_analysis_coverage_pct
)


print(
    f"Revenue covered by identifiable "
    f"customer analysis: "
    f"{customer_analysis_coverage_pct:.2f}%"
)

print(
    f"Revenue not linkable to a customer: "
    f"{customer_analysis_uncovered_pct:.2f}%"
)

In [ ]:
# ================================================================
# 10.18 — CUSTOMER ANALYSIS COVERAGE IN COMPARISON PERIODS
# ================================================================

comparison_customer_coverage = (
    comparison_df
    .groupby(
        [
            "period_month",
            "customer_is_identified"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


comparison_period_total = (
    comparison_customer_coverage
    .groupby(
        "period_month"
    )[
        "net_merchandise_revenue"
    ]
    .transform("sum")
)


comparison_customer_coverage[
    "revenue_share_pct"
] = (
    comparison_customer_coverage[
        "net_merchandise_revenue"
    ]
    / comparison_period_total
    * 100
)


display(
    comparison_customer_coverage
)

In [ ]:
# ================================================================
# VISUAL 10.4 — CUSTOMER-ID REVENUE COMPOSITION
# ================================================================

coverage_plot = (
    comparison_customer_coverage
    .copy()
)


coverage_plot[
    "Customer Status"
] = np.where(
    coverage_plot[
        "customer_is_identified"
    ],
    "Identified",
    "Unidentified"
)


coverage_pivot = (
    coverage_plot
    .pivot(
        index="period_month",
        columns="Customer Status",
        values="revenue_share_pct"
    )
    .fillna(0)
)


coverage_pivot = (
    coverage_pivot
    .reindex(
        [
            baseline_period,
            comparison_period
        ]
    )
)


ax = coverage_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 6)
)


ax.set_title(
    "Customer Identification Coverage "
    "of Merchandise Revenue"
)

ax.set_xlabel(
    "Period"
)

ax.set_ylabel(
    "Share of Net Merchandise Revenue (%)"
)

ax.set_ylim(
    0,
    100
)

ax.legend(
    title="Customer Status"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 10.19 — STEP 10 SUMMARY METRICS
# ================================================================

step10_summary = pd.DataFrame(
    {
        "Metric": [
            "Overall unidentified transaction row share (%)",
            "Overall unidentified merchandise revenue share (%)",
            f"Unidentified revenue — {baseline_period}",
            f"Unidentified revenue — {comparison_period}",
            "Change in unidentified revenue",
            "Change in unidentified revenue share (pp)",
            "Revenue covered by customer-level analysis (%)",
            "Top 10 products share of unidentified revenue (%)",
        ],

        "Value": [
            unidentified_row_pct,
            unidentified_revenue_pct,

            baseline_customer_quality[
                "unidentified_revenue"
            ],

            comparison_customer_quality_row[
                "unidentified_revenue"
            ],

            unidentified_revenue_change,
            unidentified_share_change_pp,

            customer_analysis_coverage_pct,
            unknown_top10_share,
        ],
    }
)


display(
    step10_summary
)

### Step 10 Summary — Customer-ID Data Quality and Unidentified Revenue

Step 10 assessed how missing customer identifiers affect the reliability and coverage of the customer-level analysis.

The cleaned merchandise dataset contained approximately **1.039 million merchandise transaction rows**. Of these, approximately **805,893 rows had an identifiable customer**, while approximately **233,163 rows had no customer identifier**.

This means that approximately **77.56% of merchandise transaction rows were customer-identified**, while **22.44% were unidentified**.

The missing-customer issue is therefore substantial at the transaction level: more than one in five merchandise lines cannot be assigned to an individual customer.

However, the revenue impact is somewhat smaller than the transaction-row impact.

Across the full dataset, net merchandise revenue was approximately **18.98 million**. Of this:

- approximately **16.41 million**, or **86.46%**, was linked to identified customers; and
- approximately **2.57 million**, or **13.54%**, was associated with transactions where no customer ID was available.

This distinction is important. The unidentified transactions account for a disproportionately large number of transaction rows but a smaller proportion of revenue, indicating that unidentified transactions tend to be lower-value on average.

The average merchandise revenue per transaction line was approximately **20.37 for identified customers**, compared with only approximately **11.02 for unidentified transactions**. The median values show the same pattern: approximately **11.25 for identified customer lines versus 4.96 for unidentified lines**.

This suggests that missing customer IDs are more heavily associated with smaller-value transactions rather than being distributed evenly across all transaction types.

#### Unidentified revenue increased materially in the main comparison period

The data-quality issue became more important between the two principal comparison months.

In **November 2010**, unidentified customers generated approximately **273.63k of net merchandise revenue**, representing approximately **19.54% of total merchandise revenue** for the month.

By **November 2011**, unidentified-customer revenue had increased to approximately **314.04k**, representing approximately **21.92% of merchandise revenue**.

Unidentified merchandise revenue therefore increased by approximately **40.41k**, while its share of merchandise revenue increased by approximately **2.38 percentage points**.

This finding is particularly important because the entire business recorded only approximately **32.17k of net merchandise revenue growth** between the two comparison periods.

In other words, the increase in revenue associated with unidentified customers was **larger than the company's total net merchandise revenue increase**.

This reconciles directly with the customer bridge in Step 6. Identified-customer revenue declined slightly overall, while growth in unidentified-customer revenue was sufficient to produce the positive headline result.

The increase in total revenue therefore cannot be interpreted simply as evidence of stronger performance among known customers.

#### Customer-level analytical coverage deteriorated between the two periods

Because unidentified transactions cannot be assigned longitudinally to customers, only the identified portion of revenue can participate in analyses such as:

- repeat purchasing,
- customer acquisition,
- reactivation,
- retention,
- inactivity,
- cohort behaviour,
- and customer lifetime or revenue-retention analysis.

In November 2010, approximately **80.46% of merchandise revenue was attributable to identified customers**.

In November 2011, this fell to approximately **78.08%**.

Therefore, more than **one-fifth of November 2011 merchandise revenue cannot be connected to a persistent customer identity**.

Any future retention or customer-lifecycle metrics must consequently be described as applying to the **identified-customer population**, rather than to the entire business.

#### Missing customer identification varies substantially over time

Across the 24 complete calendar months used for monthly trend analysis, unidentified revenue represented an average of approximately **12.86% of monthly merchandise revenue**.

The proportion was not stable.

The lowest unidentified revenue share occurred in **February 2010**, at approximately **7.48%**, while the highest occurred in **December 2010**, when approximately **27.65% of merchandise revenue was not linked to an identifiable customer**.

This variation shows that customer-ID completeness is not simply a fixed structural characteristic of the dataset. The quality of customer identification changes materially between months.

As a result, comparisons of customer metrics over time can be affected by changes in identification coverage as well as genuine changes in customer behaviour.

For example, an apparent decline in identified-customer revenue may partly represent a greater proportion of transactions being recorded without customer IDs rather than a true loss of those customers.

#### The unidentified-revenue problem is overwhelmingly concentrated in the United Kingdom

Geographically, the United Kingdom accounts for by far the largest absolute amount of unidentified revenue.

Approximately **2.51 million of unidentified merchandise revenue** was generated in the United Kingdom. This represents roughly **97.6% of all unidentified merchandise revenue in the dataset**.

Approximately **15.46% of total UK merchandise revenue** was unidentified, while around **24.07% of UK merchandise transaction rows** lacked a customer ID.

Because the United Kingdom also dominates overall business revenue, the absolute concentration is partly expected. Nevertheless, the scale means that improvements to customer identification within the UK would have by far the greatest impact on the reliability of customer analytics.

Eire was the next-largest source of unidentified revenue at approximately **35.56k**, but this was substantially smaller than the UK total.

Hong Kong represents a different type of data-quality issue. Its approximately **13.86k of merchandise revenue was entirely unidentified**, producing a **100% unidentified customer rate** for the market.

Other markets with notable identification gaps included **Unspecified**, where approximately **21.17% of revenue was unidentified**, while several smaller markets also had high percentages but much lower absolute economic significance.

The distinction between **absolute unidentified revenue** and **percentage unidentified revenue** is important. A small market may have very poor customer identification but little effect on total-business analytics, whereas the UK's more moderate percentage has a much greater impact because of its scale.

#### Missing customer identification is distributed across many products

The product analysis shows that unidentified revenue is not confined to one unusual SKU or a very small product group.

The product with the largest unidentified-customer revenue was **REGENCY CAKESTAND 3 TIER**, with approximately **52.93k** of unidentified revenue, representing about **16.83% of that product's total merchandise revenue**.

Other major products included:

- **PARTY BUNTING** — approximately **44.99k unidentified**, or **30.57%** of product revenue;
- **PAPER CHAIN KIT 50'S CHRISTMAS** — approximately **40.91k**, or **35.14%**;
- **CHARLOTTE BAG SUKI DESIGN** — approximately **29.88k**, or **65.97%**;
- **HOT WATER BOTTLE TEA AND SYMPATHY** — approximately **24.53k**, or **41.90%**;
- **SMALL POPCORN HOLDER** — approximately **21.25k**, or **26.92%**; and
- **RABBIT NIGHT LIGHT** — approximately **15.62k**, or **23.40%**.

This is particularly relevant because several of these products also appeared in the product-growth and substitution analyses.

For example, RABBIT NIGHT LIGHT was one of the strongest revenue-growth products identified in Step 7, but approximately **23% of its full-period revenue was generated without a customer identifier**. Customer-level interpretations involving such products therefore necessarily observe only part of their total commercial activity.

The top ten products accounted for only approximately **10.70% of all unidentified merchandise revenue**.

This relatively low concentration indicates that missing customer identification is **broadly distributed across the product portfolio**, rather than being caused primarily by a handful of problematic SKUs.

#### The data-quality issue changes how customer findings should be interpreted

The most important implication from Step 10 is not that unidentified revenue should be removed.

These transactions remain valid commercial activity and should continue contributing to merchandise revenue, country analysis, product analysis and overall financial performance.

The limitation arises specifically when the analysis asks questions that require persistent customer identity.

An unidentified transaction cannot reliably tell us whether the purchaser was:

- a new customer,
- an existing customer,
- a returning customer,
- a reactivated customer,
- or a customer who had previously appeared under another transaction.

Therefore, unidentified transactions must remain excluded from longitudinal customer-behaviour calculations rather than being grouped into a synthetic "unknown customer."

Creating one artificial unknown customer would falsely make hundreds of thousands of unrelated transactions appear to belong to a single customer and would severely distort customer concentration, repeat purchase and retention metrics.

#### Overall conclusion

Step 10 establishes that customer identification is a **material analytical limitation but not a reason to invalidate the overall revenue analysis**.

Approximately **86.46% of full-period merchandise revenue is linked to identifiable customers**, providing substantial coverage for customer analysis. However, approximately **13.54% remains outside the observable customer history**, and the missing share rises to approximately **21.92% in November 2011**.

Most importantly, unidentified-customer revenue increased by approximately **40.41k between November 2010 and November 2011**, exceeding the approximately **32.17k overall increase in net merchandise revenue**.

This means that the business's positive headline revenue movement cannot be attributed to improved performance among known customers. Identified-customer revenue actually weakened slightly, while unidentified revenue provided the offset required to produce overall growth.

The missing-customer issue is overwhelmingly concentrated geographically in the **United Kingdom**, but is distributed broadly across the product portfolio.

Accordingly, all subsequent retention, repeat-purchase, cohort and customer-inactivity analyses should explicitly state that they apply only to the **identified-customer population**.

Step 10 therefore establishes the analytical coverage boundary for the remainder of the customer-lifecycle analysis: the identified-customer data is sufficiently substantial to support meaningful behavioural analysis, but the resulting retention metrics should not be presented as if they describe 100% of company revenue.



## Step 11 — Customer Retention and Repeat-Purchase Analysis

Step 11 evaluates whether identified customers return to purchase again after their first observable merchandise order.

The analysis measures cumulative repeat purchasing within:

- 30 days,
- 60 days,
- 90 days,
- and 180 days

of the customer's first positive merchandise purchase.

Only identified customers are included because repeat purchasing requires a persistent customer identifier. As established in Step 10, these customers represent the observable customer population rather than 100% of business revenue.

The analysis is performed at the order level rather than the transaction-line level so that multiple product lines within the same invoice do not artificially increase purchase frequency.

Customers whose first observed purchase occurs at the beginning of the dataset are excluded from first-purchase cohort interpretation where the available history suggests that their true acquisition date may pre-date the dataset.

Each repeat-purchase horizon is evaluated only for customers with enough subsequent observation time to reach that horizon. This prevents recently acquired customers from being incorrectly classified as non-repeat customers simply because the dataset ends before their 30-, 60-, 90- or 180-day observation window is complete.

The purpose is to determine:

- how many observable customers purchase again,
- how quickly repeat purchasing occurs,
- how repeat behaviour changes as the observation window expands,
- and whether repeat purchase appears sufficiently strong to support customer-driven revenue growth.

In [ ]:
# ================================================================
# 11.2 — DEFINE VALID CUSTOMER PURCHASE EVENTS
# ================================================================

customer_purchase_lines = (
    df.loc[
        df["customer_is_identified"]
        & df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
    ]
    .copy()
)


print(
    f"Valid identified merchandise purchase lines: "
    f"{len(customer_purchase_lines):,}"
)

print(
    f"Unique identified customers: "
    f"{customer_purchase_lines['customer'].nunique():,}"
)

print(
    f"Unique invoices: "
    f"{customer_purchase_lines['invoice'].nunique():,}"
)

In [ ]:
# ================================================================
# 11.3 — CREATE CUSTOMER ORDER-LEVEL DATASET
# ================================================================

customer_orders = (
    customer_purchase_lines
    .groupby(
        [
            "customer",
            "invoice"
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        order_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        order_quantity=(
            "quantity",
            "sum"
        ),

        distinct_products=(
            "product",
            "nunique"
        ),

        region=(
            "region",
            "first"
        ),
    )
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Customer orders: "
    f"{len(customer_orders):,}"
)

print(
    f"Customers represented: "
    f"{customer_orders['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 11.4 — FIRST OBSERVED POSITIVE PURCHASE
# ================================================================

customer_first_purchase = (
    customer_orders
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_purchase_date=(
            "order_date",
            "min"
        )
    )
)


display(
    customer_first_purchase.head()
)

In [ ]:
# ================================================================
# 11.5 — RECONSTRUCT AND ATTACH LEFT-TRUNCATION STATUS
# ================================================================

# ---------------------------------------------------------------
# Why this is necessary
# ---------------------------------------------------------------
# Customers whose first observed transaction occurs in the first
# calendar month covered by the dataset may have purchased before
# the available history begins.
#
# We therefore flag them as potentially left-truncated.
#
# Important:
# We use the customer's FIRST OBSERVED TRANSACTION in the dataset,
# not merely their first positive purchase.
# ---------------------------------------------------------------


# Ensure date is datetime
df["date"] = pd.to_datetime(
    df["date"]
)


# ---------------------------------------------------------------
# Identify the first calendar month represented in the dataset
# ---------------------------------------------------------------

dataset_start_date = (
    df["date"]
    .min()
)

dataset_start_month = (
    dataset_start_date
    .to_period("M")
)


print(
    f"Dataset begins: "
    f"{dataset_start_date:%d %B %Y}"
)

print(
    f"Potential left-truncation month: "
    f"{dataset_start_month}"
)


# ---------------------------------------------------------------
# Find each identified customer's first observed transaction date
# ---------------------------------------------------------------

customer_first_observed = (
    df.loc[
        df["customer_is_identified"]
    ]
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        customer_first_observed_date=(
            "date",
            "min"
        )
    )
)


# ---------------------------------------------------------------
# Flag customers first observed in the dataset's opening month
# ---------------------------------------------------------------

customer_first_observed[
    "customer_is_left_truncated"
] = (
    customer_first_observed[
        "customer_first_observed_date"
    ]
    .dt.to_period("M")
    .eq(
        dataset_start_month
    )
)


# ---------------------------------------------------------------
# Attach flag to the first-purchase table
# ---------------------------------------------------------------

customer_first_purchase = (
    customer_first_purchase
    .merge(
        customer_first_observed[
            [
                "customer",
                "customer_first_observed_date",
                "customer_is_left_truncated",
            ]
        ],
        on="customer",
        how="left"
    )
)


# Any unmatched cases are conservatively treated as False.
customer_first_purchase[
    "customer_is_left_truncated"
] = (
    customer_first_purchase[
        "customer_is_left_truncated"
    ]
    .fillna(False)
)


# ---------------------------------------------------------------
# Validation
# ---------------------------------------------------------------

print(
    f"Customers in first-purchase table: "
    f"{len(customer_first_purchase):,}"
)

print(
    f"Potentially left-truncated customers: "
    f"{customer_first_purchase['customer_is_left_truncated'].sum():,}"
)

print(
    f"Customers eligible as observable acquisitions: "
    f"{(~customer_first_purchase['customer_is_left_truncated']).sum():,}"
)

In [ ]:
# ================================================================
# 11.6 — DEFINE OBSERVABLE ACQUISITION COHORT
# ================================================================

observable_first_purchase = (
    customer_first_purchase
    .loc[
        ~customer_first_purchase[
            "customer_is_left_truncated"
        ]
    ]
    .copy()
)


print(
    f"Customers eligible for first-purchase analysis: "
    f"{len(observable_first_purchase):,}"
)

In [ ]:

# ================================================================
# 11.7 — RANK CUSTOMER ORDERS
# ================================================================

customer_orders[
    "order_number"
] = (
    customer_orders
    .groupby(
        "customer"
    )
    .cumcount()
    + 1
)

In [ ]:
# ================================================================
# 11.8 — IDENTIFY SECOND PURCHASE
# ================================================================

second_orders = (
    customer_orders
    .loc[
        customer_orders[
            "order_number"
        ] == 2
    ]
    [
        [
            "customer",
            "order_date"
        ]
    ]
    .rename(
        columns={
            "order_date":
                "second_purchase_date"
        }
    )
)


customer_repeat = (
    observable_first_purchase
    .merge(
        second_orders,
        on="customer",
        how="left"
    )
)

In [ ]:

# ================================================================
# 11.9 — DAYS TO SECOND PURCHASE
# ================================================================

customer_repeat[
    "days_to_second_purchase"
] = (
    customer_repeat[
        "second_purchase_date"
    ]
    - customer_repeat[
        "first_purchase_date"
    ]
).dt.days


customer_repeat[
    "ever_repeated"
] = (
    customer_repeat[
        "second_purchase_date"
    ]
    .notna()
)

In [ ]:
# ================================================================
# 11.10 — BASIC REPEAT-PURCHASE SUMMARY
# ================================================================

customers_total = (
    len(customer_repeat)
)


customers_ever_repeated = (
    customer_repeat[
        "ever_repeated"
    ].sum()
)


ever_repeat_rate = (
    customers_ever_repeated
    / customers_total
    * 100
)


print(
    f"Observable customers: "
    f"{customers_total:,}"
)

print(
    f"Customers with a later purchase: "
    f"{customers_ever_repeated:,}"
)

print(
    f"Observed eventual repeat-purchase rate: "
    f"{ever_repeat_rate:.2f}%"
)

In [ ]:
# ================================================================
# 11.11 — DEFINE OBSERVATION END DATE
# ================================================================

analysis_end_date = (
    customer_orders[
        "order_date"
    ].max()
)


print(
    f"Customer observation ends: "
    f"{analysis_end_date:%d %B %Y}"
)

In [ ]:
# ================================================================
# 11.12 — DEFINE RETENTION / REPEAT WINDOWS
# ================================================================

repeat_windows = [
    30,
    60,
    90,
    180,
]

In [ ]:
# ================================================================
# 11.13 — CALCULATE WINDOW ELIGIBILITY
# ================================================================

for days in repeat_windows:

    customer_repeat[
        f"eligible_{days}d"
    ] = (
        customer_repeat[
            "first_purchase_date"
        ]
        + pd.Timedelta(
            days=days
        )
        <= analysis_end_date
    )

In [ ]:
# ================================================================
# 11.14 — CALCULATE CUMULATIVE REPEAT PURCHASE
# ================================================================

for days in repeat_windows:

    customer_repeat[
        f"repeat_{days}d"
    ] = (
        customer_repeat[
            "eligible_" + str(days) + "d"
        ]
        &
        customer_repeat[
            "days_to_second_purchase"
        ]
        .le(days)
    )

In [ ]:
# ================================================================
# 11.15 — REPEAT-PURCHASE RATE SUMMARY
# ================================================================

repeat_summary_records = []


for days in repeat_windows:

    eligible_mask = (
        customer_repeat[
            f"eligible_{days}d"
        ]
    )


    eligible_customers = (
        eligible_mask.sum()
    )


    repeat_customers = (
        customer_repeat.loc[
            eligible_mask,
            f"repeat_{days}d"
        ]
        .sum()
    )


    repeat_rate = (
        repeat_customers
        / eligible_customers
        * 100
        if eligible_customers > 0
        else np.nan
    )


    repeat_summary_records.append(
        {
            "window_days":
                days,

            "eligible_customers":
                eligible_customers,

            "repeat_customers":
                repeat_customers,

            "repeat_rate_pct":
                repeat_rate,

            "non_repeat_customers":
                eligible_customers
                - repeat_customers,
        }
    )


repeat_purchase_summary = pd.DataFrame(
    repeat_summary_records
)


display(
    repeat_purchase_summary
)

In [ ]:
# ================================================================
# VISUAL 11.1 — CUMULATIVE REPEAT-PURCHASE RATE
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(
    figsize=(10, 6)
)


sns.lineplot(
    data=repeat_purchase_summary,
    x="window_days",
    y="repeat_rate_pct",
    marker="o"
)


for _, row in repeat_purchase_summary.iterrows():

    plt.text(
        row["window_days"],
        row["repeat_rate_pct"],
        f"{row['repeat_rate_pct']:.1f}%",
        ha="center",
        va="bottom"
    )


plt.title(
    "Cumulative Repeat-Purchase Rate"
)

plt.xlabel(
    "Days Since First Purchase"
)

plt.ylabel(
    "Customers with a Repeat Purchase (%)"
)

plt.xticks(
    repeat_windows
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 11.16 — INCREMENTAL REPEAT BEHAVIOUR
#          USING A COMMON 180-DAY-ELIGIBLE COHORT
# ================================================================

common_repeat_cohort = (
    customer_repeat
    .loc[
        customer_repeat[
            "eligible_180d"
        ]
    ]
    .copy()
)


common_repeat_records = []

previous_rate = 0


for days in repeat_windows:

    repeat_customers = (
        common_repeat_cohort[
            "days_to_second_purchase"
        ]
        .le(days)
        .sum()
    )


    repeat_rate = (
        repeat_customers
        / len(common_repeat_cohort)
        * 100
    )


    incremental_rate = (
        repeat_rate
        - previous_rate
    )


    common_repeat_records.append(
        {
            "window_days":
                days,

            "eligible_customers":
                len(common_repeat_cohort),

            "repeat_customers":
                repeat_customers,

            "repeat_rate_pct":
                repeat_rate,

            "incremental_repeat_rate_pp":
                incremental_rate,
        }
    )


    previous_rate = repeat_rate


common_cohort_repeat_summary = (
    pd.DataFrame(
        common_repeat_records
    )
)


display(
    common_cohort_repeat_summary
)

In [ ]:
# ================================================================
# 11.17 — DISTRIBUTION OF TIME TO SECOND PURCHASE
# ================================================================

repeaters = (
    customer_repeat
    .loc[
        customer_repeat[
            "ever_repeated"
        ]
        &
        customer_repeat[
            "days_to_second_purchase"
        ]
        .ge(0)
    ]
    .copy()
)


repeat_time_summary = (
    repeaters[
        "days_to_second_purchase"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)


display(
    repeat_time_summary
)

In [ ]:
# ================================================================
# VISUAL 11.2 — DAYS TO SECOND PURCHASE
# ================================================================

repeaters_365 = (
    repeaters.loc[
        repeaters[
            "days_to_second_purchase"
        ] <= 365
    ]
)


plt.figure(
    figsize=(11, 6)
)


sns.histplot(
    data=repeaters_365,
    x="days_to_second_purchase",
    bins=40,
    kde=True
)


for threshold in [
    30,
    60,
    90,
    180,
]:

    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=1
    )


plt.title(
    "Distribution of Time to Second Purchase"
)

plt.xlabel(
    "Days from First to Second Purchase"
)

plt.ylabel(
    "Customers"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 11.18 — TIME-TO-REPEAT STATISTICS
# ================================================================

median_days_to_repeat = (
    repeaters[
        "days_to_second_purchase"
    ]
    .median()
)


mean_days_to_repeat = (
    repeaters[
        "days_to_second_purchase"
    ]
    .mean()
)


print(
    f"Median days to second purchase: "
    f"{median_days_to_repeat:.1f}"
)

print(
    f"Mean days to second purchase: "
    f"{mean_days_to_repeat:.1f}"
)

In [ ]:
# ================================================================
# 11.19 — CUSTOMER ORDER FREQUENCY
# ================================================================

customer_order_frequency = (
    customer_orders
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        total_orders=(
            "invoice",
            "nunique"
        ),

        total_purchase_revenue=(
            "order_revenue",
            "sum"
        ),
    )
)


customer_repeat = (
    customer_repeat
    .merge(
        customer_order_frequency,
        on="customer",
        how="left"
    )
)


display(
    customer_repeat[
        [
            "customer",
            "first_purchase_date",
            "days_to_second_purchase",
            "total_orders",
            "total_purchase_revenue",
        ]
    ]
    .head()
)

In [ ]:
# ================================================================
# 11.20 — ORDER-FREQUENCY GROUPS
# ================================================================

customer_repeat[
    "order_frequency_group"
] = pd.cut(
    customer_repeat[
        "total_orders"
    ],
    bins=[
        0,
        1,
        2,
        3,
        5,
        10,
        np.inf,
    ],
    labels=[
        "1 order",
        "2 orders",
        "3 orders",
        "4–5 orders",
        "6–10 orders",
        "11+ orders",
    ]
)


order_frequency_summary = (
    customer_repeat[
        "order_frequency_group"
    ]
    .value_counts(
        sort=False
    )
    .rename_axis(
        "Order Frequency"
    )
    .reset_index(
        name="Customers"
    )
)


order_frequency_summary[
    "Customer Share (%)"
] = (
    order_frequency_summary[
        "Customers"
    ]
    / order_frequency_summary[
        "Customers"
    ].sum()
    * 100
)


display(
    order_frequency_summary
)

In [ ]:
# ================================================================
# VISUAL 11.3 — CUSTOMER ORDER FREQUENCY
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=order_frequency_summary,
    x="Order Frequency",
    y="Customer Share (%)"
)


plt.title(
    "Distribution of Customer Purchase Frequency"
)

plt.xlabel(
    "Observed Number of Orders"
)

plt.ylabel(
    "Share of Customers (%)"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 11.21 — REVENUE BY CUSTOMER PURCHASE FREQUENCY
# ================================================================

customer_repeat[
    "customer_type"
] = np.where(
    customer_repeat[
        "total_orders"
    ] > 1,
    "Repeat Customer",
    "One-Time Customer"
)


customer_type_revenue = (
    customer_repeat
    .groupby(
        "customer_type",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        total_revenue=(
            "total_purchase_revenue",
            "sum"
        ),
    )
)


customer_type_revenue[
    "customer_share_pct"
] = (
    customer_type_revenue[
        "customers"
    ]
    / customer_type_revenue[
        "customers"
    ].sum()
    * 100
)


customer_type_revenue[
    "revenue_share_pct"
] = (
    customer_type_revenue[
        "total_revenue"
    ]
    / customer_type_revenue[
        "total_revenue"
    ].sum()
    * 100
)


display(
    customer_type_revenue
)

In [ ]:
# ================================================================
# VISUAL 11.4 — CUSTOMER SHARE VS REVENUE SHARE
# ================================================================

customer_type_plot = (
    customer_type_revenue
    .melt(
        id_vars="customer_type",
        value_vars=[
            "customer_share_pct",
            "revenue_share_pct",
        ],
        var_name="Measure",
        value_name="Share (%)"
    )
)


customer_type_plot[
    "Measure"
] = (
    customer_type_plot[
        "Measure"
    ]
    .replace(
        {
            "customer_share_pct":
                "Customer Share",

            "revenue_share_pct":
                "Revenue Share",
        }
    )
)


plt.figure(
    figsize=(9, 6)
)


sns.barplot(
    data=customer_type_plot,
    x="customer_type",
    y="Share (%)",
    hue="Measure"
)


plt.title(
    "Customer Share vs Revenue Share"
)

plt.xlabel(
    "Customer Type"
)

plt.ylabel(
    "Share (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 11.22 — ACQUISITION YEAR
# ================================================================

customer_repeat[
    "acquisition_year"
] = (
    customer_repeat[
        "first_purchase_date"
    ]
    .dt.year
)

In [ ]:
# ================================================================
# 11.22B — 90-DAY REPEAT RATE BY ACQUISITION YEAR
# ================================================================

yearly_repeat_90 = (
    customer_repeat
    .loc[
        customer_repeat[
            "eligible_90d"
        ]
    ]
    .groupby(
        "acquisition_year",
        as_index=False
    )
    .agg(
        eligible_customers=(
            "customer",
            "nunique"
        ),

        repeat_customers=(
            "repeat_90d",
            "sum"
        ),
    )
)


yearly_repeat_90[
    "repeat_rate_90d_pct"
] = (
    yearly_repeat_90[
        "repeat_customers"
    ]
    / yearly_repeat_90[
        "eligible_customers"
    ]
    * 100
)


display(
    yearly_repeat_90
)

In [ ]:
# ================================================================
# VISUAL 11.5 — 90-DAY REPEAT RATE BY ACQUISITION YEAR
# ================================================================

plt.figure(
    figsize=(8, 6)
)


sns.barplot(
    data=yearly_repeat_90,
    x="acquisition_year",
    y="repeat_rate_90d_pct"
)


plt.title(
    "90-Day Repeat-Purchase Rate by Acquisition Year"
)

plt.xlabel(
    "Acquisition Year"
)

plt.ylabel(
    "90-Day Repeat-Purchase Rate (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 11.24 — EARLY REPEAT VS LONGER-TERM ORDER FREQUENCY
# ================================================================

customer_repeat[
    "repeat_speed_group"
] = np.select(
    [
        customer_repeat[
            "days_to_second_purchase"
        ].le(30),

        customer_repeat[
            "days_to_second_purchase"
        ].le(90),

        customer_repeat[
            "days_to_second_purchase"
        ].notna(),
    ],

    [
        "Repeated within 30 days",
        "Repeated in 31–90 days",
        "Repeated after 90 days",
    ],

    default="No observed repeat"
)


repeat_speed_frequency = (
    customer_repeat
    .groupby(
        "repeat_speed_group",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        median_orders=(
            "total_orders",
            "median"
        ),

        average_orders=(
            "total_orders",
            "mean"
        ),

        median_customer_revenue=(
            "total_purchase_revenue",
            "median"
        ),

        average_customer_revenue=(
            "total_purchase_revenue",
            "mean"
        ),
    )
)


display(
    repeat_speed_frequency
)

In [ ]:
# ================================================================
# 11.25 — VALIDATION CHECKS
# ================================================================

# Repeat-purchase rates must be cumulative.
repeat_rates = (
    repeat_purchase_summary[
        "repeat_rate_pct"
    ].values
)


assert all(
    earlier <= later
    for earlier, later
    in zip(
        repeat_rates[:-1],
        repeat_rates[1:]
    )
), (
    "Cumulative repeat-purchase rates "
    "should not decline as the window expands."
)


# Eligible population should decrease or stay constant
# as the required observation horizon gets longer.
eligible_counts = (
    repeat_purchase_summary[
        "eligible_customers"
    ].values
)


assert all(
    earlier >= later
    for earlier, later
    in zip(
        eligible_counts[:-1],
        eligible_counts[1:]
    )
), (
    "Longer repeat windows should not have "
    "more eligible customers."
)


# Repeat customer count cannot exceed eligible count.
assert (
    repeat_purchase_summary[
        "repeat_customers"
    ]
    <= repeat_purchase_summary[
        "eligible_customers"
    ]
).all()


print(
    "✓ Step 11 repeat-purchase validation checks passed."
)

In [ ]:
print(
    repeat_purchase_summary.columns.tolist()
)

display(
    repeat_purchase_summary
)

In [ ]:
# ================================================================
# 11.26 — STEP 11 SUMMARY METRICS
# ================================================================

step11_summary = (
    repeat_purchase_summary
    [
        [
            "window_days",
            "eligible_customers",
            "repeat_customers",
            "repeat_rate_pct",
            "non_repeat_customers",
        ]
    ]
    .copy()
)


step11_summary[
    "window"
] = (
    step11_summary[
        "window_days"
    ]
    .astype(str)
    + " days"
)


step11_summary = (
    step11_summary[
        [
            "window",
            "eligible_customers",
            "repeat_customers",
            "repeat_rate_pct",
            "non_repeat_customers",
        ]
    ]
)


display(
    step11_summary
)


print(
    f"Median time to second purchase: "
    f"{median_days_to_repeat:.1f} days"
)

print(
    f"Observed customers ever repeating: "
    f"{ever_repeat_rate:.2f}%"
)


### Step 11 Summary — Customer Retention and Repeat-Purchase Behaviour

Step 11 examined longitudinal purchasing behaviour among customers whose identities could be observed reliably across the transaction history.

The analysis was conducted at the **order level rather than the transaction-line level**, ensuring that an invoice containing multiple products was treated as one purchase occasion rather than several separate purchases.

Only identified customers were eligible for the analysis. In addition, customers first observed during the opening month of the dataset were excluded from acquisition-based retention calculations because their true first purchase may have occurred before the available transaction history began.

This produced an observable acquisition population of **4,823 customers** after excluding **1,029 potentially left-truncated customers**.

The analysis then measured whether customers completed a second purchase within 30, 60, 90 and 180 days of their first observable positive merchandise purchase.

#### Repeat purchasing strengthens materially as the observation window expands

Among customers with sufficient future observation time, approximately **20.96% repeated within 30 days** of their first purchase.

The cumulative repeat-purchase rate increased to approximately:

- **34.98% within 60 days**,
- **43.76% within 90 days**, and
- **58.03% within 180 days**.

The eligible customer population naturally becomes smaller as the observation window becomes longer. Approximately **4,662 customers** had enough history to be evaluated at 30 days, compared with **3,924 customers** at the 180-day horizon.

The results therefore show that repeat purchasing is not concentrated entirely in the first few weeks after acquisition. A meaningful proportion of customers return later in the relationship.

By 90 days, fewer than half of eligible customers had made a second purchase, but by 180 days this increased to almost **three in five customers**.

This suggests that the business has a meaningful repeat-purchase base, although a significant proportion of customers still remain single-purchase customers over the observable period.

#### The typical repeat customer returns in approximately two months

Among customers who eventually made another purchase, the **median time to second purchase was 59 days**.

The mean was substantially higher, at approximately **102 days**, indicating that the distribution is positively skewed by customers who take much longer to return.

Approximately:

- 25% of repeat customers returned within **23 days**,
- 50% returned within **59 days**,
- 75% returned within approximately **138 days**, and
- 90% returned within approximately **268 days**.

This suggests that the customer relationship should not be evaluated using only a short 30-day window.

A 30-day measure captures the fastest repeat purchasers but misses a large proportion of customers who return during the following two to six months.

For this business, the **90-day and 180-day horizons provide a more complete picture of customer retention behaviour** than the 30-day measure alone.

#### Approximately one-third of observable customers purchased only once

Across the observable acquisition population, **1,524 customers, or approximately 31.60%, made only one observed order**.

The remaining **3,299 customers, or approximately 68.40%, made at least one subsequent purchase** during the available history.

The customer-frequency distribution shows considerable variation beyond the initial repeat purchase:

- **31.60%** of customers placed only one order;
- **18.16%** placed two orders;
- **12.21%** placed three orders;
- **14.78%** placed four to five orders;
- **14.66%** placed six to ten orders; and
- **8.58%** placed eleven or more orders.

Therefore, the business is not composed solely of occasional repeat purchasers. A meaningful subset of customers developed into relatively high-frequency buyers.

Approximately **23.24% of observable customers placed six or more orders**, indicating the presence of a commercially important recurring customer segment.

#### Repeat customers generate the overwhelming majority of observed customer revenue

The commercial importance of repeat purchasing is considerably greater than the customer-count percentages alone suggest.

Repeat customers represented approximately **68.40% of observable customers**, but generated approximately **7.85 million**, or **93.60% of the purchase revenue generated by the observable acquisition population**.

In contrast, one-time customers represented approximately **31.60% of customers** but generated only approximately **536.05k**, or **6.40% of observed purchase revenue**.

This is one of the strongest customer findings in the notebook.

It demonstrates that the economic value of the identified customer base is overwhelmingly concentrated among customers who purchase more than once.

The distinction is substantial: repeat customers are not merely more numerous; they account for almost all of the revenue generated by this observable acquisition population.

This means that customer retention and repeat purchasing are central to revenue sustainability.

#### Faster repeat purchasing is associated with higher subsequent customer value

Customers who repeated quickly also displayed substantially greater observed purchasing activity over the remainder of the dataset.

Customers who repeated within **30 days** ultimately placed an average of approximately **7.41 orders**, compared with:

- approximately **6.49 orders** for customers whose second purchase occurred between 31 and 90 days;
- approximately **3.93 orders** for customers who repeated after 90 days; and
- exactly **1 order** for customers with no observed repeat.

The same pattern appears in customer revenue.

Customers who repeated within 30 days generated average observed purchase revenue of approximately **3.33k per customer**, compared with:

- approximately **2.36k** for customers repeating within 31–90 days;
- approximately **1.61k** for customers repeating after 90 days; and
- approximately **352** for customers with no observed repeat.

Median customer revenue shows the same direction.

These results indicate that rapid repeat purchasing is strongly **associated with** higher subsequent customer frequency and higher observed customer revenue.

However, this relationship should not be interpreted causally. Customers who repeat quickly may already represent inherently higher-value customers rather than early repeat purchasing itself causing the higher value.

#### Ninety-day repeat behaviour was broadly stable across acquisition years

A broad comparison of observable acquisition years showed little deterioration in 90-day repeat performance.

Customers acquired in **2010** recorded an approximately **43.56% 90-day repeat-purchase rate**, compared with approximately **44.46% for eligible customers acquired in 2011**.

The difference is less than one percentage point.

This suggests that, at an annual level, there was no evidence of a major deterioration in short-term repeat behaviour between the two acquisition years.

However, annual averages can conceal important differences between individual acquisition months. A more detailed cohort analysis is therefore required before concluding that retention quality was completely stable.

That analysis is addressed in the next section.

#### The timing profile shows that retention should not be judged too early

The cumulative pattern has an important operational implication.

Only approximately **21% of eligible customers repeat within 30 days**, yet the proportion rises to approximately **58% by 180 days**.

This means that classifying customers as “lost” shortly after their first purchase would materially overstate customer attrition.

A customer who has not returned within 30 or even 60 days may still return later in the purchasing cycle.

Accordingly, future customer-inactivity definitions should use empirically grounded thresholds rather than assuming that the absence of a purchase in the following month represents churn.

The distribution of time to second purchase suggests that a **90-day window is useful for early retention monitoring**, while **180 days provides a stronger measure of established repeat behaviour**.

#### Methodological interpretation

The retention results apply only to the **identified and observable customer population**.

As established in Step 10, a material proportion of merchandise revenue is generated without a customer identifier and therefore cannot participate in longitudinal customer analysis.

The results should consequently be reported as:

> repeat-purchase behaviour among identifiable customers with sufficient observable history

rather than as the retention rate of the entire business.

The analysis also measures **observed repeat purchasing**, not contractual retention. Because this is a transactional retail dataset, customers do not formally cancel or remain subscribed. A customer may simply purchase irregularly.

For that reason, terms such as "repeat purchase", "active", and "inactive" are more appropriate than treating every purchasing gap as conventional churn.

#### Overall conclusion

Step 11 establishes that the business possesses a **substantial and economically important repeat-customer base**.

Approximately **20.96% of eligible customers repeat within 30 days**, rising to **43.76% by 90 days and 58.03% by 180 days**.

The median returning customer makes a second purchase after approximately **59 days**, demonstrating that meaningful customer recurrence occurs beyond the first month following acquisition.

At the broader customer-history level, approximately **68.40% of observable customers eventually make another purchase**, while approximately **31.60% remain one-time buyers during the available history**.

Most importantly, repeat customers generate approximately **93.60% of observed purchase revenue**, despite representing only around two-thirds of observable customers.

The economic implication is therefore clear: the performance of the business is heavily dependent on converting first-time buyers into repeat purchasers and sustaining the behaviour of customers who already purchase repeatedly.

The next analytical question is whether this repeat behaviour is consistent across acquisition cohorts or whether certain periods produce materially stronger or weaker customer retention.

Step 12 therefore examines customer retention by acquisition cohort.



## Step 12 — Customer Cohort Retention Analysis

Step 12 evaluates repeat-purchase behaviour by customer acquisition cohort.

Customers are grouped according to the month of their first observable positive merchandise purchase. For each cohort, the analysis measures the proportion of customers who remain purchase-active in subsequent months.

The purpose is to determine whether customer quality and retention are stable across acquisition periods or whether particular cohorts perform materially better or worse.

Only identified customers with observable acquisition dates are included. Customers flagged as potentially left-truncated are excluded because their true acquisition date may pre-date the available transaction history.

Retention in this section refers to observed purchasing activity, not contractual retention. A customer is considered active in a given cohort month if they place at least one positive merchandise order during that month.

The analysis focuses on:

- cohort size,
- month-1, month-2, month-3 and later repeat activity,
- differences between early and later acquisition cohorts,
- and whether apparent retention deterioration is caused by insufficient observation time rather than genuine behavioural weakness.

In [ ]:
# ================================================================
# 12.2 — DEFINE CUSTOMER ACQUISITION COHORT MONTH
#        USING LAST COMPLETE CALENDAR MONTH
# ================================================================

# Last complete calendar month in the dataset.
# December 2011 is partial and should not be used for cohort retention.
cohort_analysis_end_month = pd.Period(
    "2011-11",
    freq="M"
)


observable_customer_cohorts = (
    customer_first_purchase
    .loc[
        ~customer_first_purchase[
            "customer_is_left_truncated"
        ]
    ]
    [
        [
            "customer",
            "first_purchase_date"
        ]
    ]
    .copy()
)


observable_customer_cohorts[
    "cohort_month"
] = (
    observable_customer_cohorts[
        "first_purchase_date"
    ]
    .dt.to_period("M")
)


# Exclude customers whose first observable purchase
# occurs after the last complete analysis month.
observable_customer_cohorts = (
    observable_customer_cohorts
    .loc[
        observable_customer_cohorts[
            "cohort_month"
        ] <= cohort_analysis_end_month
    ]
    .copy()
)


print(
    f"Last complete cohort-analysis month: "
    f"{cohort_analysis_end_month}"
)

print(
    f"Observable acquisition customers: "
    f"{len(observable_customer_cohorts):,}"
)

print(
    f"Acquisition cohorts: "
    f"{observable_customer_cohorts['cohort_month'].nunique():,}"
)

In [ ]:
# ================================================================
# 12.3 — ATTACH COHORT MONTH TO CUSTOMER ORDERS
#        AND EXCLUDE PARTIAL DECEMBER 2011 ACTIVITY
# ================================================================

cohort_orders = (
    customer_orders
    .merge(
        observable_customer_cohorts,
        on="customer",
        how="inner"
    )
)


cohort_orders[
    "order_month"
] = (
    cohort_orders[
        "order_date"
    ]
    .dt.to_period("M")
)


# Retain only complete observation months.
# December 2011 is excluded because the source ends on 9 December.
cohort_orders = (
    cohort_orders
    .loc[
        cohort_orders[
            "order_month"
        ] <= cohort_analysis_end_month
    ]
    .copy()
)


print(
    f"Orders retained through "
    f"{cohort_analysis_end_month}: "
    f"{len(cohort_orders):,}"
)

In [ ]:
# ================================================================
# 12.4 — CALCULATE MONTHS SINCE ACQUISITION
# ================================================================

cohort_orders[
    "cohort_age_month"
] = (
    (
        cohort_orders[
            "order_month"
        ].dt.year
        -
        cohort_orders[
            "cohort_month"
        ].dt.year
    )
    * 12
    +
    (
        cohort_orders[
            "order_month"
        ].dt.month
        -
        cohort_orders[
            "cohort_month"
        ].dt.month
    )
)


assert (
    cohort_orders[
        "cohort_age_month"
    ] >= 0
).all()

In [ ]:
# ================================================================
# 12.5 — ACTIVE CUSTOMERS BY COHORT AGE
# ================================================================

cohort_activity = (
    cohort_orders
    [
        [
            "customer",
            "cohort_month",
            "cohort_age_month"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "cohort_month",
            "cohort_age_month"
        ],
        as_index=False
    )
    .agg(
        active_customers=(
            "customer",
            "nunique"
        )
    )
)


display(
    cohort_activity.head()
)

In [ ]:
# ================================================================
# 12.6 — CALCULATE COHORT SIZE
# ================================================================

cohort_sizes = (
    cohort_activity
    .loc[
        cohort_activity[
            "cohort_age_month"
        ] == 0
    ]
    [
        [
            "cohort_month",
            "active_customers"
        ]
    ]
    .rename(
        columns={
            "active_customers":
                "cohort_size"
        }
    )
)


display(
    cohort_sizes.head()
)

In [ ]:
# ================================================================
# 12.7 — CALCULATE COHORT RETENTION
# ================================================================

cohort_retention = (
    cohort_activity
    .merge(
        cohort_sizes,
        on="cohort_month",
        how="left"
    )
)


cohort_retention[
    "retention_rate"
] = (
    cohort_retention[
        "active_customers"
    ]
    / cohort_retention[
        "cohort_size"
    ]
)


cohort_retention[
    "retention_rate_pct"
] = (
    cohort_retention[
        "retention_rate"
    ]
    * 100
)

In [ ]:
# ================================================================
# 12.8 — BUILD COHORT RETENTION MATRIX
# ================================================================

cohort_retention_matrix = (
    cohort_retention
    .pivot(
        index="cohort_month",
        columns="cohort_age_month",
        values="retention_rate"
    )
)


display(
    cohort_retention_matrix.head()
)

In [ ]:
# ================================================================
# VISUAL 12.1 — CUSTOMER COHORT RETENTION HEATMAP
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


# Limit the first view to 12 months after acquisition
# so the heatmap remains readable.
heatmap_months = list(
    range(
        0,
        min(
            13,
            cohort_retention_matrix.shape[1]
        )
    )
)


cohort_heatmap = (
    cohort_retention_matrix[
        [
            col
            for col in heatmap_months
            if col in cohort_retention_matrix.columns
        ]
    ]
    .copy()
)


plt.figure(
    figsize=(14, 10)
)


sns.heatmap(
    cohort_heatmap,
    annot=True,
    fmt=".0%",
    linewidths=0.5,
    cbar_kws={
        "label":
            "Active Customer Rate"
    }
)


plt.title(
    "Customer Cohort Retention by Month Since Acquisition"
)

plt.xlabel(
    "Months Since First Purchase"
)

plt.ylabel(
    "Acquisition Cohort"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 12.9 — KEY MONTHLY RETENTION CHECKPOINTS
# ================================================================

retention_checkpoints = [
    1,
    2,
    3,
    6,
]


cohort_checkpoint_table = (
    cohort_retention
    .loc[
        cohort_retention[
            "cohort_age_month"
        ].isin(
            retention_checkpoints
        )
    ]
    .pivot(
        index="cohort_month",
        columns="cohort_age_month",
        values="retention_rate_pct"
    )
    .rename(
        columns={
            1: "month_1_retention_pct",
            2: "month_2_retention_pct",
            3: "month_3_retention_pct",
            6: "month_6_retention_pct",
        }
    )
)


display(
    cohort_checkpoint_table
)

In [ ]:
# ================================================================
# 12.10 — CHECK COHORT ELIGIBILITY BY AGE
# ================================================================

cohort_eligibility = (
    cohort_sizes[
        [
            "cohort_month",
            "cohort_size"
        ]
    ]
    .copy()
)


cohort_eligibility[
    "observable_months"
] = (
    (
        cohort_analysis_end_month.year
        -
        cohort_eligibility[
            "cohort_month"
        ].dt.year
    )
    * 12
    +
    (
        cohort_analysis_end_month.month
        -
        cohort_eligibility[
            "cohort_month"
        ].dt.month
    )
)


display(
    cohort_eligibility.tail()
)

In [ ]:
# ================================================================
# 12.11 — WEIGHTED COHORT RETENTION AT KEY AGES
# ================================================================

weighted_cohort_retention_records = []


for month_age in [
    1,
    2,
    3,
    6,
    12,
]:

    eligible_cohorts = (
        cohort_eligibility
        .loc[
            cohort_eligibility[
                "observable_months"
            ] >= month_age
        ]
        [
            "cohort_month"
        ]
    )


    eligible_activity = (
        cohort_retention
        .loc[
            cohort_retention[
                "cohort_month"
            ].isin(
                eligible_cohorts
            )
            &
            (
                cohort_retention[
                    "cohort_age_month"
                ] == month_age
            )
        ]
    )


    total_active = (
        eligible_activity[
            "active_customers"
        ]
        .sum()
    )


    eligible_cohort_sizes = (
        cohort_sizes
        .loc[
            cohort_sizes[
                "cohort_month"
            ].isin(
                eligible_cohorts
            )
        ][
            "cohort_size"
        ]
        .sum()
    )


    weighted_retention = (
        total_active
        / eligible_cohort_sizes
        * 100
        if eligible_cohort_sizes > 0
        else np.nan
    )


    weighted_cohort_retention_records.append(
        {
            "cohort_age_month":
                month_age,

            "eligible_customers":
                eligible_cohort_sizes,

            "active_customers":
                total_active,

            "weighted_retention_pct":
                weighted_retention,
        }
    )


weighted_cohort_retention = (
    pd.DataFrame(
        weighted_cohort_retention_records
    )
)


display(
    weighted_cohort_retention
)

In [ ]:
# ================================================================
# VISUAL 12.2 — WEIGHTED MONTHLY COHORT RETENTION
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.lineplot(
    data=weighted_cohort_retention,
    x="cohort_age_month",
    y="weighted_retention_pct",
    marker="o"
)


for _, row in weighted_cohort_retention.iterrows():

    plt.text(
        row["cohort_age_month"],
        row["weighted_retention_pct"],
        f"{row['weighted_retention_pct']:.1f}%",
        ha="center",
        va="bottom"
    )


plt.title(
    "Weighted Customer Retention by Months Since Acquisition"
)

plt.xlabel(
    "Months Since Acquisition"
)

plt.ylabel(
    "Active Customers (%)"
)

plt.xticks(
    weighted_cohort_retention[
        "cohort_age_month"
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 12.12 — MONTH-3 RETENTION BY ACQUISITION COHORT
# ================================================================

month3_cohort_retention = (
    cohort_retention
    .loc[
        cohort_retention[
            "cohort_age_month"
        ] == 3
    ]
    [
        [
            "cohort_month",
            "cohort_size",
            "active_customers",
            "retention_rate_pct",
        ]
    ]
    .copy()
)


month3_cohort_retention[
    "cohort_month_date"
] = (
    month3_cohort_retention[
        "cohort_month"
    ]
    .dt.to_timestamp()
)


display(
    month3_cohort_retention
)

In [ ]:
# ================================================================
# VISUAL 12.3 — MONTH-3 RETENTION BY ACQUISITION COHORT
# ================================================================

plt.figure(
    figsize=(12, 6)
)


sns.lineplot(
    data=month3_cohort_retention,
    x="cohort_month_date",
    y="retention_rate_pct",
    marker="o"
)


plt.title(
    "Month-3 Retention by Acquisition Cohort"
)

plt.xlabel(
    "Acquisition Cohort"
)

plt.ylabel(
    "Month-3 Active Customer Rate (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 12.13 — ACQUISITION COHORT SIZE
# ================================================================

cohort_size_table = (
    cohort_sizes
    .copy()
)


cohort_size_table[
    "cohort_month_date"
] = (
    cohort_size_table[
        "cohort_month"
    ]
    .dt.to_timestamp()
)


display(
    cohort_size_table
)

In [ ]:
# ================================================================
# VISUAL 12.4 — CUSTOMER ACQUISITION VOLUME
# ================================================================

plt.figure(
    figsize=(12, 6)
)


sns.barplot(
    data=cohort_size_table,
    x="cohort_month_date",
    y="cohort_size"
)


plt.title(
    "Observable Customer Acquisition by Month"
)

plt.xlabel(
    "Acquisition Month"
)

plt.ylabel(
    "New Observable Customers"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 12.14 — STRONGEST AND WEAKEST MATURE COHORTS
# ================================================================

minimum_cohort_size = 50


meaningful_month3_cohorts = (
    month3_cohort_retention
    .loc[
        month3_cohort_retention[
            "cohort_size"
        ] >= minimum_cohort_size
    ]
    .copy()
)


strongest_month3_cohorts = (
    meaningful_month3_cohorts
    .sort_values(
        "retention_rate_pct",
        ascending=False
    )
    .head(5)
)


weakest_month3_cohorts = (
    meaningful_month3_cohorts
    .sort_values(
        "retention_rate_pct",
        ascending=True
    )
    .head(5)
)


print(
    "Strongest Month-3 cohorts:"
)

display(
    strongest_month3_cohorts
)


print(
    "Weakest Month-3 cohorts:"
)

display(
    weakest_month3_cohorts
)

In [ ]:
# ================================================================
# 12.15 — COHORT RETENTION BY ACQUISITION YEAR
# ================================================================

month3_cohort_retention[
    "acquisition_year"
] = (
    month3_cohort_retention[
        "cohort_month_date"
    ]
    .dt.year
)


yearly_month3_retention = (
    month3_cohort_retention
    .groupby(
        "acquisition_year",
        as_index=False
    )
    .apply(
        lambda group: pd.Series(
            {
                "eligible_customers":
                    group[
                        "cohort_size"
                    ].sum(),

                "active_customers":
                    group[
                        "active_customers"
                    ].sum(),

                "weighted_month3_retention_pct":
                    (
                        group[
                            "active_customers"
                        ].sum()
                        /
                        group[
                            "cohort_size"
                        ].sum()
                        * 100
                    ),
            }
        ),
        include_groups=False
    )
    .reset_index()
)


display(
    yearly_month3_retention
)

In [ ]:
yearly_month3_retention = (
    month3_cohort_retention
    .groupby(
        "acquisition_year"
    )
    .agg(
        eligible_customers=(
            "cohort_size",
            "sum"
        ),
        active_customers=(
            "active_customers",
            "sum"
        ),
    )
    .reset_index()
)


yearly_month3_retention[
    "weighted_month3_retention_pct"
] = (
    yearly_month3_retention[
        "active_customers"
    ]
    / yearly_month3_retention[
        "eligible_customers"
    ]
    * 100
)

In [ ]:
# ================================================================
# 12.16 — CUSTOMER REVENUE BY ACQUISITION COHORT
# ================================================================

customer_total_revenue = (
    customer_orders
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        total_customer_revenue=(
            "order_revenue",
            "sum"
        )
    )
)


cohort_customer_revenue = (
    observable_customer_cohorts
    .merge(
        customer_total_revenue,
        on="customer",
        how="left"
    )
)


cohort_revenue_summary = (
    cohort_customer_revenue
    .groupby(
        "cohort_month",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        total_revenue=(
            "total_customer_revenue",
            "sum"
        ),

        average_customer_revenue=(
            "total_customer_revenue",
            "mean"
        ),

        median_customer_revenue=(
            "total_customer_revenue",
            "median"
        ),
    )
)


display(
    cohort_revenue_summary
)

In [ ]:
# ================================================================
# 12.17 — RETENTION VS CUSTOMER REVENUE
# ================================================================

cohort_value_relationship = (
    month3_cohort_retention
    .merge(
        cohort_revenue_summary,
        on="cohort_month",
        how="left"
    )
)


display(
    cohort_value_relationship[
        [
            "cohort_month",
            "cohort_size",
            "retention_rate_pct",
            "average_customer_revenue",
            "median_customer_revenue",
        ]
    ]
)

In [ ]:
# ================================================================
# VISUAL 12.5 — COHORT RETENTION VS CUSTOMER VALUE
# ================================================================

plt.figure(
    figsize=(10, 7)
)


sns.scatterplot(
    data=cohort_value_relationship,
    x="retention_rate_pct",
    y="average_customer_revenue",
    size="cohort_size",
    sizes=(40, 350),
    alpha=0.75
)


plt.title(
    "Month-3 Cohort Retention vs Average Customer Revenue"
)

plt.xlabel(
    "Month-3 Retention Rate (%)"
)

plt.ylabel(
    "Average Observed Customer Revenue"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 12.18 — VALIDATION CHECKS
# ================================================================

# Month 0 should always be 100%.
month_zero = (
    cohort_retention
    .loc[
        cohort_retention[
            "cohort_age_month"
        ] == 0,
        "retention_rate"
    ]
)


assert np.allclose(
    month_zero,
    1.0
), (
    "Every cohort should have 100% activity "
    "in its acquisition month."
)


# Active customers cannot exceed cohort size.
assert (
    cohort_retention[
        "active_customers"
    ]
    <= cohort_retention[
        "cohort_size"
    ]
).all()


# Retention rates must remain between 0 and 100%.
assert (
    cohort_retention[
        "retention_rate_pct"
    ]
    .between(
        0,
        100
    )
).all()


print(
    "✓ Step 12 cohort validation checks passed."
)

In [ ]:
# ================================================================
# 12.19 — STEP 12 SUMMARY METRICS
# ================================================================

step12_summary = (
    weighted_cohort_retention
    .copy()
)


step12_summary[
    "cohort_age"
] = (
    "Month "
    + step12_summary[
        "cohort_age_month"
    ]
    .astype(str)
)


step12_summary = (
    step12_summary[
        [
            "cohort_age",
            "eligible_customers",
            "active_customers",
            "weighted_retention_pct",
        ]
    ]
)


display(
    step12_summary
)

### Step 12 Summary — Customer Cohort Retention Analysis

Step 12 examined customer retention by acquisition cohort to determine whether customers acquired in different periods displayed materially different post-acquisition purchasing behaviour.

Customers were grouped according to the month of their first observable positive merchandise purchase. As in Step 11, customers flagged as potentially left-truncated were excluded because their true acquisition date may pre-date the available transaction history.

Monthly cohort retention was defined as the proportion of each original acquisition cohort that placed at least one positive merchandise order in a specified month after acquisition.

To avoid right-censoring and incomplete-period bias, **November 2011 was used as the final complete observation month**. Transactions from December 2011 were excluded from cohort-retention calculations because the source data ends on 9 December 2011 and therefore does not represent a complete calendar month.

This correction is important because including partial December activity would systematically understate retention among the most recent cohorts.

#### The cohort analysis covers a broad but uneven acquisition history

After excluding potentially left-truncated customers and restricting the analysis to complete observation months, the cohort analysis covers observable acquisitions from **January 2010 through November 2011**.

Customer acquisition volume varied materially across these months.

Approximately **3,290 observable customers were acquired during 2010**, equivalent to an average of roughly **274 customers per monthly cohort**.

Across the first eleven complete months of 2011, approximately **1,505 observable customers were acquired**, or roughly **137 customers per cohort**.

This indicates that acquisition volume was materially lower in 2011 than in 2010.

The largest observable acquisition cohort was **March 2010**, with approximately **426 customers**, followed by **October 2010 with approximately 373 customers** and **February 2010 with approximately 359 customers**.

This variation in cohort size is important when interpreting retention performance. A very small cohort can produce an unusually high or low retention percentage from only a few customers, while large cohorts provide more stable evidence.

#### Monthly cohort retention settles at approximately one-fifth of the original cohort

When customer activity is weighted by original cohort size, approximately **20.89% of eligible customers were active in Month 1** following acquisition.

The weighted activity rate was approximately:

- **20.89% in Month 1**
- **21.47% in Month 2**
- **20.90% in Month 3**
- **18.22% in Month 6**
- **18.23% in Month 12**

These figures should not be confused with the cumulative repeat-purchase rates calculated in Step 11.

Step 11 asks:

> Has the customer made at least one repeat purchase by a specified number of days?

Step 12 instead asks:

> Is the customer actively purchasing in this specific month after acquisition?

A customer may therefore purchase in Month 1, skip Months 2 and 3, and return in Month 4. That customer would be counted as active in Months 1 and 4 but inactive in Months 2 and 3.

For this reason, cohort-retention percentages are not expected to increase cumulatively and should not be interpreted as a conventional survival curve.

The results suggest that approximately **one-fifth of each original acquisition cohort remains purchase-active during a typical early post-acquisition month**, with activity declining only moderately at longer horizons.

The persistence of approximately **18% activity at Months 6 and 12** indicates that a meaningful minority of customers continue purchasing well beyond the initial acquisition period.

#### Retention performance varies substantially between acquisition cohorts

Although the weighted Month-3 retention rate was approximately **20.90% overall**, individual cohorts performed very differently.

Among cohorts with at least 50 customers, the strongest Month-3 retention rates were approximately:

- **August 2010 — 32.72%**
- **January 2010 — 31.71%**
- **July 2010 — 28.96%**
- **February 2010 — 28.69%**
- **July 2011 — 27.72%**

By comparison, the weakest meaningful Month-3 cohorts included:

- **December 2010 — 9.21%**
- **November 2010 — 9.57%**
- **October 2010 — 12.60%**
- **September 2010 — 13.03%**
- **April 2010 — 15.79%**

The spread between the strongest and weakest cohorts is therefore substantial.

The best-performing cohorts retained more than **30% of customers at Month 3**, while the weakest meaningful cohorts retained fewer than **10%**.

This demonstrates that the overall retention average conceals significant variation in the quality and subsequent behaviour of customers acquired in different periods.

#### A clear retention deterioration appears among late-2010 cohorts

The most pronounced pattern in the cohort analysis occurs among customers acquired during the later months of 2010.

Month-3 retention declined sharply from approximately:

- **32.72% for the August 2010 cohort**
to
- **13.03% for September 2010**
- **12.60% for October 2010**
- **9.57% for November 2010**
- **9.21% for December 2010**

This represents a major weakening in post-acquisition customer activity over a relatively short period.

The decline is particularly important because it was not limited to very small cohorts.

For example, the **October 2010 cohort contained approximately 373 customers**, yet only about **12.6% were active three months later**.

Similarly, the **November 2010 cohort contained approximately 324 customers**, but Month-3 activity was only approximately **9.6%**.

The retention weakness therefore affected economically meaningful customer populations rather than being an artefact of small sample sizes.

#### Late-2010 weakness should not automatically be interpreted as permanent customer-quality deterioration

The cohort analysis establishes that retention weakened among late-2010 acquisitions, but it does not establish why.

Several plausible commercial explanations could produce the same pattern, including:

- seasonal or Christmas-related one-off purchasing;
- promotional acquisition campaigns;
- changes in customer mix;
- shifts in product assortment;
- changes in first-order composition;
- unusually high gifting activity;
- or a greater proportion of customers purchasing for a specific seasonal occasion.

The evidence therefore supports the conclusion that **late-2010 acquisition cohorts displayed materially weaker Month-3 retention**, but it does not support attributing that weakness to any one cause without additional analysis.

This distinction is important because the November comparison occurs during a highly seasonal trading period.

#### Retention quality recovered during 2011

The sharp late-2010 deterioration did not continue throughout 2011.

Month-3 retention among several 2011 cohorts returned to the high-teens and mid-twenties.

Examples include approximately:

- **January 2011 — 19.44%**
- **March 2011 — 19.66%**
- **April 2011 — 20.00%**
- **June 2011 — 26.85%**
- **July 2011 — 27.72%**
- **August 2011 — 26.17%**

This recovery suggests that the weakness observed in late 2010 was concentrated within a particular acquisition period rather than representing a permanent deterioration in the customer base.

The recovery is particularly important because it prevents the analysis from concluding incorrectly that retention was progressively worsening over time.

#### Overall 2011 cohort quality was broadly comparable with 2010

When Month-3 retention is weighted by cohort size, eligible 2010 cohorts recorded an overall Month-3 activity rate of approximately **20.70%**.

Eligible 2011 cohorts recorded approximately **21.63%**.

The difference is less than one percentage point.

Therefore, despite the severe weakness of several late-2010 cohorts, the broader year-level evidence does not indicate that customers acquired in 2011 were systematically lower quality than those acquired in 2010.

If anything, weighted Month-3 retention was slightly stronger in 2011.

This finding is consistent with Step 11, where 90-day cumulative repeat-purchase behaviour was also broadly stable between the two acquisition years.

The evidence therefore points toward **period-specific cohort weakness rather than structural deterioration in customer retention quality**.

#### Acquisition volume and retention quality appear to be separate dimensions

The analysis does not indicate a meaningful relationship between cohort size and Month-3 retention.

The correlation between acquisition cohort size and Month-3 retention was approximately **-0.01**, effectively indicating no meaningful linear relationship.

Large acquisition cohorts were therefore not systematically associated with weak retention, while small cohorts were not systematically associated with stronger retention.

This is analytically important because it suggests that changes in acquisition volume should not automatically be interpreted as changes in customer quality.

The lower acquisition volume observed in 2011 did not appear to have been purchased at the cost of weaker retention.

Management should therefore track acquisition scale and retention performance independently.

#### Cohort analysis reveals information that the overall repeat rate cannot

Step 11 established that a substantial proportion of customers eventually purchase again.

Step 12 shows that this behaviour is not evenly distributed across acquisition periods.

Two acquisition months can produce similar numbers of new customers but materially different levels of subsequent activity.

For that reason, a single overall retention percentage is insufficient for managing customer quality.

A more effective customer-performance framework should monitor at least:

- acquisition cohort size;
- Month-1 activity;
- Month-3 activity;
- Month-6 activity;
- cumulative repeat purchase;
- and revenue generated by each cohort.

This allows management to distinguish between:

1. an acquisition-volume problem;
2. a post-acquisition retention problem;
3. or a combination of both.

#### The late-2010 cohorts deserve targeted investigation

The clearest analytical follow-up from Step 12 is the weakness of cohorts acquired from approximately **September through December 2010**.

These cohorts should be investigated against variables such as:

- first-order value;
- products purchased on the first order;
- country;
- basket size;
- seasonal product exposure;
- acquisition month;
- subsequent purchase timing;
- and one-time versus repeat-customer behaviour.

If a large proportion of these customers entered through seasonal products or Christmas-related purchases, weaker Month-3 retention may reflect the nature of the acquisition occasion rather than poor customer experience.

That question should therefore remain open until the subsequent customer and seasonal analyses are completed.

#### Methodological interpretation

Several methodological boundaries are important when interpreting these findings.

First, cohort retention represents **observed purchasing activity**, not formal contractual retention.

A customer who does not transact during a particular month is classified as inactive for that month, but this does not prove that the customer has permanently left the business.

Second, cohort retention is not cumulative.

A customer can move in and out of monthly activity over time.

Third, right-censoring has been controlled by evaluating each cohort only where sufficient complete observation history exists.

The incomplete December 2011 period was excluded so that recently acquired customers are not unfairly classified as inactive simply because the dataset ends before a full month can be observed.

Finally, as established in Step 10, the analysis applies only to customers with persistent identifiers. Unidentified transactions cannot be assigned to acquisition cohorts and are therefore outside the observable retention population.

#### Overall conclusion

Step 12 shows that the business has a meaningful recurring customer base, but **retention quality varies significantly according to when customers were acquired**.

Portfolio-wide monthly activity stabilises at approximately **20% during the first three post-acquisition months**, before settling at approximately **18% at longer horizons**.

However, individual cohorts vary considerably around these averages.

The strongest Month-3 cohorts retained more than **30% of their original customers**, while the weakest meaningful cohorts retained fewer than **10%**.

The most significant weakness occurred among customers acquired between approximately **September and December 2010**, where Month-3 retention fell sharply.

Importantly, this deterioration was not permanent.

Retention quality recovered across several 2011 cohorts, and weighted Month-3 retention was broadly stable between years, increasing slightly from approximately **20.70% in 2010 to 21.63% in 2011**.

The evidence therefore does not support a conclusion that customer retention was structurally deteriorating.

Instead, it indicates that **specific acquisition periods produced materially weaker cohorts**, particularly in late 2010.

This distinction has important management implications. The business should not treat customer acquisition volume alone as evidence of growth quality. Acquisition should be evaluated together with the subsequent retention behaviour of each cohort.

The next analytical question is whether customers who stop purchasing are genuinely lost, temporarily inactive, or likely to return after longer purchasing gaps.

Step 13 therefore examines **customer inactivity and churn-style behaviour using empirically observed purchase-gap patterns rather than assuming that any short period without a transaction represents churn**.



## Step 13 — Customer Inactivity and Churn-Style Behaviour

Step 13 examines periods of customer inactivity and evaluates whether customers who stop purchasing appear permanently lost or frequently return after extended gaps.

Because this is a transactional retail dataset rather than a subscription business, the absence of a purchase does not constitute formal churn. Customers may purchase irregularly and return after long periods of inactivity.

The analysis therefore avoids assigning a permanent churn label. Instead, customers are classified according to the time elapsed since their most recent positive merchandise purchase.

Drawing on the repeat-purchase patterns identified in Step 11, the analysis uses two practical inactivity thresholds:

- **90 days** — early inactivity / at-risk threshold
- **180 days** — extended inactivity threshold

These thresholds are analytical conventions rather than proof of customer loss.

The analysis investigates:

- the distribution of gaps between purchases,
- the number of customers currently active or inactive,
- the historical economic contribution of inactive customers,
- the frequency with which customers return after 90- and 180-day gaps,
- and whether long inactivity should reasonably be interpreted as permanent churn.

The final complete observation date is November 30, 2011 so that the partial December 2011 trading period does not distort customer activity status.

In [ ]:
# ================================================================
# 13.2 — DEFINE FINAL COMPLETE OBSERVATION DATE
# ================================================================

inactivity_analysis_end_date = pd.Timestamp(
    "2011-11-30"
)


print(
    f"Inactivity analysis end date: "
    f"{inactivity_analysis_end_date:%d %B %Y}"
)

In [ ]:
# ================================================================
# 13.3 — CUSTOMER ORDERS THROUGH FINAL COMPLETE PERIOD
# ================================================================

customer_orders_complete = (
    customer_orders
    .loc[
        customer_orders[
            "order_date"
        ] <= inactivity_analysis_end_date
    ]
    .copy()
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice"
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Orders included in inactivity analysis: "
    f"{len(customer_orders_complete):,}"
)

print(
    f"Identified customers included: "
    f"{customer_orders_complete['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 13.4 — CALCULATE INTERPURCHASE GAPS
# ================================================================

customer_orders_complete[
    "previous_order_date"
] = (
    customer_orders_complete
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .shift(1)
)


customer_orders_complete[
    "days_since_previous_order"
] = (
    customer_orders_complete[
        "order_date"
    ]
    -
    customer_orders_complete[
        "previous_order_date"
    ]
).dt.days


interpurchase_gaps = (
    customer_orders_complete
    .loc[
        customer_orders_complete[
            "days_since_previous_order"
        ].notna()
    ]
    .copy()
)


display(
    interpurchase_gaps[
        [
            "customer",
            "order_date",
            "previous_order_date",
            "days_since_previous_order",
        ]
    ]
    .head()
)

In [ ]:
# ================================================================
# 13.5 — INTERPURCHASE GAP SUMMARY
# ================================================================

gap_summary = (
    interpurchase_gaps[
        "days_since_previous_order"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)


display(
    gap_summary
)

In [ ]:
# ================================================================
# VISUAL 13.1 — INTERPURCHASE GAP DISTRIBUTION
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


gap_plot_data = (
    interpurchase_gaps
    .loc[
        interpurchase_gaps[
            "days_since_previous_order"
        ] <= 365
    ]
)


plt.figure(
    figsize=(11, 6)
)


sns.histplot(
    data=gap_plot_data,
    x="days_since_previous_order",
    bins=45,
    kde=True
)


plt.axvline(
    90,
    linestyle="--",
    linewidth=1.5,
    label="90-day inactivity threshold"
)


plt.axvline(
    180,
    linestyle="--",
    linewidth=1.5,
    label="180-day extended inactivity threshold"
)


plt.title(
    "Distribution of Time Between Customer Purchases"
)

plt.xlabel(
    "Days Between Consecutive Orders"
)

plt.ylabel(
    "Observed Purchase Intervals"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 13.6 — CUSTOMER LAST PURCHASE DATE
# ================================================================

customer_last_purchase = (
    customer_orders_complete
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_purchase_date=(
            "order_date",
            "min"
        ),

        last_purchase_date=(
            "order_date",
            "max"
        ),

        total_orders=(
            "invoice",
            "nunique"
        ),

        historical_revenue=(
            "order_revenue",
            "sum"
        ),
    )
)

In [ ]:
# ================================================================
# 13.7 — DAYS SINCE LAST PURCHASE
# ================================================================

customer_last_purchase[
    "days_since_last_purchase"
] = (
    inactivity_analysis_end_date
    -
    customer_last_purchase[
        "last_purchase_date"
    ]
).dt.days


display(
    customer_last_purchase.head()
)

In [ ]:
# ================================================================
# 13.8 — CUSTOMER INACTIVITY SEGMENTS
# ================================================================

customer_last_purchase[
    "inactivity_segment"
] = pd.cut(
    customer_last_purchase[
        "days_since_last_purchase"
    ],
    bins=[
        -1,
        90,
        180,
        np.inf,
    ],
    labels=[
        "Active — ≤90 days",
        "Recently inactive — 91–180 days",
        "Long inactive — >180 days",
    ]
)

In [ ]:
# ================================================================
# 13.9 — INACTIVITY SEGMENT SUMMARY
# ================================================================

inactivity_summary = (
    customer_last_purchase
    .groupby(
        "inactivity_segment",
        observed=True,
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        historical_revenue=(
            "historical_revenue",
            "sum"
        ),

        median_orders=(
            "total_orders",
            "median"
        ),

        average_orders=(
            "total_orders",
            "mean"
        ),

        median_days_since_purchase=(
            "days_since_last_purchase",
            "median"
        ),
    )
)


inactivity_summary[
    "customer_share_pct"
] = (
    inactivity_summary[
        "customers"
    ]
    / inactivity_summary[
        "customers"
    ].sum()
    * 100
)


inactivity_summary[
    "historical_revenue_share_pct"
] = (
    inactivity_summary[
        "historical_revenue"
    ]
    / inactivity_summary[
        "historical_revenue"
    ].sum()
    * 100
)


display(
    inactivity_summary
)

In [ ]:
# ================================================================
# VISUAL 13.2 — CUSTOMER INACTIVITY STATUS
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=inactivity_summary,
    x="inactivity_segment",
    y="customer_share_pct"
)


plt.title(
    "Customer Base by Current Inactivity Status"
)

plt.xlabel(
    "Customer Status"
)

plt.ylabel(
    "Share of Identified Customers (%)"
)

plt.xticks(
    rotation=15
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 13.10 — CUSTOMER SHARE VS HISTORICAL REVENUE SHARE
# ================================================================

inactivity_value_plot = (
    inactivity_summary
    .melt(
        id_vars="inactivity_segment",
        value_vars=[
            "customer_share_pct",
            "historical_revenue_share_pct",
        ],
        var_name="Measure",
        value_name="Share (%)"
    )
)


inactivity_value_plot[
    "Measure"
] = (
    inactivity_value_plot[
        "Measure"
    ]
    .replace(
        {
            "customer_share_pct":
                "Customer Share",

            "historical_revenue_share_pct":
                "Historical Revenue Share",
        }
    )
)

In [ ]:
# ================================================================
# VISUAL 13.3 — INACTIVITY CUSTOMER VS REVENUE SHARE
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=inactivity_value_plot,
    x="inactivity_segment",
    y="Share (%)",
    hue="Measure"
)


plt.title(
    "Customer Share vs Historical Revenue Share "
    "by Inactivity Status"
)

plt.xlabel(
    "Customer Status"
)

plt.ylabel(
    "Share (%)"
)

plt.xticks(
    rotation=15
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 13.11 — NEXT PURCHASE DATE AND GAP
# ================================================================

customer_orders_complete[
    "next_order_date"
] = (
    customer_orders_complete
    .groupby(
        "customer"
    )[
        "order_date"
    ]
    .shift(-1)
)


customer_orders_complete[
    "days_to_next_order"
] = (
    customer_orders_complete[
        "next_order_date"
    ]
    -
    customer_orders_complete[
        "order_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 13.12 — 90-DAY INACTIVITY EPISODES
# ================================================================

customer_orders_complete[
    "had_90d_observation"
] = (
    customer_orders_complete[
        "order_date"
    ]
    + pd.Timedelta(
        days=90
    )
    <= inactivity_analysis_end_date
)


customer_orders_complete[
    "reached_90d_inactivity"
] = (
    customer_orders_complete[
        "had_90d_observation"
    ]
    &
    (
        customer_orders_complete[
            "next_order_date"
        ].isna()
        |
        (
            customer_orders_complete[
                "days_to_next_order"
            ] > 90
        )
    )
)


customer_orders_complete[
    "reactivated_after_90d"
] = (
    customer_orders_complete[
        "reached_90d_inactivity"
    ]
    &
    customer_orders_complete[
        "next_order_date"
    ].notna()
)

In [ ]:
# ================================================================
# 13.13 — 180-DAY INACTIVITY EPISODES
# ================================================================

customer_orders_complete[
    "had_180d_observation"
] = (
    customer_orders_complete[
        "order_date"
    ]
    + pd.Timedelta(
        days=180
    )
    <= inactivity_analysis_end_date
)


customer_orders_complete[
    "reached_180d_inactivity"
] = (
    customer_orders_complete[
        "had_180d_observation"
    ]
    &
    (
        customer_orders_complete[
            "next_order_date"
        ].isna()
        |
        (
            customer_orders_complete[
                "days_to_next_order"
            ] > 180
        )
    )
)


customer_orders_complete[
    "reactivated_after_180d"
] = (
    customer_orders_complete[
        "reached_180d_inactivity"
    ]
    &
    customer_orders_complete[
        "next_order_date"
    ].notna()
)

In [ ]:
# ================================================================
# 13.14 — REACTIVATION AFTER EXTENDED INACTIVITY
# ================================================================

reactivation_records = []


for days in [
    90,
    180,
]:

    inactive_flag = (
        f"reached_{days}d_inactivity"
    )

    reactivated_flag = (
        f"reactivated_after_{days}d"
    )


    inactivity_episodes = (
        customer_orders_complete[
            inactive_flag
        ].sum()
    )


    reactivated_episodes = (
        customer_orders_complete[
            reactivated_flag
        ].sum()
    )


    reactivation_rate = (
        reactivated_episodes
        / inactivity_episodes
        * 100
        if inactivity_episodes > 0
        else np.nan
    )


    reactivation_records.append(
        {
            "inactivity_threshold_days":
                days,

            "inactivity_episodes":
                inactivity_episodes,

            "reactivated_episodes":
                reactivated_episodes,

            "reactivation_rate_pct":
                reactivation_rate,
        }
    )


reactivation_summary = pd.DataFrame(
    reactivation_records
)


display(
    reactivation_summary
)

In [ ]:
# ================================================================
# 13.15 — UNIQUE CUSTOMERS REACTIVATING AFTER LONG GAPS
# ================================================================

unique_reactivation_summary = pd.DataFrame(
    {
        "Metric": [
            "Customers ever returning after >90-day gap",
            "Customers ever returning after >180-day gap",
        ],

        "Customers": [
            customer_orders_complete.loc[
                customer_orders_complete[
                    "days_to_next_order"
                ] > 90,
                "customer"
            ].nunique(),

            customer_orders_complete.loc[
                customer_orders_complete[
                    "days_to_next_order"
                ] > 180,
                "customer"
            ].nunique(),
        ],
    }
)


display(
    unique_reactivation_summary
)

In [ ]:
# ================================================================
# VISUAL 13.4 — REACTIVATION AFTER INACTIVITY
# ================================================================

plt.figure(
    figsize=(8, 6)
)


sns.barplot(
    data=reactivation_summary,
    x="inactivity_threshold_days",
    y="reactivation_rate_pct"
)


plt.title(
    "Observed Reactivation After Extended Inactivity"
)

plt.xlabel(
    "Inactivity Threshold (Days)"
)

plt.ylabel(
    "Reactivation Rate (%)"
)

plt.xticks(
    ticks=[
        0,
        1
    ],
    labels=[
        "90 Days",
        "180 Days"
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 13.16 — PROFILE CURRENTLY LONG-INACTIVE CUSTOMERS
# ================================================================

long_inactive_customers = (
    customer_last_purchase
    .loc[
        customer_last_purchase[
            "inactivity_segment"
        ]
        == "Long inactive — >180 days"
    ]
    .copy()
)


long_inactive_profile = (
    long_inactive_customers
    [
        [
            "customer",
            "first_purchase_date",
            "last_purchase_date",
            "days_since_last_purchase",
            "total_orders",
            "historical_revenue",
        ]
    ]
    .sort_values(
        "historical_revenue",
        ascending=False
    )
)


display(
    long_inactive_profile.head(20)
)

In [ ]:
# ================================================================
# 13.17 — HIGH-VALUE LONG-INACTIVE CUSTOMERS
# ================================================================

historical_revenue_75th = (
    customer_last_purchase[
        "historical_revenue"
    ]
    .quantile(0.75)
)


high_value_long_inactive = (
    long_inactive_customers
    .loc[
        long_inactive_customers[
            "historical_revenue"
        ] >= historical_revenue_75th
    ]
    .sort_values(
        "historical_revenue",
        ascending=False
    )
)


print(
    f"75th percentile customer historical revenue: "
    f"{historical_revenue_75th:,.2f}"
)

print(
    f"High-value customers currently inactive >180 days: "
    f"{len(high_value_long_inactive):,}"
)


display(
    high_value_long_inactive.head(20)
)

In [ ]:
# ================================================================
# 13.18 — INACTIVITY BY CUSTOMER FREQUENCY
# ================================================================

customer_last_purchase[
    "frequency_group"
] = pd.cut(
    customer_last_purchase[
        "total_orders"
    ],
    bins=[
        0,
        1,
        2,
        5,
        10,
        np.inf,
    ],
    labels=[
        "1 order",
        "2 orders",
        "3–5 orders",
        "6–10 orders",
        "11+ orders",
    ]
)


inactivity_frequency_matrix = (
    customer_last_purchase
    .groupby(
        [
            "frequency_group",
            "inactivity_segment"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
)


inactivity_frequency_pct = (
    inactivity_frequency_matrix
    .div(
        inactivity_frequency_matrix
        .sum(axis=1),
        axis=0
    )
    * 100
)


display(
    inactivity_frequency_pct
)

In [ ]:
# ================================================================
# VISUAL 13.5 — INACTIVITY BY PURCHASE FREQUENCY
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.heatmap(
    inactivity_frequency_pct,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    cbar_kws={
        "label":
            "Share of Frequency Group (%)"
    }
)


plt.title(
    "Customer Inactivity Status by Purchase Frequency"
)

plt.xlabel(
    "Current Inactivity Status"
)

plt.ylabel(
    "Observed Customer Order Frequency"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 13.19 — VALIDATION CHECKS
# ================================================================

# Every identified customer must have exactly one inactivity status.
assert (
    customer_last_purchase[
        "inactivity_segment"
    ].notna()
).all()


# Customer counts must reconcile.
assert (
    inactivity_summary[
        "customers"
    ].sum()
    ==
    customer_last_purchase[
        "customer"
    ].nunique()
)


# Revenue shares should reconcile to approximately 100%.
assert np.isclose(
    inactivity_summary[
        "historical_revenue_share_pct"
    ].sum(),
    100,
    atol=0.01
)


# Reactivated episodes cannot exceed inactivity episodes.
assert (
    reactivation_summary[
        "reactivated_episodes"
    ]
    <=
    reactivation_summary[
        "inactivity_episodes"
    ]
).all()


print(
    "✓ Step 13 inactivity validation checks passed."
)

In [ ]:
# ================================================================
# 13.20 — STEP 13 SUMMARY METRICS
# ================================================================

step13_summary = (
    inactivity_summary
    [
        [
            "inactivity_segment",
            "customers",
            "customer_share_pct",
            "historical_revenue",
            "historical_revenue_share_pct",
            "median_orders",
            "median_days_since_purchase",
        ]
    ]
    .copy()
)


display(
    step13_summary
)


print(
    "\nReactivation summary:"
)

display(
    reactivation_summary
)


print(
    f"\nHigh-value customers inactive >180 days: "
    f"{len(high_value_long_inactive):,}"
)

### Step 13 Summary — Customer Inactivity and Churn-Style Behaviour

Step 13 examined customer inactivity to determine whether customers who stop purchasing should reasonably be considered permanently lost or whether long gaps between purchases are a normal feature of customer behaviour.

Because this is a transactional retail business rather than a subscription model, inactivity was deliberately separated from formal churn. A customer who has not purchased recently may still return after several months.

The analysis therefore used two practical inactivity thresholds:

- **90 days** as an early inactivity or at-risk threshold; and
- **180 days** as an extended inactivity threshold.

These thresholds are analytical conventions informed by the repeat-purchase behaviour observed in Step 11. They should not be interpreted as evidence that a customer has permanently churned.

To avoid distortion from the partial December 2011 trading period, customer activity status was evaluated as of **30 November 2011**.

#### The observed customer base contains substantial long-term inactivity

A total of approximately **5,821 identified customers** had completed at least one positive merchandise order by the end of the complete observation period.

As of 30 November 2011:

- **2,845 customers, or approximately 48.87%, had purchased within the previous 90 days**;
- **607 customers, or approximately 10.43%, had been inactive for 91–180 days**; and
- **2,369 customers, or approximately 40.70%, had been inactive for more than 180 days**.

On customer count alone, this appears to indicate a very large inactive population.

However, customer counts provide only part of the picture.

The active customer group was responsible for a disproportionately large share of historical purchasing activity, while many long-inactive customers had historically purchased relatively infrequently.

#### Active customers account for the overwhelming majority of historical purchase revenue

Customers active within the previous 90 days represented approximately **48.87% of the identified customer base**, but accounted for approximately **13.26 million**, or **80.02% of historical positive merchandise purchase revenue** generated by the customer population included in this analysis.

Customers inactive for 91–180 days represented approximately **10.43% of customers** and approximately **6.32% of historical purchase revenue**.

Customers inactive for more than 180 days represented approximately **40.70% of customers**, but only approximately **13.66% of historical purchase revenue**.

This distinction is significant.

The long-inactive population is large in customer-count terms, but its historical economic contribution is much smaller than its share of customers.

This suggests that inactivity is disproportionately concentrated among lower-frequency and historically lower-value customers.

The median customer currently active within 90 days had completed approximately **5 orders**, compared with approximately **3 orders** among recently inactive customers and only **2 orders** among customers inactive for more than 180 days.

The average order-frequency pattern is even more pronounced. Currently active customers averaged approximately **9.38 observed orders**, compared with approximately **4.77 orders** among customers inactive for 91–180 days and only approximately **2.60 orders** among the long-inactive group.

The evidence therefore indicates that frequent customers are substantially more likely to remain active.

#### Purchase gaps are often much longer than a conventional monthly churn definition would allow

Across the complete order history, approximately **29,902 observed intervals between consecutive customer purchases** were available for analysis.

The median gap between purchases was approximately **25 days**, but the distribution was highly skewed.

Approximately:

- 25% of purchase gaps were **7 days or less**;
- 50% were **25 days or less**;
- 75% were **62 days or less**;
- 90% were approximately **136 days or less**; and
- 95% were approximately **206 days or less**.

The mean interpurchase interval was approximately **51.8 days**.

The long upper tail is particularly important.

A meaningful proportion of customers who eventually purchase again do so only after several months of inactivity. Therefore, the absence of a purchase for one month—or even several months—does not provide sufficient evidence that a customer has permanently left the business.

This reinforces the conclusion from Step 11 that customer behaviour in this dataset is naturally irregular.

#### A 90-day inactivity threshold is useful for risk monitoring, but not for declaring churn

The most important finding from Step 13 is the frequency with which customers returned after exceeding the inactivity thresholds.

Across the observable order history, approximately **8,057 purchase episodes progressed beyond 90 days without another purchase**.

Of these, approximately **5,069 episodes subsequently resulted in another customer purchase**.

This represents an observed reactivation rate of approximately **62.91% following a 90-day inactivity episode**.

In other words, almost two-thirds of the observable 90-day inactivity episodes eventually ended with the customer purchasing again.

This makes it inappropriate to describe a customer as permanently churned after only 90 days without a transaction.

A 90-day threshold is better interpreted as:

> **a customer entering an at-risk or inactive state**

rather than evidence of permanent customer loss.

#### Even 180 days of inactivity does not reliably indicate permanent churn

The same issue remains visible at the more conservative 180-day threshold.

Approximately **4,277 observable purchase episodes reached more than 180 days without another transaction**.

Of these, approximately **1,902 were eventually followed by another purchase**.

This represents an observed reactivation rate of approximately **44.47% after a 180-day inactivity episode**.

This is a particularly important result.

Almost **45% of observable 180-day inactivity episodes ultimately ended in reactivation**.

Therefore, even six months without a purchase does not reliably establish permanent churn in this customer base.

The evidence strongly supports using terminology such as:

- active,
- inactive,
- long inactive,
- and reactivated,

rather than automatically labelling customers as churned.

#### Long-gap reactivation is widespread across the customer population

The reactivation phenomenon is not being driven by only a small handful of customers.

Approximately **2,983 unique customers** were observed making at least one subsequent purchase after a gap exceeding **90 days**.

Approximately **1,681 unique customers** returned after at least one gap exceeding **180 days**.

This provides strong behavioural evidence that long purchasing gaps are a common characteristic of the customer base.

The result changes how customer inactivity should be interpreted operationally.

A customer who has stopped purchasing for several months may still have meaningful future purchase potential, particularly if the customer has an established purchase history.

#### One-time buyers are substantially more likely to become long inactive

The relationship between customer purchase frequency and current inactivity status is particularly strong.

Among customers with only **one observed order**, approximately:

- **23.70% were active within 90 days**; and
- **68.98% were long inactive for more than 180 days**.

Among customers with exactly **two orders**, approximately **53.83% were long inactive**.

The long-inactive proportion falls sharply as purchase frequency increases.

Among customers with **3–5 orders**, approximately **35.29% were long inactive**.

Among customers with **6–10 orders**, only approximately **16.11% were long inactive**.

Among customers with **11 or more orders**, only approximately **6.22% were long inactive**, while approximately **86.96% remained active within the previous 90 days**.

This is one of the clearest behavioural relationships identified in the notebook.

Customers who develop repeated purchasing habits are dramatically less likely to become long inactive.

The implication is that the transition from first purchase into repeated purchasing appears strategically important not only for customer revenue, as established in Step 11, but also for continued customer activity.

#### Long-inactive customers are not economically homogeneous

Although long-inactive customers are generally lower frequency, the group still contains a smaller number of historically valuable customers.

The 75th percentile of historical customer purchase revenue was approximately **2,208.32**.

Using this threshold to identify historically higher-value customers, approximately **167 customers currently inactive for more than 180 days** had previously generated revenue at or above the overall customer 75th-percentile level.

Collectively, these high-value long-inactive customers had generated approximately **1.06 million in historical positive merchandise purchase revenue**.

This represents a materially different commercial problem from the large population of low-frequency, one-time customers.

The long-inactive base should therefore not be approached as one homogeneous group.

A more useful distinction is between:

1. low-frequency customers who became inactive after one or two purchases; and
2. historically valuable repeat customers who later became inactive.

The second category is likely to be far more strategically relevant for future reactivation efforts.

However, their historical revenue should **not** be interpreted as automatically recoverable or as current lost revenue.

#### Historical revenue associated with inactive customers is not the same as revenue leakage

Customers currently inactive for more than 180 days generated approximately **2.26 million of historical purchase revenue**.

It would be incorrect to describe this entire amount as lost revenue.

Historical revenue records what those customers purchased in the past. It does not tell us how much they would have purchased had they remained active.

Similarly, the approximately **1.06 million generated historically by high-value long-inactive customers** represents a useful indicator of prior customer importance, not a recovery forecast.

The next stage of the analysis must therefore quantify revenue movement more carefully by distinguishing:

- customer acquisition,
- expansion,
- contraction,
- inactivity,
- and reactivation.

This distinction prevents historical customer value from being incorrectly converted into a claim of revenue loss.

#### The evidence does not support a simple permanent-churn definition

Taken together, the results provide strong evidence against using short inactivity windows as permanent churn definitions.

A meaningful proportion of customers return after both 90-day and 180-day purchasing gaps.

Approximately **62.91% of observable 90-day inactivity episodes** were eventually followed by another purchase, while approximately **44.47% of observable 180-day inactivity episodes** were followed by reactivation.

This means that even an apparently long period of inactivity frequently represents a temporary gap rather than the end of the customer relationship.

For this business, customer status is therefore better understood as a continuum:

> **Active → Recently Inactive → Long Inactive → Potentially Reactivated**

rather than as a binary retained/churned framework.

#### Overall conclusion

Step 13 demonstrates that customer inactivity is a significant feature of the business, but **inactivity and permanent churn are not equivalent**.

As of 30 November 2011, approximately **48.87% of identified customers were active within 90 days**, approximately **10.43% were inactive for 91–180 days**, and approximately **40.70% had been inactive for more than 180 days**.

However, the economic importance of these groups differs substantially.

The currently active population generated approximately **80.02% of historical purchase revenue**, while the much larger long-inactive population generated only approximately **13.66%**.

Customer purchase frequency is also strongly associated with inactivity. Nearly **69% of one-order customers were long inactive**, compared with only around **6% of customers with 11 or more orders**.

Most importantly, long inactivity frequently does not represent permanent customer loss.

Approximately **63% of observable 90-day inactivity episodes** and approximately **44% of observable 180-day inactivity episodes** ultimately ended with another purchase.

The business should therefore avoid treating customers as permanently churned solely because they have crossed a fixed inactivity threshold.

A more useful customer-management framework would treat 90 days as an **early inactivity or at-risk signal**, 180 days as **extended inactivity**, and use prior purchase frequency, historical customer value and previous reactivation behaviour to determine which customers warrant intervention.

The next step is therefore not to assume that all inactive-customer revenue has been lost.

Step 14 examines **Revenue Retention and Leakage**, decomposing revenue movement into customer acquisition, reactivation, expansion, contraction and inactivity so that the economic effects of customer behaviour can be quantified more precisely.



## Step 14 — Revenue Retention and Leakage

Step 14 quantifies how customer behaviour translated into revenue movement between November 2010 and November 2011.

The analysis focuses on identified customers only because customer-level revenue movement requires persistent customer identifiers.

Rather than treating every inactive customer as permanently churned, the analysis decomposes the revenue bridge into:

- retained revenue from baseline customers,
- expansion among existing customers,
- contraction among existing customers,
- revenue absent from customers who were active in the baseline period but not positive in the comparison period,
- revenue contributed by newly acquired customers,
- revenue from customers acquired after the baseline period,
- reactivated-customer revenue,
- and other negative customer movements where applicable.

The objective is to distinguish gross revenue leakage from the customer growth sources that offset it.

Because this is a transactional retail business rather than a subscription model, the resulting retention measures should be interpreted as period-to-period customer revenue retention analogues rather than formal recurring-revenue retention metrics.

In [ ]:
# ================================================================
# 14.2 — IDENTIFIED CUSTOMER REVENUE BY COMPARISON PERIOD
# ================================================================

identified_customer_period_revenue = (
    df.loc[
        df["customer_is_identified"]
        & df["is_merchandise"]
        & df["period_month"].isin(
            [
                baseline_period,
                comparison_period
            ]
        )
    ]
    .groupby(
        [
            "customer",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


display(
    identified_customer_period_revenue.head()
)

In [ ]:
# ================================================================
# 14.3 — CUSTOMER REVENUE BRIDGE TABLE
# ================================================================

customer_revenue_bridge = (
    identified_customer_period_revenue
    .pivot(
        index="customer",
        columns="period_month",
        values="net_merchandise_revenue"
    )
    .fillna(0)
    .reset_index()
)


customer_revenue_bridge = (
    customer_revenue_bridge
    .rename(
        columns={
            baseline_period:
                "baseline_revenue",

            comparison_period:
                "comparison_revenue",
        }
    )
)


customer_revenue_bridge[
    "revenue_change"
] = (
    customer_revenue_bridge[
        "comparison_revenue"
    ]
    - customer_revenue_bridge[
        "baseline_revenue"
    ]
)


display(
    customer_revenue_bridge.head()
)

In [ ]:
# ================================================================
# 14.4 — ATTACH FIRST POSITIVE PURCHASE DATE
# ================================================================

first_positive_purchase = (
    customer_purchase_lines
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_positive_purchase_date=(
            "date",
            "min"
        )
    )
)


customer_revenue_bridge = (
    customer_revenue_bridge
    .merge(
        first_positive_purchase,
        on="customer",
        how="left"
    )
)

In [ ]:
# ================================================================
# 14.5 — DEFINE PERIOD BOUNDARIES
# ================================================================

baseline_start = pd.Timestamp(
    f"{baseline_period}-01"
)

baseline_end = (
    baseline_start
    + pd.offsets.MonthEnd(0)
)


comparison_start = pd.Timestamp(
    f"{comparison_period}-01"
)

comparison_end = (
    comparison_start
    + pd.offsets.MonthEnd(0)
)


print(
    f"Baseline period: "
    f"{baseline_start:%d %b %Y} "
    f"to {baseline_end:%d %b %Y}"
)

print(
    f"Comparison period: "
    f"{comparison_start:%d %b %Y} "
    f"to {comparison_end:%d %b %Y}"
)

In [ ]:
# ================================================================
# 14.6 — CLASSIFY CUSTOMER REVENUE MOVEMENT
# ================================================================

def classify_customer_revenue_movement(row):

    baseline = row["baseline_revenue"]
    comparison = row["comparison_revenue"]
    first_purchase = row["first_positive_purchase_date"]


    # ------------------------------------------------------------
    # Existing baseline customers
    # ------------------------------------------------------------

    if baseline > 0 and comparison > baseline:
        return "Existing customer expansion"


    if baseline > 0 and 0 < comparison < baseline:
        return "Existing customer contraction"


    if baseline > 0 and np.isclose(
        comparison,
        baseline
    ):
        return "Existing customer stable"


    if baseline > 0 and comparison <= 0:
        return "Baseline customer inactive / negative"


    # ------------------------------------------------------------
    # Customers producing positive comparison-period revenue
    # without positive baseline revenue
    # ------------------------------------------------------------

    if baseline <= 0 and comparison > 0:

        if (
            pd.notna(first_purchase)
            and
            comparison_start
            <= first_purchase
            <= comparison_end
        ):
            return "New in comparison month"


        if (
            pd.notna(first_purchase)
            and
            baseline_end
            < first_purchase
            < comparison_start
        ):
            return "Acquired since baseline"


        if (
            pd.notna(first_purchase)
            and
            first_purchase
            < baseline_start
        ):
            return "Reactivated customer"


        return "Other positive customer movement"


    # ------------------------------------------------------------
    # Negative / credit-only comparison movements
    # ------------------------------------------------------------

    if comparison < 0:
        return "Returns / credit-only movement"


    return "No positive revenue movement"


customer_revenue_bridge[
    "movement_segment"
] = (
    customer_revenue_bridge
    .apply(
        classify_customer_revenue_movement,
        axis=1
    )
)

In [ ]:
# ================================================================
# 14.7 — CUSTOMER MOVEMENT SUMMARY
# ================================================================

customer_movement_summary = (
    customer_revenue_bridge
    .groupby(
        "movement_segment",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        baseline_revenue=(
            "baseline_revenue",
            "sum"
        ),

        comparison_revenue=(
            "comparison_revenue",
            "sum"
        ),

        revenue_change=(
            "revenue_change",
            "sum"
        ),
    )
    .sort_values(
        "revenue_change",
        ascending=False
    )
)


display(
    customer_movement_summary
)

In [ ]:
# ================================================================
# 14.8 — BASELINE IDENTIFIED CUSTOMER REVENUE
# ================================================================

baseline_identified_revenue = (
    customer_revenue_bridge[
        "baseline_revenue"
    ]
    .sum()
)


comparison_identified_revenue = (
    customer_revenue_bridge[
        "comparison_revenue"
    ]
    .sum()
)


identified_revenue_change = (
    comparison_identified_revenue
    - baseline_identified_revenue
)


print(
    f"Baseline identified revenue: "
    f"{baseline_identified_revenue:,.2f}"
)

print(
    f"Comparison identified revenue: "
    f"{comparison_identified_revenue:,.2f}"
)

print(
    f"Identified customer revenue change: "
    f"{identified_revenue_change:+,.2f}"
)

In [ ]:
# ================================================================
# 14.9 — RETAINED BASELINE CUSTOMER REVENUE
# ================================================================

baseline_positive_customers = (
    customer_revenue_bridge
    .loc[
        customer_revenue_bridge[
            "baseline_revenue"
        ] > 0
    ]
    .copy()
)


baseline_positive_customers[
    "retained_revenue"
] = np.where(
    baseline_positive_customers[
        "comparison_revenue"
    ] > 0,

    np.minimum(
        baseline_positive_customers[
            "baseline_revenue"
        ],
        baseline_positive_customers[
            "comparison_revenue"
        ]
    ),

    0
)


retained_revenue = (
    baseline_positive_customers[
        "retained_revenue"
    ]
    .sum()
)


print(
    f"Retained baseline revenue: "
    f"{retained_revenue:,.2f}"
)

In [ ]:
# ================================================================
# 14.10 — EXISTING CUSTOMER EXPANSION
# ================================================================

baseline_positive_customers[
    "expansion_revenue"
] = np.where(
    baseline_positive_customers[
        "comparison_revenue"
    ]
    >
    baseline_positive_customers[
        "baseline_revenue"
    ],

    baseline_positive_customers[
        "comparison_revenue"
    ]
    -
    baseline_positive_customers[
        "baseline_revenue"
    ],

    0
)


expansion_revenue = (
    baseline_positive_customers[
        "expansion_revenue"
    ]
    .sum()
)


print(
    f"Existing-customer expansion revenue: "
    f"{expansion_revenue:,.2f}"
)

In [ ]:
# ================================================================
# 14.11 — EXISTING CUSTOMER CONTRACTION
# ================================================================

baseline_positive_customers[
    "contraction_leakage"
] = np.where(
    (
        baseline_positive_customers[
            "comparison_revenue"
        ] > 0
    )
    &
    (
        baseline_positive_customers[
            "comparison_revenue"
        ]
        <
        baseline_positive_customers[
            "baseline_revenue"
        ]
    ),

    baseline_positive_customers[
        "baseline_revenue"
    ]
    -
    baseline_positive_customers[
        "comparison_revenue"
    ],

    0
)


contraction_leakage = (
    baseline_positive_customers[
        "contraction_leakage"
    ]
    .sum()
)


print(
    f"Revenue contraction leakage: "
    f"{contraction_leakage:,.2f}"
)

In [ ]:
# ================================================================
# 14.12 — BASELINE CUSTOMER INACTIVITY LEAKAGE
# ================================================================

baseline_positive_customers[
    "inactivity_leakage"
] = np.where(
    baseline_positive_customers[
        "comparison_revenue"
    ] <= 0,

    baseline_positive_customers[
        "baseline_revenue"
    ],

    0
)


inactivity_leakage = (
    baseline_positive_customers[
        "inactivity_leakage"
    ]
    .sum()
)


print(
    f"Baseline revenue absent from inactive customers: "
    f"{inactivity_leakage:,.2f}"
)

In [ ]:
# ================================================================
# 14.13 — GROSS CUSTOMER REVENUE LEAKAGE
# ================================================================

gross_customer_leakage = (
    contraction_leakage
    + inactivity_leakage
)


gross_leakage_rate = (
    gross_customer_leakage
    / baseline_identified_revenue
    * 100
)


print(
    f"Gross customer revenue leakage: "
    f"{gross_customer_leakage:,.2f}"
)

print(
    f"Gross leakage rate: "
    f"{gross_leakage_rate:.2f}%"
)

In [ ]:
# ================================================================
# 14.14 — GROSS PERIOD REVENUE RETENTION ANALOGUE
# ================================================================

gross_revenue_retention_pct = (
    (
        baseline_identified_revenue
        - gross_customer_leakage
    )
    / baseline_identified_revenue
    * 100
)


print(
    f"Gross period revenue retention: "
    f"{gross_revenue_retention_pct:.2f}%"
)

In [ ]:
# ================================================================
# 14.15 — EXISTING CUSTOMER REVENUE AFTER EXPANSION
# ================================================================

existing_customer_comparison_revenue = (
    baseline_positive_customers[
        "comparison_revenue"
    ]
    .sum()
)


existing_customer_net_retention_pct = (
    existing_customer_comparison_revenue
    / baseline_identified_revenue
    * 100
)


print(
    f"Comparison revenue from baseline customers: "
    f"{existing_customer_comparison_revenue:,.2f}"
)

print(
    f"Existing-customer net retention analogue: "
    f"{existing_customer_net_retention_pct:.2f}%"
)

In [ ]:
# ================================================================
# 14.16 — CUSTOMER GROWTH INFLOWS
# ================================================================

growth_segments = [
    "New in comparison month",
    "Acquired since baseline",
    "Reactivated customer",
    "Other positive customer movement",
]


customer_growth_inflows = (
    customer_revenue_bridge
    .loc[
        customer_revenue_bridge[
            "movement_segment"
        ].isin(
            growth_segments
        )
    ]
    .groupby(
        "movement_segment",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        comparison_revenue=(
            "comparison_revenue",
            "sum"
        ),
    )
)


display(
    customer_growth_inflows
)

In [ ]:
# ================================================================
# 14.17 — TOTAL CUSTOMER GROWTH INFLOWS
# ================================================================

total_customer_growth_inflows = (
    customer_growth_inflows[
        "comparison_revenue"
    ]
    .sum()
)


print(
    f"Total acquisition/reactivation inflow: "
    f"{total_customer_growth_inflows:,.2f}"
)

In [ ]:
# ================================================================
# 14.18 — LEAKAGE OFFSET RATIO
# ================================================================

leakage_offset_ratio = (
    total_customer_growth_inflows
    / gross_customer_leakage
    if gross_customer_leakage > 0
    else np.nan
)


leakage_offset_pct = (
    leakage_offset_ratio
    * 100
)


print(
    f"Customer growth inflows offset "
    f"{leakage_offset_pct:.2f}% "
    f"of gross customer leakage."
)

In [ ]:
# ================================================================
# 14.19 — TOTAL POSITIVE CUSTOMER OFFSETS
# ================================================================

total_positive_customer_offsets = (
    expansion_revenue
    + total_customer_growth_inflows
)


total_offset_ratio = (
    total_positive_customer_offsets
    / gross_customer_leakage
    if gross_customer_leakage > 0
    else np.nan
)


print(
    f"Expansion + acquisition/reactivation "
    f"offset {total_offset_ratio * 100:.2f}% "
    f"of gross customer leakage."
)

In [ ]:
# ================================================================
# 14.20 — PREPARE CUSTOMER REVENUE WATERFALL
# ================================================================

waterfall_components = pd.DataFrame(
    {
        "Component": [
            "Baseline Identified Revenue",
            "Existing Customer Expansion",
            "Customer Contraction",
            "Inactive Baseline Customers",
            "New / Acquired / Reactivated",
            "Comparison Identified Revenue",
        ],

        "Value": [
            baseline_identified_revenue,
            expansion_revenue,
            -contraction_leakage,
            -inactivity_leakage,
            total_customer_growth_inflows,
            comparison_identified_revenue,
        ],
    }
)


display(
    waterfall_components
)

In [ ]:
# ================================================================
# VISUAL 14.1 — CUSTOMER REVENUE WATERFALL
# ================================================================

import matplotlib.pyplot as plt
import numpy as np


components = [
    "Baseline\nRevenue",
    "Expansion",
    "Contraction",
    "Inactivity",
    "New / Acquired /\nReactivated",
    "Comparison\nRevenue",
]


values = [
    baseline_identified_revenue,
    expansion_revenue,
    -contraction_leakage,
    -inactivity_leakage,
    total_customer_growth_inflows,
    comparison_identified_revenue,
]


# ---------------------------------------------------------------
# Calculate waterfall starting positions
# ---------------------------------------------------------------

starts = [
    0
]


running_total = (
    baseline_identified_revenue
)


for value in values[1:-1]:

    starts.append(
        running_total
    )

    running_total += value


starts.append(0)


# ---------------------------------------------------------------
# Plot
# ---------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(13, 7)
)


for i, (
    label,
    value,
    start
) in enumerate(
    zip(
        components,
        values,
        starts
    )
):

    # First and final bars are totals.
    if i in [
        0,
        len(values) - 1
    ]:

        bottom = 0
        height = value

    else:

        if value >= 0:
            bottom = start
            height = value

        else:
            bottom = start + value
            height = abs(value)


    ax.bar(
        i,
        height,
        bottom=bottom
    )


    label_position = (
        bottom + height
    )


    ax.text(
        i,
        label_position,
        f"{value / 1000:,.1f}k",
        ha="center",
        va="bottom",
        fontsize=9
    )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    range(
        len(components)
    )
)

ax.set_xticklabels(
    components
)


ax.set_title(
    "Identified Customer Revenue Bridge: "
    "November 2010 to November 2011"
)

ax.set_ylabel(
    "Net Merchandise Revenue"
)


plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 14.21 — LARGEST CUSTOMER CONTRACTION
# ================================================================

largest_customer_contractions = (
    customer_revenue_bridge
    .loc[
        customer_revenue_bridge[
            "movement_segment"
        ]
        == "Existing customer contraction"
    ]
    .sort_values(
        "revenue_change",
        ascending=True
    )
    [
        [
            "customer",
            "baseline_revenue",
            "comparison_revenue",
            "revenue_change",
        ]
    ]
    .head(20)
)


display(
    largest_customer_contractions
)

In [ ]:
# ================================================================
# 14.22 — LARGEST INACTIVITY LEAKAGE CUSTOMERS
# ================================================================

largest_inactivity_leakage = (
    customer_revenue_bridge
    .loc[
        (
            customer_revenue_bridge[
                "baseline_revenue"
            ] > 0
        )
        &
        (
            customer_revenue_bridge[
                "comparison_revenue"
            ] <= 0
        )
    ]
    .sort_values(
        "baseline_revenue",
        ascending=False
    )
    [
        [
            "customer",
            "baseline_revenue",
            "comparison_revenue",
        ]
    ]
    .head(20)
)


display(
    largest_inactivity_leakage
)

In [ ]:
# ================================================================
# 14.23 — CUSTOMER LEAKAGE CONCENTRATION
# ================================================================

customer_leakage = (
    baseline_positive_customers
    .copy()
)


customer_leakage[
    "total_leakage"
] = (
    customer_leakage[
        "contraction_leakage"
    ]
    +
    customer_leakage[
        "inactivity_leakage"
    ]
)


customer_leakage = (
    customer_leakage
    .loc[
        customer_leakage[
            "total_leakage"
        ] > 0
    ]
    .sort_values(
        "total_leakage",
        ascending=False
    )
)


top_1_leakage_share = (
    customer_leakage[
        "total_leakage"
    ]
    .head(1)
    .sum()
    / gross_customer_leakage
    * 100
)


top_5_leakage_share = (
    customer_leakage[
        "total_leakage"
    ]
    .head(5)
    .sum()
    / gross_customer_leakage
    * 100
)


top_10_leakage_share = (
    customer_leakage[
        "total_leakage"
    ]
    .head(10)
    .sum()
    / gross_customer_leakage
    * 100
)


print(
    f"Top 1 customer share of leakage: "
    f"{top_1_leakage_share:.2f}%"
)

print(
    f"Top 5 customer share of leakage: "
    f"{top_5_leakage_share:.2f}%"
)

print(
    f"Top 10 customer share of leakage: "
    f"{top_10_leakage_share:.2f}%"
)

In [ ]:
# ================================================================
# VISUAL 14.2 — TOP CUSTOMER REVENUE LEAKAGE
# ================================================================

top_customer_leakage = (
    customer_leakage
    .head(15)
    .copy()
)


top_customer_leakage[
    "customer"
] = (
    top_customer_leakage[
        "customer"
    ]
    .astype(str)
)


plt.figure(
    figsize=(11, 8)
)


sns.barplot(
    data=top_customer_leakage,
    y="customer",
    x="total_leakage"
)


plt.title(
    "Largest Customer-Level Revenue Leakage"
)

plt.xlabel(
    "Period-to-Period Revenue Leakage"
)

plt.ylabel(
    "Customer ID"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 14.24 — LEAKAGE COMPOSITION
# ================================================================

leakage_composition = pd.DataFrame(
    {
        "Leakage Type": [
            "Existing Customer Contraction",
            "Baseline Customer Inactivity",
        ],

        "Revenue Leakage": [
            contraction_leakage,
            inactivity_leakage,
        ],
    }
)


leakage_composition[
    "Leakage Share (%)"
] = (
    leakage_composition[
        "Revenue Leakage"
    ]
    / leakage_composition[
        "Revenue Leakage"
    ].sum()
    * 100
)


display(
    leakage_composition
)

In [ ]:
# ================================================================
# VISUAL 14.3 — LEAKAGE COMPOSITION
# ================================================================

plt.figure(
    figsize=(9, 5)
)


sns.barplot(
    data=leakage_composition,
    y="Leakage Type",
    x="Revenue Leakage"
)


plt.title(
    "Composition of Gross Customer Revenue Leakage"
)

plt.xlabel(
    "Revenue Leakage"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 14.25 — CUSTOMER REVENUE GROWTH SOURCES
# ================================================================

growth_source_summary = (
    customer_movement_summary
    .loc[
        customer_movement_summary[
            "movement_segment"
        ].isin(
            [
                "Existing customer expansion",
                "Reactivated customer",
                "Acquired since baseline",
                "New in comparison month",
            ]
        )
    ]
    [
        [
            "movement_segment",
            "customers",
            "revenue_change",
        ]
    ]
    .copy()
)


display(
    growth_source_summary
)

In [ ]:
# ================================================================
# VISUAL 14.4 — SOURCES OF POSITIVE CUSTOMER REVENUE MOVEMENT
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=growth_source_summary,
    y="movement_segment",
    x="revenue_change"
)


plt.title(
    "Sources of Positive Identified-Customer Revenue Movement"
)

plt.xlabel(
    "Positive Revenue Contribution"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 14.26 — BUILD EXACT CUSTOMER REVENUE BRIDGE COMPONENTS
# ================================================================

exact_customer_bridge = (
    customer_revenue_bridge
    .copy()
)


# ---------------------------------------------------------------
# Existing-customer expansion
# Baseline positive and comparison revenue higher
# ---------------------------------------------------------------

exact_customer_bridge[
    "expansion_change"
] = np.where(
    (
        exact_customer_bridge[
            "baseline_revenue"
        ] > 0
    )
    &
    (
        exact_customer_bridge[
            "comparison_revenue"
        ]
        >
        exact_customer_bridge[
            "baseline_revenue"
        ]
    ),

    exact_customer_bridge[
        "revenue_change"
    ],

    0
)


# ---------------------------------------------------------------
# Existing-customer contraction
# Customer remains positive but spends less
# ---------------------------------------------------------------

exact_customer_bridge[
    "contraction_change"
] = np.where(
    (
        exact_customer_bridge[
            "baseline_revenue"
        ] > 0
    )
    &
    (
        exact_customer_bridge[
            "comparison_revenue"
        ] > 0
    )
    &
    (
        exact_customer_bridge[
            "comparison_revenue"
        ]
        <
        exact_customer_bridge[
            "baseline_revenue"
        ]
    ),

    exact_customer_bridge[
        "revenue_change"
    ],

    0
)


# ---------------------------------------------------------------
# Baseline customer inactive / negative
#
# Important:
# Use the FULL actual revenue change.
#
# If baseline = 500 and comparison = -20,
# the real movement is -520, not merely -500.
# ---------------------------------------------------------------

exact_customer_bridge[
    "inactivity_change"
] = np.where(
    (
        exact_customer_bridge[
            "baseline_revenue"
        ] > 0
    )
    &
    (
        exact_customer_bridge[
            "comparison_revenue"
        ] <= 0
    ),

    exact_customer_bridge[
        "revenue_change"
    ],

    0
)


# ---------------------------------------------------------------
# Positive revenue from customers without positive baseline revenue
# ---------------------------------------------------------------

exact_customer_bridge[
    "growth_inflow_change"
] = np.where(
    (
        exact_customer_bridge[
            "baseline_revenue"
        ] <= 0
    )
    &
    (
        exact_customer_bridge[
            "comparison_revenue"
        ] > 0
    ),

    exact_customer_bridge[
        "revenue_change"
    ],

    0
)

In [ ]:
# ================================================================
# 14.27 — RESIDUAL CREDIT / ACCOUNTING MOVEMENTS
# ================================================================

exact_customer_bridge[
    "core_bridge_change"
] = (
    exact_customer_bridge[
        "expansion_change"
    ]
    +
    exact_customer_bridge[
        "contraction_change"
    ]
    +
    exact_customer_bridge[
        "inactivity_change"
    ]
    +
    exact_customer_bridge[
        "growth_inflow_change"
    ]
)


exact_customer_bridge[
    "residual_credit_adjustment"
] = (
    exact_customer_bridge[
        "revenue_change"
    ]
    -
    exact_customer_bridge[
        "core_bridge_change"
    ]
)


# ---------------------------------------------------------------
# Summarise exact components
# ---------------------------------------------------------------

exact_expansion_change = (
    exact_customer_bridge[
        "expansion_change"
    ]
    .sum()
)


exact_contraction_change = (
    exact_customer_bridge[
        "contraction_change"
    ]
    .sum()
)


exact_inactivity_change = (
    exact_customer_bridge[
        "inactivity_change"
    ]
    .sum()
)


exact_growth_inflow_change = (
    exact_customer_bridge[
        "growth_inflow_change"
    ]
    .sum()
)


residual_credit_adjustment = (
    exact_customer_bridge[
        "residual_credit_adjustment"
    ]
    .sum()
)


print(
    f"Expansion contribution: "
    f"{exact_expansion_change:+,.2f}"
)

print(
    f"Contraction contribution: "
    f"{exact_contraction_change:+,.2f}"
)

print(
    f"Inactivity / negative contribution: "
    f"{exact_inactivity_change:+,.2f}"
)

print(
    f"Acquisition / reactivation contribution: "
    f"{exact_growth_inflow_change:+,.2f}"
)

print(
    f"Residual credit/accounting adjustment: "
    f"{residual_credit_adjustment:+,.2f}"
)

In [ ]:
# ================================================================
# 14.28 — EXACT CUSTOMER BRIDGE VALIDATION
# ================================================================

fully_reconstructed_change = (
    exact_expansion_change
    +
    exact_contraction_change
    +
    exact_inactivity_change
    +
    exact_growth_inflow_change
    +
    residual_credit_adjustment
)


print(
    f"Reconstructed identified-customer change: "
    f"{fully_reconstructed_change:+,.2f}"
)

print(
    f"Actual identified-customer change: "
    f"{identified_revenue_change:+,.2f}"
)

print(
    f"Difference: "
    f"{fully_reconstructed_change - identified_revenue_change:+,.8f}"
)


# ---------------------------------------------------------------
# Exact reconciliation
# ---------------------------------------------------------------

assert np.isclose(
    fully_reconstructed_change,
    identified_revenue_change,
    atol=0.01
), (
    "Customer revenue bridge does not reconcile."
)


# ---------------------------------------------------------------
# Baseline + movement = comparison
# ---------------------------------------------------------------

assert np.isclose(
    (
        baseline_identified_revenue
        +
        identified_revenue_change
    ),
    comparison_identified_revenue,
    atol=0.01
), (
    "Baseline revenue plus change does not "
    "equal comparison revenue."
)


# ---------------------------------------------------------------
# Direct customer-level check
# ---------------------------------------------------------------

assert np.isclose(
    exact_customer_bridge[
        "revenue_change"
    ].sum(),
    identified_revenue_change,
    atol=0.01
), (
    "Customer-level revenue changes do not "
    "reconcile to the period-level result."
)


print(
    "✓ Step 14 customer revenue bridge "
    "reconciles exactly."
)

In [ ]:
# ================================================================
# 14.29 — STEP 14 SUMMARY METRICS
# ================================================================

step14_summary = pd.DataFrame(
    {
        "Metric": [
            "Baseline identified revenue",
            "Comparison identified revenue",
            "Identified revenue change",

            "Retained baseline revenue",
            "Existing-customer expansion",

            "Customer contraction leakage",
            "Customer inactivity leakage",
            "Gross customer leakage",

            "Gross leakage rate (%)",
            "Gross period revenue retention (%)",
            "Existing-customer net retention (%)",

            "Acquisition/reactivation inflows",
            "Growth inflows as % of leakage",
            "Expansion + growth inflows as % of leakage",

            "Top 10 customer share of leakage (%)",
        ],

        "Value": [
            baseline_identified_revenue,
            comparison_identified_revenue,
            identified_revenue_change,

            retained_revenue,
            expansion_revenue,

            contraction_leakage,
            inactivity_leakage,
            gross_customer_leakage,

            gross_leakage_rate,
            gross_revenue_retention_pct,
            existing_customer_net_retention_pct,

            total_customer_growth_inflows,
            leakage_offset_pct,
            total_offset_ratio * 100,

            top_10_leakage_share,
        ],
    }
)


display(
    step14_summary
)

### Step 14 Summary — Revenue Retention and Leakage

Step 14 examined how identified-customer revenue changed between November 2010 and November 2011 and decomposed that movement into retained revenue, customer expansion, customer contraction, inactivity, acquisition and reactivation.

The purpose was to distinguish genuine customer revenue retention from the offsetting effects of new and returning customers.

Because this is a transactional retail business rather than a subscription business, the retention measures in this section should be interpreted as **period-to-period customer revenue retention analogues** rather than formal recurring-revenue metrics.

The analysis focuses only on identified customers because customer-level revenue movements cannot be attributed reliably where no persistent customer identifier exists.

#### Identified-customer revenue declined slightly

Identified customers generated approximately **1.127 million** of net merchandise revenue in November 2010.

In November 2011, identified-customer revenue was approximately **1.119 million**.

This represents a decline of approximately **8.24k**, or roughly **0.73%**.

This result is important because total net merchandise revenue across the business increased over the same period.

The positive company-level result therefore did not originate from stronger overall performance among identifiable customers.

As established in Step 10, unidentified-customer revenue increased sufficiently to offset the small decline in the identifiable customer base.

#### Only around 41% of baseline identified revenue was retained without allowing for expansion

Approximately **471.88k of baseline identified-customer revenue** was retained from customers who remained positive in the comparison period, when retained revenue is capped at the smaller of baseline and comparison-period revenue.

Against baseline identified revenue of approximately 1.127 million, the gross period revenue retention analogue was approximately **41.28%**.

This means that approximately **58.72% of baseline identified revenue was not retained at its original level** before considering customer expansion.

The gross leakage was therefore substantial.

However, this should not be interpreted as permanent customer loss.

Step 13 demonstrated that many customers return after long periods of inactivity. The metric instead describes how much November 2010 customer revenue failed to appear at the same level in November 2011.

#### Gross customer revenue leakage was approximately 661.76k

The total gross period-to-period customer revenue leakage was approximately **661.76k**.

This leakage came from two main sources:

- **existing-customer contraction — approximately 241.79k**
- **baseline customer inactivity — approximately 419.97k**

Inactivity therefore accounted for approximately **63.46% of gross customer leakage**, while contraction accounted for approximately **36.54%**.

The larger issue was therefore not simply that active customers spent less.

A greater share of the leakage came from customers who had generated positive revenue in November 2010 but generated no positive net merchandise revenue in November 2011.

This makes customer inactivity the dominant period-to-period leakage mechanism.

#### Existing-customer expansion materially softened the leakage

Although many baseline customers contracted or became inactive, other continuing customers increased their spending.

Existing-customer expansion contributed approximately **138.92k of additional revenue**.

After allowing for this expansion, baseline customers generated approximately **609.83k in November 2011**, equivalent to an existing-customer net retention analogue of approximately **54.11%**.

This means that expansion among successful existing customers recovered a meaningful portion of the revenue lost through contraction and inactivity, but was not sufficient to restore the original baseline customer revenue base.

#### Acquisition and reactivation provided more than 510k of revenue

Positive customer inflows from customers who were not generating positive baseline-period revenue contributed approximately **510.02k** in the comparison month.

The principal contributors were:

- **Reactivated customers — approximately 229.41k**
- **Customers acquired since the baseline period — approximately 201.68k**
- **Customers new in November 2011 — approximately 79.89k**

Reactivated customers were therefore the single largest external source of positive customer revenue movement.

This reinforces the Step 13 finding that inactivity should not be treated automatically as permanent churn.

Customers who had previously been absent from the comparison baseline generated more revenue than customers newly acquired during November 2011 itself.

The commercial value of reactivation is therefore clearly visible in the revenue bridge.

#### Customer growth inflows offset about three-quarters of gross leakage

Acquisition and reactivation inflows of approximately **510.02k** offset around **77.07% of the gross customer leakage** of approximately 661.76k.

When the approximately **138.92k of existing-customer expansion** is added, total positive customer offsets reached approximately **648.94k**.

Together, expansion, acquisition and reactivation offset approximately **98.06% of gross customer leakage**.

This is one of the most important findings in Step 14.

The business experienced substantial gross erosion within the identified customer base, but almost all of that erosion was replaced by:

- stronger spending from some existing customers,
- reactivation,
- and acquisition.

The resulting net change was therefore relatively small compared with the scale of movement beneath it.

#### The stable headline concealed very large customer turnover

The identified-customer revenue decline was only approximately **8.24k**, but the underlying movements were much larger.

On the negative side:

- approximately **241.79k** was lost through customer contraction;
- approximately **419.97k** was absent because baseline customers were inactive;
- and small additional negative credit movements also occurred.

On the positive side:

- approximately **138.92k** came from existing-customer expansion;
- approximately **229.41k** came from reactivated customers;
- approximately **201.68k** came from customers acquired after the baseline;
- and approximately **79.89k** came from customers newly acquired during the comparison month.

The relatively small final movement therefore masks a large amount of underlying customer turnover.

The business was not simply maintaining the same customers at roughly the same spending level.

Instead, significant leakage from parts of the customer base was being continuously replaced by expansion, acquisition and reactivation elsewhere.

#### Reactivation is economically more important than new-month acquisition

Reactivated customers contributed approximately **229.41k**, compared with approximately **79.89k from customers first acquired during November 2011**.

Reactivation therefore generated almost three times the revenue of customers who were brand new in the comparison month.

This finding is strategically important.

It suggests that commercially valuable growth does not come only from acquiring completely new customers.

Re-engaging previous customers is also a major revenue source and, during this comparison period, was substantially more important than same-month new acquisition.

This is consistent with the behavioural evidence in Step 13, where many customers returned after long inactivity gaps.

#### Leakage was not excessively concentrated in only a few customers

The largest single customer accounted for approximately **2.70% of gross customer revenue leakage**.

The top five customers accounted for approximately **8.16%**, while the top ten accounted for approximately **12.60%**.

The top twenty accounted for approximately **18.51%**.

This indicates that customer leakage was relatively distributed rather than being caused primarily by one or two unusually large accounts.

The largest individual leakage came from customer **15838**, whose revenue fell by approximately **17.86k** between the two periods.

Customer **16684** declined by approximately **10.80k**, while customer **12415** declined by approximately **10.37k**.

Other large losses arose from customers who generated meaningful baseline revenue but no positive comparison-period revenue.

The broad distribution of leakage suggests that the issue is portfolio-wide rather than purely account-specific.

#### Inactivity was more important than contraction

Approximately **419.97k**, or **63.46% of gross leakage**, arose from customers who were positive in the baseline period but produced no positive revenue in the comparison month.

Approximately **241.79k**, or **36.54%**, came from customers who remained active but generated less revenue.

This distinction matters because the appropriate commercial response differs.

Customer contraction suggests a need to understand declining spend among still-active customers.

Customer inactivity suggests a need to understand purchasing gaps, reactivation opportunities and whether the customer is temporarily dormant or structurally disengaged.

Because Step 13 showed that long periods of inactivity frequently end in reactivation, the inactivity component should not automatically be interpreted as permanently lost revenue.

#### Gross retention appears weak, but the customer ecosystem is highly dynamic

A gross period revenue retention level of approximately **41.28%** may initially appear very weak.

However, this metric is being applied to a retail business with irregular purchasing patterns across two individual months separated by one year.

It should therefore not be interpreted in the same way as Gross Revenue Retention in a recurring subscription model.

The analysis does not ask whether a contractual revenue stream remained active.

It asks how much revenue generated by identifiable customers in November 2010 was reproduced by those same customers in November 2011.

In a seasonal and transactional environment, this is a much stricter test.

The low gross retention therefore demonstrates substantial customer revenue turnover, but not necessarily customer relationship termination.

#### The identified customer base was economically dependent on replacement mechanisms

The combined evidence from Steps 6, 11, 12 and 13 becomes particularly important here.

Step 11 showed that repeat customers generate the overwhelming majority of observed customer revenue.

Step 12 showed that retention quality varies materially by acquisition cohort.

Step 13 showed that customers frequently reactivate after long periods of inactivity.

Step 14 now demonstrates how those behaviours affect revenue.

The customer base experienced substantial leakage through inactivity and contraction, but growth among continuing customers, new acquisitions and especially reactivated customers replaced almost all of it.

The business therefore appears to rely on a combination of:

1. retaining part of the existing customer base;
2. expanding spend among selected continuing customers;
3. reactivating dormant customers;
4. and continuously adding newly acquired customers.

No single mechanism was sufficient on its own.

#### Small credit-related movements were kept separate from core customer behaviour

A small number of customers generated negative or credit-only revenue movements.

These accounting effects are retained in the full reconciliation so that the identified customer bridge ties exactly back to the underlying net merchandise revenue movement.

They should not, however, be interpreted as customer acquisition, contraction or inactivity.

Similarly, a small positive movement arose where negative baseline customer revenue reverted toward zero.

These edge cases are economically minor relative to the main customer movements, but separating them prevents accounting adjustments from being misinterpreted as behavioural growth.

#### Overall conclusion

Step 14 shows that the apparently stable identified-customer revenue result conceals **substantial gross customer revenue turnover**.

Identified-customer revenue declined only approximately **8.24k** between November 2010 and November 2011, but beneath that result the business experienced approximately **661.76k of gross customer leakage**.

Approximately **419.97k** of that leakage resulted from baseline customers becoming inactive in the comparison month, while approximately **241.79k** resulted from customers who remained active but spent less.

These losses were almost entirely offset by positive customer movements.

Existing-customer expansion added approximately **138.92k**, while acquisition and reactivation contributed approximately **510.02k**.

Together, these positive movements replaced approximately **98% of gross customer leakage**.

The most important strategic conclusion is therefore not that customer revenue was simply stable.

Rather, the business maintained its customer revenue position through a high degree of internal turnover.

Substantial revenue left through inactivity and contraction, but was continually replaced through customer expansion, reactivation and acquisition.

Reactivation was particularly important, contributing approximately **229.41k**, making it a larger source of positive customer revenue movement than new customers acquired during the comparison month.

The evidence therefore suggests that future revenue improvement could potentially come from reducing the scale of existing-customer leakage while preserving the acquisition and reactivation mechanisms that are already offsetting most of that loss.

Step 15 therefore evaluates **retention-improvement and revenue-recovery scenarios**, using the measured leakage as the analytical base while clearly distinguishing scenario modelling from forecast revenue.


## Step 15 — Retention-Improvement and Revenue-Recovery Scenarios

Step 15 evaluates the potential economic significance of reducing measured customer revenue leakage.

The analysis does not forecast future revenue and does not assume that historical customer revenue can automatically be recovered.

Instead, it uses the period-to-period leakage quantified in Step 14 as a sensitivity-analysis base and asks:

> What would the comparison-period revenue result have looked like if a specified percentage of observed customer contraction or inactivity leakage had been avoided?

The scenarios are deliberately illustrative.

They model recovery rates of 5%, 10%, 15% and 20% of measured leakage and distinguish between:

- existing-customer contraction leakage; and
- baseline-customer inactivity leakage.

The analysis also shows how each hypothetical recovery level would affect:

- identified-customer revenue,
- net merchandise revenue,
- customer revenue retention,
- and the November 2010 versus November 2011 revenue-growth result.

These values should be interpreted as scenario sensitivities rather than forecasts, targets or guaranteed recoverable revenue.

In [ ]:
# ================================================================
# 15.2 — LOCK MEASURED REVENUE-LEAKAGE BASE
# ================================================================

scenario_contraction_leakage = (
    contraction_leakage
)

scenario_inactivity_leakage = (
    inactivity_leakage
)

scenario_gross_leakage = (
    scenario_contraction_leakage
    + scenario_inactivity_leakage
)


print(
    f"Contraction leakage base: "
    f"{scenario_contraction_leakage:,.2f}"
)

print(
    f"Inactivity leakage base: "
    f"{scenario_inactivity_leakage:,.2f}"
)

print(
    f"Gross leakage scenario base: "
    f"{scenario_gross_leakage:,.2f}"
)

In [ ]:
# ================================================================
# 15.3 — COMPARISON-PERIOD MERCHANDISE REVENUE BASE
# ================================================================

baseline_total_merchandise_revenue = (
    df.loc[
        df["period_month"]
        == baseline_period,
        "net_merchandise_revenue"
    ]
    .sum()
)


comparison_total_merchandise_revenue = (
    df.loc[
        df["period_month"]
        == comparison_period,
        "net_merchandise_revenue"
    ]
    .sum()
)


actual_total_revenue_change = (
    comparison_total_merchandise_revenue
    - baseline_total_merchandise_revenue
)


actual_total_growth_pct = (
    actual_total_revenue_change
    / baseline_total_merchandise_revenue
    * 100
)


print(
    f"Baseline merchandise revenue: "
    f"{baseline_total_merchandise_revenue:,.2f}"
)

print(
    f"Comparison merchandise revenue: "
    f"{comparison_total_merchandise_revenue:,.2f}"
)

print(
    f"Actual revenue change: "
    f"{actual_total_revenue_change:+,.2f}"
)

print(
    f"Actual growth rate: "
    f"{actual_total_growth_pct:+.2f}%"
)

In [ ]:
# ================================================================
# 15.4 — DEFINE ILLUSTRATIVE RECOVERY SCENARIOS
# ================================================================

recovery_rates = [
    0.05,
    0.10,
    0.15,
    0.20,
]

In [ ]:
# ================================================================
# 15.5 — SIMPLE GROSS LEAKAGE RECOVERY SCENARIOS
# ================================================================

recovery_scenarios = []


for recovery_rate in recovery_rates:

    recovered_revenue = (
        scenario_gross_leakage
        * recovery_rate
    )


    scenario_identified_revenue = (
        comparison_identified_revenue
        + recovered_revenue
    )


    scenario_total_merchandise_revenue = (
        comparison_total_merchandise_revenue
        + recovered_revenue
    )


    scenario_total_change = (
        scenario_total_merchandise_revenue
        - baseline_total_merchandise_revenue
    )


    scenario_growth_pct = (
        scenario_total_change
        / baseline_total_merchandise_revenue
        * 100
    )


    recovery_scenarios.append(
        {
            "recovery_rate":
                recovery_rate,

            "recovery_rate_pct":
                recovery_rate * 100,

            "recovered_revenue":
                recovered_revenue,

            "scenario_identified_revenue":
                scenario_identified_revenue,

            "scenario_total_merchandise_revenue":
                scenario_total_merchandise_revenue,

            "scenario_total_revenue_change":
                scenario_total_change,

            "scenario_growth_pct":
                scenario_growth_pct,
        }
    )


gross_recovery_scenarios = pd.DataFrame(
    recovery_scenarios
)


display(
    gross_recovery_scenarios
)

In [ ]:
# ================================================================
# 15.6 — SCENARIO UPLIFT VS ACTUAL RESULT
# ================================================================

gross_recovery_scenarios[
    "incremental_revenue_vs_actual"
] = (
    gross_recovery_scenarios[
        "scenario_total_merchandise_revenue"
    ]
    - comparison_total_merchandise_revenue
)


gross_recovery_scenarios[
    "growth_improvement_pp"
] = (
    gross_recovery_scenarios[
        "scenario_growth_pct"
    ]
    - actual_total_growth_pct
)


display(
    gross_recovery_scenarios
)

In [ ]:
# ================================================================
# VISUAL 15.1 — REVENUE RECOVERY SENSITIVITY
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(
    figsize=(10, 6)
)


sns.lineplot(
    data=gross_recovery_scenarios,
    x="recovery_rate_pct",
    y="recovered_revenue",
    marker="o"
)


for _, row in gross_recovery_scenarios.iterrows():

    plt.text(
        row["recovery_rate_pct"],
        row["recovered_revenue"],
        f"{row['recovered_revenue'] / 1000:,.1f}k",
        ha="center",
        va="bottom"
    )


plt.title(
    "Illustrative Revenue Recovery from Reduced Customer Leakage"
)

plt.xlabel(
    "Share of Measured Leakage Avoided (%)"
)

plt.ylabel(
    "Illustrative Revenue Uplift"
)

plt.xticks(
    gross_recovery_scenarios[
        "recovery_rate_pct"
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 15.7 — CONTRACTION VS INACTIVITY RECOVERY GRID
# ================================================================

contraction_recovery_rates = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
]


inactivity_recovery_rates = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
]


recovery_grid_records = []


for contraction_rate in contraction_recovery_rates:

    for inactivity_rate in inactivity_recovery_rates:

        contraction_recovery = (
            scenario_contraction_leakage
            * contraction_rate
        )

        inactivity_recovery = (
            scenario_inactivity_leakage
            * inactivity_rate
        )

        total_recovery = (
            contraction_recovery
            + inactivity_recovery
        )


        scenario_revenue = (
            comparison_total_merchandise_revenue
            + total_recovery
        )


        scenario_change = (
            scenario_revenue
            - baseline_total_merchandise_revenue
        )


        scenario_growth = (
            scenario_change
            / baseline_total_merchandise_revenue
            * 100
        )


        recovery_grid_records.append(
            {
                "contraction_recovery_pct":
                    contraction_rate * 100,

                "inactivity_recovery_pct":
                    inactivity_rate * 100,

                "contraction_revenue_recovered":
                    contraction_recovery,

                "inactivity_revenue_recovered":
                    inactivity_recovery,

                "total_revenue_recovered":
                    total_recovery,

                "scenario_total_revenue":
                    scenario_revenue,

                "scenario_growth_pct":
                    scenario_growth,
            }
        )


recovery_grid = pd.DataFrame(
    recovery_grid_records
)


display(
    recovery_grid.head(10)
)

In [ ]:
# ================================================================
# 15.8 — RECOVERY SCENARIO MATRIX
# ================================================================

recovery_heatmap = (
    recovery_grid
    .pivot(
        index="inactivity_recovery_pct",
        columns="contraction_recovery_pct",
        values="total_revenue_recovered"
    )
)


display(
    recovery_heatmap
)

In [ ]:
# ================================================================
# VISUAL 15.2 — RECOVERY SCENARIO HEATMAP
# ================================================================

plt.figure(
    figsize=(10, 7)
)


sns.heatmap(
    recovery_heatmap,
    annot=True,
    fmt=",.0f",
    linewidths=0.5,
    cbar_kws={
        "label":
            "Illustrative Revenue Uplift"
    }
)


plt.title(
    "Revenue Sensitivity to Reduced "
    "Contraction and Inactivity Leakage"
)

plt.xlabel(
    "Contraction Leakage Avoided (%)"
)

plt.ylabel(
    "Inactivity Leakage Avoided (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 15.9 — RECOVERY VALUE BY LEAKAGE TYPE
# ================================================================

leakage_type_scenarios = []


for recovery_rate in recovery_rates:

    leakage_type_scenarios.append(
        {
            "Recovery Rate (%)":
                recovery_rate * 100,

            "Leakage Type":
                "Customer Contraction",

            "Illustrative Recovery":
                scenario_contraction_leakage
                * recovery_rate,
        }
    )


    leakage_type_scenarios.append(
        {
            "Recovery Rate (%)":
                recovery_rate * 100,

            "Leakage Type":
                "Customer Inactivity",

            "Illustrative Recovery":
                scenario_inactivity_leakage
                * recovery_rate,
        }
    )


leakage_type_scenario_df = pd.DataFrame(
    leakage_type_scenarios
)


display(
    leakage_type_scenario_df
)

In [ ]:
# ================================================================
# VISUAL 15.3 — RECOVERY SENSITIVITY BY LEAKAGE TYPE
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.lineplot(
    data=leakage_type_scenario_df,
    x="Recovery Rate (%)",
    y="Illustrative Recovery",
    hue="Leakage Type",
    marker="o"
)


plt.title(
    "Illustrative Revenue Recovery by Leakage Type"
)

plt.xlabel(
    "Share of Leakage Avoided (%)"
)

plt.ylabel(
    "Illustrative Revenue Uplift"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 15.10 — SCENARIO GROSS PERIOD REVENUE RETENTION
# ================================================================

retention_scenario_records = []


for recovery_rate in recovery_rates:

    recovered_leakage = (
        scenario_gross_leakage
        * recovery_rate
    )


    adjusted_leakage = (
        scenario_gross_leakage
        - recovered_leakage
    )


    scenario_gross_retention = (
        (
            baseline_identified_revenue
            - adjusted_leakage
        )
        / baseline_identified_revenue
        * 100
    )


    retention_scenario_records.append(
        {
            "recovery_rate_pct":
                recovery_rate * 100,

            "remaining_leakage":
                adjusted_leakage,

            "scenario_gross_period_retention_pct":
                scenario_gross_retention,
        }
    )


retention_recovery_scenarios = (
    pd.DataFrame(
        retention_scenario_records
    )
)


display(
    retention_recovery_scenarios
)

In [ ]:
# ================================================================
# VISUAL 15.4 — GROSS RETENTION SENSITIVITY
# ================================================================

plt.figure(
    figsize=(9, 6)
)


sns.lineplot(
    data=retention_recovery_scenarios,
    x="recovery_rate_pct",
    y="scenario_gross_period_retention_pct",
    marker="o"
)


plt.axhline(
    gross_revenue_retention_pct,
    linestyle="--",
    linewidth=1,
    label="Actual gross period retention"
)


plt.title(
    "Illustrative Gross Revenue Retention Improvement"
)

plt.xlabel(
    "Share of Gross Leakage Avoided (%)"
)

plt.ylabel(
    "Gross Period Revenue Retention (%)"
)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 15.11 — INACTIVITY-ONLY RECOVERY SCENARIOS
# ================================================================

inactivity_only_records = []


for recovery_rate in recovery_rates:

    recovered_inactivity_revenue = (
        scenario_inactivity_leakage
        * recovery_rate
    )


    scenario_revenue = (
        comparison_total_merchandise_revenue
        + recovered_inactivity_revenue
    )


    scenario_growth = (
        (
            scenario_revenue
            - baseline_total_merchandise_revenue
        )
        / baseline_total_merchandise_revenue
        * 100
    )


    inactivity_only_records.append(
        {
            "recovery_rate_pct":
                recovery_rate * 100,

            "recovered_revenue":
                recovered_inactivity_revenue,

            "scenario_total_revenue":
                scenario_revenue,

            "scenario_growth_pct":
                scenario_growth,
        }
    )


inactivity_recovery_scenarios = (
    pd.DataFrame(
        inactivity_only_records
    )
)


display(
    inactivity_recovery_scenarios
)

In [ ]:
# ================================================================
# 15.12 — CONTRACTION-ONLY RECOVERY SCENARIOS
# ================================================================

contraction_only_records = []


for recovery_rate in recovery_rates:

    recovered_contraction_revenue = (
        scenario_contraction_leakage
        * recovery_rate
    )


    scenario_revenue = (
        comparison_total_merchandise_revenue
        + recovered_contraction_revenue
    )


    scenario_growth = (
        (
            scenario_revenue
            - baseline_total_merchandise_revenue
        )
        / baseline_total_merchandise_revenue
        * 100
    )


    contraction_only_records.append(
        {
            "recovery_rate_pct":
                recovery_rate * 100,

            "recovered_revenue":
                recovered_contraction_revenue,

            "scenario_total_revenue":
                scenario_revenue,

            "scenario_growth_pct":
                scenario_growth,
        }
    )


contraction_recovery_scenarios = (
    pd.DataFrame(
        contraction_only_records
    )
)


display(
    contraction_recovery_scenarios
)

In [ ]:
# ================================================================
# 15.13 — NAMED ILLUSTRATIVE SCENARIOS
# ================================================================

named_scenarios = pd.DataFrame(
    {
        "Scenario": [
            "Actual",
            "Conservative",
            "Moderate",
            "Strong",
        ],

        "Contraction Recovery Rate": [
            0.00,
            0.05,
            0.10,
            0.15,
        ],

        "Inactivity Recovery Rate": [
            0.00,
            0.05,
            0.10,
            0.15,
        ],
    }
)

In [ ]:
# ================================================================
# 15.13B — CALCULATE NAMED SCENARIO RESULTS
# ================================================================

named_scenarios[
    "Contraction Recovery"
] = (
    named_scenarios[
        "Contraction Recovery Rate"
    ]
    * scenario_contraction_leakage
)


named_scenarios[
    "Inactivity Recovery"
] = (
    named_scenarios[
        "Inactivity Recovery Rate"
    ]
    * scenario_inactivity_leakage
)


named_scenarios[
    "Total Revenue Recovery"
] = (
    named_scenarios[
        "Contraction Recovery"
    ]
    +
    named_scenarios[
        "Inactivity Recovery"
    ]
)


named_scenarios[
    "Scenario Merchandise Revenue"
] = (
    comparison_total_merchandise_revenue
    +
    named_scenarios[
        "Total Revenue Recovery"
    ]
)


named_scenarios[
    "Scenario Revenue Change"
] = (
    named_scenarios[
        "Scenario Merchandise Revenue"
    ]
    -
    baseline_total_merchandise_revenue
)


named_scenarios[
    "Scenario Growth (%)"
] = (
    named_scenarios[
        "Scenario Revenue Change"
    ]
    / baseline_total_merchandise_revenue
    * 100
)


display(
    named_scenarios
)

In [ ]:
# ================================================================
# 15.14 — GROWTH IMPROVEMENT VS ACTUAL RESULT
# ================================================================

named_scenarios[
    "Growth Improvement vs Actual (pp)"
] = (
    named_scenarios[
        "Scenario Growth (%)"
    ]
    - actual_total_growth_pct
)


display(
    named_scenarios[
        [
            "Scenario",
            "Total Revenue Recovery",
            "Scenario Merchandise Revenue",
            "Scenario Growth (%)",
            "Growth Improvement vs Actual (pp)",
        ]
    ]
)

In [ ]:
# ================================================================
# 15.15 — REQUIRED RECOVERY FOR TARGET GROWTH SCENARIOS
# ================================================================

illustrative_growth_levels = [
    0.03,
    0.05,
    0.07,
]


required_recovery_records = []


for growth_level in illustrative_growth_levels:

    required_revenue = (
        baseline_total_merchandise_revenue
        * (
            1
            + growth_level
        )
    )


    required_incremental_revenue = (
        required_revenue
        - comparison_total_merchandise_revenue
    )


    required_leakage_recovery_pct = (
        required_incremental_revenue
        / scenario_gross_leakage
        * 100
    )


    required_recovery_records.append(
        {
            "illustrative_growth_level_pct":
                growth_level * 100,

            "required_merchandise_revenue":
                required_revenue,

            "additional_revenue_required":
                required_incremental_revenue,

            "share_of_gross_leakage_required_pct":
                required_leakage_recovery_pct,
        }
    )


required_recovery_for_growth = pd.DataFrame(
    required_recovery_records
)


display(
    required_recovery_for_growth
)

In [ ]:
# ================================================================
# 15.16 — SCENARIO BOUNDARY VALIDATION
# ================================================================

assert (
    gross_recovery_scenarios[
        "recovered_revenue"
    ]
    <= scenario_gross_leakage
).all()


assert (
    recovery_grid[
        "contraction_revenue_recovered"
    ]
    <= scenario_contraction_leakage
).all()


assert (
    recovery_grid[
        "inactivity_revenue_recovered"
    ]
    <= scenario_inactivity_leakage
).all()


print(
    "✓ All Step 15 recovery scenarios remain "
    "within observed leakage boundaries."
)

In [ ]:
# ================================================================
# 15.17 — SCENARIO RECONCILIATION CHECKS
# ================================================================

for _, row in gross_recovery_scenarios.iterrows():

    expected_revenue = (
        comparison_total_merchandise_revenue
        + row[
            "recovered_revenue"
        ]
    )


    assert np.isclose(
        expected_revenue,
        row[
            "scenario_total_merchandise_revenue"
        ],
        atol=0.01
    )


print(
    "✓ Step 15 scenario revenue calculations reconcile."
)

In [ ]:
# ================================================================
# 15.18 — STEP 15 SUMMARY METRICS
# ================================================================

step15_summary = (
    gross_recovery_scenarios
    [
        [
            "recovery_rate_pct",
            "recovered_revenue",
            "scenario_total_merchandise_revenue",
            "scenario_growth_pct",
            "growth_improvement_pp",
        ]
    ]
    .copy()
)


step15_summary[
    "Scenario"
] = (
    step15_summary[
        "recovery_rate_pct"
    ]
    .astype(int)
    .astype(str)
    + "% leakage reduction"
)


step15_summary = (
    step15_summary[
        [
            "Scenario",
            "recovered_revenue",
            "scenario_total_merchandise_revenue",
            "scenario_growth_pct",
            "growth_improvement_pp",
        ]
    ]
)


display(
    step15_summary
)

### Step 15 Summary — Retention-Improvement and Revenue-Recovery Scenarios

Step 15 tested the revenue sensitivity of reducing the **661.76k of measured customer revenue leakage** identified in Step 14. These calculations are illustrative scenarios rather than forecasts or guaranteed recoverable revenue.

Actual net merchandise revenue increased from approximately **1.401m in November 2010 to 1.433m in November 2011**, representing growth of approximately **2.30%**.

Even relatively small reductions in measured leakage would have materially changed that result. Avoiding revenue equivalent to:

- **5% of leakage** would add approximately **33.09k**, increasing illustrative growth to **4.66%**;
- **10%** would add approximately **66.18k**, increasing growth to **7.02%**;
- **15%** would add approximately **99.26k**, increasing growth to **9.38%**; and
- **20%** would add approximately **132.35k**, increasing growth to approximately **11.75%**.

This demonstrates the scale of the retention opportunity relative to the company's actual revenue increase. The approximately **33.09k uplift from reducing leakage by only 5% is slightly larger than the entire actual merchandise revenue increase of 32.17k** between the two periods.

#### Inactivity represents the larger recovery lever

Of the gross leakage base, approximately **419.97k resulted from customer inactivity**, compared with approximately **241.79k from customer contraction**.

As a result, the same percentage improvement produces substantially more revenue when applied to inactivity. For example, a 10% reduction would correspond to approximately:

- **42.00k from inactivity leakage**, compared with
- **24.18k from contraction leakage**.

Inactivity therefore provides approximately **1.7 times the mathematical revenue leverage** of contraction at an equivalent recovery rate.

#### Small improvements would materially strengthen revenue retention

The Step 14 gross period revenue-retention analogue was approximately **41.28%**.

Illustratively reducing gross leakage by:

- **5%** would increase it to approximately **44.21%**;
- **10%** to approximately **47.15%**;
- **15%** to approximately **50.09%**; and
- **20%** to approximately **53.02%**.

These are sensitivity calculations rather than predicted retention outcomes, but they demonstrate how strongly the revenue result responds to relatively small changes in customer leakage.

#### The required improvement to change the headline growth result is relatively modest

The reverse scenario analysis shows that, all else equal:

- achieving **3% merchandise revenue growth** would require additional revenue equivalent to only about **1.49% of measured leakage**;
- achieving **5% growth** would require approximately **37.86k**, equivalent to about **5.72% of leakage**; and
- achieving **7% growth** would require approximately **65.87k**, equivalent to about **9.95% of leakage**.

The key conclusion is therefore that the business did not need to eliminate customer leakage to produce a materially stronger revenue result. Even modest reductions in contraction and inactivity would have had a meaningful effect.

Overall, Step 15 shows that **customer retention represents a substantial economic lever**. The analysis does not prove that the modelled amounts are recoverable, but it demonstrates that reducing even a small fraction of observed leakage could materially improve the company's period-to-period revenue performance.



## Step 16 — New-Product Sustainability

Step 16 evaluates whether newly active merchandise products generated durable commercial value after their initial appearance.

Earlier analysis showed that newly active products contributed materially to revenue growth. However, a product appearing in one comparison month does not necessarily indicate a sustainable portfolio improvement.

This section therefore tracks products from their first positive merchandise-revenue month and measures whether they remain commercially active in subsequent months.

The analysis focuses on:

- product survival after 1, 3, 6 and 12 months;
- revenue retained relative to the launch month;
- the proportion of products that become one-month spikes;
- and whether a small number of newly active products account for most sustainable product growth.

Only positive merchandise activity is used to define product activation. Returns, cancellations and non-merchandise adjustments do not count as evidence that a product was commercially active.

In [ ]:
# ================================================================
# 16.2 — DEFINE POSITIVE PRODUCT ACTIVITY
# ================================================================

positive_product_sales = (
    df.loc[
        df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
    ]
    .copy()
)


print(
    f"Positive merchandise rows: "
    f"{len(positive_product_sales):,}"
)

print(
    f"Unique merchandise products: "
    f"{positive_product_sales['product'].nunique():,}"
)

In [ ]:
# ================================================================
# 16.3 — MONTHLY PRODUCT REVENUE
# ================================================================

monthly_product_revenue = (
    positive_product_sales
    .groupby(
        [
            "product",
            "product_description",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        positive_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        quantity_sold=(
            "quantity",
            "sum"
        ),

        customer_count=(
            "customer",
            "nunique"
        ),
    )
)


display(
    monthly_product_revenue.head()
)

In [ ]:
# ================================================================
# 16.4 — RESTRICT TO COMPLETE CALENDAR MONTHS
# ================================================================

product_analysis_end_month = pd.Period(
    "2011-11",
    freq="M"
)


monthly_product_revenue[
    "period_month_period"
] = (
    monthly_product_revenue[
        "period_month"
    ]
    .astype("period[M]")
)


monthly_product_complete = (
    monthly_product_revenue
    .loc[
        monthly_product_revenue[
            "period_month_period"
        ] <= product_analysis_end_month
    ]
    .copy()
)

In [ ]:
# ================================================================
# 16.5 — FIRST POSITIVE PRODUCT MONTH
# ================================================================

product_first_active = (
    monthly_product_complete
    .groupby(
        "product",
        as_index=False
    )
    .agg(
        first_active_month=(
            "period_month_period",
            "min"
        )
    )
)


display(
    product_first_active.head()
)

In [ ]:
# ================================================================
# 16.6 — IDENTIFY LEFT-TRUNCATED PRODUCTS
# ================================================================

product_dataset_start_month = (
    monthly_product_complete[
        "period_month_period"
    ]
    .min()
)


product_first_active[
    "product_is_left_truncated"
] = (
    product_first_active[
        "first_active_month"
    ]
    .eq(
        product_dataset_start_month
    )
)


print(
    f"Opening-month products: "
    f"{product_first_active['product_is_left_truncated'].sum():,}"
)

In [ ]:
# ================================================================
# 16.7 — OBSERVABLE PRODUCT-LAUNCH POPULATION
# ================================================================

observable_product_launches = (
    product_first_active
    .loc[
        ~product_first_active[
            "product_is_left_truncated"
        ]
    ]
    .copy()
)


print(
    f"Observable product launches: "
    f"{len(observable_product_launches):,}"
)

In [ ]:
# ================================================================
# 16.8 — ATTACH PRODUCT-LAUNCH MONTH
# ================================================================

product_lifecycle = (
    monthly_product_complete
    .merge(
        observable_product_launches[
            [
                "product",
                "first_active_month"
            ]
        ],
        on="product",
        how="inner"
    )
)

In [ ]:
# ================================================================
# 16.9 — CALCULATE MONTHS SINCE PRODUCT ACTIVATION
# ================================================================

product_lifecycle[
    "product_age_month"
] = (
    (
        product_lifecycle[
            "period_month_period"
        ].dt.year
        -
        product_lifecycle[
            "first_active_month"
        ].dt.year
    )
    * 12
    +
    (
        product_lifecycle[
            "period_month_period"
        ].dt.month
        -
        product_lifecycle[
            "first_active_month"
        ].dt.month
    )
)


assert (
    product_lifecycle[
        "product_age_month"
    ] >= 0
).all()

In [ ]:
# ================================================================
# 16.10 — PRODUCT LAUNCH-MONTH REVENUE
# ================================================================

launch_month_metrics = (
    product_lifecycle
    .loc[
        product_lifecycle[
            "product_age_month"
        ] == 0
    ]
    [
        [
            "product",
            "product_description",
            "first_active_month",
            "positive_revenue",
            "quantity_sold",
            "customer_count",
        ]
    ]
    .rename(
        columns={
            "positive_revenue":
                "launch_revenue",

            "quantity_sold":
                "launch_quantity",

            "customer_count":
                "launch_customers",
        }
    )
)


display(
    launch_month_metrics.head()
)

In [ ]:
# ================================================================
# 16.11 — PRODUCT REVENUE BY AGE
# ================================================================

product_age_revenue = (
    product_lifecycle
    .pivot_table(
        index="product",
        columns="product_age_month",
        values="positive_revenue",
        aggfunc="sum",
        fill_value=0
    )
)


display(
    product_age_revenue.head()
)

In [ ]:
# ================================================================
# 16.12 — PRODUCT HORIZON ELIGIBILITY
# ================================================================

product_launch_eligibility = (
    observable_product_launches
    .copy()
)


product_launch_eligibility[
    "observable_months"
] = (
    (
        product_analysis_end_month.year
        -
        product_launch_eligibility[
            "first_active_month"
        ].dt.year
    )
    * 12
    +
    (
        product_analysis_end_month.month
        -
        product_launch_eligibility[
            "first_active_month"
        ].dt.month
    )
)


display(
    product_launch_eligibility.head()
)

In [ ]:
# ================================================================
# 16.13 — PRODUCT SURVIVAL BY AGE
# ================================================================

product_survival_records = []


for month_age in [
    1,
    3,
    6,
    12,
]:

    eligible_products = (
        product_launch_eligibility
        .loc[
            product_launch_eligibility[
                "observable_months"
            ] >= month_age,
            "product"
        ]
    )


    eligible_count = (
        eligible_products.nunique()
    )


    active_products = (
        product_lifecycle
        .loc[
            product_lifecycle[
                "product"
            ].isin(
                eligible_products
            )
            &
            (
                product_lifecycle[
                    "product_age_month"
                ] == month_age
            )
            &
            (
                product_lifecycle[
                    "positive_revenue"
                ] > 0
            ),
            "product"
        ]
        .nunique()
    )


    survival_rate = (
        active_products
        / eligible_count
        * 100
        if eligible_count > 0
        else np.nan
    )


    product_survival_records.append(
        {
            "month_age":
                month_age,

            "eligible_products":
                eligible_count,

            "active_products":
                active_products,

            "survival_rate_pct":
                survival_rate,
        }
    )


product_survival_summary = pd.DataFrame(
    product_survival_records
)


display(
    product_survival_summary
)

In [ ]:
# ================================================================
# VISUAL 16.1 — PRODUCT SURVIVAL BY MONTH SINCE ACTIVATION
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(
    figsize=(9, 6)
)


sns.lineplot(
    data=product_survival_summary,
    x="month_age",
    y="survival_rate_pct",
    marker="o"
)


for _, row in product_survival_summary.iterrows():

    plt.text(
        row["month_age"],
        row["survival_rate_pct"],
        f"{row['survival_rate_pct']:.1f}%",
        ha="center",
        va="bottom"
    )


plt.title(
    "New-Product Survival After Initial Activation"
)

plt.xlabel(
    "Months Since First Positive Sales"
)

plt.ylabel(
    "Products Still Active (%)"
)

plt.xticks(
    [
        1,
        3,
        6,
        12
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 16.14 — PRODUCT REPEAT-ACTIVITY AFTER LAUNCH
# ================================================================

post_launch_activity = (
    product_lifecycle
    .loc[
        product_lifecycle[
            "product_age_month"
        ] > 0
    ]
    .groupby(
        "product",
        as_index=False
    )
    .agg(
        post_launch_active_months=(
            "period_month_period",
            "nunique"
        ),

        post_launch_revenue=(
            "positive_revenue",
            "sum"
        ),
    )
)


product_sustainability = (
    launch_month_metrics
    .merge(
        product_launch_eligibility[
            [
                "product",
                "observable_months"
            ]
        ],
        on="product",
        how="left"
    )
    .merge(
        post_launch_activity,
        on="product",
        how="left"
    )
)


product_sustainability[
    [
        "post_launch_active_months",
        "post_launch_revenue"
    ]
] = (
    product_sustainability[
        [
            "post_launch_active_months",
            "post_launch_revenue"
        ]
    ]
    .fillna(0)
)

In [ ]:
# ================================================================
# 16.15 — IDENTIFY ONE-MONTH PRODUCT SPIKES
# ================================================================

product_sustainability[
    "one_month_only"
] = (
    product_sustainability[
        "post_launch_active_months"
    ] == 0
)


one_month_product_count = (
    product_sustainability[
        "one_month_only"
    ]
    .sum()
)


one_month_product_share_pct = (
    one_month_product_count
    / len(product_sustainability)
    * 100
)


print(
    f"One-month-only products: "
    f"{one_month_product_count:,}"
)

print(
    f"Share of observable launches: "
    f"{one_month_product_share_pct:.2f}%"
)

In [ ]:
# ================================================================
# 16.16 — REVENUE RETENTION BY PRODUCT AGE
# ================================================================

product_revenue_retention_records = []


for month_age in [
    1,
    3,
    6,
    12,
]:

    eligible_products = (
        product_launch_eligibility
        .loc[
            product_launch_eligibility[
                "observable_months"
            ] >= month_age,
            "product"
        ]
    )


    eligible_launch = (
        launch_month_metrics
        .loc[
            launch_month_metrics[
                "product"
            ].isin(
                eligible_products
            )
        ]
        [
            [
                "product",
                "launch_revenue"
            ]
        ]
    )


    horizon_revenue = (
        product_lifecycle
        .loc[
            product_lifecycle[
                "product"
            ].isin(
                eligible_products
            )
            &
            (
                product_lifecycle[
                    "product_age_month"
                ] == month_age
            )
        ]
        .groupby(
            "product",
            as_index=False
        )
        .agg(
            horizon_revenue=(
                "positive_revenue",
                "sum"
            )
        )
    )


    horizon_table = (
        eligible_launch
        .merge(
            horizon_revenue,
            on="product",
            how="left"
        )
        .fillna(
            {
                "horizon_revenue": 0
            }
        )
    )


    launch_total = (
        horizon_table[
            "launch_revenue"
        ]
        .sum()
    )


    horizon_total = (
        horizon_table[
            "horizon_revenue"
        ]
        .sum()
    )


    revenue_retention_pct = (
        horizon_total
        / launch_total
        * 100
        if launch_total > 0
        else np.nan
    )


    product_revenue_retention_records.append(
        {
            "month_age":
                month_age,

            "eligible_products":
                len(horizon_table),

            "launch_revenue":
                launch_total,

            "horizon_revenue":
                horizon_total,

            "revenue_retention_pct":
                revenue_retention_pct,
        }
    )


product_revenue_retention = (
    pd.DataFrame(
        product_revenue_retention_records
    )
)


display(
    product_revenue_retention
)

In [ ]:
# ================================================================
# VISUAL 16.2 — REVENUE RETENTION AFTER PRODUCT ACTIVATION
# ================================================================

plt.figure(
    figsize=(9, 6)
)


sns.lineplot(
    data=product_revenue_retention,
    x="month_age",
    y="revenue_retention_pct",
    marker="o"
)


plt.title(
    "Revenue Retained Relative to Product Launch Month"
)

plt.xlabel(
    "Months Since First Positive Sales"
)

plt.ylabel(
    "Revenue Relative to Launch Month (%)"
)

plt.xticks(
    [
        1,
        3,
        6,
        12
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 16.17 — CLASSIFY PRODUCT SUSTAINABILITY
# ================================================================

def classify_product_sustainability(row):

    active_months = (
        row[
            "post_launch_active_months"
        ]
    )

    observable_months = (
        row[
            "observable_months"
        ]
    )


    if observable_months < 3:
        return "Too recent to assess"


    if active_months == 0:
        return "One-month only"


    if active_months <= 2:
        return "Short-lived"


    if active_months <= 5:
        return "Moderately persistent"


    return "Persistent"


product_sustainability[
    "sustainability_class"
] = (
    product_sustainability
    .apply(
        classify_product_sustainability,
        axis=1
    )
)

In [ ]:
# ================================================================
# 16.18 — PRODUCT SUSTAINABILITY SUMMARY
# ================================================================

product_sustainability_summary = (
    product_sustainability
    .groupby(
        "sustainability_class",
        as_index=False
    )
    .agg(
        products=(
            "product",
            "nunique"
        ),

        launch_revenue=(
            "launch_revenue",
            "sum"
        ),

        post_launch_revenue=(
            "post_launch_revenue",
            "sum"
        ),
    )
)


product_sustainability_summary[
    "product_share_pct"
] = (
    product_sustainability_summary[
        "products"
    ]
    / product_sustainability_summary[
        "products"
    ].sum()
    * 100
)


display(
    product_sustainability_summary
)

In [ ]:
# ================================================================
# VISUAL 16.3 — PRODUCT SUSTAINABILITY CLASSIFICATION
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=product_sustainability_summary,
    x="sustainability_class",
    y="product_share_pct"
)


plt.title(
    "Sustainability of Observable Product Launches"
)

plt.xlabel(
    "Product Sustainability Class"
)

plt.ylabel(
    "Share of Observable Product Launches (%)"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 16.19 — NOVEMBER 2011 NEWLY ACTIVE PRODUCTS
# ================================================================

nov_2011_new_products = (
    product_sustainability
    .loc[
        product_sustainability[
            "first_active_month"
        ]
        == pd.Period(
            comparison_period,
            freq="M"
        )
    ]
    .copy()
)


print(
    f"Products first active in {comparison_period}: "
    f"{len(nov_2011_new_products):,}"
)


display(
    nov_2011_new_products
    .sort_values(
        "launch_revenue",
        ascending=False
    )
    .head(20)
)

In [ ]:
# ================================================================
# 16.20 — UNOBSERVABLE SUSTAINABILITY OF NOV-2011 LAUNCHES
# ================================================================

nov_2011_launch_revenue = (
    nov_2011_new_products[
        "launch_revenue"
    ]
    .sum()
)


print(
    f"Revenue from products first active in "
    f"{comparison_period}: "
    f"{nov_2011_launch_revenue:,.2f}"
)

print(
    "Future sustainability cannot be observed "
    "within the available source history."
)

In [ ]:
# ================================================================
# 16.21 — DEFINE MATURE PRODUCT LAUNCHES
# ================================================================

mature_product_launches = (
    product_sustainability
    .loc[
        product_sustainability[
            "observable_months"
        ] >= 6
    ]
    .copy()
)


print(
    f"Product launches with at least "
    f"6 months observation: "
    f"{len(mature_product_launches):,}"
)

In [ ]:
# ================================================================
# 16.22 — MOST SUCCESSFUL MATURE PRODUCT LAUNCHES
# ================================================================

top_sustainable_products = (
    mature_product_launches
    .sort_values(
        "post_launch_revenue",
        ascending=False
    )
    [
        [
            "product",
            "product_description",
            "first_active_month",
            "launch_revenue",
            "post_launch_active_months",
            "post_launch_revenue",
            "sustainability_class",
        ]
    ]
    .head(20)
)


display(
    top_sustainable_products
)

In [ ]:
# ================================================================
# 16.23 — LARGE LAUNCHES WITH WEAK PERSISTENCE
# ================================================================

large_launch_threshold = (
    mature_product_launches[
        "launch_revenue"
    ]
    .quantile(0.75)
)


large_weak_launches = (
    mature_product_launches
    .loc[
        (
            mature_product_launches[
                "launch_revenue"
            ] >= large_launch_threshold
        )
        &
        (
            mature_product_launches[
                "post_launch_active_months"
            ] <= 2
        )
    ]
    .sort_values(
        "launch_revenue",
        ascending=False
    )
)


display(
    large_weak_launches[
        [
            "product",
            "product_description",
            "first_active_month",
            "launch_revenue",
            "post_launch_active_months",
            "post_launch_revenue",
        ]
    ]
    .head(20)
)

In [ ]:
# ================================================================
# 16.24 — LAUNCH SIZE VS POST-LAUNCH VALUE
# ================================================================

plt.figure(
    figsize=(10, 7)
)


sns.scatterplot(
    data=mature_product_launches,
    x="launch_revenue",
    y="post_launch_revenue",
    size="post_launch_active_months",
    sizes=(20, 300),
    alpha=0.7
)


plt.title(
    "Launch-Month Revenue vs Subsequent Product Revenue"
)

plt.xlabel(
    "Launch-Month Revenue"
)

plt.ylabel(
    "Post-Launch Revenue"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 16.25 — LAUNCH SIZE / SUSTAINABILITY CORRELATION
# ================================================================

launch_post_revenue_correlation = (
    mature_product_launches[
        [
            "launch_revenue",
            "post_launch_revenue"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)


print(
    f"Correlation between launch revenue "
    f"and post-launch revenue: "
    f"{launch_post_revenue_correlation:.3f}"
)

In [ ]:
# ================================================================
# 16.26 — VALIDATION CHECKS
# ================================================================

# Launch month should always be age zero.
assert (
    product_lifecycle[
        "product_age_month"
    ] >= 0
).all()


# Survival rates must remain within 0–100%.
assert (
    product_survival_summary[
        "survival_rate_pct"
    ]
    .between(
        0,
        100
    )
).all()


# Active products cannot exceed eligible products.
assert (
    product_survival_summary[
        "active_products"
    ]
    <=
    product_survival_summary[
        "eligible_products"
    ]
).all()


# One-month products cannot have post-launch revenue.
assert np.isclose(
    product_sustainability.loc[
        product_sustainability[
            "one_month_only"
        ],
        "post_launch_revenue"
    ]
    .sum(),
    0,
    atol=0.01
)


print(
    "✓ Step 16 product sustainability checks passed."
)

In [ ]:
# ================================================================
# 16.27 — STEP 16 SUMMARY METRICS
# ================================================================

step16_summary = pd.DataFrame(
    {
        "Metric": [
            "Observable product launches",
            "One-month-only products",
            "One-month-only product share (%)",
            "Mature launches with ≥6 months observation",
            f"Products first active in {comparison_period}",
            f"Revenue from {comparison_period} first-active products",
            "Launch vs post-launch revenue correlation",
        ],

        "Value": [
            len(product_sustainability),
            one_month_product_count,
            one_month_product_share_pct,
            len(mature_product_launches),
            len(nov_2011_new_products),
            nov_2011_launch_revenue,
            launch_post_revenue_correlation,
        ],
    }
)


display(
    step16_summary
)


print(
    "\nProduct survival:"
)

display(
    product_survival_summary
)


print(
    "\nRevenue retention by product age:"
)

display(
    product_revenue_retention
)

### Step 16 Summary — New-Product Sustainability

Step 16 examined whether products that first became commercially active during the observable dataset continued generating revenue after launch or disappeared shortly afterwards.

After excluding products already active in the opening month because their true launch date is unknown, the analysis identified approximately **1,716 observable product launches**.

#### Most product launches demonstrated meaningful persistence

Product survival remained relatively high after first activation:

- **83.6%** of eligible products were still active one month after launch;
- **79.0%** were active at Month 3;
- **72.0%** remained active at Month 6; and
- **66.2%** were still generating positive sales at Month 12.

Among all observable launches, approximately **59.9% were classified as persistent**, while only a relatively small proportion of sufficiently mature launches appeared to be genuine one-month or short-lived products.

This indicates that the product portfolio generally succeeded in converting new product introductions into continuing commercial activity rather than relying primarily on temporary launch spikes.

#### Revenue durability was also strong

The products that survived did not merely remain technically active at very low sales levels.

Collectively, eligible products generated revenue equivalent to approximately:

- **117.2% of launch-month revenue at Month 1**;
- **96.1% at Month 3**;
- **87.3% at Month 6**; and
- **81.8% at Month 12**.

The Month-1 result above 100% indicates that, collectively, eligible products generated more revenue one month after launch than during their initial activation month.

Even after twelve months, the relevant product cohorts were generating more than **80% of their original launch-month revenue**, providing evidence of meaningful product-level durability.

#### Several launches developed into major long-term products

The strongest sustainable launches included products such as **REGENCY CAKESTAND 3 TIER**, **RABBIT NIGHT LIGHT**, **JUMBO BAG PINK POLKADOT**, **PAPER CHAIN KIT VINTAGE CHRISTMAS**, and **JUMBO BAG VINTAGE DOILY**.

For example, **REGENCY CAKESTAND 3 TIER** generated approximately **9.89k in its first observed sales month** but subsequently produced more than **315k of additional positive merchandise revenue**.

However, launch size alone was not a strong predictor of long-term commercial success. The correlation between launch-month revenue and subsequent revenue was only approximately **0.19**, indicating that a large initial launch did not necessarily guarantee a successful long-term product.

#### An important distinction from Step 7 is required

Step 7 identified approximately **751 products as newly active in November 2011 relative to November 2010**. That does **not** mean all 751 were genuinely new products.

Many had traded previously but happened to have no positive revenue in the November 2010 baseline month.

Using the stricter Step 16 definition of a genuine first-observed product activation, only approximately **34 products first appeared in November 2011**, generating approximately **7.52k of positive launch revenue**.

Because November 2011 is the final complete month in the dataset, the subsequent sustainability of those 34 products **cannot be observed**.

### Overall conclusion

The historical evidence supports the conclusion that newly introduced products were generally capable of producing sustained revenue. Approximately **72% of eligible launches remained active after six months and 66% after twelve months**, while twelve-month cohort revenue remained approximately **82% of launch-month revenue**.

However, the analysis does **not** establish that the specific products first introduced in November 2011 subsequently succeeded, because the dataset ends before a complete post-launch observation period is available.

Step 16 therefore provides strong evidence of **historical product-launch sustainability**, while the future sustainability of the final-period launches remains unverified.


## Step 17 — Cross-Selling and Product Affinity

Step 17 examines product relationships within customer purchase baskets to identify potential cross-selling patterns.

The analysis measures how frequently products are purchased together and distinguishes simple co-occurrence from stronger-than-expected product affinity.

Three principal association measures are used:

- **Support** — the proportion of eligible baskets containing both products;
- **Confidence** — the probability that Product B appears when Product A is purchased; and
- **Lift** — how much more frequently the two products appear together than would be expected if their purchases were independent.

A lift above 1 indicates positive association, while a lift near 1 suggests that the apparent relationship may simply reflect product popularity.

The analysis focuses on positive merchandise purchases and excludes cancellations, returns and non-merchandise transactions.

Association does not establish causation or prove that one product caused the purchase of another. The findings should therefore be interpreted as observed basket relationships and potential cross-selling opportunities.

In [ ]:
# ================================================================
# 17.2 — BUILD POSITIVE MERCHANDISE BASKET LINES
# ================================================================

basket_lines = (
    df.loc[
        df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
    ]
    [
        [
            "invoice",
            "product",
            "product_description",
            "date",
            "period_month",
        ]
    ]
    .drop_duplicates(
        subset=[
            "invoice",
            "product"
        ]
    )
    .copy()
)


print(
    f"Eligible basket-product rows: "
    f"{len(basket_lines):,}"
)

print(
    f"Eligible invoices: "
    f"{basket_lines['invoice'].nunique():,}"
)

print(
    f"Unique products: "
    f"{basket_lines['product'].nunique():,}"
)

In [ ]:
# ================================================================
# 17.3 — BASKET SIZE DISTRIBUTION
# ================================================================

basket_size = (
    basket_lines
    .groupby(
        "invoice",
        as_index=False
    )
    .agg(
        distinct_products=(
            "product",
            "nunique"
        )
    )
)


display(
    basket_size[
        "distinct_products"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)

In [ ]:
# ================================================================
# VISUAL 17.1 — DISTINCT PRODUCTS PER BASKET
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


basket_plot = (
    basket_size
    .loc[
        basket_size[
            "distinct_products"
        ] <=
        basket_size[
            "distinct_products"
        ].quantile(0.99)
    ]
)


plt.figure(
    figsize=(10, 6)
)


sns.histplot(
    data=basket_plot,
    x="distinct_products",
    bins=30
)


plt.title(
    "Distribution of Distinct Products per Purchase Basket"
)

plt.xlabel(
    "Distinct Products per Invoice"
)

plt.ylabel(
    "Number of Invoices"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 17.4 — PRODUCT BASKET FREQUENCY
# ================================================================

total_baskets = (
    basket_lines[
        "invoice"
    ]
    .nunique()
)


product_basket_frequency = (
    basket_lines
    .groupby(
        [
            "product",
            "product_description"
        ],
        as_index=False
    )
    .agg(
        baskets=(
            "invoice",
            "nunique"
        )
    )
)


product_basket_frequency[
    "support_pct"
] = (
    product_basket_frequency[
        "baskets"
    ]
    / total_baskets
    * 100
)


product_basket_frequency = (
    product_basket_frequency
    .sort_values(
        "baskets",
        ascending=False
    )
)


display(
    product_basket_frequency.head(20)
)

In [ ]:
# ================================================================
# 17.5 — DEFINE AFFINITY PRODUCT UNIVERSE
# ================================================================

minimum_product_baskets = 50


affinity_products = (
    product_basket_frequency
    .loc[
        product_basket_frequency[
            "baskets"
        ] >= minimum_product_baskets,
        "product"
    ]
)


affinity_basket_lines = (
    basket_lines
    .loc[
        basket_lines[
            "product"
        ].isin(
            affinity_products
        )
    ]
    .copy()
)


print(
    f"Products appearing in at least "
    f"{minimum_product_baskets} baskets: "
    f"{affinity_products.nunique():,}"
)

In [ ]:
# ================================================================
# 17.6 — BUILD PRODUCT SET PER INVOICE
# ================================================================

invoice_baskets = (
    affinity_basket_lines
    .groupby(
        "invoice"
    )[
        "product"
    ]
    .apply(
        lambda x: sorted(
            set(x)
        )
    )
)


display(
    invoice_baskets.head()
)

In [ ]:
# ================================================================
# 17.7 — GENERATE PRODUCT PAIRS
# ================================================================

from itertools import combinations
from collections import Counter


pair_counter = Counter()


for products in invoice_baskets:

    if len(products) < 2:
        continue

    pair_counter.update(
        combinations(
            products,
            2
        )
    )


product_pairs = pd.DataFrame(
    [
        {
            "product_a": pair[0],
            "product_b": pair[1],
            "pair_baskets": count,
        }
        for pair, count
        in pair_counter.items()
    ]
)


print(
    f"Observed product pairs: "
    f"{len(product_pairs):,}"
)


display(
    product_pairs.head()
)

In [ ]:
# ================================================================
# 17.8 — ATTACH PRODUCT FREQUENCIES
# ================================================================

product_frequency_lookup = (
    product_basket_frequency
    .set_index(
        "product"
    )[
        "baskets"
    ]
)


product_pairs[
    "product_a_baskets"
] = (
    product_pairs[
        "product_a"
    ]
    .map(
        product_frequency_lookup
    )
)


product_pairs[
    "product_b_baskets"
] = (
    product_pairs[
        "product_b"
    ]
    .map(
        product_frequency_lookup
    )
)

In [ ]:
# ================================================================
# 17.9 — PRODUCT AFFINITY METRICS
# ================================================================

product_pairs[
    "pair_support_pct"
] = (
    product_pairs[
        "pair_baskets"
    ]
    / total_baskets
    * 100
)


product_pairs[
    "confidence_a_to_b_pct"
] = (
    product_pairs[
        "pair_baskets"
    ]
    / product_pairs[
        "product_a_baskets"
    ]
    * 100
)


product_pairs[
    "confidence_b_to_a_pct"
] = (
    product_pairs[
        "pair_baskets"
    ]
    / product_pairs[
        "product_b_baskets"
    ]
    * 100
)


product_pairs[
    "lift"
] = (
    (
        product_pairs[
            "pair_baskets"
        ]
        / total_baskets
    )
    /
    (
        (
            product_pairs[
                "product_a_baskets"
            ]
            / total_baskets
        )
        *
        (
            product_pairs[
                "product_b_baskets"
            ]
            / total_baskets
        )
    )
)

In [ ]:
# ================================================================
# 17.10 — ATTACH PRODUCT DESCRIPTIONS
# ================================================================

product_description_lookup = (
    product_basket_frequency
    .drop_duplicates(
        "product"
    )
    .set_index(
        "product"
    )[
        "product_description"
    ]
)


product_pairs[
    "product_a_description"
] = (
    product_pairs[
        "product_a"
    ]
    .map(
        product_description_lookup
    )
)


product_pairs[
    "product_b_description"
] = (
    product_pairs[
        "product_b"
    ]
    .map(
        product_description_lookup
    )
)

In [ ]:
# ================================================================
# 17.11 — FILTER MEANINGFUL PRODUCT PAIRS
# ================================================================

minimum_pair_baskets = 20


meaningful_product_pairs = (
    product_pairs
    .loc[
        product_pairs[
            "pair_baskets"
        ] >= minimum_pair_baskets
    ]
    .copy()
)


print(
    f"Pairs appearing together in at least "
    f"{minimum_pair_baskets} baskets: "
    f"{len(meaningful_product_pairs):,}"
)

In [ ]:
# ================================================================
# 17.12 — STRONGEST PRODUCT AFFINITY PAIRS
# ================================================================

strongest_affinity_pairs = (
    meaningful_product_pairs
    .sort_values(
        [
            "lift",
            "pair_baskets"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "product_a",
            "product_a_description",
            "product_b",
            "product_b_description",
            "pair_baskets",
            "pair_support_pct",
            "confidence_a_to_b_pct",
            "confidence_b_to_a_pct",
            "lift",
        ]
    ]
    .head(30)
)


display(
    strongest_affinity_pairs
)

In [ ]:
# ================================================================
# 17.13 — MOST FREQUENT PRODUCT PAIRS
# ================================================================

most_frequent_pairs = (
    meaningful_product_pairs
    .sort_values(
        "pair_baskets",
        ascending=False
    )
    [
        [
            "product_a_description",
            "product_b_description",
            "pair_baskets",
            "pair_support_pct",
            "lift",
        ]
    ]
    .head(20)
)


display(
    most_frequent_pairs
)

In [ ]:
# ================================================================
# VISUAL 17.2 — STRONG PRODUCT AFFINITY PAIRS
# ================================================================

top_affinity_plot = (
    strongest_affinity_pairs
    .head(15)
    .copy()
)


top_affinity_plot[
    "pair_label"
] = (
    top_affinity_plot[
        "product_a_description"
    ]
    .str.slice(
        0,
        28
    )
    +
    " + "
    +
    top_affinity_plot[
        "product_b_description"
    ]
    .str.slice(
        0,
        28
    )
)


plt.figure(
    figsize=(12, 9)
)


sns.barplot(
    data=top_affinity_plot,
    y="pair_label",
    x="lift"
)


plt.axvline(
    1,
    linestyle="--",
    linewidth=1
)


plt.title(
    "Strongest Supported Product Affinity Relationships"
)

plt.xlabel(
    "Lift"
)

plt.ylabel(
    "Product Pair"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 17.14 — DIRECTIONAL CROSS-SELL CONFIDENCE
# ================================================================

directional_affinity = (
    meaningful_product_pairs
    .copy()
)


directional_affinity[
    "stronger_direction"
] = np.where(
    directional_affinity[
        "confidence_a_to_b_pct"
    ]
    >=
    directional_affinity[
        "confidence_b_to_a_pct"
    ],

    "A → B",
    "B → A"
)


directional_affinity[
    "max_confidence_pct"
] = (
    directional_affinity[
        [
            "confidence_a_to_b_pct",
            "confidence_b_to_a_pct",
        ]
    ]
    .max(axis=1)
)


top_directional_cross_sell = (
    directional_affinity
    .loc[
        directional_affinity[
            "lift"
        ] > 1
    ]
    .sort_values(
        [
            "max_confidence_pct",
            "pair_baskets"
        ],
        ascending=False
    )
    .head(30)
)


display(
    top_directional_cross_sell[
        [
            "product_a_description",
            "product_b_description",
            "pair_baskets",
            "confidence_a_to_b_pct",
            "confidence_b_to_a_pct",
            "lift",
            "stronger_direction",
        ]
    ]
)

In [ ]:
# ================================================================
# 17.15 — CLASSIFY CROSS-SELL EVIDENCE
# ================================================================

def classify_affinity_evidence(row):

    if (
        row["pair_baskets"] >= 100
        and row["lift"] >= 2.0
        and row["max_confidence_pct"] >= 20
    ):
        return "Strong"


    if (
        row["pair_baskets"] >= 50
        and row["lift"] >= 1.5
        and row["max_confidence_pct"] >= 15
    ):
        return "Moderate"


    if (
        row["pair_baskets"] >= 20
        and row["lift"] > 1.0
    ):
        return "Weak"


    return "Insufficient"


directional_affinity[
    "affinity_evidence"
] = (
    directional_affinity
    .apply(
        classify_affinity_evidence,
        axis=1
    )
)


affinity_evidence_summary = (
    directional_affinity
    .groupby(
        "affinity_evidence",
        as_index=False
    )
    .agg(
        pairs=(
            "product_a",
            "size"
        )
    )
)


display(
    affinity_evidence_summary
)

In [ ]:
# ================================================================
# 17.16 — TOP PRODUCT AFFINITY MATRIX
# ================================================================

top_affinity_products = (
    product_basket_frequency
    .head(20)[
        "product"
    ]
    .tolist()
)


top_pair_subset = (
    product_pairs
    .loc[
        product_pairs[
            "product_a"
        ].isin(
            top_affinity_products
        )
        &
        product_pairs[
            "product_b"
        ].isin(
            top_affinity_products
        )
    ]
)


affinity_matrix = pd.DataFrame(
    1.0,
    index=top_affinity_products,
    columns=top_affinity_products
)


for _, row in top_pair_subset.iterrows():

    affinity_matrix.loc[
        row["product_a"],
        row["product_b"]
    ] = row["lift"]

    affinity_matrix.loc[
        row["product_b"],
        row["product_a"]
    ] = row["lift"]

In [ ]:
affinity_labels = {
    product:
        str(
            product_description_lookup.get(
                product,
                product
            )
        )[:25]

    for product in top_affinity_products
}


affinity_matrix = (
    affinity_matrix
    .rename(
        index=affinity_labels,
        columns=affinity_labels
    )
)

In [ ]:
# ================================================================
# VISUAL 17.3 — MAJOR PRODUCT AFFINITY HEATMAP
# ================================================================

plt.figure(
    figsize=(14, 11)
)


sns.heatmap(
    affinity_matrix,
    cmap="viridis",
    linewidths=0.3,
    cbar_kws={
        "label":
            "Lift"
    }
)


plt.title(
    "Basket Affinity Among Major Products"
)

plt.xlabel(
    ""
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 17.17 — IDENTIFY TOP REVENUE-GROWTH PRODUCTS
# Rebuilt directly from the canonical cleaned dataset
# ================================================================

product_growth_table = (
    df.loc[
        df["is_merchandise"]
        & df["period_month"].isin(
            [
                baseline_period,
                comparison_period
            ]
        )
    ]
    .groupby(
        [
            "product",
            "product_description",
            "period_month"
        ],
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


# ---------------------------------------------------------------
# Pivot baseline and comparison revenue
# ---------------------------------------------------------------

product_growth_table = (
    product_growth_table
    .pivot_table(
        index=[
            "product",
            "product_description"
        ],
        columns="period_month",
        values="net_merchandise_revenue",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


# ---------------------------------------------------------------
# Ensure both period columns exist
# ---------------------------------------------------------------

for period in [
    baseline_period,
    comparison_period
]:

    if period not in product_growth_table.columns:

        product_growth_table[
            period
        ] = 0


# ---------------------------------------------------------------
# Rename for clarity
# ---------------------------------------------------------------

product_growth_table = (
    product_growth_table
    .rename(
        columns={
            baseline_period:
                "baseline_revenue",

            comparison_period:
                "comparison_revenue",
        }
    )
)


# ---------------------------------------------------------------
# Calculate revenue change
# ---------------------------------------------------------------

product_growth_table[
    "revenue_change"
] = (
    product_growth_table[
        "comparison_revenue"
    ]
    -
    product_growth_table[
        "baseline_revenue"
    ]
)


# ---------------------------------------------------------------
# Identify top 30 positive-growth products
# ---------------------------------------------------------------

top_growth_products_table = (
    product_growth_table
    .loc[
        product_growth_table[
            "revenue_change"
        ] > 0
    ]
    .sort_values(
        "revenue_change",
        ascending=False
    )
    .head(30)
    .copy()
)


top_growth_products = (
    top_growth_products_table[
        "product"
    ]
    .tolist()
)


display(
    top_growth_products_table[
        [
            "product",
            "product_description",
            "baseline_revenue",
            "comparison_revenue",
            "revenue_change",
        ]
    ]
)


print(
    f"Top growth products selected: "
    f"{len(top_growth_products):,}"
)

In [ ]:
# ================================================================
# 17.18 — BASKET BREADTH AROUND GROWTH PRODUCTS
# ================================================================

basket_size_lookup = (
    basket_size
    .set_index(
        "invoice"
    )[
        "distinct_products"
    ]
)


growth_product_baskets = (
    basket_lines
    .loc[
        basket_lines[
            "product"
        ].isin(
            top_growth_products
        )
    ]
    [
        [
            "invoice",
            "product",
            "product_description"
        ]
    ]
    .drop_duplicates()
)


growth_product_baskets[
    "basket_size"
] = (
    growth_product_baskets[
        "invoice"
    ]
    .map(
        basket_size_lookup
    )
)


growth_product_breadth = (
    growth_product_baskets
    .groupby(
        [
            "product",
            "product_description"
        ],
        as_index=False
    )
    .agg(
        baskets=(
            "invoice",
            "nunique"
        ),

        average_basket_size=(
            "basket_size",
            "mean"
        ),

        median_basket_size=(
            "basket_size",
            "median"
        ),
    )
)


display(
    growth_product_breadth
    .sort_values(
        "average_basket_size",
        ascending=False
    )
    .head(20)
)

In [ ]:
# ================================================================
# 17.19 — GROWTH PRODUCT BASKET BREADTH VS PORTFOLIO
# ================================================================

portfolio_average_basket_size = (
    basket_size[
        "distinct_products"
    ]
    .mean()
)


growth_product_average_basket_size = (
    growth_product_baskets[
        "basket_size"
    ]
    .mean()
)


print(
    f"Portfolio average basket breadth: "
    f"{portfolio_average_basket_size:.2f} products"
)

print(
    f"Average basket breadth when a top-growth product appears: "
    f"{growth_product_average_basket_size:.2f} products"
)

In [ ]:
# ================================================================
# 17.20 — TOP COMPANIONS OF GROWTH PRODUCTS
# ================================================================

growth_affinity_pairs = (
    directional_affinity
    .loc[
        directional_affinity[
            "product_a"
        ].isin(
            top_growth_products
        )
        |
        directional_affinity[
            "product_b"
        ].isin(
            top_growth_products
        )
    ]
    .copy()
)


growth_affinity_pairs = (
    growth_affinity_pairs
    .sort_values(
        [
            "lift",
            "pair_baskets"
        ],
        ascending=[
            False,
            False
        ]
    )
)


display(
    growth_affinity_pairs[
        [
            "product_a_description",
            "product_b_description",
            "pair_baskets",
            "pair_support_pct",
            "confidence_a_to_b_pct",
            "confidence_b_to_a_pct",
            "lift",
            "affinity_evidence",
        ]
    ]
    .head(30)
)

In [ ]:
# ================================================================
# VISUAL 17.4 — GROWTH PRODUCTS AND BASKET BREADTH
# CLEAN / KERNEL-SAFE VERSION
# ================================================================

import matplotlib.pyplot as plt
import seaborn as sns


# Close any figures left open by earlier notebook cells
plt.close("all")


fig, ax = plt.subplots(
    figsize=(10, 7)
)


sns.scatterplot(
    data=growth_product_breadth,
    x="baskets",
    y="average_basket_size",
    s=90,
    alpha=0.75,
    ax=ax
)


# Portfolio benchmark
ax.axhline(
    y=portfolio_average_basket_size,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Portfolio average "
        f"({portfolio_average_basket_size:.1f} products)"
    )
)


ax.set_title(
    "Basket Breadth Associated with Top Revenue-Growth Products"
)

ax.set_xlabel(
    "Number of Baskets Containing Product"
)

ax.set_ylabel(
    "Average Distinct Products in Basket"
)

ax.legend()

plt.tight_layout()
plt.show()

# Explicitly release the figure
plt.close(fig)

In [ ]:
# ================================================================
# 17.21 — POTENTIAL PRODUCT BUNDLE CANDIDATES
# ================================================================

bundle_candidates = (
    directional_affinity
    .loc[
        (
            directional_affinity[
                "lift"
            ] >= 1.5
        )
        &
        (
            directional_affinity[
                "pair_baskets"
            ] >= 50
        )
    ]
    .sort_values(
        [
            "pair_baskets",
            "lift"
        ],
        ascending=False
    )
)


display(
    bundle_candidates[
        [
            "product_a_description",
            "product_b_description",
            "pair_baskets",
            "confidence_a_to_b_pct",
            "confidence_b_to_a_pct",
            "lift",
            "affinity_evidence",
        ]
    ]
    .head(30)
)

In [ ]:
# ================================================================
# 17.22 — PERIOD-SPECIFIC PRODUCT PAIR METRICS
# ================================================================

def calculate_period_pair_metrics(
    basket_data,
    period
):

    period_lines = (
        basket_data
        .loc[
            basket_data[
                "period_month"
            ] == period
        ]
        .copy()
    )


    period_baskets = (
        period_lines[
            "invoice"
        ]
        .nunique()
    )


    product_counts = (
        period_lines
        .groupby(
            "product"
        )[
            "invoice"
        ]
        .nunique()
    )


    basket_sets = (
        period_lines
        .groupby(
            "invoice"
        )[
            "product"
        ]
        .apply(
            lambda x:
                sorted(
                    set(x)
                )
        )
    )


    pair_counts = Counter()


    for products in basket_sets:

        if len(products) >= 2:

            pair_counts.update(
                combinations(
                    products,
                    2
                )
            )


    records = []


    for (
        product_a,
        product_b
    ), pair_count in pair_counts.items():

        a_count = (
            product_counts.get(
                product_a,
                0
            )
        )

        b_count = (
            product_counts.get(
                product_b,
                0
            )
        )


        if (
            a_count == 0
            or b_count == 0
            or period_baskets == 0
        ):
            continue


        lift = (
            (
                pair_count
                / period_baskets
            )
            /
            (
                (
                    a_count
                    / period_baskets
                )
                *
                (
                    b_count
                    / period_baskets
                )
            )
        )


        records.append(
            {
                "product_a":
                    product_a,

                "product_b":
                    product_b,

                "pair_baskets":
                    pair_count,

                "lift":
                    lift,
            }
        )


    return pd.DataFrame(
        records
    )

In [ ]:
nov_2010_affinity = (
    calculate_period_pair_metrics(
        basket_lines,
        baseline_period
    )
)


nov_2011_affinity = (
    calculate_period_pair_metrics(
        basket_lines,
        comparison_period
    )
)

In [ ]:
# ================================================================
# 17.23 — VALIDATION CHECKS
# ================================================================

assert (
    product_pairs[
        "pair_baskets"
    ] > 0
).all()


assert (
    product_pairs[
        "lift"
    ] > 0
).all()


assert (
    product_pairs[
        "confidence_a_to_b_pct"
    ]
    .between(
        0,
        100
    )
).all()


assert (
    product_pairs[
        "confidence_b_to_a_pct"
    ]
    .between(
        0,
        100
    )
).all()


assert (
    product_pairs[
        "pair_baskets"
    ]
    <=
    product_pairs[
        "product_a_baskets"
    ]
).all()


assert (
    product_pairs[
        "pair_baskets"
    ]
    <=
    product_pairs[
        "product_b_baskets"
    ]
).all()


print(
    "✓ Step 17 product-affinity validation checks passed."
)

In [ ]:
# ================================================================
# 17.24 — STEP 17 SUMMARY METRICS
# ================================================================

step17_summary = pd.DataFrame(
    {
        "Metric": [
            "Eligible baskets",
            "Unique merchandise products",
            "Products meeting affinity-frequency threshold",
            "Observed product pairs",
            "Meaningful pairs with ≥20 shared baskets",
            "Potential bundle candidates",
            "Portfolio average basket breadth",
            "Top-growth product basket breadth",
        ],

        "Value": [
            total_baskets,
            basket_lines[
                "product"
            ].nunique(),

            affinity_products.nunique(),

            len(
                product_pairs
            ),

            len(
                meaningful_product_pairs
            ),

            len(
                bundle_candidates
            ),

            portfolio_average_basket_size,

            growth_product_average_basket_size,
        ],
    }
)


display(
    step17_summary
)


print(
    "\nStrongest supported affinities:"
)


display(
    strongest_affinity_pairs.head(15)
)

### Step 17 Summary — Cross-Selling and Product Affinity

Step 17 examined which products are purchased together and whether those relationships are stronger than would be expected from product popularity alone.

The analysis covered approximately **39,519 eligible purchase baskets** and **4,899 merchandise products**. Basket breadth was substantial: the median basket contained **15 distinct products**, the 75th percentile contained **28 products**, and approximately **91.9% of baskets contained more than one product**.

This confirms that the business operates in a strongly multi-product purchasing environment, making cross-selling analysis commercially relevant.

#### Several product families show strong and repeated affinity

The most commercially important relationships combine both high co-purchase frequency and meaningful lift.

Examples include:

- **JUMBO BAG PINK POLKADOT + JUMBO BAG RED RETROSPOT** — purchased together in approximately **1,426 baskets**, with lift of about **6.29x**;
- **JUMBO STORAGE BAG SUKI + JUMBO BAG RED RETROSPOT** — approximately **1,324 shared baskets**, lift **5.63x**;
- **RED HANGING HEART T-LIGHT HOLDER + WHITE HANGING HEART T-LIGHT HOLDER** — approximately **1,234 shared baskets**, lift **5.16x**;
- **LUNCH BAG RED RETROSPOT + LUNCH BAG BLACK SKULL** — approximately **1,202 shared baskets**, lift **6.61x**;
- **PACK OF 72 RETROSPOT CAKE CASES + 60 TEATIME FAIRY CAKE CASES** — approximately **1,197 shared baskets**, lift **7.13x**.

These are stronger findings than simple co-purchase counts because lift shows that the combinations occur substantially more often than expected if the products were purchased independently.

#### Some smaller product families show exceptionally strong affinity

Several lower-volume product families produced very high lift values.

For example:

- **SET/6 FRUIT SALAD PAPER CUPS + SET/6 FRUIT SALAD PAPER PLATES** appeared together in 109 baskets with lift of approximately **221x**;
- **CAST IRON HOOK GARDEN FORK + CAST IRON HOOK GARDEN TROWEL** appeared together in 147 baskets with lift of approximately **195x**;
- **ENAMEL PINK TEA CONTAINER + ENAMEL PINK COFFEE CONTAINER** produced lift of approximately **170x**;
- **DOLLY GIRL CHILDRENS BOWL + DOLLY GIRL CHILDRENS CUP** produced lift of approximately **161x**.

These relationships suggest clear product-family purchasing behaviour and are potential candidates for recommendations, merchandising or bundle testing.

However, very high lift should not be interpreted alone. Commercial relevance still depends on transaction volume.

#### Top-growth products tend to appear in broader baskets

Across the full portfolio, the average basket contained approximately **25.1 distinct products**.

When one of the top 30 revenue-growth products was present, the average basket contained approximately **79.2 products**.

This is a very large descriptive difference and suggests that major growth products often participate in broader, high-item-count purchases.

However, this does **not** prove that the growth products caused larger baskets. The relationship may partly reflect customer type, wholesale buying patterns or large-order behaviour.

### Overall conclusion

Step 17 provides strong evidence that cross-selling is already embedded in customer purchasing behaviour.

The business has both:

- high-volume product combinations with meaningful lift; and
- smaller product families with exceptionally strong affinity.

The most actionable opportunities are therefore not simply the pairs with the highest lift, but the combinations that combine **strong association, sufficient transaction volume and meaningful directional confidence**.

These relationships can support future testing of product recommendations, bundles, merchandising placement and targeted cross-sell campaigns, but they should be treated as observed purchase associations rather than proof of causal cross-selling effects.


## Step 18 — Seasonality and Monthly Revenue Trends

Step 18 examines monthly revenue patterns across the complete observation period to determine whether the November 2010 versus November 2011 comparison reflects a broader commercial trend or is materially influenced by seasonal trading patterns.

The analysis focuses on complete calendar months only and therefore excludes the partial December 2011 period.

The section evaluates:

- monthly net merchandise revenue;
- year-over-year monthly growth where comparable periods exist;
- monthly gross versus net merchandise revenue;
- return and cancellation drag;
- active customer counts;
- revenue per active customer;
- active product counts;
- and the seasonal position of November relative to other months.

The objective is not to estimate a formal seasonal model, because only two annual cycles are available. Instead, the analysis identifies recurring monthly patterns and determines how much confidence can reasonably be placed on a November-to-November comparison.

In [ ]:
# ================================================================
# 18.2 — DEFINE COMPLETE MONTHLY ANALYSIS WINDOW
# ================================================================

monthly_analysis_start = pd.Period(
    "2010-01",
    freq="M"
)

monthly_analysis_end = pd.Period(
    "2011-11",
    freq="M"
)

In [ ]:
# ================================================================
# 18.3 — MONTHLY REVENUE SUMMARY
# ================================================================

# Create a reliable Period[M] version of the month field.
# This avoids Arrow string vs Period comparison errors.

df[
    "period_month_period"
] = pd.to_datetime(
    df[
        "period_month"
    ].astype(str)
).dt.to_period("M")


monthly_revenue = (
    df.loc[
        df[
            "period_month_period"
        ].between(
            monthly_analysis_start,
            monthly_analysis_end
        )
    ]
    .groupby(
        "period_month_period",
        as_index=False
    )
    .agg(
        net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        gross_merchandise_revenue=(
            "gross_merchandise_revenue",
            "sum"
        ),

        non_merchandise_revenue=(
            "non_merchandise_revenue",
            "sum"
        ),

        all_in_revenue=(
            "all_in_revenue",
            "sum"
        ),
    )
    .rename(
        columns={
            "period_month_period":
                "period_month"
        }
    )
)


monthly_revenue = (
    monthly_revenue
    .sort_values(
        "period_month"
    )
    .reset_index(
        drop=True
    )
)


display(
    monthly_revenue.head()
)

In [ ]:
# ================================================================
# 18.4 — MONTHLY GROWTH METRICS
# ================================================================

monthly_revenue[
    "month_over_month_change_pct"
] = (
    monthly_revenue[
        "net_merchandise_revenue"
    ]
    .pct_change()
    * 100
)


monthly_revenue[
    "year_over_year_change_pct"
] = (
    monthly_revenue[
        "net_merchandise_revenue"
    ]
    .pct_change(
        periods=12
    )
    * 100
)

In [ ]:
# ================================================================
# VISUAL 18.1 — MONTHLY NET MERCHANDISE REVENUE
# ================================================================

import seaborn as sns
import matplotlib.pyplot as plt


monthly_revenue[
    "month_label"
] = (
    monthly_revenue[
        "period_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(14, 6)
)


sns.lineplot(
    data=monthly_revenue,
    x="month_label",
    y="net_merchandise_revenue",
    marker="o"
)


plt.title(
    "Monthly Net Merchandise Revenue"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Net Merchandise Revenue"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.5 — STRONGEST AND WEAKEST MONTHS
# ================================================================

strongest_months = (
    monthly_revenue
    .nlargest(
        5,
        "net_merchandise_revenue"
    )
)


weakest_months = (
    monthly_revenue
    .nsmallest(
        5,
        "net_merchandise_revenue"
    )
)


print("Strongest months:")
display(
    strongest_months[
        [
            "period_month",
            "net_merchandise_revenue"
        ]
    ]
)


print("\nWeakest months:")
display(
    weakest_months[
        [
            "period_month",
            "net_merchandise_revenue"
        ]
    ]
)

In [ ]:
# ================================================================
# 18.6 — CALENDAR-MONTH SEASONAL PROFILE
# ================================================================

monthly_revenue[
    "calendar_month"
] = (
    monthly_revenue[
        "period_month"
    ].dt.month
)


monthly_revenue[
    "calendar_month_name"
] = (
    monthly_revenue[
        "period_month"
    ]
    .dt.strftime("%b")
)


calendar_month_profile = (
    monthly_revenue
    .groupby(
        [
            "calendar_month",
            "calendar_month_name"
        ],
        as_index=False
    )
    .agg(
        average_net_revenue=(
            "net_merchandise_revenue",
            "mean"
        ),

        median_net_revenue=(
            "net_merchandise_revenue",
            "median"
        ),

        observations=(
            "period_month",
            "size"
        ),
    )
    .sort_values(
        "calendar_month"
    )
)


display(
    calendar_month_profile
)

In [ ]:
# ================================================================
# VISUAL 18.2 — CALENDAR-MONTH REVENUE PROFILE
# ================================================================

plt.figure(
    figsize=(11, 6)
)


sns.barplot(
    data=calendar_month_profile,
    x="calendar_month_name",
    y="average_net_revenue"
)


plt.title(
    "Average Net Merchandise Revenue by Calendar Month"
)

plt.xlabel(
    "Calendar Month"
)

plt.ylabel(
    "Average Net Merchandise Revenue"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.7 — NOVEMBER RELATIVE TO YEARLY MONTHLY AVERAGE
# ================================================================

monthly_revenue[
    "year"
] = (
    monthly_revenue[
        "period_month"
    ].dt.year
)


yearly_monthly_average = (
    monthly_revenue
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        average_monthly_revenue=(
            "net_merchandise_revenue",
            "mean"
        )
    )
)


november_revenue = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "calendar_month"
        ] == 11
    ]
    [
        [
            "year",
            "period_month",
            "net_merchandise_revenue"
        ]
    ]
)


november_context = (
    november_revenue
    .merge(
        yearly_monthly_average,
        on="year",
        how="left"
    )
)


november_context[
    "november_vs_year_average_pct"
] = (
    (
        november_context[
            "net_merchandise_revenue"
        ]
        /
        november_context[
            "average_monthly_revenue"
        ]
    )
    - 1
) * 100


display(
    november_context
)

In [ ]:
# ================================================================
# 18.8 — GROSS / NET MONTHLY REVENUE
# ================================================================

monthly_revenue[
    "gross_to_net_drag"
] = (
    monthly_revenue[
        "gross_merchandise_revenue"
    ]
    -
    monthly_revenue[
        "net_merchandise_revenue"
    ]
)


monthly_revenue[
    "gross_to_net_drag_pct"
] = (
    monthly_revenue[
        "gross_to_net_drag"
    ]
    /
    monthly_revenue[
        "gross_merchandise_revenue"
    ]
    * 100
)

In [ ]:
# ================================================================
# VISUAL 18.3 — GROSS VS NET MONTHLY REVENUE
# ================================================================

gross_net_plot = (
    monthly_revenue
    .melt(
        id_vars=[
            "period_month",
            "month_label"
        ],
        value_vars=[
            "gross_merchandise_revenue",
            "net_merchandise_revenue"
        ],
        var_name="revenue_type",
        value_name="revenue"
    )
)


gross_net_plot[
    "revenue_type"
] = (
    gross_net_plot[
        "revenue_type"
    ]
    .replace(
        {
            "gross_merchandise_revenue":
                "Gross Merchandise Revenue",

            "net_merchandise_revenue":
                "Net Merchandise Revenue",
        }
    )
)


plt.figure(
    figsize=(14, 6)
)


sns.lineplot(
    data=gross_net_plot,
    x="month_label",
    y="revenue",
    hue="revenue_type",
    marker="o"
)


plt.title(
    "Gross vs Net Merchandise Revenue by Month"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Merchandise Revenue"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.9 — MONTHS WITH LARGEST GROSS-TO-NET DRAG
# ================================================================

largest_drag_months = (
    monthly_revenue
    .sort_values(
        "gross_to_net_drag",
        ascending=False
    )
    [
        [
            "period_month",
            "gross_merchandise_revenue",
            "net_merchandise_revenue",
            "gross_to_net_drag",
            "gross_to_net_drag_pct",
        ]
    ]
    .head(10)
)


display(
    largest_drag_months
)

In [ ]:
# ================================================================
# 18.10 — MONTHLY ACTIVE CUSTOMER METRICS
# FULL SELF-CONTAINED VERSION
# ================================================================

# Active identified customers by month
monthly_customer_activity = (
    df.loc[
        df["customer_is_identified"]
        & df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
        & df["period_month_period"].between(
            monthly_analysis_start,
            monthly_analysis_end
        )
    ]
    .groupby(
        "period_month_period",
        as_index=False
    )
    .agg(
        active_customers=(
            "customer",
            "nunique"
        )
    )
    .rename(
        columns={
            "period_month_period":
                "period_month"
        }
    )
)


# Identified-customer net merchandise revenue by month
monthly_identified_revenue = (
    df.loc[
        df["customer_is_identified"]
        & df["is_merchandise"]
        & df["period_month_period"].between(
            monthly_analysis_start,
            monthly_analysis_end
        )
    ]
    .groupby(
        "period_month_period",
        as_index=False
    )
    .agg(
        identified_net_merchandise_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
    .rename(
        columns={
            "period_month_period":
                "period_month"
        }
    )
)


# Merge customer counts with identified revenue
monthly_customer_activity = (
    monthly_customer_activity
    .merge(
        monthly_identified_revenue,
        on="period_month",
        how="left"
    )
)


# Revenue per active identified customer
monthly_customer_activity[
    "revenue_per_active_customer"
] = (
    monthly_customer_activity[
        "identified_net_merchandise_revenue"
    ]
    /
    monthly_customer_activity[
        "active_customers"
    ]
)


# Plotting label
monthly_customer_activity[
    "month_label"
] = (
    monthly_customer_activity[
        "period_month"
    ]
    .astype(str)
)


display(
    monthly_customer_activity.head()
)

print(
    monthly_customer_activity.columns.tolist()
)

In [ ]:
# ================================================================
# VISUAL 18.5 — REVENUE PER ACTIVE CUSTOMER
# ================================================================

# Create plotting label from Period[M] field
monthly_customer_activity[
    "month_label"
] = (
    monthly_customer_activity[
        "period_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(14, 6)
)


sns.lineplot(
    data=monthly_customer_activity,
    x="month_label",
    y="revenue_per_active_customer",
    marker="o"
)


plt.title(
    "Monthly Revenue per Active Identified Customer"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Revenue per Active Identified Customer"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.12 — MONTHLY ACTIVE PRODUCT COUNT
# ================================================================

monthly_product_activity = (
    df.loc[
        df["is_merchandise"]
        & df["quantity"].gt(0)
        & df["unit_price"].gt(0)
        & ~df["is_cancellation"]
        & df["period_month_period"].between(
            monthly_analysis_start,
            monthly_analysis_end
        )
    ]
    .groupby(
        "period_month_period",
        as_index=False
    )
    .agg(
        active_products=(
            "product",
            "nunique"
        )
    )
    .rename(
        columns={
            "period_month_period":
                "period_month"
        }
    )
)


display(
    monthly_product_activity.head()
)

In [ ]:
# ================================================================
# VISUAL 18.6 — ACTIVE PRODUCT COUNT
# ================================================================

monthly_product_activity[
    "month_label"
] = (
    monthly_product_activity[
        "period_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(14, 6)
)


sns.lineplot(
    data=monthly_product_activity,
    x="month_label",
    y="active_products",
    marker="o"
)


plt.title(
    "Monthly Active Merchandise Products"
)

plt.xlabel(
    "Month"
)

plt.ylabel(
    "Active Products"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.13 — MATCHED-MONTH YEAR-OVER-YEAR GROWTH
# ================================================================

yoy_monthly_growth = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "year_over_year_change_pct"
        ].notna()
    ]
    [
        [
            "period_month",
            "net_merchandise_revenue",
            "year_over_year_change_pct",
        ]
    ]
    .copy()
)


display(
    yoy_monthly_growth
)

In [ ]:
# ================================================================
# VISUAL 18.7 — MATCHED-MONTH YOY REVENUE GROWTH
# ================================================================

yoy_monthly_growth[
    "month_label"
] = (
    yoy_monthly_growth[
        "period_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(12, 6)
)


sns.barplot(
    data=yoy_monthly_growth,
    x="month_label",
    y="year_over_year_change_pct"
)


plt.axhline(
    0,
    linewidth=1
)


plt.title(
    "Year-over-Year Net Merchandise Revenue Growth"
)

plt.xlabel(
    "Comparison Month"
)

plt.ylabel(
    "Year-over-Year Growth (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.14 — NOVEMBER REVENUE RANK WITHIN YEAR
# ================================================================

monthly_revenue[
    "revenue_rank_within_year"
] = (
    monthly_revenue
    .groupby(
        "year"
    )[
        "net_merchandise_revenue"
    ]
    .rank(
        method="min",
        ascending=False
    )
)


november_rank = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "calendar_month"
        ] == 11
    ]
    [
        [
            "year",
            "period_month",
            "net_merchandise_revenue",
            "revenue_rank_within_year",
        ]
    ]
)


display(
    november_rank
)

In [ ]:
# ================================================================
# 18.15 — Q4 VS NON-Q4 REVENUE
# ================================================================

monthly_revenue[
    "quarter"
] = (
    monthly_revenue[
        "period_month"
    ].dt.quarter
)


monthly_revenue[
    "season_group"
] = np.where(
    monthly_revenue[
        "quarter"
    ] == 4,
    "Q4",
    "Non-Q4"
)


q4_comparison = (
    monthly_revenue
    .groupby(
        "season_group",
        as_index=False
    )
    .agg(
        average_monthly_revenue=(
            "net_merchandise_revenue",
            "mean"
        ),

        median_monthly_revenue=(
            "net_merchandise_revenue",
            "median"
        ),

        months=(
            "period_month",
            "size"
        ),
    )
)


display(
    q4_comparison
)

In [ ]:
# ================================================================
# 18.16 — MONTHLY REVENUE VOLATILITY
# ================================================================

monthly_mean_revenue = (
    monthly_revenue[
        "net_merchandise_revenue"
    ]
    .mean()
)


monthly_std_revenue = (
    monthly_revenue[
        "net_merchandise_revenue"
    ]
    .std()
)


monthly_cv_pct = (
    monthly_std_revenue
    / monthly_mean_revenue
    * 100
)


print(
    f"Average monthly net merchandise revenue: "
    f"{monthly_mean_revenue:,.2f}"
)

print(
    f"Monthly revenue standard deviation: "
    f"{monthly_std_revenue:,.2f}"
)

print(
    f"Coefficient of variation: "
    f"{monthly_cv_pct:.2f}%"
)

In [ ]:
# ================================================================
# 18.17 — REVENUE DRIVER CORRELATIONS
# ================================================================

monthly_driver_table = (
    monthly_revenue[
        [
            "period_month",
            "net_merchandise_revenue"
        ]
    ]
    .merge(
        monthly_customer_activity[
            [
                "period_month",
                "active_customers",
                "revenue_per_active_customer",
            ]
        ],
        on="period_month",
        how="left"
    )
    .merge(
        monthly_product_activity[
            [
                "period_month",
                "active_products"
            ]
        ],
        on="period_month",
        how="left"
    )
)


monthly_driver_correlations = (
    monthly_driver_table[
        [
            "net_merchandise_revenue",
            "active_customers",
            "revenue_per_active_customer",
            "active_products",
        ]
    ]
    .corr()
)


display(
    monthly_driver_correlations
)

In [ ]:
# ================================================================
# VISUAL 18.8 — MONTHLY DRIVER CORRELATION HEATMAP
# ================================================================

plt.figure(
    figsize=(8, 6)
)


sns.heatmap(
    monthly_driver_correlations,
    annot=True,
    fmt=".2f",
    linewidths=0.5
)


plt.title(
    "Monthly Revenue Driver Correlations"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 18.18 — VALIDATE NOVEMBER REVENUE AGAINST STEP 4
# ================================================================

nov_2010_revenue_check = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "period_month"
        ] == baseline_period,
        "net_merchandise_revenue"
    ]
    .sum()
)


nov_2011_revenue_check = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "period_month"
        ] == comparison_period,
        "net_merchandise_revenue"
    ]
    .sum()
)


print(
    f"November 2010 check: "
    f"{nov_2010_revenue_check:,.2f}"
)

print(
    f"November 2011 check: "
    f"{nov_2011_revenue_check:,.2f}"
)


assert np.isclose(
    nov_2010_revenue_check,
    baseline_total_merchandise_revenue,
    atol=0.01
)


assert np.isclose(
    nov_2011_revenue_check,
    comparison_total_merchandise_revenue,
    atol=0.01
)


print(
    "✓ Step 18 November revenue reconciles with earlier analysis."
)

In [ ]:
# ================================================================
# 18.19 — STEP 18 SUMMARY METRICS
# ================================================================

nov_2010_row = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "period_month"
        ] == baseline_period
    ]
    .iloc[0]
)


nov_2011_row = (
    monthly_revenue
    .loc[
        monthly_revenue[
            "period_month"
        ] == comparison_period
    ]
    .iloc[0]
)


step18_summary = pd.DataFrame(
    {
        "Metric": [
            "Average monthly net merchandise revenue",
            "Monthly revenue coefficient of variation (%)",
            "November 2010 revenue",
            "November 2011 revenue",
            "November YoY growth (%)",
            "November 2010 rank within year",
            "November 2011 rank within year",
            "November 2010 vs annual monthly average (%)",
            "November 2011 vs annual monthly average (%)",
        ],

        "Value": [
            monthly_mean_revenue,
            monthly_cv_pct,
            nov_2010_row[
                "net_merchandise_revenue"
            ],
            nov_2011_row[
                "net_merchandise_revenue"
            ],
            nov_2011_row[
                "year_over_year_change_pct"
            ],
            nov_2010_row[
                "revenue_rank_within_year"
            ],
            nov_2011_row[
                "revenue_rank_within_year"
            ],
            november_context.loc[
                november_context[
                    "year"
                ] == 2010,
                "november_vs_year_average_pct"
            ].iloc[0],
            november_context.loc[
                november_context[
                    "year"
                ] == 2011,
                "november_vs_year_average_pct"
            ].iloc[0],
        ],
    }
)


display(
    step18_summary
)

#### Customer activity explains much of the monthly revenue movement

Active identified customers increased from approximately **1,606 in November 2010 to 1,660 in November 2011**, an increase of roughly **3.4%**.

Across the monthly series, revenue movement was strongly associated with changes in the number of active customers and with the breadth of the active product portfolio.

Revenue per active customer was recalculated using identified-customer revenue so that the numerator and denominator refer to the same customer population. This provides a cleaner measure of customer spending intensity and should be interpreted alongside active-customer volume rather than as an independent causal driver.

The overall evidence indicates that monthly revenue variation was driven by a combination of:

- changes in the number of active customers;
- changes in revenue generated per active identified customer; and
- changes in the number of actively selling products.

Among these factors, active-customer volume appears to be particularly important.


## Step 19 — Concentration and Risk Synthesis

Step 19 consolidates the principal concentration and dependency risks identified throughout the analysis.

Revenue growth can appear commercially healthy while still becoming more dependent on a narrow group of countries, customers, products or unidentified transactions.

The analysis therefore compares November 2010 with November 2011 across four principal dimensions:

- geographic concentration;
- identified-customer concentration;
- product concentration;
- and unidentified-customer revenue exposure.

Herfindahl-Hirschman concentration measures are also used for geography to distinguish simple top-market dependency from broader portfolio concentration.

The purpose is not to create an arbitrary composite risk score. Instead, each concentration measure is evaluated separately so that the source and direction of risk remain transparent.

In [ ]:
# ================================================================
# 19.2 — BUILD ANCHOR-PERIOD MERCHANDISE DATA
# ================================================================

# ---------------------------------------------------------------
# Ensure canonical monthly Period field exists
# ---------------------------------------------------------------

if "period_month_period" not in df.columns:

    df[
        "period_month_period"
    ] = (
        pd.to_datetime(
            df["period_month"].astype(str),
            errors="coerce"
        )
        .dt.to_period("M")
    )


baseline_period_p = pd.Period(
    str(baseline_period),
    freq="M"
)

comparison_period_p = pd.Period(
    str(comparison_period),
    freq="M"
)


# ---------------------------------------------------------------
# Validate required Step 19 fields
# ---------------------------------------------------------------

required_step19_columns = [
    "period_month_period",
    "region",
    "customer",
    "customer_is_identified",
    "product",
    "product_description",
    "is_merchandise",
    "net_merchandise_revenue",
]


missing_step19_columns = [
    column
    for column in required_step19_columns
    if column not in df.columns
]


if missing_step19_columns:

    raise KeyError(
        "Step 19 is missing required columns: "
        f"{missing_step19_columns}"
    )


# ---------------------------------------------------------------
# Anchor-period merchandise population
# ---------------------------------------------------------------

risk_period_data = (
    df.loc[
        df["is_merchandise"]
        &
        df["period_month_period"].isin(
            [
                baseline_period_p,
                comparison_period_p,
            ]
        )
    ]
    .copy()
)


# Give missing geography an explicit category so it is not
# silently removed from concentration calculations.
risk_period_data[
    "region_analysis"
] = (
    risk_period_data[
        "region"
    ]
    .astype("string")
    .fillna("Unknown Region")
    .str.strip()
)


print(
    f"Baseline period: {baseline_period_p}"
)

print(
    f"Comparison period: {comparison_period_p}"
)

print(
    f"Merchandise rows included: "
    f"{len(risk_period_data):,}"
)

print(
    f"Regions represented: "
    f"{risk_period_data['region_analysis'].nunique():,}"
)

In [ ]:
# ================================================================
# 19.3 — GEOGRAPHIC CONCENTRATION
# ================================================================

region_period_revenue = (
    risk_period_data
    .groupby(
        [
            "period_month_period",
            "region_analysis"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        net_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


geographic_concentration_records = []


for period in [
    baseline_period_p,
    comparison_period_p,
]:

    temp = (
        region_period_revenue
        .loc[
            region_period_revenue[
                "period_month_period"
            ] == period
        ]
        .copy()
    )


    total_revenue = (
        temp[
            "net_revenue"
        ]
        .sum()
    )


    if np.isclose(
        total_revenue,
        0
    ):

        raise ValueError(
            f"Net merchandise revenue is zero "
            f"for {period}; geographic shares "
            f"cannot be calculated."
        )


    temp[
        "revenue_share"
    ] = (
        temp[
            "net_revenue"
        ]
        / total_revenue
    )


    # -----------------------------------------------------------
    # UK share
    # -----------------------------------------------------------

    uk_share_pct = (
        temp.loc[
            temp[
                "region_analysis"
            ]
            .astype(str)
            .str.strip()
            .str.casefold()
            .eq(
                "united kingdom"
            ),
            "revenue_share"
        ]
        .sum()
        * 100
    )


    # -----------------------------------------------------------
    # Top-five geographic markets
    # -----------------------------------------------------------

    top5_region_share_pct = (
        temp[
            "revenue_share"
        ]
        .nlargest(5)
        .sum()
        * 100
    )


    # -----------------------------------------------------------
    # HHI on percentage revenue shares
    # -----------------------------------------------------------

    country_hhi = (
        (
            temp[
                "revenue_share"
            ]
            * 100
        ) ** 2
    ).sum()


    geographic_concentration_records.append(
        {
            "period":
                period,

            "uk_share_pct":
                uk_share_pct,

            "top5_country_share_pct":
                top5_region_share_pct,

            "country_hhi":
                country_hhi,
        }
    )


country_concentration = pd.DataFrame(
    geographic_concentration_records
)


display(
    country_concentration
)

In [ ]:
# ================================================================
# 19.4 — IDENTIFIED CUSTOMER CONCENTRATION
# ================================================================

customer_period_revenue = (
    risk_period_data
    .loc[
        risk_period_data[
            "customer_is_identified"
        ]
    ]
    .groupby(
        [
            "period_month_period",
            "customer"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        net_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


customer_concentration_records = []


for period in [
    baseline_period_p,
    comparison_period_p,
]:

    temp = (
        customer_period_revenue
        .loc[
            (
                customer_period_revenue[
                    "period_month_period"
                ] == period
            )
            &
            (
                customer_period_revenue[
                    "net_revenue"
                ] > 0
            )
        ]
        .copy()
    )


    positive_revenue = (
        temp[
            "net_revenue"
        ]
        .sum()
    )


    if positive_revenue <= 0:

        raise ValueError(
            f"No positive identified-customer "
            f"revenue available for {period}."
        )


    temp[
        "revenue_share"
    ] = (
        temp[
            "net_revenue"
        ]
        / positive_revenue
    )


    customer_concentration_records.append(
        {
            "period":
                period,

            "top1_customer_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(1)
                .sum()
                * 100,

            "top5_customer_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(5)
                .sum()
                * 100,

            "top10_customer_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(10)
                .sum()
                * 100,
        }
    )


customer_concentration = pd.DataFrame(
    customer_concentration_records
)


display(
    customer_concentration
)

In [ ]:
# ================================================================
# 19.5 — PRODUCT CONCENTRATION
# ================================================================

product_period_revenue = (
    risk_period_data
    .groupby(
        [
            "period_month_period",
            "product",
            "product_description"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        net_revenue=(
            "net_merchandise_revenue",
            "sum"
        )
    )
)


product_concentration_records = []


for period in [
    baseline_period_p,
    comparison_period_p,
]:

    temp = (
        product_period_revenue
        .loc[
            (
                product_period_revenue[
                    "period_month_period"
                ] == period
            )
            &
            (
                product_period_revenue[
                    "net_revenue"
                ] > 0
            )
        ]
        .copy()
    )


    positive_revenue = (
        temp[
            "net_revenue"
        ]
        .sum()
    )


    if positive_revenue <= 0:

        raise ValueError(
            f"No positive product revenue "
            f"available for {period}."
        )


    temp[
        "revenue_share"
    ] = (
        temp[
            "net_revenue"
        ]
        / positive_revenue
    )


    product_concentration_records.append(
        {
            "period":
                period,

            "top1_product_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(1)
                .sum()
                * 100,

            "top5_product_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(5)
                .sum()
                * 100,

            "top10_product_share_pct":
                temp[
                    "revenue_share"
                ]
                .nlargest(10)
                .sum()
                * 100,
        }
    )


product_concentration = pd.DataFrame(
    product_concentration_records
)


display(
    product_concentration
)

In [ ]:
# ================================================================
# 19.6 — UNIDENTIFIED CUSTOMER REVENUE EXPOSURE
# ================================================================

unknown_revenue_records = []


for period in [
    baseline_period_p,
    comparison_period_p,
]:

    temp = (
        risk_period_data
        .loc[
            risk_period_data[
                "period_month_period"
            ] == period
        ]
        .copy()
    )


    total_net_revenue = (
        temp[
            "net_merchandise_revenue"
        ]
        .sum()
    )


    unidentified_net_revenue = (
        temp.loc[
            ~temp[
                "customer_is_identified"
            ],
            "net_merchandise_revenue"
        ]
        .sum()
    )


    unknown_revenue_records.append(
        {
            "period":
                period,

            "unidentified_revenue":
                unidentified_net_revenue,

            "unidentified_revenue_share_pct":
                (
                    unidentified_net_revenue
                    / total_net_revenue
                    * 100
                )
                if not np.isclose(
                    total_net_revenue,
                    0
                )
                else np.nan,
        }
    )


unidentified_revenue_risk = pd.DataFrame(
    unknown_revenue_records
)


display(
    unidentified_revenue_risk
)

In [ ]:
# ================================================================
# 19.7 — BUILD CONSOLIDATED CONCENTRATION TABLE
# ================================================================

concentration_summary = (
    country_concentration
    .merge(
        customer_concentration,
        on="period",
        how="left"
    )
    .merge(
        product_concentration,
        on="period",
        how="left"
    )
    .merge(
        unidentified_revenue_risk[
            [
                "period",
                "unidentified_revenue_share_pct",
            ]
        ],
        on="period",
        how="left"
    )
)


display(
    concentration_summary
)

In [ ]:
# ================================================================
# 19.8 — CONCENTRATION CHANGE SUMMARY
# ================================================================

metric_labels = {
    "uk_share_pct":
        "UK revenue share",

    "top5_country_share_pct":
        "Top 5 geographic market share",

    "country_hhi":
        "Geographic revenue HHI",

    "top1_customer_share_pct":
        "Top 1 customer share",

    "top5_customer_share_pct":
        "Top 5 customer share",

    "top10_customer_share_pct":
        "Top 10 customer share",

    "top1_product_share_pct":
        "Top 1 product share",

    "top5_product_share_pct":
        "Top 5 product share",

    "top10_product_share_pct":
        "Top 10 product share",

    "unidentified_revenue_share_pct":
        "Unidentified customer revenue share",
}


baseline_row = (
    concentration_summary
    .loc[
        concentration_summary[
            "period"
        ] == baseline_period_p
    ]
    .iloc[0]
)


comparison_row = (
    concentration_summary
    .loc[
        concentration_summary[
            "period"
        ] == comparison_period_p
    ]
    .iloc[0]
)


risk_change_records = []


for metric, label in metric_labels.items():

    baseline_value = (
        baseline_row[
            metric
        ]
    )

    comparison_value = (
        comparison_row[
            metric
        ]
    )


    risk_change_records.append(
        {
            "metric":
                metric,

            "risk_dimension":
                label,

            "baseline":
                baseline_value,

            "comparison":
                comparison_value,

            "change":
                comparison_value
                - baseline_value,
        }
    )


risk_change_summary = pd.DataFrame(
    risk_change_records
)


display(
    risk_change_summary
)

In [ ]:
# ================================================================
# 19.9 — CLASSIFY SHARE-METRIC RISK DIRECTION
# ================================================================

risk_change_summary[
    "risk_direction"
] = "Interpret separately"


share_metric_mask = (
    risk_change_summary[
        "metric"
    ] != "country_hhi"
)


risk_change_summary.loc[
    share_metric_mask,
    "risk_direction"
] = np.select(
    [
        risk_change_summary.loc[
            share_metric_mask,
            "change"
        ] > 0.10,

        risk_change_summary.loc[
            share_metric_mask,
            "change"
        ] < -0.10,
    ],
    [
        "Increased",
        "Reduced",
    ],
    default="Broadly stable"
)


risk_change_summary.loc[
    risk_change_summary[
        "metric"
    ] == "country_hhi",
    "risk_direction"
] = "Interpret HHI change separately"


display(
    risk_change_summary
)

In [ ]:
# ================================================================
# VISUAL 19.1 — CONCENTRATION CHANGE
# ================================================================

share_metrics = (
    risk_change_summary
    .loc[
        risk_change_summary[
            "metric"
        ] != "country_hhi"
    ]
    .copy()
)


concentration_plot = (
    share_metrics
    .melt(
        id_vars=[
            "risk_dimension"
        ],
        value_vars=[
            "baseline",
            "comparison"
        ],
        var_name="period",
        value_name="share_pct"
    )
)


concentration_plot[
    "period"
] = (
    concentration_plot[
        "period"
    ]
    .replace(
        {
            "baseline":
                str(baseline_period),

            "comparison":
                str(comparison_period),
        }
    )
)


plt.figure(
    figsize=(12, 8)
)


sns.barplot(
    data=concentration_plot,
    y="risk_dimension",
    x="share_pct",
    hue="period"
)


plt.title(
    "Revenue Concentration and Dependency Measures"
)

plt.xlabel(
    "Revenue Share (%)"
)

plt.ylabel(
    ""
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 19.2 — GEOGRAPHIC CONCENTRATION HHI
# ================================================================

hhi_plot = (
    country_concentration
    .copy()
)


hhi_plot[
    "period_label"
] = (
    hhi_plot[
        "period"
    ]
    .astype(str)
)


plt.figure(
    figsize=(8, 5)
)


sns.barplot(
    data=hhi_plot,
    x="period_label",
    y="country_hhi"
)


plt.title(
    "Geographic Revenue Concentration"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Country HHI"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 19.10 — LARGEST GEOGRAPHIC EXPOSURES
# ================================================================

comparison_country_exposure = (
    region_period_revenue
    .loc[
        region_period_revenue[
            "period_month_period"
        ] == comparison_period_p
    ]
    .sort_values(
        "net_revenue",
        ascending=False
    )
    .copy()
)


comparison_total_region_revenue = (
    comparison_country_exposure[
        "net_revenue"
    ]
    .sum()
)


comparison_country_exposure[
    "revenue_share_pct"
] = (
    comparison_country_exposure[
        "net_revenue"
    ]
    / comparison_total_region_revenue
    * 100
)


display(
    comparison_country_exposure[
        [
            "region_analysis",
            "net_revenue",
            "revenue_share_pct",
        ]
    ]
    .head(10)
)

In [ ]:
# ================================================================
# 19.11 — LARGEST CUSTOMER EXPOSURES
# ================================================================

comparison_customer_exposure = (
    customer_period_revenue
    .loc[
        (
            customer_period_revenue[
                "period_month_period"
            ] == comparison_period_p
        )
        &
        (
            customer_period_revenue[
                "net_revenue"
            ] > 0
        )
    ]
    .sort_values(
        "net_revenue",
        ascending=False
    )
    .copy()
)


comparison_customer_exposure[
    "revenue_share_pct"
] = (
    comparison_customer_exposure[
        "net_revenue"
    ]
    /
    comparison_customer_exposure[
        "net_revenue"
    ].sum()
    * 100
)


display(
    comparison_customer_exposure.head(10)
)

In [ ]:
# ================================================================
# 19.12 — LARGEST PRODUCT EXPOSURES
# ================================================================

comparison_product_exposure = (
    product_period_revenue
    .loc[
        (
            product_period_revenue[
                "period_month_period"
            ] == comparison_period_p
        )
        &
        (
            product_period_revenue[
                "net_revenue"
            ] > 0
        )
    ]
    .sort_values(
        "net_revenue",
        ascending=False
    )
    .copy()
)


comparison_product_exposure[
    "revenue_share_pct"
] = (
    comparison_product_exposure[
        "net_revenue"
    ]
    /
    comparison_product_exposure[
        "net_revenue"
    ].sum()
    * 100
)


display(
    comparison_product_exposure.head(10)
)

In [ ]:
# ================================================================
# 19.13 — CUSTOMER LEAKAGE CONCENTRATION
# ================================================================

if (
    "customer_leakage" in globals()
    and
    "gross_customer_leakage" in globals()
):

    top_1_leakage_share = (
        customer_leakage[
            "total_leakage"
        ]
        .head(1)
        .sum()
        / gross_customer_leakage
        * 100
    )


    top_5_leakage_share = (
        customer_leakage[
            "total_leakage"
        ]
        .head(5)
        .sum()
        / gross_customer_leakage
        * 100
    )


    top_10_leakage_share = (
        customer_leakage[
            "total_leakage"
        ]
        .head(10)
        .sum()
        / gross_customer_leakage
        * 100
    )


    leakage_concentration_summary = pd.DataFrame(
        {
            "Metric": [
                "Top 1 customer share of leakage",
                "Top 5 customer share of leakage",
                "Top 10 customer share of leakage",
            ],

            "Share (%)": [
                top_1_leakage_share,
                top_5_leakage_share,
                top_10_leakage_share,
            ],
        }
    )


    display(
        leakage_concentration_summary
    )

else:

    print(
        "Step 14 leakage variables are unavailable. "
        "Run Step 14 before Step 19."
    )

In [ ]:
# ================================================================
# 19.14 — EXECUTIVE RISK SYNTHESIS
# ================================================================

risk_synthesis = pd.DataFrame(
    {
        "Risk Area": [
            "Geographic dependency",
            "Customer concentration",
            "Product concentration",
            "Customer identification",
            "Customer revenue leakage",
            "Seasonality",
        ],

        "Evidence": [
            (
                f"UK share "
                f"{comparison_row['uk_share_pct']:.2f}% "
                f"and geographic HHI "
                f"{comparison_row['country_hhi']:.0f}"
            ),

            (
                f"Top 10 customers represent "
                f"{comparison_row['top10_customer_share_pct']:.2f}% "
                f"of positive identified-customer revenue"
            ),

            (
                f"Top 10 products represent "
                f"{comparison_row['top10_product_share_pct']:.2f}% "
                f"of positive product revenue"
            ),

            (
                f"{comparison_row['unidentified_revenue_share_pct']:.2f}% "
                f"of net merchandise revenue is unidentified"
            ),

            (
                f"Gross customer leakage measured at "
                f"{gross_customer_leakage:,.0f}"
                if "gross_customer_leakage" in globals()
                else
                "See Step 14"
            ),

            (
                "November is a peak trading month; "
                "see Step 18"
            ),
        ],

        "Interpretation": [
            "High geographic dependency",
            "Moderate customer dependency",
            "Relatively diversified product base",
            "Material customer-data limitation",
            "Broadly distributed customer erosion",
            "Material seasonal exposure",
        ],
    }
)


display(
    risk_synthesis
)

In [ ]:
# ================================================================
# 19.15 — VALIDATION CHECKS
# ================================================================

share_columns = [
    "uk_share_pct",
    "top5_country_share_pct",
    "top1_customer_share_pct",
    "top5_customer_share_pct",
    "top10_customer_share_pct",
    "top1_product_share_pct",
    "top5_product_share_pct",
    "top10_product_share_pct",
    "unidentified_revenue_share_pct",
]


for column in share_columns:

    assert (
        concentration_summary[
            column
        ]
        .between(
            0,
            100
        )
        .all()
    ), f"{column} is outside 0–100%."


# Customer concentration hierarchy
assert (
    concentration_summary[
        "top1_customer_share_pct"
    ]
    <=
    concentration_summary[
        "top5_customer_share_pct"
    ]
).all()


assert (
    concentration_summary[
        "top5_customer_share_pct"
    ]
    <=
    concentration_summary[
        "top10_customer_share_pct"
    ]
).all()


# Product concentration hierarchy
assert (
    concentration_summary[
        "top1_product_share_pct"
    ]
    <=
    concentration_summary[
        "top5_product_share_pct"
    ]
).all()


assert (
    concentration_summary[
        "top5_product_share_pct"
    ]
    <=
    concentration_summary[
        "top10_product_share_pct"
    ]
).all()


# Geographic market revenue should reconcile to merchandise revenue
for period in [
    baseline_period_p,
    comparison_period_p,
]:

    region_total = (
        region_period_revenue.loc[
            region_period_revenue[
                "period_month_period"
            ] == period,
            "net_revenue"
        ]
        .sum()
    )


    merchandise_total = (
        risk_period_data.loc[
            risk_period_data[
                "period_month_period"
            ] == period,
            "net_merchandise_revenue"
        ]
        .sum()
    )


    assert np.isclose(
        region_total,
        merchandise_total,
        atol=0.01
    ), (
        f"Geographic revenue does not reconcile "
        f"for {period}."
    )


print(
    "✓ Step 19 concentration calculations validated."
)

In [ ]:
# ================================================================
# 19.16 — STEP 19 SUMMARY METRICS
# ================================================================

step19_summary = pd.DataFrame(
    {
        "Metric": [
            "UK revenue share (%)",
            "Top 5 geographic market share (%)",
            "Geographic HHI",
            "Top 10 customer share (%)",
            "Top 10 product share (%)",
            "Unidentified revenue share (%)",
        ],

        str(baseline_period): [
            baseline_row[
                "uk_share_pct"
            ],

            baseline_row[
                "top5_country_share_pct"
            ],

            baseline_row[
                "country_hhi"
            ],

            baseline_row[
                "top10_customer_share_pct"
            ],

            baseline_row[
                "top10_product_share_pct"
            ],

            baseline_row[
                "unidentified_revenue_share_pct"
            ],
        ],

        str(comparison_period): [
            comparison_row[
                "uk_share_pct"
            ],

            comparison_row[
                "top5_country_share_pct"
            ],

            comparison_row[
                "country_hhi"
            ],

            comparison_row[
                "top10_customer_share_pct"
            ],

            comparison_row[
                "top10_product_share_pct"
            ],

            comparison_row[
                "unidentified_revenue_share_pct"
            ],
        ],
    }
)


step19_summary[
    "Change"
] = (
    step19_summary[
        str(comparison_period)
    ]
    -
    step19_summary[
        str(baseline_period)
    ]
)


display(
    step19_summary
)

### Step 19 Summary — Concentration and Risk Synthesis

Step 19 consolidated the principal dependency and concentration risks across geography, customers, products and customer identification.

The overall result is mixed:

- the business is **highly concentrated geographically**;
- but comparatively **well diversified across individual customers and products**;
- while the share of revenue that cannot be linked to an identified customer increased materially.

#### Geographic dependency is the clearest structural concentration risk

The United Kingdom accounted for approximately **87.54% of net merchandise revenue in November 2010** and **88.24% in November 2011**.

This represents an increase of approximately **0.70 percentage points**.

The top five geographic markets collectively increased from approximately **95.27% to 95.70% of revenue**.

Geographic HHI also increased from approximately **7,682 to 7,803**, confirming that the revenue base became slightly more geographically concentrated rather than more diversified.

The UK alone generated approximately **1.264 million of the 1.433 million November 2011 merchandise revenue**.

The next-largest markets were much smaller:

- **Eire — approximately 29.5k**
- **France — approximately 28.3k**
- **Netherlands — approximately 25.6k**
- **Germany — approximately 23.3k**

The geographic risk is therefore not simply that the UK is the largest market. It is that the UK is overwhelmingly dominant relative to every other country.

#### Customer concentration remains relatively moderate

Customer concentration was much lower than geographic concentration.

Among customers with positive identified revenue:

- the largest customer accounted for approximately **2.71% of positive identified-customer revenue in November 2010**, declining to approximately **2.48% in November 2011**;
- the top five customers increased slightly from approximately **9.84% to 10.20%**;
- the top ten customers increased from approximately **14.89% to 15.25%**.

The largest identified customer in November 2011 generated approximately **27.8k**, or only about **2.48% of positive identified-customer revenue**.

This indicates that the business was not dependent on one or two dominant named customers.

Although top-five and top-ten concentration increased slightly, the overall customer base remained comparatively diversified.

#### Product concentration also remained moderate

Product concentration was similarly limited.

The largest product represented approximately:

- **2.02% of positive product revenue in November 2010**
- **2.40% in November 2011**

The top five product share declined from approximately **8.02% to 7.29%**, while the top ten product share declined from approximately **11.93% to 11.35%**.

This suggests that product revenue became slightly more distributed across the portfolio even though the single largest product became somewhat more important.

The largest product in November 2011 was SKU **23084**, generating approximately **34.4k**, equivalent to about **2.40% of positive product revenue**.

The product portfolio therefore does not appear excessively dependent on a small number of SKUs.

#### Customer identification risk increased

The proportion of merchandise revenue associated with customers without a persistent customer identifier increased from approximately:

- **19.54% in November 2010**
to
- **21.92% in November 2011**

This is an increase of approximately **2.38 percentage points**.

In absolute terms, unidentified merchandise revenue increased from approximately **273.6k to 314.0k**.

This is significant because customer-level analyses such as retention, cohort behaviour, churn-style inactivity and customer revenue leakage can only be performed on identified customers.

The increasing unidentified share therefore represents an analytical and customer-management risk even though the revenue itself is economically real.

#### The risk profile is therefore asymmetric

The business does not appear highly concentrated across individual customers or products.

Instead, the primary structural dependency is geographic.

Revenue is highly dependent on the United Kingdom while being relatively dispersed across many individual customers and many products.

At the same time, the growing unidentified-customer share limits the organisation's ability to understand the behaviour of more than one-fifth of merchandise revenue.

This produces a risk profile characterised by:

- **high geographic concentration**;
- **moderate customer concentration**;
- **moderate-to-low product concentration**;
- and **material customer-identification exposure**.

#### Overall conclusion

Step 19 shows that the company's main concentration risk is **not dependence on a small number of customers or products**.

It is dependence on a single geographic market.

UK revenue increased from approximately **87.54% to 88.24% of total merchandise revenue**, while geographic HHI rose from approximately **7,682 to 7,803**.

By contrast, the top ten identified customers accounted for only approximately **15.25% of positive identified-customer revenue**, and the top ten products accounted for approximately **11.35% of positive product revenue**.

The portfolio is therefore comparatively diversified at customer and product level but remains highly geographically concentrated.

A second material risk is data visibility: unidentified-customer revenue increased to approximately **21.92% of merchandise revenue in November 2011**, reducing the proportion of the business that can be analysed reliably at customer level.

The key risk conclusion is therefore:

> **The business is diversified across customers and products, but structurally dependent on the United Kingdom and increasingly constrained by unidentified-customer revenue.**



## Step 20 — Final Findings Validation

Step 20 audits the principal conclusions developed throughout the notebook before they are carried into the final report.

Each major claim is classified according to the strength of the available evidence:

- **Supported** — directly demonstrated by the analysis;
- **Partially Supported** — directionally supported but requires qualification;
- **Unsupported** — the available evidence does not establish the claim;
- **Contradicted** — the analysis provides evidence against the claim; or
- **Unverified** — the dataset or observation window is insufficient to test the claim.

This step does not generate new commercial conclusions. Its purpose is to distinguish measured findings from interpretation, assumptions and claims that exceed what the data can establish.

In [ ]:
# ================================================================
# 20.2 — BUILD FINAL FINDINGS VALIDATION REGISTER
# ================================================================

findings_validation = pd.DataFrame(
    [
        # --------------------------------------------------------
        # REVENUE
        # --------------------------------------------------------
        {
            "step": 4,
            "theme": "Revenue",
            "claim":
                "Net merchandise revenue increased between "
                "November 2010 and November 2011.",

            "status": "Supported",

            "evidence":
                "Net merchandise revenue increased from approximately "
                "1.401m to 1.433m, equivalent to about 2.30%.",

            "qualification":
                "This is a matched November-to-November comparison."
        },

        {
            "step": 4,
            "theme": "Revenue",
            "claim":
                "The business experienced strong underlying revenue growth.",

            "status": "Partially Supported",

            "evidence":
                "Revenue increased, but November growth was only about "
                "2.30% and Jan-Nov growth was approximately 2.18%.",

            "qualification":
                "Positive growth is supported; describing it as strong "
                "would overstate the measured result."
        },

        # --------------------------------------------------------
        # COUNTRIES / GEOGRAPHY
        # --------------------------------------------------------
        {
            "step": 5,
            "theme": "Geography",
            "claim":
                "The United Kingdom remained the dominant revenue market.",

            "status": "Supported",

            "evidence":
                "The UK represented approximately 87.54% of November "
                "2010 revenue and 88.24% in November 2011.",

            "qualification":
                "The dependency increased slightly."
        },

        {
            "step": 9,
            "theme": "Geography",
            "claim":
                "The business became geographically more diversified.",

            "status": "Contradicted",

            "evidence":
                "UK share, top-five market share and geographic HHI "
                "all increased slightly.",

            "qualification":
                "The evidence indicates slightly greater geographic "
                "concentration, not diversification."
        },

        # --------------------------------------------------------
        # CUSTOMER MOVEMENT
        # --------------------------------------------------------
        {
            "step": 6,
            "theme": "Customers",
            "claim":
                "New and returning customers materially offset revenue "
                "losses from other customers.",

            "status": "Supported",

            "evidence":
                "Acquired, newly acquired and reactivated customers "
                "contributed substantial positive comparison-period revenue.",

            "qualification":
                "This finding applies to identified customers."
        },

        {
            "step": 6,
            "theme": "Customers",
            "claim":
                "The identified customer base itself produced the overall "
                "business revenue increase.",

            "status": "Contradicted",

            "evidence":
                "Identified-customer revenue declined by approximately "
                "8.24k while unidentified merchandise revenue increased "
                "by approximately 40.41k.",

            "qualification":
                "Overall merchandise growth was therefore supported by "
                "the unidentified-customer portion of the business."
        },

        # --------------------------------------------------------
        # PRODUCT DRIVERS
        # --------------------------------------------------------
        {
            "step": 7,
            "theme": "Products",
            "claim":
                "Product additions materially supported November 2011 revenue.",

            "status": "Supported",

            "evidence":
                "Products active in November 2011 but not the November "
                "2010 baseline contributed substantial positive movement.",

            "qualification":
                "'Newly active' does not necessarily mean genuinely new "
                "to the business."
        },

        {
            "step": 7,
            "theme": "Products",
            "claim":
                "All products newly active in November 2011 were new launches.",

            "status": "Contradicted",

            "evidence":
                "Step 16 distinguished first-observed launches from products "
                "that were simply inactive in the November 2010 baseline.",

            "qualification":
                "Only genuine first-observed products should be described "
                "as launches."
        },

        # --------------------------------------------------------
        # SUBSTITUTION
        # --------------------------------------------------------
        {
            "step": 8,
            "theme": "Products",
            "claim":
                "Product substitution explains the overall revenue movement.",

            "status": "Unsupported",

            "evidence":
                "Targeted switching relationships were observed, but only "
                "a limited number provided strong substitution evidence.",

            "qualification":
                "Substitution is supported for selected product pairs, "
                "not as a portfolio-wide explanation."
        },

        {
            "step": 8,
            "theme": "Products",
            "claim":
                "Some closely related products show evidence of customer switching.",

            "status": "Supported",

            "evidence":
                "Several specific pairs showed customer overlap, exclusive "
                "switching and lift materially above background adoption.",

            "qualification":
                "Association and switching evidence do not prove management-"
                "directed substitution."
        },

        # --------------------------------------------------------
        # DATA QUALITY
        # --------------------------------------------------------
        {
            "step": 10,
            "theme": "Data quality",
            "claim":
                "Missing customer identification is economically material.",

            "status": "Supported",

            "evidence":
                "Approximately 21.92% of November 2011 merchandise revenue "
                "was associated with unidentified customers.",

            "qualification":
                "Customer-level retention and lifecycle analysis cannot "
                "cover this revenue population."
        },

        {
            "step": 10,
            "theme": "Data quality",
            "claim":
                "Customer-level findings describe the entire business.",

            "status": "Contradicted",

            "evidence":
                "A material share of transactions cannot be assigned to "
                "persistent customer IDs.",

            "qualification":
                "Customer conclusions apply only to the identified population."
        },

        # --------------------------------------------------------
        # REPEAT PURCHASE
        # --------------------------------------------------------
        {
            "step": 11,
            "theme": "Retention",
            "claim":
                "Repeat purchasing is economically important.",

            "status": "Supported",

            "evidence":
                "Repeat customers generated the overwhelming majority of "
                "observable identified-customer purchase revenue.",

            "qualification":
                "Repeat behaviour is measured within observable history."
        },

        {
            "step": 11,
            "theme": "Retention",
            "claim":
                "Most observable customers eventually make another purchase.",

            "status": "Supported",

            "evidence":
                "Approximately 68% of observable acquisition customers "
                "eventually repeated within the available history.",

            "qualification":
                "This is an observed repeat rate, not a lifetime probability."
        },

        # --------------------------------------------------------
        # COHORTS
        # --------------------------------------------------------
        {
            "step": 12,
            "theme": "Retention",
            "claim":
                "Late-2010 acquisition cohorts had weaker short-term retention.",

            "status": "Supported",

            "evidence":
                "September-December 2010 cohorts displayed materially lower "
                "Month-3 activity than several earlier cohorts.",

            "qualification":
                "The analysis does not establish why those cohorts were weaker."
        },

        {
            "step": 12,
            "theme": "Retention",
            "claim":
                "Customer retention structurally deteriorated throughout 2011.",

            "status": "Contradicted",

            "evidence":
                "Retention recovered among several 2011 cohorts and weighted "
                "Month-3 performance was broadly comparable with 2010.",

            "qualification":
                "The weakness was concentrated in specific acquisition periods."
        },

        # --------------------------------------------------------
        # INACTIVITY / CHURN
        # --------------------------------------------------------
        {
            "step": 13,
            "theme": "Retention",
            "claim":
                "A customer inactive for 90 days should be considered permanently churned.",

            "status": "Contradicted",

            "evidence":
                "A substantial proportion of observed 90-day inactivity "
                "episodes later resulted in another purchase.",

            "qualification":
                "90 days is better interpreted as an inactivity or at-risk signal."
        },

        {
            "step": 13,
            "theme": "Retention",
            "claim":
                "Even 180 days of inactivity reliably proves permanent churn.",

            "status": "Contradicted",

            "evidence":
                "A material proportion of customers returned after gaps "
                "greater than 180 days.",

            "qualification":
                "Use 'long inactive' rather than permanent churn."
        },

        # --------------------------------------------------------
        # REVENUE RETENTION
        # --------------------------------------------------------
        {
            "step": 14,
            "theme": "Revenue retention",
            "claim":
                "The relatively stable identified-customer revenue result "
                "conceals substantial underlying customer turnover.",

            "status": "Supported",

            "evidence":
                "Large contraction and inactivity leakage was offset by "
                "expansion, acquisition and reactivation.",

            "qualification":
                "These are period-to-period revenue movements rather than "
                "formal subscription churn metrics."
        },

        {
            "step": 14,
            "theme": "Revenue retention",
            "claim":
                "All revenue associated with inactive customers is permanently lost.",

            "status": "Contradicted",

            "evidence":
                "Step 13 demonstrated frequent reactivation after long periods "
                "of inactivity.",

            "qualification":
                "Step 14 measures period-to-period leakage, not lifetime loss."
        },

        # --------------------------------------------------------
        # RECOVERY SCENARIOS
        # --------------------------------------------------------
        {
            "step": 15,
            "theme": "Scenarios",
            "claim":
                "Reducing customer leakage could materially improve revenue performance.",

            "status": "Supported",

            "evidence":
                "Scenario analysis shows that even small reductions in measured "
                "leakage produce material revenue sensitivity.",

            "qualification":
                "The analysis demonstrates mathematical sensitivity, not forecast "
                "or guaranteed recoverability."
        },

        {
            "step": 15,
            "theme": "Scenarios",
            "claim":
                "The model proves that 5%-20% of customer leakage will be recovered.",

            "status": "Unsupported",

            "evidence":
                "The percentages were deliberately illustrative recovery scenarios.",

            "qualification":
                "No causal intervention or forecast model was estimated."
        },

        # --------------------------------------------------------
        # NEW PRODUCT SUSTAINABILITY
        # --------------------------------------------------------
        {
            "step": 16,
            "theme": "Products",
            "claim":
                "Earlier observable product launches generally showed meaningful persistence.",

            "status": "Supported",

            "evidence":
                "A substantial majority of eligible product launches remained "
                "active at later monthly horizons.",

            "qualification":
                "This refers to historically observable launches."
        },

        {
            "step": 16,
            "theme": "Products",
            "claim":
                "The products first launched in November 2011 proved sustainable.",

            "status": "Unverified",

            "evidence":
                "November 2011 is the final complete month in the dataset, "
                "so no complete post-launch month is available.",

            "qualification":
                "Their subsequent sustainability cannot be tested."
        },

        # --------------------------------------------------------
        # CROSS SELL
        # --------------------------------------------------------
        {
            "step": 17,
            "theme": "Cross-selling",
            "claim":
                "The business exhibits meaningful multi-product purchasing behaviour.",

            "status": "Supported",

            "evidence":
                "The overwhelming majority of baskets contain multiple products "
                "and numerous product pairs show materially positive lift.",

            "qualification":
                "Basket association does not prove causal cross-selling."
        },

        {
            "step": 17,
            "theme": "Cross-selling",
            "claim":
                "High-growth products cause customers to purchase larger baskets.",

            "status": "Unsupported",

            "evidence":
                "Growth products appeared in broader baskets, but the analysis "
                "is descriptive and does not establish causality.",

            "qualification":
                "Customer type or wholesale-order behaviour may explain part "
                "of the relationship."
        },

        # --------------------------------------------------------
        # SEASONALITY
        # --------------------------------------------------------
        {
            "step": 18,
            "theme": "Seasonality",
            "claim":
                "November is a peak revenue month.",

            "status": "Supported",

            "evidence":
                "November ranked as the highest-revenue month in both 2010 and 2011.",

            "qualification":
                "The source contains fewer than two complete annual cycles."
        },

        {
            "step": 18,
            "theme": "Seasonality",
            "claim":
                "The notebook proves a stable long-run seasonal pattern.",

            "status": "Unsupported",

            "evidence":
                "Only a limited number of complete annual cycles are available.",

            "qualification":
                "The evidence supports descriptive seasonality, not robust "
                "long-run seasonal modelling."
        },

        # --------------------------------------------------------
        # CONCENTRATION
        # --------------------------------------------------------
        {
            "step": 19,
            "theme": "Risk",
            "claim":
                "Geographic concentration is the clearest structural concentration risk.",

            "status": "Supported",

            "evidence":
                "UK revenue share exceeded 88% in November 2011 and geographic "
                "HHI increased from the baseline period.",

            "qualification":
                "This is materially higher concentration than observed at "
                "customer or product level."
        },

        {
            "step": 19,
            "theme": "Risk",
            "claim":
                "The business is highly dependent on a small number of individual customers.",

            "status": "Contradicted",

            "evidence":
                "The top ten identified customers represented only approximately "
                "15% of positive identified-customer revenue.",

            "qualification":
                "Customer concentration is relatively moderate compared with geography."
        },

        {
            "step": 19,
            "theme": "Risk",
            "claim":
                "The business is highly dependent on only a few products.",

            "status": "Contradicted",

            "evidence":
                "The top ten products represented only approximately 11% "
                "of positive product revenue.",

            "qualification":
                "Product-level revenue is comparatively diversified."
        },
    ]
)


display(
    findings_validation
)

In [ ]:
# ================================================================
# 20.3 — VALIDATE FINDING STATUS VALUES
# ================================================================

allowed_statuses = {
    "Supported",
    "Partially Supported",
    "Unsupported",
    "Contradicted",
    "Unverified",
}


unexpected_statuses = (
    set(
        findings_validation[
            "status"
        ].unique()
    )
    -
    allowed_statuses
)


assert not unexpected_statuses, (
    f"Unexpected validation statuses: "
    f"{unexpected_statuses}"
)


print(
    "✓ All finding classifications use approved status labels."
)

In [ ]:
# ================================================================
# 20.4 — FINDINGS VALIDATION SUMMARY
# ================================================================

validation_summary = (
    findings_validation
    .groupby(
        "status",
        as_index=False
    )
    .agg(
        findings=(
            "claim",
            "size"
        )
    )
)


validation_summary[
    "share_pct"
] = (
    validation_summary[
        "findings"
    ]
    / validation_summary[
        "findings"
    ].sum()
    * 100
)


status_order = [
    "Supported",
    "Partially Supported",
    "Unsupported",
    "Contradicted",
    "Unverified",
]


validation_summary[
    "status"
] = pd.Categorical(
    validation_summary[
        "status"
    ],
    categories=status_order,
    ordered=True
)


validation_summary = (
    validation_summary
    .sort_values(
        "status"
    )
    .reset_index(
        drop=True
    )
)


display(
    validation_summary
)

In [ ]:
# ================================================================
# VISUAL 20.1 — FINAL FINDINGS VALIDATION STATUS
# ================================================================

plt.figure(
    figsize=(10, 6)
)


sns.barplot(
    data=validation_summary,
    y="status",
    x="findings"
)


plt.title(
    "Final Analytical Findings by Evidence Status"
)

plt.xlabel(
    "Number of Findings"
)

plt.ylabel(
    "Evidence Classification"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 20.5 — SUPPORTED FINAL FINDINGS
# ================================================================

supported_findings = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ] == "Supported"
    ]
    [
        [
            "step",
            "theme",
            "claim",
            "evidence",
            "qualification",
        ]
    ]
    .sort_values(
        [
            "step",
            "theme"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    supported_findings
)

In [ ]:
# ================================================================
# 20.6 — PARTIALLY SUPPORTED FINDINGS
# ================================================================

partially_supported_findings = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ] == "Partially Supported"
    ]
    [
        [
            "step",
            "theme",
            "claim",
            "evidence",
            "qualification",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    partially_supported_findings
)

In [ ]:
# ================================================================
# 20.7 — EXCLUDE UNSUPPORTED / CONTRADICTED CLAIMS
# ================================================================

claims_not_for_final_report = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ].isin(
            [
                "Unsupported",
                "Contradicted",
            ]
        )
    ]
    [
        [
            "step",
            "theme",
            "status",
            "claim",
            "evidence",
            "qualification",
        ]
    ]
    .sort_values(
        [
            "status",
            "step"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    claims_not_for_final_report
)

In [ ]:
# ================================================================
# 20.8 — UNVERIFIED FINDINGS
# ================================================================

unverified_findings = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ] == "Unverified"
    ]
    [
        [
            "step",
            "theme",
            "claim",
            "evidence",
            "qualification",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    unverified_findings
)

In [ ]:
# ================================================================
# 20.9 — VALIDATION BY ANALYTICAL THEME
# ================================================================

theme_validation = (
    findings_validation
    .groupby(
        [
            "theme",
            "status"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
)


# Ensure all status columns exist
for status in status_order:

    if status not in theme_validation.columns:

        theme_validation[
            status
        ] = 0


theme_validation = (
    theme_validation[
        status_order
    ]
)


display(
    theme_validation
)

In [ ]:
# ================================================================
# VISUAL 20.2 — VALIDATION STATUS BY THEME
# ================================================================

plt.figure(
    figsize=(11, 8)
)


sns.heatmap(
    theme_validation,
    annot=True,
    fmt="d",
    linewidths=0.5
)


plt.title(
    "Evidence Classification by Analytical Theme"
)

plt.xlabel(
    "Validation Status"
)

plt.ylabel(
    "Analytical Theme"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 20.10 — FINAL EVIDENCE-BACKED CONCLUSION REGISTER
# ================================================================

final_conclusion_register = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ].isin(
            [
                "Supported",
                "Partially Supported",
            ]
        )
    ]
    [
        [
            "step",
            "theme",
            "status",
            "claim",
            "evidence",
            "qualification",
        ]
    ]
    .sort_values(
        "step"
    )
    .reset_index(
        drop=True
    )
)


display(
    final_conclusion_register
)

In [ ]:
# ================================================================
# 20.11 — CLAIM CORRECTION REGISTER
# ================================================================

correction_register = (
    findings_validation
    .loc[
        findings_validation[
            "status"
        ].isin(
            [
                "Contradicted",
                "Unsupported",
                "Unverified",
            ]
        )
    ]
    [
        [
            "step",
            "theme",
            "status",
            "claim",
            "qualification",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    correction_register
)

In [ ]:
# ================================================================
# 20.12 — FINAL VALIDATION CHECKS
# ================================================================

assert (
    findings_validation[
        "claim"
    ]
    .notna()
    .all()
)


assert (
    findings_validation[
        "evidence"
    ]
    .notna()
    .all()
)


assert (
    findings_validation[
        "qualification"
    ]
    .notna()
    .all()
)


assert (
    findings_validation[
        "claim"
    ]
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate claims exist in the findings register."
)


assert (
    len(
        findings_validation
    )
    ==
    len(
        final_conclusion_register
    )
    +
    len(
        correction_register
    )
)


print(
    "✓ Step 20 findings-validation register is complete."
)

### Step 20 Summary — Final Findings Validation

Step 20 audited the major conclusions developed throughout the analysis and separated directly supported findings from claims requiring qualification or exclusion.

The validation confirms that the principal findings are well supported:

- merchandise revenue increased modestly between November 2010 and November 2011;
- the business remains overwhelmingly dependent on the United Kingdom;
- customer and product concentration are substantially lower than geographic concentration;
- repeat purchasing and customer reactivation are economically important;
- late-2010 acquisition cohorts demonstrated weaker retention, but this did not develop into a persistent 2011 deterioration;
- inactivity should not be interpreted automatically as permanent churn;
- substantial customer revenue leakage is being offset by expansion, acquisition and reactivation;
- historical product introductions generally demonstrate meaningful persistence;
- multi-product purchasing and product affinity are strongly present;
- and November is a structurally strong trading month.

The validation also identifies several claims that the data does not support.

The analysis does not establish that the business became geographically more diversified, that customer inactivity represents permanent churn, that product substitution explains portfolio-wide revenue movement, that large baskets are caused by specific growth products, or that illustrative revenue-recovery scenarios constitute forecasts.

Similarly, the sustainability of products first introduced in November 2011 remains unverified because the dataset does not contain a complete subsequent observation month.

The final report should therefore distinguish carefully between measured findings, descriptive associations and scenario-based interpretation.

Overall, the strongest evidence describes a business with modest revenue growth, substantial customer turnover beneath the headline result, strong repeat and reactivation behaviour, diversified customer and product exposure, significant seasonality, and a major structural dependency on the United Kingdom.


## Step 21 — Gaps and Opportunities

Step 21 translates the validated findings from the preceding analysis into an evidence-backed opportunity register.

The objective is not to prescribe guaranteed commercial outcomes. Instead, it identifies areas where the observed data suggests that management attention, experimentation or improved measurement may create value.

Each opportunity is linked to a finding already validated in Step 20 and is classified by:

- opportunity area;
- supporting evidence;
- primary gap or problem;
- proposed management response;
- recommended KPI;
- evidence strength;
- and implementation horizon.

The section distinguishes between commercial opportunities and data-quality opportunities so that analytical limitations are not mistaken for customer or market behaviour.

In [ ]:
# ================================================================
# 21.2 — EVIDENCE-BACKED OPPORTUNITY REGISTER
# ================================================================

opportunity_register = pd.DataFrame(
    [
        {
            "opportunity_area":
                "Customer identification",

            "opportunity_type":
                "Data / CRM",

            "evidence":
                "Approximately 21.92% of November 2011 merchandise "
                "revenue was associated with unidentified customers.",

            "gap":
                "A material share of revenue cannot be linked to a "
                "persistent customer identity.",

            "management_response":
                "Improve customer-ID capture, transaction matching and "
                "CRM linkage at point of sale.",

            "recommended_kpi":
                "Unidentified merchandise revenue share (%)",

            "evidence_strength":
                "High",

            "horizon":
                "Immediate"
        },

        {
            "opportunity_area":
                "Geographic diversification",

            "opportunity_type":
                "Commercial / Market",

            "evidence":
                "The United Kingdom generated approximately 88.24% "
                "of November 2011 merchandise revenue.",

            "gap":
                "Revenue is overwhelmingly dependent on one market.",

            "management_response":
                "Evaluate expansion opportunities in existing non-UK "
                "markets before pursuing entirely new geographies.",

            "recommended_kpi":
                "UK revenue share and non-UK revenue growth",

            "evidence_strength":
                "High",

            "horizon":
                "Structural"
        },

        {
            "opportunity_area":
                "Customer reactivation",

            "opportunity_type":
                "Commercial / Retention",

            "evidence":
                "Substantial numbers of customers returned after periods "
                "of 90 and even 180 days of inactivity.",

            "gap":
                "Inactive customers may be treated as lost even though "
                "many remain commercially recoverable.",

            "management_response":
                "Create inactivity-triggered reactivation campaigns rather "
                "than treating inactivity automatically as permanent churn.",

            "recommended_kpi":
                "90-day and 180-day reactivation rate",

            "evidence_strength":
                "High",

            "horizon":
                "Near-term"
        },

        {
            "opportunity_area":
                "Customer leakage recovery",

            "opportunity_type":
                "Commercial / Retention",

            "evidence":
                "Substantial revenue contraction and inactivity leakage "
                "was observed between the two November periods.",

            "gap":
                "Revenue growth is being offset by meaningful erosion "
                "from previously active customers.",

            "management_response":
                "Prioritise customers showing material revenue contraction "
                "or inactivity for targeted retention activity.",

            "recommended_kpi":
                "Recovered leakage revenue and retained-customer revenue",

            "evidence_strength":
                "High",

            "horizon":
                "Near-term"
        },

        {
            "opportunity_area":
                "Early repeat purchase",

            "opportunity_type":
                "Commercial / Retention",

            "evidence":
                "Approximately 43.76% of eligible observable customers "
                "repeated within 90 days and approximately 58.03% within "
                "180 days.",

            "gap":
                "A significant proportion of newly observed customers "
                "do not repeat within the first several months.",

            "management_response":
                "Test post-first-purchase interventions designed to "
                "accelerate the second purchase.",

            "recommended_kpi":
                "30-, 60-, 90- and 180-day repeat rate",

            "evidence_strength":
                "High",

            "horizon":
                "Near-term"
        },

        {
            "opportunity_area":
                "Cohort-quality monitoring",

            "opportunity_type":
                "Analytics / Retention",

            "evidence":
                "Late-2010 acquisition cohorts showed weaker short-term "
                "retention than several other cohorts.",

            "gap":
                "Headline acquisition volume can conceal differences in "
                "the quality of customers acquired in different periods.",

            "management_response":
                "Track acquisition cohorts routinely by Month-1, Month-3, "
                "Month-6 and Month-12 activity.",

            "recommended_kpi":
                "Cohort retention by acquisition month",

            "evidence_strength":
                "High",

            "horizon":
                "Immediate"
        },

        {
            "opportunity_area":
                "Product cross-selling",

            "opportunity_type":
                "Commercial / Product",

            "evidence":
                "Multi-product purchasing is widespread and several product "
                "pairs show strong positive affinity.",

            "gap":
                "Observed product affinities may not yet be used "
                "systematically in merchandising or recommendations.",

            "management_response":
                "Test evidence-backed bundles, recommendations and "
                "adjacent-product merchandising.",

            "recommended_kpi":
                "Bundle attach rate and average basket breadth",

            "evidence_strength":
                "High",

            "horizon":
                "Near-term"
        },

        {
            "opportunity_area":
                "Product launch governance",

            "opportunity_type":
                "Commercial / Product",

            "evidence":
                "Historically observable product launches generally "
                "showed meaningful persistence beyond launch month.",

            "gap":
                "Launch success should be assessed beyond first-month revenue.",

            "management_response":
                "Measure product performance at 1-, 3-, 6- and 12-month "
                "post-launch horizons.",

            "recommended_kpi":
                "Product survival and post-launch revenue retention",

            "evidence_strength":
                "High",

            "horizon":
                "Immediate"
        },

        {
            "opportunity_area":
                "Seasonal commercial planning",

            "opportunity_type":
                "Commercial / Planning",

            "evidence":
                "November was the highest-revenue month in both 2010 "
                "and 2011 and Q4 revenue was materially elevated.",

            "gap":
                "Seasonal demand can create inventory, service and "
                "customer-management pressure if not planned explicitly.",

            "management_response":
                "Plan inventory, CRM activity and operational capacity "
                "ahead of the September-November trading build.",

            "recommended_kpi":
                "Monthly revenue, stock availability and return drag",

            "evidence_strength":
                "High",

            "horizon":
                "Recurring"
        },

        {
            "opportunity_area":
                "Returns and cancellation control",

            "opportunity_type":
                "Commercial / Operations",

            "evidence":
                "Gross-to-net revenue drag varies materially by month and "
                "improved between November 2010 and November 2011.",

            "gap":
                "High-return or cancellation months can materially weaken "
                "otherwise healthy gross merchandise performance.",

            "management_response":
                "Monitor return and cancellation drag by month, product "
                "and customer segment.",

            "recommended_kpi":
                "Gross-to-net merchandise revenue drag (%)",

            "evidence_strength":
                "High",

            "horizon":
                "Immediate"
        },
    ]
)


display(
    opportunity_register
)

In [ ]:
# ================================================================
# 21.3 — COMMERCIAL VS DATA / ANALYTICS OPPORTUNITIES
# ================================================================

commercial_opportunities = (
    opportunity_register
    .loc[
        opportunity_register[
            "opportunity_type"
        ]
        .str.startswith(
            "Commercial"
        )
    ]
    .reset_index(
        drop=True
    )
)


data_opportunities = (
    opportunity_register
    .loc[
        ~opportunity_register[
            "opportunity_type"
        ]
        .str.startswith(
            "Commercial"
        )
    ]
    .reset_index(
        drop=True
    )
)


print(
    f"Commercial opportunities: "
    f"{len(commercial_opportunities)}"
)

print(
    f"Data / analytics opportunities: "
    f"{len(data_opportunities)}"
)


display(
    commercial_opportunities
)


display(
    data_opportunities
)

In [ ]:
# ================================================================
# 21.4 — IMMEDIATE PRIORITY REGISTER
# ================================================================

immediate_priorities = (
    opportunity_register
    .loc[
        opportunity_register[
            "horizon"
        ].isin(
            [
                "Immediate",
                "Near-term"
            ]
        )
    ]
    [
        [
            "opportunity_area",
            "opportunity_type",
            "gap",
            "management_response",
            "recommended_kpi",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    immediate_priorities
)

In [ ]:
# ================================================================
# 21.5 — ANALYTICAL AND DATA GAPS
# ================================================================

data_gap_register = pd.DataFrame(
    [
        {
            "gap_area":
                "Customer identification",

            "observed_problem":
                "A material portion of merchandise revenue has no "
                "persistent customer identifier.",

            "analytical_consequence":
                "Retention, reactivation and customer-level revenue "
                "analysis cannot cover the entire business.",

            "recommended_improvement":
                "Improve customer-ID capture and transaction matching."
        },

        {
            "gap_area":
                "Observation horizon",

            "observed_problem":
                "The dataset contains fewer than two complete annual cycles.",

            "analytical_consequence":
                "Long-run seasonality cannot be estimated robustly.",

            "recommended_improvement":
                "Extend the historical transaction series."
        },

        {
            "gap_area":
                "Right censoring",

            "observed_problem":
                "Customers and products entering near the end of the dataset "
                "have limited follow-up time.",

            "analytical_consequence":
                "Repeat, retention and product-survival outcomes may be "
                "unobservable for recent entrants.",

            "recommended_improvement":
                "Use horizon-specific eligibility and extend future observation."
        },

        {
            "gap_area":
                "Product substitution",

            "observed_problem":
                "Transaction data shows switching patterns but not customer intent.",

            "analytical_consequence":
                "Observed switching cannot prove deliberate substitution.",

            "recommended_improvement":
                "Combine transaction evidence with product lifecycle, stock "
                "availability and merchandising information."
        },

        {
            "gap_area":
                "Cross-selling causality",

            "observed_problem":
                "Product affinity identifies association rather than causal impact.",

            "analytical_consequence":
                "Higher basket breadth cannot automatically be attributed "
                "to specific products.",

            "recommended_improvement":
                "Use controlled bundle or recommendation experiments."
        },

        {
            "gap_area":
                "Recovery forecasting",

            "observed_problem":
                "Step 15 uses illustrative recovery scenarios.",

            "analytical_consequence":
                "Scenario revenue should not be interpreted as forecast revenue.",

            "recommended_improvement":
                "Measure actual campaign response before constructing "
                "recovery forecasts."
        },
    ]
)


display(
    data_gap_register
)

In [ ]:
# ================================================================
# 21.6 — MANAGEMENT ACTION REGISTER
# ================================================================

management_action_register = (
    opportunity_register[
        [
            "opportunity_area",
            "management_response",
            "recommended_kpi",
            "horizon",
            "evidence_strength",
        ]
    ]
    .rename(
        columns={
            "opportunity_area":
                "Opportunity",

            "management_response":
                "Recommended Action",

            "recommended_kpi":
                "Primary KPI",

            "horizon":
                "Horizon",

            "evidence_strength":
                "Evidence Strength",
        }
    )
)


display(
    management_action_register
)

In [ ]:
# ================================================================
# VISUAL 21.1 — OPPORTUNITIES BY IMPLEMENTATION HORIZON
# ================================================================

horizon_summary = (
    opportunity_register
    .groupby(
        "horizon",
        as_index=False
    )
    .agg(
        opportunities=(
            "opportunity_area",
            "size"
        )
    )
)


horizon_order = [
    "Immediate",
    "Near-term",
    "Recurring",
    "Structural",
]


horizon_summary[
    "horizon"
] = pd.Categorical(
    horizon_summary[
        "horizon"
    ],
    categories=horizon_order,
    ordered=True
)


horizon_summary = (
    horizon_summary
    .sort_values(
        "horizon"
    )
)


plt.figure(
    figsize=(9, 5)
)


sns.barplot(
    data=horizon_summary,
    x="horizon",
    y="opportunities"
)


plt.title(
    "Evidence-Backed Opportunities by Implementation Horizon"
)

plt.xlabel(
    "Implementation Horizon"
)

plt.ylabel(
    "Number of Opportunities"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 21.7 — CUSTOMER RETENTION OPPORTUNITY SNAPSHOT
# ================================================================

retention_opportunity_snapshot = pd.DataFrame(
    {
        "Metric": [
            "30-day repeat rate",
            "60-day repeat rate",
            "90-day repeat rate",
            "180-day repeat rate",
            "Observed eventual repeat rate",
        ],

        "Value (%)": [
            20.96,
            34.98,
            43.76,
            58.03,
            68.40,
        ],

        "Interpretation": [
            "Most customers have not repeated within 30 days.",
            "Repeat purchasing continues to build materially after 30 days.",
            "More than half have not yet repeated by 90 days.",
            "Repeat behaviour continues materially through six months.",
            "A majority eventually repeat within observable history.",
        ],
    }
)


display(
    retention_opportunity_snapshot
)

In [ ]:
# ================================================================
# VISUAL 21.2 — CUMULATIVE REPEAT PURCHASE DEVELOPMENT
# ================================================================

repeat_opportunity_plot = pd.DataFrame(
    {
        "days": [
            30,
            60,
            90,
            180
        ],

        "repeat_rate_pct": [
            20.08,
            33.87,
            43.07,
            58.03,
        ],
    }
)


plt.figure(
    figsize=(9, 5)
)


sns.lineplot(
    data=repeat_opportunity_plot,
    x="days",
    y="repeat_rate_pct",
    marker="o"
)


plt.title(
    "Cumulative Repeat Purchase Development"
)

plt.xlabel(
    "Days Since First Observable Purchase"
)

plt.ylabel(
    "Customers Repeating (%)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 21.8 — GEOGRAPHIC OPPORTUNITY SNAPSHOT
# ================================================================

geographic_opportunity = pd.DataFrame(
    {
        "Metric": [
            "UK revenue share",
            "Non-UK revenue share",
            "Top 5 geographic market share",
        ],

        "November 2011 (%)": [
            comparison_row[
                "uk_share_pct"
            ],

            100
            -
            comparison_row[
                "uk_share_pct"
            ],

            comparison_row[
                "top5_country_share_pct"
            ],
        ],
    }
)


display(
    geographic_opportunity
)

In [ ]:
# ================================================================
# 21.9 — CUSTOMER VISIBILITY OPPORTUNITY
# ================================================================

customer_visibility = pd.DataFrame(
    {
        "Metric": [
            "Identified merchandise revenue share",
            "Unidentified merchandise revenue share",
        ],

        "November 2011 (%)": [
            100
            -
            comparison_row[
                "unidentified_revenue_share_pct"
            ],

            comparison_row[
                "unidentified_revenue_share_pct"
            ],
        ],
    }
)


display(
    customer_visibility
)

In [ ]:
# ================================================================
# 21.10 — OPPORTUNITY HIERARCHY
# ================================================================

opportunity_hierarchy = pd.DataFrame(
    {
        "Tier": [
            "Tier 1",
            "Tier 1",
            "Tier 1",
            "Tier 2",
            "Tier 2",
            "Tier 2",
            "Tier 3",
            "Tier 3",
        ],

        "Opportunity": [
            "Improve customer identification",
            "Reduce customer leakage / strengthen reactivation",
            "Improve early repeat purchase",
            "Exploit evidence-backed cross-selling",
            "Strengthen cohort-quality monitoring",
            "Manage seasonal trading proactively",
            "Develop non-UK revenue over the long term",
            "Strengthen product launch measurement",
        ],

        "Reason": [
            "Directly improves visibility over more than one-fifth of revenue.",
            "Addresses material observed customer revenue erosion.",
            "Large proportion of customers do not repeat within early horizons.",
            "Strong multi-product behaviour and affinity already exist.",
            "Retention quality varies materially across acquisition cohorts.",
            "Revenue is strongly concentrated in late-year trading.",
            "Reduces the largest structural concentration exposure.",
            "Improves decisions on which new products deserve continued support.",
        ],
    }
)


display(
    opportunity_hierarchy
)

In [ ]:
# ================================================================
# 21.11 — VALIDATION CHECKS
# ================================================================

required_opportunity_fields = [
    "opportunity_area",
    "opportunity_type",
    "evidence",
    "gap",
    "management_response",
    "recommended_kpi",
    "evidence_strength",
    "horizon",
]


assert (
    opportunity_register[
        required_opportunity_fields
    ]
    .notna()
    .all()
    .all()
), (
    "Opportunity register contains missing fields."
)


assert (
    opportunity_register[
        "opportunity_area"
    ]
    .duplicated()
    .sum()
    == 0
), (
    "Duplicate opportunity areas exist."
)


assert (
    set(
        opportunity_register[
            "evidence_strength"
        ]
    )
    <=
    {
        "High",
        "Moderate",
        "Low",
    }
)


assert (
    set(
        opportunity_register[
            "horizon"
        ]
    )
    <=
    {
        "Immediate",
        "Near-term",
        "Recurring",
        "Structural",
    }
)


print(
    "✓ Step 21 opportunity register validated."
)

### Step 21 Summary — Gaps and Opportunities

Step 21 converts the validated analytical findings into an evidence-backed management opportunity set.

The strongest opportunities are concentrated in customer visibility, retention, repeat purchasing, cross-selling and long-term geographic risk reduction.

#### 1. Improve customer identification

The most immediate analytical gap is customer visibility.

Approximately **21.92% of November 2011 merchandise revenue** could not be linked to an identified customer, up from approximately **19.54% in November 2010**.

This limits the ability to measure retention, reactivation, customer value and revenue leakage across the full business.

Improving customer identification would therefore create value beyond data quality alone: it would increase the proportion of revenue that can be managed through customer-level CRM and lifecycle programmes.

#### 2. Treat inactivity as a reactivation opportunity, not automatic churn

The customer analysis shows that inactivity does not necessarily represent permanent customer loss.

Material numbers of customers returned after both 90-day and 180-day periods of inactivity.

This creates an opportunity to introduce systematic inactivity-triggered CRM programmes, particularly for customers with meaningful historical revenue.

The appropriate KPI is therefore not simply churn, but **reactivation after defined inactivity intervals**.

#### 3. Accelerate the second purchase

Repeat-purchase behaviour develops progressively rather than immediately.

Among customers with sufficient observation history, approximately **20% repeated within 30 days, 34% within 60 days, 43% within 90 days and 58% within 180 days**.

This indicates a meaningful opportunity to improve the transition from first purchase to second purchase.

The commercial objective should therefore include both increasing eventual repeat behaviour and shortening the time required for customers to repeat.

#### 4. Reduce customer revenue leakage

The November revenue bridge revealed substantial contraction and inactivity leakage beneath a relatively stable overall revenue result.

Growth from acquisition, reactivation and expansion offset much of this erosion.

This means improving retention does not require eliminating customer loss completely. Even modest improvements in measured leakage could be commercially meaningful.

However, the recovery scenarios from Step 15 remain sensitivities rather than forecasts.

#### 5. Use product affinity for cross-selling experiments

Basket analysis showed strong multi-product purchasing behaviour and several product pairs with materially elevated affinity.

This supports controlled testing of product bundles, adjacent-product recommendations and merchandising combinations.

The evidence supports association, but does not prove that recommending one product will cause purchase of another. Cross-selling initiatives should therefore be tested experimentally and measured using attach rate and incremental basket value.

#### 6. Improve product-launch measurement

Historically observable product launches generally demonstrated meaningful persistence beyond their launch month.

This suggests that product decisions should not be based on launch-month revenue alone.

Products should instead be monitored at defined post-launch horizons such as one, three, six and twelve months.

Products first observed in November 2011 cannot yet be evaluated for sustainability because the dataset does not contain sufficient subsequent complete months.

#### 7. Plan explicitly for seasonality

November was the highest-revenue month in both 2010 and 2011, with a broader revenue build visible through the later part of the year.

This creates operational opportunities around inventory planning, customer activation, merchandising and returns management ahead of the peak trading period.

The available history supports descriptive seasonality, although it is not long enough to establish a robust long-run seasonal model.

#### 8. Reduce long-term geographic dependency

The largest structural risk remains geographic concentration.

Approximately **88.24% of November 2011 merchandise revenue** came from the United Kingdom.

The opportunity is therefore not necessarily to reduce UK activity, but to grow viable non-UK markets at a faster rate over time so that overall revenue becomes less dependent on a single geography.

### Overall opportunity conclusion

The analysis does not suggest that the business lacks customers or products.

Instead, the larger opportunities lie in **managing the existing revenue base more effectively**.

The strongest evidence supports five priorities:

1. improve customer identification;
2. strengthen repeat purchase and reactivation;
3. reduce customer revenue leakage;
4. exploit validated product affinities through controlled cross-selling tests; and
5. reduce geographic dependency over the longer term.

The central commercial opportunity is therefore to convert an already broad customer and product base into a more visible, repeatable and resilient revenue system.

## Step 22 — Executive Conclusions, Recommendations and ML Roadmap

Step 22 consolidates the validated analytical findings into a final decision-ready set of conclusions and recommendations.

It also identifies which business problems are suitable for predictive machine-learning models and which should be addressed through operational, data-quality or strategic interventions.

The purpose is to ensure that subsequent modelling is driven by demonstrated business problems rather than by the availability of algorithms.

The modelling roadmap therefore prioritises problems where:

- a clearly defined target can be constructed from the transaction history;
- predictions can lead to a specific management action;
- sufficient historical behaviour exists for model training;
- target leakage can be controlled;
- and model performance can be evaluated using future, unseen periods.

In [ ]:
# ================================================================
# 22.2 — EXECUTIVE CONCLUSION REGISTER
# ================================================================

executive_conclusions = pd.DataFrame(
    [
        {
            "Area": "Revenue performance",
            "Conclusion":
                "Net merchandise revenue increased modestly between "
                "November 2010 and November 2011.",
            "Implication":
                "Headline growth is positive but should not be interpreted "
                "as broad-based structural improvement."
        },

        {
            "Area": "Customer dynamics",
            "Conclusion":
                "Substantial acquisition, reactivation and expansion "
                "offset material customer contraction and inactivity.",
            "Implication":
                "The business experiences meaningful customer turnover "
                "beneath the relatively stable revenue headline."
        },

        {
            "Area": "Repeat purchase",
            "Conclusion":
                "Repeat behaviour develops materially over the first "
                "180 days after observable acquisition.",
            "Implication":
                "Accelerating the second purchase represents a major "
                "customer-management opportunity."
        },

        {
            "Area": "Inactivity",
            "Conclusion":
                "Customer inactivity does not reliably imply permanent churn.",
            "Implication":
                "Inactive customers should be treated as reactivation "
                "candidates rather than automatically classified as lost."
        },

        {
            "Area": "Customer leakage",
            "Conclusion":
                "Large revenue leakage from contraction and inactivity "
                "is being offset by other sources of growth.",
            "Implication":
                "Predicting customer deterioration before it occurs could "
                "create substantial commercial value."
        },

        {
            "Area": "Products",
            "Conclusion":
                "Revenue is relatively diversified across products and "
                "historical product introductions often persist beyond launch.",
            "Implication":
                "Product management should focus on sustainability and "
                "cross-selling rather than reliance on a few hero products."
        },

        {
            "Area": "Cross-selling",
            "Conclusion":
                "Multi-product purchasing is widespread and several "
                "product combinations show strong affinity.",
            "Implication":
                "There is a credible basis for personalised or basket-level "
                "product recommendations."
        },

        {
            "Area": "Seasonality",
            "Conclusion":
                "Revenue displays a pronounced late-year build, with "
                "November the strongest observed month.",
            "Implication":
                "Forecasting and inventory planning should explicitly "
                "incorporate seasonal demand."
        },

        {
            "Area": "Geographic concentration",
            "Conclusion":
                "Revenue remains overwhelmingly dependent on the "
                "United Kingdom.",
            "Implication":
                "Geographic diversification is primarily a strategic "
                "growth problem rather than an ML prediction problem."
        },

        {
            "Area": "Customer identification",
            "Conclusion":
                "A material and increasing share of merchandise revenue "
                "cannot be connected to persistent customer identities.",
            "Implication":
                "Improved identity capture is required before customer-level "
                "analytics can represent the entire business."
        },
    ]
)

display(executive_conclusions)

In [ ]:
# ================================================================
# 22.3 — FINAL MANAGEMENT RECOMMENDATION REGISTER
# ================================================================

recommendation_register = pd.DataFrame(
    [
        {
            "Priority": 1,
            "Recommendation":
                "Improve customer identification and transaction linkage.",
            "Why":
                "More than one-fifth of November 2011 merchandise revenue "
                "cannot be analysed reliably at customer level.",
            "Action":
                "Improve persistent customer-ID capture and CRM integration.",
            "ML_candidate": "No — data infrastructure first"
        },

        {
            "Priority": 2,
            "Recommendation":
                "Predict which customers are likely to purchase again.",
            "Why":
                "A substantial proportion of customers fail to repeat quickly.",
            "Action":
                "Prioritise first-purchase customers for targeted lifecycle activity.",
            "ML_candidate": "Yes"
        },

        {
            "Priority": 3,
            "Recommendation":
                "Predict reactivation among inactive customers.",
            "Why":
                "Many customers return after 90- and 180-day inactivity periods.",
            "Action":
                "Rank inactive customers by probability of returning.",
            "ML_candidate": "Yes"
        },

        {
            "Priority": 4,
            "Recommendation":
                "Detect customers at risk of material revenue contraction.",
            "Why":
                "Revenue leakage from contraction and inactivity is substantial.",
            "Action":
                "Create an early-warning retention system.",
            "ML_candidate": "Yes"
        },

        {
            "Priority": 5,
            "Recommendation":
                "Develop next-best-product recommendations.",
            "Why":
                "Strong basket affinity and multi-product purchasing exist.",
            "Action":
                "Use customer and basket history to recommend relevant products.",
            "ML_candidate": "Yes"
        },

        {
            "Priority": 6,
            "Recommendation":
                "Forecast product and monthly demand.",
            "Why":
                "Revenue is strongly seasonal and operational planning matters.",
            "Action":
                "Forecast future demand at appropriate product/time granularity.",
            "ML_candidate": "Yes"
        },

        {
            "Priority": 7,
            "Recommendation":
                "Grow viable non-UK markets.",
            "Why":
                "UK revenue concentration remains extremely high.",
            "Action":
                "Use market-level strategy and experimentation to increase "
                "non-UK contribution.",
            "ML_candidate": "Not initially"
        },
    ]
)

display(recommendation_register)

In [ ]:
# ================================================================
# 22.4 — ML SUITABILITY MATRIX
# ================================================================

ml_suitability = pd.DataFrame(
    [
        {
            "Business Problem": "Early repeat purchase",
            "ML Suitability": "Very High",
            "Recommended Approach":
                "Binary classification + survival analysis",
            "Prediction":
                "Will this customer make another purchase within X days?"
        },

        {
            "Business Problem": "Customer reactivation",
            "ML Suitability": "Very High",
            "Recommended Approach":
                "Binary classification / survival analysis",
            "Prediction":
                "Will this inactive customer return within the next X days?"
        },

        {
            "Business Problem": "Revenue leakage",
            "ML Suitability": "Very High",
            "Recommended Approach":
                "Classification + regression",
            "Prediction":
                "Will this customer materially contract or become inactive?"
        },

        {
            "Business Problem": "Time to next purchase",
            "ML Suitability": "Very High",
            "Recommended Approach":
                "Survival analysis",
            "Prediction":
                "How long until the customer's next purchase?"
        },

        {
            "Business Problem": "Cross-selling",
            "ML Suitability": "High",
            "Recommended Approach":
                "Recommender system / ranking model",
            "Prediction":
                "Which product is the customer most likely to buy next?"
        },

        {
            "Business Problem": "Future customer value",
            "ML Suitability": "High",
            "Recommended Approach":
                "Regression / probabilistic CLV",
            "Prediction":
                "How much revenue is this customer likely to generate?"
        },

        {
            "Business Problem": "Demand forecasting",
            "ML Suitability": "Moderate",
            "Recommended Approach":
                "Time-series forecasting",
            "Prediction":
                "What demand should be expected in future periods?"
        },

        {
            "Business Problem": "Unknown customer identification",
            "ML Suitability": "Low with current data",
            "Recommended Approach":
                "Identity resolution only if additional identifiers exist",
            "Prediction":
                "Which transactions belong to the same real customer?"
        },

        {
            "Business Problem": "Geographic diversification",
            "ML Suitability": "Low initially",
            "Recommended Approach":
                "Market strategy / experimentation",
            "Prediction":
                "Not primarily a prediction problem."
        },
    ]
)

display(ml_suitability)

In [ ]:
# ================================================================
# 22.5 — RECOMMENDED ML DEVELOPMENT SEQUENCE
# ================================================================

ml_roadmap = pd.DataFrame(
    {
        "Stage": [
            1,
            2,
            3,
            4,
            5,
            6,
            7,
        ],

        "Model": [
            "Repeat Purchase Propensity",
            "Time-to-Next-Purchase",
            "Reactivation Propensity",
            "Revenue Leakage Risk",
            "Future Customer Value",
            "Next-Best-Product",
            "Demand Forecasting",
        ],

        "Primary Business Objective": [
            "Accelerate second purchase",
            "Predict purchase timing",
            "Recover inactive customers",
            "Prevent revenue deterioration",
            "Prioritise high-value customers",
            "Increase basket value and breadth",
            "Improve operational planning",
        ],

        "Priority": [
            "Highest",
            "Highest",
            "Highest",
            "Highest",
            "High",
            "High",
            "Medium",
        ],
    }
)


display(ml_roadmap)

### Machine-Learning Modelling Principles

All predictive models developed from this analysis will follow the following principles.

**1. Temporal validation**

Models must be trained on earlier periods and evaluated on later unseen periods. Random train/test splits should not be the primary evaluation method because they can leak future customer behaviour into training.

**2. Leakage-safe features**

Every feature must be observable at the exact point in time when the prediction would be made.

Information occurring after the prediction date must never be included in model inputs.

**3. Censoring awareness**

Customers without sufficient future observation time must not automatically be labelled as negative outcomes. Horizon-based models must include only customers with adequate follow-up, while time-to-event models should explicitly account for right censoring.

**4. Business-value evaluation**

Accuracy alone is insufficient.

Models should be evaluated using metrics appropriate to the decision problem, including precision, recall, PR-AUC, lift, calibration and expected revenue value.

**5. Baseline comparison**

Every machine-learning model must outperform a simple benchmark before it is considered useful.

**6. Interpretability**

Customer-facing intervention models should provide understandable drivers of prediction so that predictions can support actionable decisions.

**7. Association is not causation**

Predictive models estimate likelihood. They do not prove that an intervention will cause the desired commercial outcome. Campaign effectiveness should ultimately be measured through controlled experiments.


### Step 22 Summary — Executive Conclusions and Predictive Analytics Direction

The completed analysis shows that the business's most important opportunity is not simply generating more transactions, but improving the predictability and management of existing customer behaviour.

Headline merchandise revenue increased modestly, but the underlying customer analysis revealed substantial movement beneath that result. Acquisition, reactivation and customer expansion offset meaningful contraction and inactivity leakage.

This makes customer behaviour the strongest candidate for the next analytical stage.

The recommended predictive programme should therefore begin with repeat-purchase propensity, time-to-next-purchase, customer reactivation and revenue-leakage prediction.

These models directly address problems demonstrated in the descriptive analysis and can produce operationally actionable outputs.

A conventional permanent-churn model is not recommended as the starting point. The transaction history shows that many customers return after extended inactivity, meaning inactivity should be treated as a time-dependent customer state rather than definitive churn.

Survival analysis is particularly relevant because it can estimate purchase timing while correctly accounting for customers whose eventual behaviour cannot yet be observed.

Product-affinity findings also support development of a next-best-product recommendation system, initially using association and collaborative-filtering approaches rather than unnecessarily complex algorithms.

Demand forecasting should follow later. The available transaction history demonstrates seasonality, but the relatively short time series means simple forecasting benchmarks should be established before deploying more complex models.

Two major findings should not initially be treated as ML problems.

First, unidentified customer revenue is fundamentally a data-capture and identity-management issue. Predictive modelling cannot reliably reconstruct customer identities without additional identifiers.

Second, geographic concentration is primarily a strategic market-development problem. Market attractiveness and non-UK growth opportunities should be analysed before attempting predictive geographic models.

The recommended modelling sequence is therefore:

1. repeat-purchase propensity;
2. time-to-next-purchase;
3. reactivation propensity;
4. revenue-leakage prediction;
5. future customer value;
6. next-best-product recommendations; and
7. demand forecasting.

The objective of the next phase is therefore not to build machine learning for its own sake.

It is to convert the strongest validated findings into predictive systems that answer three commercially important questions:

> **Who should we act on?**

> **When should we act?**

> **What should we offer them?**

Those three questions provide the foundation for moving from descriptive analytics into decision intelligence.

## Step 23 — Machine-Learning Problem Definition and Dataset Architecture

Step 23 defines the predictive problems that will be developed from the validated analytical findings.

No model is trained at this stage.

Instead, each machine-learning problem is specified in terms of:

- eligible population;
- prediction point;
- feature or observation window;
- target definition;
- prediction horizon;
- censoring rule;
- exclusion criteria;
- preferred validation approach;
- evaluation metrics;
- and the management action that would follow a prediction.

This architecture is necessary to prevent target leakage and to ensure that each model solves a genuine commercial problem.

The first modelling priority will be repeat-purchase propensity because its target is clearly observable, directly linked to the retention findings, and actionable through post-purchase customer intervention.

In [ ]:
# ================================================================
# 23.2 — MODELLING OBSERVATION BOUNDARY
# ================================================================

modeling_end_date = pd.Timestamp(
    "2011-11-30 23:59:59"
)


modeling_data = (
    df.loc[
        df["date"] <= modeling_end_date
    ]
    .copy()
)


print(
    f"Modelling observation end: "
    f"{modeling_end_date.date()}"
)

print(
    f"Rows available for modelling: "
    f"{len(modeling_data):,}"
)

print(
    f"Observed date range: "
    f"{modeling_data['date'].min().date()} "
    f"to "
    f"{modeling_data['date'].max().date()}"
)

In [ ]:
# ================================================================
# 23.3 — POSITIVE IDENTIFIED MERCHANDISE PURCHASES
# ================================================================

positive_customer_purchases = (
    modeling_data.loc[
        modeling_data["customer_is_identified"]
        & modeling_data["is_merchandise"]
        & modeling_data["quantity"].gt(0)
        & modeling_data["unit_price"].gt(0)
        & ~modeling_data["is_cancellation"]
    ]
    .copy()
)


print(
    f"Positive identified merchandise rows: "
    f"{len(positive_customer_purchases):,}"
)

print(
    f"Identified customers: "
    f"{positive_customer_purchases['customer'].nunique():,}"
)

In [ ]:
# ================================================================
# 23.4 — CUSTOMER ORDER-LEVEL DATASET
# ================================================================

customer_orders = (
    positive_customer_purchases
    .groupby(
        [
            "customer",
            "invoice"
        ],
        as_index=False
    )
    .agg(
        order_date=(
            "date",
            "min"
        ),

        order_revenue=(
            "net_merchandise_revenue",
            "sum"
        ),

        order_units=(
            "quantity",
            "sum"
        ),

        distinct_products=(
            "product",
            "nunique"
        ),

        average_unit_price=(
            "unit_price",
            "mean"
        ),

        region=(
            "region",
            "first"
        ),
    )
)


customer_orders = (
    customer_orders
    .sort_values(
        [
            "customer",
            "order_date",
            "invoice"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    customer_orders.head()
)

print(
    f"Customer orders: "
    f"{len(customer_orders):,}"
)

In [ ]:
# ================================================================
# 23.5 — ORDER STRUCTURE VALIDATION
# ================================================================

invoice_customer_counts = (
    positive_customer_purchases
    .groupby(
        "invoice"
    )[
        "customer"
    ]
    .nunique()
)


multi_customer_invoices = (
    invoice_customer_counts
    .gt(1)
    .sum()
)


print(
    f"Invoices linked to more than one "
    f"identified customer: "
    f"{multi_customer_invoices:,}"
)

In [ ]:
# ================================================================
# 23.6 — ML PROBLEM DEFINITION REGISTER
# ================================================================

ml_problem_register = pd.DataFrame(
    [
        {
            "model_id": "M1",
            "model_name":
                "90-Day Repeat Purchase Propensity",

            "population":
                "Customers immediately after first observable purchase",

            "prediction_point":
                "Completion of first observable order",

            "feature_window":
                "First order only",

            "target":
                "Second purchase occurs within 90 days",

            "prediction_horizon_days":
                90,

            "primary_method":
                "Binary classification",

            "primary_action":
                "Prioritise post-purchase CRM intervention",

            "primary_metric":
                "PR-AUC + Lift@Top Decile",

            "priority":
                "Highest",
        },

        {
            "model_id": "M2",
            "model_name":
                "Time-to-Next-Purchase",

            "population":
                "Customers after an observable purchase",

            "prediction_point":
                "Immediately after purchase",

            "feature_window":
                "Customer history up to prediction date",

            "target":
                "Days until next purchase",

            "prediction_horizon_days":
                None,

            "primary_method":
                "Survival analysis",

            "primary_action":
                "Determine optimal timing of customer intervention",

            "primary_metric":
                "Concordance Index",

            "priority":
                "Highest",
        },

        {
            "model_id": "M3",
            "model_name":
                "Reactivation Propensity",

            "population":
                "Previously purchasing customers inactive for >=90 days",

            "prediction_point":
                "90-day inactivity threshold",

            "feature_window":
                "All historical behaviour before inactivity threshold",

            "target":
                "Customer purchases again within next 90 days",

            "prediction_horizon_days":
                90,

            "primary_method":
                "Binary classification",

            "primary_action":
                "Prioritise inactive customers for reactivation",

            "primary_metric":
                "PR-AUC + Lift@Top Decile",

            "priority":
                "Highest",
        },

        {
            "model_id": "M4",
            "model_name":
                "Customer Revenue Leakage Risk",

            "population":
                "Currently active identified customers",

            "prediction_point":
                "Rolling customer snapshot date",

            "feature_window":
                "Previous 90 days",

            "target":
                "Material revenue contraction or inactivity "
                "during next 90 days",

            "prediction_horizon_days":
                90,

            "primary_method":
                "Classification + regression",

            "primary_action":
                "Early retention intervention",

            "primary_metric":
                "PR-AUC + revenue-weighted lift",

            "priority":
                "Highest",
        },

        {
            "model_id": "M5",
            "model_name":
                "Future Customer Value",

            "population":
                "Active identified customers",

            "prediction_point":
                "Rolling customer snapshot",

            "feature_window":
                "Historical behaviour before snapshot",

            "target":
                "Net merchandise revenue over next 180 days",

            "prediction_horizon_days":
                180,

            "primary_method":
                "Regression / probabilistic CLV",

            "primary_action":
                "Prioritise retention and acquisition economics",

            "primary_metric":
                "MAE + ranking performance",

            "priority":
                "High",
        },

        {
            "model_id": "M6",
            "model_name":
                "Next-Best-Product Recommendation",

            "population":
                "Customers with prior purchase history",

            "prediction_point":
                "After each completed order",

            "feature_window":
                "Products purchased before prediction point",

            "target":
                "Product purchased in next order",

            "prediction_horizon_days":
                None,

            "primary_method":
                "Recommender / ranking",

            "primary_action":
                "Personalised cross-sell recommendations",

            "primary_metric":
                "Precision@K / Recall@K",

            "priority":
                "High",
        },

        {
            "model_id": "M7",
            "model_name":
                "Demand Forecasting",

            "population":
                "Product or portfolio time series",

            "prediction_point":
                "End of each completed time period",

            "feature_window":
                "Historical demand only",

            "target":
                "Future demand/revenue",

            "prediction_horizon_days":
                None,

            "primary_method":
                "Time-series forecasting",

            "primary_action":
                "Inventory and operational planning",

            "primary_metric":
                "MAE / WAPE vs naive baseline",

            "priority":
                "Medium",
        },
    ]
)


display(
    ml_problem_register
)

#### 23.7 — Model 1: lock the exact prediction problem

This should be our first actual model.
At the moment a customer's first observable order is completed, predict whether they will make another purchase within 90 days.

The key phrase is at that moment.
Anything happening later is forbidden from the features.

In [ ]:
# ================================================================
# 23.8 — FIRST AND SECOND OBSERVABLE ORDERS
# ================================================================

customer_orders[
    "order_number"
] = (
    customer_orders
    .groupby(
        "customer"
    )
    .cumcount()
    + 1
)


first_orders = (
    customer_orders.loc[
        customer_orders[
            "order_number"
        ] == 1
    ]
    .copy()
)


second_orders = (
    customer_orders.loc[
        customer_orders[
            "order_number"
        ] == 2
    ][
        [
            "customer",
            "order_date"
        ]
    ]
    .rename(
        columns={
            "order_date":
                "second_order_date"
        }
    )
)


first_orders = (
    first_orders
    .merge(
        second_orders,
        on="customer",
        how="left"
    )
)


first_orders = (
    first_orders
    .rename(
        columns={
            "order_date":
                "first_order_date",

            "invoice":
                "first_invoice",

            "order_revenue":
                "first_order_value",

            "order_units":
                "first_order_units",

            "distinct_products":
                "first_order_product_count",

            "average_unit_price":
                "first_order_average_unit_price",

            "region":
                "first_order_region",
        }
    )
)


display(
    first_orders.head()
)

In [ ]:
# ================================================================
# 23.9 — DAYS TO SECOND PURCHASE
# ================================================================

first_orders[
    "days_to_second_purchase"
] = (
    first_orders[
        "second_order_date"
    ]
    -
    first_orders[
        "first_order_date"
    ]
).dt.days

In [ ]:
# ================================================================
# 23.10 — 90-DAY OUTCOME ELIGIBILITY
# ================================================================

first_orders[
    "followup_days_available"
] = (
    modeling_end_date.normalize()
    -
    first_orders[
        "first_order_date"
    ].dt.normalize()
).dt.days


first_orders[
    "eligible_90d"
] = (
    first_orders[
        "followup_days_available"
    ] >= 90
)


print(
    f"Total first observable customers: "
    f"{len(first_orders):,}"
)

print(
    f"Eligible for 90-day modelling: "
    f"{first_orders['eligible_90d'].sum():,}"
)

print(
    f"Right-censored before 90 days: "
    f"{(~first_orders['eligible_90d']).sum():,}"
)

In [ ]:
# ================================================================
# 23.11 — MODEL 1 TARGET
# ================================================================

repeat_model_base = (
    first_orders.loc[
        first_orders[
            "eligible_90d"
        ]
    ]
    .copy()
)


repeat_model_base[
    "repeat_within_90d"
] = (
    repeat_model_base[
        "days_to_second_purchase"
    ]
    .le(90)
    &
    repeat_model_base[
        "days_to_second_purchase"
    ]
    .notna()
).astype(int)

In [ ]:
# ================================================================
# 23.12 — TARGET BALANCE
# ================================================================

repeat_target_summary = (
    repeat_model_base[
        "repeat_within_90d"
    ]
    .value_counts(
        normalize=False
    )
    .rename_axis(
        "repeat_within_90d"
    )
    .reset_index(
        name="customers"
    )
)


repeat_target_summary[
    "share_pct"
] = (
    repeat_target_summary[
        "customers"
    ]
    /
    repeat_target_summary[
        "customers"
    ].sum()
    * 100
)


display(
    repeat_target_summary
)

In [ ]:
# ================================================================
# 23.13 — REMOVE LEFT-TRUNCATED FIRST OBSERVATIONS
# ================================================================

dataset_start_month = (
    modeling_data[
        "date"
    ]
    .min()
    .to_period("M")
)


repeat_model_base[
    "first_order_month"
] = (
    repeat_model_base[
        "first_order_date"
    ]
    .dt.to_period("M")
)


repeat_model_base[
    "is_left_truncated"
] = (
    repeat_model_base[
        "first_order_month"
    ]
    ==
    dataset_start_month
)


repeat_model_base = (
    repeat_model_base.loc[
        ~repeat_model_base[
            "is_left_truncated"
        ]
    ]
    .copy()
)


print(
    f"Model-ready observable acquisitions: "
    f"{len(repeat_model_base):,}"
)

In [ ]:
# ================================================================
# 23.14 — FIRST-ORDER TEMPORAL FEATURES
# ================================================================

repeat_model_base[
    "first_order_month_number"
] = (
    repeat_model_base[
        "first_order_date"
    ]
    .dt.month
)


repeat_model_base[
    "first_order_weekday"
] = (
    repeat_model_base[
        "first_order_date"
    ]
    .dt.dayofweek
)


repeat_model_base[
    "first_order_hour"
] = (
    repeat_model_base[
        "first_order_date"
    ]
    .dt.hour
)


repeat_model_base[
    "first_order_is_weekend"
] = (
    repeat_model_base[
        "first_order_weekday"
    ]
    .isin(
        [
            5,
            6
        ]
    )
    .astype(int)
)

In [ ]:
# ================================================================
# 23.15 — FIRST-ORDER BASKET FEATURES
# ================================================================

first_invoice_lookup = (
    repeat_model_base[
        [
            "customer",
            "first_invoice"
        ]
    ]
)


first_order_lines = (
    positive_customer_purchases
    .merge(
        first_invoice_lookup,
        left_on=[
            "customer",
            "invoice"
        ],
        right_on=[
            "customer",
            "first_invoice"
        ],
        how="inner"
    )
)


first_order_extra_features = (
    first_order_lines
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_order_max_unit_price=(
            "unit_price",
            "max"
        ),

        first_order_min_unit_price=(
            "unit_price",
            "min"
        ),

        first_order_price_std=(
            "unit_price",
            "std"
        ),

        first_order_max_line_quantity=(
            "quantity",
            "max"
        ),
    )
)


repeat_model_base = (
    repeat_model_base
    .merge(
        first_order_extra_features,
        on="customer",
        how="left"
    )
)

#### 23.16 — Product popularity features without leakage

This requires care.
We cannot calculate product popularity using the entire dataset because that would allow future information into past observations.
For the first model version, I recommend leaving global product-popularity features out.
Start with purely first-order characteristics.
That gives us a clean baseline.
Later we can construct historical-only product popularity as of each first-order date.

In [ ]:
# ================================================================
# 23.17 — INITIAL MODEL 1 FEATURE SET
# ================================================================

repeat_model_features = [
    "first_order_value",
    "first_order_units",
    "first_order_product_count",
    "first_order_average_unit_price",
    "first_order_max_unit_price",
    "first_order_min_unit_price",
    "first_order_price_std",
    "first_order_max_line_quantity",
    "first_order_month_number",
    "first_order_weekday",
    "first_order_hour",
    "first_order_is_weekend",
    "first_order_region",
]


repeat_model_target = (
    "repeat_within_90d"
)

In [ ]:
# ================================================================
# 23.18 — TARGET-LEAKAGE BLACKLIST
# ================================================================

forbidden_repeat_model_features = [
    "second_order_date",
    "days_to_second_purchase",
    "future_order_count",
    "future_revenue",
    "customer_lifetime_revenue",
    "eventual_repeat",
    "repeat_within_180d",
    "last_purchase_date",
    "future_product_count",
]


print(
    "Forbidden post-prediction features:"
)

for feature in forbidden_repeat_model_features:
    print(
        f" - {feature}"
    )

In [ ]:
# ================================================================
# 23.19 — MODEL 1 ANALYTICAL DATASET
# ================================================================

repeat_model_dataset = (
    repeat_model_base[
        [
            "customer",
            "first_order_date",
            *repeat_model_features,
            repeat_model_target,
        ]
    ]
    .copy()
)


display(
    repeat_model_dataset.head()
)


print(
    f"Rows: "
    f"{len(repeat_model_dataset):,}"
)

print(
    f"Predictor columns: "
    f"{len(repeat_model_features):,}"
)

In [ ]:
# ================================================================
# 23.20 — MODEL 1 FEATURE MISSINGNESS
# ================================================================

model_missingness = (
    repeat_model_dataset[
        repeat_model_features
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


display(
    model_missingness
)

In [ ]:
# ================================================================
# 23.21 — MODEL 1 DATASET VALIDATION
# ================================================================

assert (
    repeat_model_dataset[
        "customer"
    ]
    .nunique()
    ==
    len(
        repeat_model_dataset
    )
), (
    "Model 1 must contain one row per customer."
)


assert (
    repeat_model_dataset[
        repeat_model_target
    ]
    .isin(
        [
            0,
            1
        ]
    )
    .all()
), (
    "Target must be binary."
)


assert (
    repeat_model_dataset[
        "first_order_date"
    ]
    .max()
    <=
    modeling_end_date
    -
    pd.Timedelta(
        days=90
    )
), (
    "A customer without a complete 90-day "
    "follow-up entered the modelling dataset."
)


print(
    "✓ Model 1 analytical dataset passed "
    "initial validation checks."
)

In [ ]:
# ================================================================
# 23.22 — INSPECT ACQUISITION TIMELINE FOR TEMPORAL SPLIT
# ================================================================

acquisition_month_counts = (
    repeat_model_dataset
    .assign(
        acquisition_month=(
            repeat_model_dataset[
                "first_order_date"
            ]
            .dt.to_period("M")
        )
    )
    .groupby(
        "acquisition_month",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "size"
        ),

        repeat_rate_90d=(
            repeat_model_target,
            "mean"
        ),
    )
)


acquisition_month_counts[
    "repeat_rate_90d"
] = (
    acquisition_month_counts[
        "repeat_rate_90d"
    ]
    * 100
)


display(
    acquisition_month_counts
)

In [ ]:
# ================================================================
# VISUAL 23.1 — ACQUISITION POPULATION BY MONTH
# ================================================================

acquisition_month_counts[
    "month_label"
] = (
    acquisition_month_counts[
        "acquisition_month"
    ]
    .astype(str)
)


plt.figure(
    figsize=(13, 6)
)


sns.lineplot(
    data=acquisition_month_counts,
    x="month_label",
    y="customers",
    marker="o"
)


plt.title(
    "Observable Model-Eligible Customer Acquisitions by Month"
)

plt.xlabel(
    "First Purchase Month"
)

plt.ylabel(
    "Customers"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# VISUAL 23.2 — 90-DAY REPEAT RATE BY ACQUISITION MONTH
# ================================================================

plt.figure(
    figsize=(13, 6)
)


sns.lineplot(
    data=acquisition_month_counts,
    x="month_label",
    y="repeat_rate_90d",
    marker="o"
)


plt.title(
    "90-Day Repeat Rate by Acquisition Month"
)

plt.xlabel(
    "First Purchase Month"
)

plt.ylabel(
    "90-Day Repeat Rate (%)"
)

plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 23.23 — MODEL 1 EVALUATION FRAMEWORK
# ================================================================

model1_evaluation_metrics = pd.DataFrame(
    {
        "Metric": [
            "PR-AUC",
            "ROC-AUC",
            "Precision",
            "Recall",
            "F1 Score",
            "Brier Score",
            "Lift@Top 10%",
            "Lift@Top 20%",
            "Calibration",
        ],

        "Purpose": [
            "Primary discrimination metric for repeat propensity",
            "General ranking/discrimination",
            "How many targeted customers actually repeat",
            "How many eventual repeaters the model identifies",
            "Balance between precision and recall",
            "Probability accuracy",
            "Commercial concentration of repeaters in highest scores",
            "Commercial concentration in broader priority group",
            "Whether predicted probabilities match observed outcomes",
        ],

        "Priority": [
            "Primary",
            "Secondary",
            "Secondary",
            "Secondary",
            "Secondary",
            "Primary",
            "Primary",
            "Primary",
            "Primary",
        ],
    }
)


display(
    model1_evaluation_metrics
)

In [ ]:
# ================================================================
# 23.24 — MODEL 1 BASELINES
# ================================================================

model1_baselines = pd.DataFrame(
    {
        "Baseline": [
            "Overall repeat-rate baseline",
            "First-order-value heuristic",
            "Logistic regression",
        ],

        "Purpose": [
            "Naive probability benchmark",
            "Simple business-rule benchmark",
            "Interpretable statistical ML baseline",
        ],
    }
)


display(
    model1_baselines
)

In [ ]:
# ================================================================
# 23.25 — MODEL 1 CANDIDATE ALGORITHMS
# ================================================================

model1_candidates = pd.DataFrame(
    {
        "Sequence": [
            1,
            2,
            3,
            4,
        ],

        "Model": [
            "Logistic Regression",
            "Random Forest",
            "Gradient Boosted Trees",
            "Calibrated Best Model",
        ],

        "Role": [
            "Interpretable benchmark",
            "Non-linear benchmark",
            "Primary performance candidate",
            "Reliable probability output",
        ],
    }
)


display(
    model1_candidates
)

### Architecture of Subsequent Models

The same temporal discipline will be applied to the remaining models.

#### M2 — Time-to-Next-Purchase

Each eligible customer-order event becomes a prediction point.

Features contain only information observable up to that order.

The target consists of:

- duration until the next purchase; and
- whether the next purchase was observed or right-censored.

This dataset will support survival modelling rather than forcing unobserved customers into a binary churn label.

#### M3 — Reactivation Propensity

A modelling row will be created when a previously purchasing customer reaches a defined inactivity threshold, initially 90 days.

Features will summarise all customer behaviour available up to that threshold.

The target will indicate whether the customer purchases during the following 90 days.

Customers without a complete post-threshold observation window will be excluded from the binary model or handled through survival methods.

#### M4 — Revenue Leakage Risk

Customer snapshots will be created at fixed dates.

Features will describe activity during a historical observation window, initially 90 days.

The target will describe behaviour during the subsequent 90 days.

Two formulations should eventually be tested:

1. classification — whether the customer experiences material contraction or inactivity; and
2. regression — the customer's future net merchandise revenue.

#### M5 — Future Customer Value

Historical customer behaviour at a fixed snapshot date will be used to predict revenue over a future 180-day period.

Because the available dataset is relatively short, the model will require stricter eligibility rules than the 90-day models.

#### M6 — Next-Best-Product

Customer purchase history before an order will be used to rank products likely to appear in the customer's next order.

Evaluation will use temporal holdout of later purchases rather than random customer-product interactions.

#### M7 — Demand Forecasting

Forecast models will use chronological training and testing only.

Simple seasonal-naive and moving-average forecasts must be established before more complex forecasting models are accepted.

In [ ]:
# ================================================================
# 23.28 — STEP 23 VALIDATION
# ================================================================

assert (
    repeat_model_dataset[
        "customer"
    ].duplicated().sum()
    == 0
), (
    "Duplicate customer rows exist in Model 1."
)


assert (
    repeat_model_dataset[
        "first_order_date"
    ]
    .notna()
    .all()
)


assert (
    repeat_model_dataset[
        repeat_model_target
    ]
    .isin(
        [0, 1]
    )
    .all()
)


assert (
    (
        modeling_end_date.normalize()
        -
        repeat_model_dataset[
            "first_order_date"
        ].dt.normalize()
    )
    .dt.days
    .ge(90)
    .all()
)


print(
    "✓ Step 23 ML architecture and "
    "Model 1 dataset passed validation."
)

### Step 23 Summary — ML Problem Definition and Dataset Architecture

Step 23 established the modelling architecture required to move from descriptive analytics into predictive decision support.

Seven potential modelling problems were identified, but repeat-purchase propensity will be developed first because it combines strong business relevance, a clearly observable target and an immediate management action.

The first model will predict whether an observable newly acquired customer makes a second purchase within 90 days of their first order.

Only information available at completion of the first order will be used as model input.

Customers without at least 90 days of future observation are excluded from the binary modelling population rather than incorrectly labelled as non-repeaters. Customers first observed during the opening month of the dataset are also excluded because their true acquisition history may pre-date the available records.

This produces a leakage-safe modelling design in which:

- one row represents one observable acquired customer;
- the observation point is the first completed order;
- predictors describe that order only;
- the target is second purchase within 90 days;
- and validation will be chronological rather than primarily random.

The broader modelling roadmap follows the same principle.

Repeat-purchase, reactivation and revenue-leakage models will use explicit future prediction windows. Time-to-next-purchase modelling will use survival methods to account for right censoring. Recommendation models will be evaluated using future orders, while demand forecasts will be judged against simple chronological forecasting baselines.

The key methodological conclusion is that the modelling phase will treat time as part of the problem definition rather than merely another feature.

This is particularly important because customer inactivity, repeat behaviour and revenue leakage are all time-dependent outcomes.

The next modelling stage can therefore proceed with a clearly defined first objective:

> **Using only information known after a customer's first observable order, predict the probability that the customer will make another purchase within 90 days.**

In [ ]:
# Run once in Notebook 1

df.to_parquet(
    "cleaned_online_retail.parquet",
    index=False
)